# GSF-3120 - Système de Gestion de Portefeuille Factoriel V3
## Actions et Obligations Nord-Américaines

**Équipe:** Laurie Fiset, Benjamin Caron, Bang Tuyet Tra, Besta Soni

**Capital fictif:** 100 millions CAD

---

### Architecture du système

| # | Module | Description |
|---|--------|-------------|
| 0 | **Installations & Imports** | Dépendances et configuration de l'environnement |
| 1 | **Configuration** | Paramètres de la stratégie (facteurs, contraintes, seuils) |
| 2 | **Data Loader** | Univers complet ~750 titres (S&P 500, TSX, obligations, ETFs) |
| 3 | **Modèle factoriel** | 6 facteurs : Momentum, Value, Profitability, Low Vol, Investment, PEAD |
| 3.1 | **Analyse factorielle** | Validation empirique (IC, corrélations, monotonie, quintiles) |
| 4 | **Gestion du risque** | Détection de régime multi-signaux et overlay adaptatif |
| 5 | **Optimisation** | Mean-Variance (Markowitz) avec contraintes de liquidité et turnover |
| 5.1 | **Sélection obligataire** | Allocation obligataire intelligente par régime |
| 6 | **Backtester** | Walk-forward rolling avec cash buffer et contrôle du turnover |
| 7 | **Performance** | Métriques complètes, attribution, ratios ajustés au risque |
| 7.1 | **Stress Testing** | Validation sur périodes de crise (COVID, 2022, SVB, etc.) |
| 8 | **Visualisation** | Dashboard interactif et graphiques de performance |
| 9 | **Exécution du backtest** | Backtest complet 2006-2025 + analyses avancées |
| 10 | **Trading live** | Algorithme de rééquilibrage en temps réel (CSV conforme) |
| 11 | **Validation liquidité** | Tests des contraintes de liquidité et de l'optimisation |
| 12 | **Validation régimes** | Pouvoir prédictif out-of-sample du détecteur de régimes |
| 13 | **Diagnostic overlay** | Performance et pertinence de l'overlay adaptatif |
| 14 | **Test A/B overlay** | Comparaison overlay ON vs OFF |
| 15 | **Annexes** | Tests unitaires et guide de modification des contraintes |


---
# 0. Installations et imports
---

**IMPORTANT:** vérifier les API ici


In [1]:
# =============================================================================
# CELLULE 0: INSTALLATION ET IMPORTS
# =============================================================================

!pip install -q yfinance pandas numpy scipy scikit-learn cvxpy plotly tqdm statsmodels requests pandas_market_calendars
!pip install -q hmmlearn
!pip install -q cvxpy
!pip install --upgrade nbformat

import numpy as np
import pandas as pd
import pandas_market_calendars as mcal
import itertools
import math
import yfinance as yf
from datetime import datetime, timedelta
from typing import Dict, List, Tuple, Optional, Any
from collections import Counter
from dataclasses import dataclass
from enum import Enum
import warnings
warnings.filterwarnings('ignore')
import plotly.io as pio
import copy, types, time

import scipy.stats as stats
from scipy.stats import ttest_1samp, f_oneway, spearmanr
import statsmodels.api as sm
from sklearn.covariance import LedoitWolf
from tqdm import tqdm

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import os
import requests

# Validation des ouverture des marchés
TSX_CAL = mcal.get_calendar("TSX")
NYSE_CAL = mcal.get_calendar("NYSE")

def is_session(cal, date_yyyy_mm_dd):
    day = pd.Timestamp(date_yyyy_mm_dd)
    sched = cal.schedule(start_date=day, end_date=day)
    return len(sched) > 0


# HMM optionnel
try:
    from hmmlearn.hmm import GaussianHMM
    HMM_AVAILABLE = True
    print("[OK] HMM disponible (hmmlearn)")
except ImportError:
    HMM_AVAILABLE = False
    print("[ATTENTION] hmmlearn non installé - HMM désactivé")

# Optimisation convexe
try:
    import cvxpy as cp
    CVXPY_AVAILABLE = True
    print("[OK] CVXPY disponible pour optimisation mean-variance")
except ImportError:
    CVXPY_AVAILABLE = False
    print("[ATTENTION] CVXPY non installé - optimisation simplifiée uniquement")

# La clé FRED est définie dans Config.API_KEYS['fred']
# et sera injectée dans os.environ après chargement de Config (Cell 5)

print("[OK] Packages importés")




[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
[OK] HMM disponible (hmmlearn)
[OK] CVXPY disponible pour optimisation mean-variance
[OK] Packages importés


---
# 1. CONFIGURATION ET PARAMÈTRES
---

**IMPORTANT:** Modifier cette section pour ajuster la stratégie sans toucher au reste du code.

In [2]:
# =============================================================================
# CELLULE 1: CONFIGURATION GLOBALE
# =============================================================================

class Config:
    """Configuration centralisée - """

    # --- Paramètres de base ---
    PORTFOLIO_VALUE: float = 100_000_000  # 100M CAD
    PORTFOLIO_CURRENCY: str = 'CAD'
    RISK_FREE_RATE: float = 0.05  # 5% par défaut, sera mis à jour depuis FRED
    DATE_TODAY = "15/03/2026" # Format JJ/MM/AAAA

    # --- Cash Buffer  ---
    CASH_BUFFER = {
        'enabled': True,
        'percentage': 0.025,  # 2.5% de cash permanent
        'earn_risk_free': False,  # Le cash gagne le taux sans risque
        'ticker': 'CASH.TO', # ETF money-market reel (Yahoo)
    }

    # --- Sélection des actions ---
    STOCK_SELECTION = {
        'n_stocks': 25,
        'max_weight': 0.10,  # 10% max par position
        'min_weight': 0.01, # 1% min par position
        'buffer_size': 50,
    }

    # --- Contrôle du Turnover  ---
    TURNOVER_CONTROL = {
    'weight_smoothing': 0.40,              # Réduit pour mieux exploiter les signaux factoriels
    'smoothing_on_regime_change': 0.15,    # Smoothing réduit lors d'un changement de régime
    'rebalance_threshold': 0.0225,
    'min_trade_size': 0.01,
    'max_daily_turnover': 0.20,
    'no_trade_band_stocks': 0.015,         # 1.5 % — bande de tolérance pour les actions
    'no_trade_band_bonds': 0.005,          # 0.5 % — bande de tolérance pour les obligations
}

    # --- Rebalancement Intelligent ---
    REBALANCING = {
        'schedule': 'weekly',            # 'daily', 'weekly', 'monthly'
        'day_of_week': 0,                # 0=Lundi, 4=Vendredi
        'emergency_on_regime_change': False,
        'regime_confirmation_days': 3,   # Jours pour confirmer changement
        'max_emergency_rebal_per_month': 2,  # Limite les rebals d'urgence
        'enabled': True,  # DÉSACTIVER le rebalancement intelligent
    }

    # --- Devises ---
    CURRENCY = {
        'base_currency': 'CAD',
        'fx_ticker': 'USDCAD=X',
        'apply_fx_conversion': True,
    }

    # --- FACTEURS (Total = 100%) ---
    # ================================================================
    # MODELE FACTORIEL A 6 FACTEURS
    # ================================================================
    # Facteurs selectionnes sur base de la litterature academique :
    #   - Momentum, Value, Profitability : primes robustes sur 150+ ans
    #     (CFA Institute 2024, van Vliet et al. 2025)
    #   - Low Volatility : meilleur Sharpe ratio ajuste, defensive
    #     (Pim van Vliet, Robeco Conservative Formula)
    #   - Investment/CMA : entreprises conservatrices surperforment
    #     (Fama & French 2015)
    #   - PEAD : anomalie distincte du momentum, non subsumee par FF5
    #     (Ball & Brown 1968, Bernard & Thomas 1989)
    #
    # Les poids sont ajustes dynamiquement selon le regime (voir
    # FACTOR_REGIME_WEIGHTS ci-dessous).
    # ================================================================

    # Poids calibres sur la litterature academique, PAS sur le grid search (voir rapport section 9)
    FACTORS = {
        'momentum': {
            'enabled': True,
            'weight': 0.24,
            'lookback': 252,
            'skip_recent': 21,
            'description': 'Rendement 12m skip 1m (Jegadeesh & Titman 1993). '
                       'IC le plus eleve dans nos tests. Robuste sur 150+ ans '
                       'de donnees globales (van Vliet et al. 2025).',
        },
        'value': {
            'enabled': True,
            'weight': 0.35,
            'description': 'Earnings Yield = NI(TTM) / Market Cap via Compustat. '
                       'Correlation negative avec momentum (~-0.3), diversifie '
                       'le signal. Definition modernisee evitant les biais '
                       'des intangibles (Arnott et al. 2021).',
        },
        'profitability': {
            'enabled': True,
            'weight': 0.15,
            'description': 'GPOA = Gross Profit / Total Assets (Novy-Marx 2013). '
                       'Seul facteur FF5 ayant systematiquement delivre un '
                       'premium sur toutes les periodes (CFA Institute 2022). '
                       'Donnees Compustat via WRDS.',
        },
        'low_volatility': {
            'enabled': True,
            'weight': 0.08,
            'lookback': 252,
            'description': 'Inverse de la volatilite realisee 12 mois. '
                       'Alpha de 2.1% apres controle pour tous les autres '
                       'facteurs (van Vliet, CFA Institute 2024). '
                       'Reduit le drawdown du portefeuille.',
        },
        'investment': {
            'enabled': True,
            'weight': 0.08,
            'description': 'CMA = Conservative Minus Aggressive. Inverse de '
                       'la croissance des actifs totaux YoY via Compustat '
                       '(Fama & French 2015). Entreprises conservatrices '
                       'surperforment les agressives.',
        },
        'pead': {
            'enabled': True,
            'weight': 0.10,
            'description': 'Post-Earnings Announcement Drift. SUE = surprise '
                       'de resultats normalisee (Bernard & Thomas 1989). '
                       'Anomalie distincte du momentum, non subsumee par FF5. '
                       'Signal via dates de publication Compustat/WRDS.',
        },
    }

    # Poids factoriels dynamiques selon le regime de marche
    # En RISK_OFF : surponderer low_vol et PEAD (defensifs)
    # En RISK_ON : surponderer momentum et value (offensifs)
    FACTOR_REGIME_WEIGHTS = {
        'RISK_ON': {
            'momentum': 0.28,
            'value': 0.35,
            'profitability': 0.15,
            'low_volatility': 0.02,
            'investment': 0.08,
            'pead': 0.12,
        },
        'RISK_OFF': {
            'momentum': 0.12,
            'value': 0.30,
            'profitability': 0.15,
            'low_volatility': 0.15,
            'investment': 0.08,
            'pead': 0.20,
        },
    }

    # --- ESG INTEGRATION ---------------------------------------------------------
    # Nous appliquons un "ESG tilt" : les entreprises avec
    # de meilleurs scores ESG reçoivent une pondération légèrement plus élevée.

    ESG = {
        'enabled': True,

        # Intensité de l'effet ESG sur les pondérations
        'tilt_strength': 0.35,

        # Méthode utilisée
        'method': 'esg_tilt'
    }

    # --- DIVERSIFICATION SECTORIELLE ------------------------------------
    # Plafond par secteur GICS pour limiter la concentration sectorielle.
    # Justification : avec une strategie momentum, le portefeuille tend a se
    # concentrer dans 2-3 secteurs. Le plafond de 25% par secteur assure
    # qu'aucun secteur ne domine excessivement le portefeuille.
    SECTOR_DIVERSIFICATION = {
        'enabled': True,
        'max_sector_weight': 0.25,
    }

    # =============================================================================
    # JUSTIFICATION DES POIDS FACTORIELS — MODÈLE 6 FACTEURS
    # =============================================================================
    #
    # Poids calibrés sur la littérature académique (PAS sur le grid search).
    #
    # Momentum 12m-1m (24%)
    #   Jegadeesh & Titman (1993), JF ; van Vliet et al. (2025).
    #   Prime robuste sur 150+ ans de données globales. Skip 21j.
    #
    # Value — Earnings Yield (35%)
    #   NI(TTM) / Market Cap via Compustat. Corrélation négative avec
    #   momentum (~−0.3) → bonne diversification du signal.
    #
    # Profitability — GPOA (15%)
    #   Novy-Marx (2013), JFE ; Fama & French (2015).
    #   Signal le plus stable ; premium sur toutes les périodes (CFA 2022).
    #
    # Low Volatility (8%)
    #   Inverse vol. réalisée 12m. Alpha 2.1% après contrôle facteurs
    #   (van Vliet, CFA 2024). Réduit le drawdown du portefeuille.
    #
    # Investment / CMA (8%)
    #   Inverse croissance actifs totaux YoY (Fama & French 2015).
    #   Entreprises conservatrices surperforment les agressives.
    #
    # PEAD — Post-Earnings Drift (10%)
    #   SUE normalisé (Bernard & Thomas 1989). Anomalie distincte du
    #   momentum, non subsumée par FF5. Signal via Compustat/WRDS.
    # =============================================================================

    # --- Allocations par régime (AVEC cash buffer intégré) ---
    REGIME_ALLOCATIONS = {
    'RISK_ON': {
        'stocks': 0.725,
        'bonds': 0.250,
        'cash': 0.025,
    },
    'RISK_OFF': {
        'stocks': 0.585,
        'bonds': 0.39,
        'cash': 0.025,
    },
    # NEUTRAL supprimé (système binaire) plus efficace apres tests
}

    # Garde-fou absolu (applique APRES overlay + smoothing)
    MIN_BOND_ALLOC = 0.25

    # --- Risque ---
    RISK = {
        'covariance_method': 'shrinkage',
        'cov_lookback': 252,
        'var_confidence': 0.95,
    }

    # --- Transaction costs ---

    # --- Contraintes mandat / compliance (Jan 2026) ---
    CONSTRAINTS = {
        # Autoriser les ETF crypto (si mandat le permet)
        'allow_crypto_etfs': True,  # selon restrictions mandat
        # Limites sur le poids total du portefeuille (pas seulement sur la poche actions)
        'max_exotic_etfs_weight': 0.03,   # Réduit pour limiter les ETFs commodités sans alpha factoriel
        'max_crypto_etfs_weight': 0.05,   # conservateur (selon restrictions mandat)
        # Turnover cap (optionnel, en fraction du portefeuille par rebal)
        'turnover_cap': None,             # ex: 0.25 pour 25% par rebal
    }

    # --- Contraintes de Liquidité
    LIQUIDITY = {
        'enabled': True,
        # Fraction max du volume quotidien moyen qu'une position peut représenter
        # Ex: 0.10 = position max de 10% du volume journalier moyen
        'max_position_adv_fraction': 0.10,
        # Volume quotidien moyen minimum en $ pour qu'un titre soit éligible
        'min_avg_daily_volume_usd': 1_000_000,
        # Fenêtre pour calculer le volume moyen (jours)
        'adv_lookback_days': 20,
        # Nombre de jours max pour liquider une position
        'max_days_to_liquidate': 5,
    }

    # --- Paramètres Optimisation Mean-Variance ---
    OPTIMIZATION = {
        'method': 'mean_variance',  # 'simple' ou 'mean_variance'
        'objective': 'max_sharpe',  # 'max_sharpe', 'min_variance', 'target_return'
        'target_volatility': 0.15,  # Volatilité cible annualisée (si applicable)
        'risk_aversion': 0.5,       # Réduit pour plus de diversification (défaut: 2.0)
        'regularization': 0.01,     # Régularisation L2 pour stabilité
    }

    # --- Management Fees ---
    MANAGEMENT_FEES = {
        'mgmt_fee_annual': 0.0080,    # 0.80% par an
        'hurdle_rate_annual': 0.12,   # 12% hurdle annuel
        'perf_fee_rate': 0.20,        # 20% sur surperformance
    }

    TRANSACTION_COSTS = {
        'enabled': True,
        'stocks_bps': 25,
        'bonds_bps': 9,
        'fixed_cost': 1000,
    }

    # --- Backtesting ---
    BACKTEST = {
        'start_date': datetime(2010, 1, 1),  # Réduit pour diminuer le risque d'overfitting
        'end_date': datetime.strptime(DATE_TODAY, '%d/%m/%Y'),
        'rolling_window': {
            'enabled': True,
            'train_months': 24,
            'test_months': 6,
            'step_months': 3,
        },
        'rebalance_freq': 'daily',   # on calcule tous les jours
        'rebalance_day': 0,
    }

    # --- Régime ---
    REGIME = {
        'enabled': True,
        'sma_period': 200,
        'vix_risk_off_threshold': 30,
        'vix_risk_on_threshold': 18,
        'confirmation_days': 5,
    }

    # --- Configuration Détection de Régimes Multi-Signaux ---
    REGIME_DETECTION = {
        'enabled': True,
        'signal_weights': {
            'volatility': 0.25,
            'trend': 0.25,
            'yield_curve': 0.15,
            'credit': 0.10,
            'breadth': 0.10,
            'hmm': 0.15,
        },
        'thresholds': {'risk_on': 0.05, 'risk_off': -0.05},
        'vix': {'risk_off': 30, 'risk_on': 18, 'extreme': 40},
        'yield_curve': {'inversion_threshold': 0, 'steep_threshold': 1.0, 'lookback': 252},
        'credit': {'tight_percentile': 20, 'wide_percentile': 80, 'lookback': 252},
        'trend': {'sma_fast': 50, 'sma_slow': 200, 'momentum_lookback': 126},
        'breadth': {'pct_above_sma_bullish': 60, 'pct_above_sma_bearish': 40},
        'confirmation_days': 5,
        'hmm': {
            'enabled': True,           # ACTIVER le HMM
            'n_states': 2,             # 2 états: High Vol / Low Vol
            'n_iter': 100,             # Itérations EM
            'lookback': 504,           # 2 ans de données
            'weight_in_composite': 0.15,  # Poids du signal HMM (15%)
            'retrain_frequency': 52,   # Ré-entraîner chaque année (52 semaines)
            },
        # Paramètres multiplicateur d'exposition
        'exposure': {
            # Multiplicateurs plus agressifs
            'min_multiplier': 0.70,      # Réduire à 70% en RISK_OFF fort
            'max_multiplier': 1.10,      # Augmenter à 110% en RISK_ON fort
            'confidence_threshold': 0.25,  # Seuil plus bas pour agir plus souvent
            # Nouveaux paramètres de calibration
            'base_multiplier': 1.0,       # Multiplicateur de base
            'scale_with_confidence': True, # Échelonner selon la confiance
        },
    }

    # --- Données Fondamentales ---
    USE_FUNDAMENTAL_DATA = True
    USE_BINARY_REGIME_WEIGHTS = True
    FUNDAMENTAL_DATA = {
        'enabled': True,
        'max_tickers': 300,  # Limite pour éviter timeouts
        'cache_days': 30,     # Durée du cache
    }

    # --- Clés API ---
    API_KEYS = {
        'fred': os.environ.get('FRED_API_KEY', 'd9ae42249b7f708b8826a755c3a99e98'),
        # Financial Modeling Prep : cle gratuite (250 requetes/jour)
        # Utilisee comme fallback si le fichier Compustat CSV est absent.
        'fmp': 'demo',
    }

    # --- Exclusions  ---
    EXCLUSIONS = {
        'enabled': False,  # Activer pour utiliser les exclusions
        'tickers': [],     # Tickers spécifiques à exclure (ex: ['META', 'GOOGL'])
        'sectors': [],     # Secteurs GICS à exclure (ex: ['Energy', 'Tobacco'])
        'industries': [],  # Industries spécifiques
        # Exclusions ESG prédéfinies
        'esg_presets': {
            'tobacco': ['PM', 'MO', 'BTI', 'IMBBY'],
            'weapons': ['LMT', 'RTX', 'NOC', 'GD', 'BA', 'HII', 'LHX'],
            'gambling': ['MGM', 'LVS', 'WYNN', 'CZR', 'DKNG', 'PENN'],
            'fossil_fuels': ['XOM', 'CVX', 'COP', 'EOG', 'SLB', 'OXY', 'PSX', 'VLO', 'MPC'],
        },
        'apply_esg_presets': [],  # Liste des presets à appliquer: ['tobacco', 'weapons']
    }

    # --- FRED ---
    FRED = {
        'enabled': True,
        'api_key_env': 'FRED_API_KEY',
        'series': {
            'treasury_3m': 'DGS3MO',
            'treasury_2y': 'DGS2',
            'treasury_10y': 'DGS10',
            'vix': 'VIXCLS',
            'fed_funds': 'FEDFUNDS',
            'credit_spread_hy': 'BAMLH0A0HYM2',
        },
    }

    @classmethod
    def validate(cls) -> bool:
        errors = []

        # Vérifier poids facteurs
        total_weight = sum(f['weight'] for f in cls.FACTORS.values() if f.get('enabled'))
        if abs(total_weight - 1.0) > 0.001:
            errors.append(f"Poids facteurs = {total_weight:.2%}, devrait être 100%")

        # Vérifier allocations par régime
        for regime, alloc in cls.REGIME_ALLOCATIONS.items():
            total = alloc['stocks'] + alloc['bonds'] + alloc.get('cash', 0)
            if abs(total - 1.0) > 0.001:
                errors.append(f"Allocation {regime} = {total:.2%}")

        if errors:
            for e in errors:
                print(f"[ERREUR CONFIG] {e}")
            return False
        print("[OK] Configuration V3 validée")
        return True

    @classmethod
    def summary(cls):
        print("="*70)
        print("CONFIGURATION DU PORTEFEUILLE ")
        print("="*70)
        print(f"Capital: ${cls.PORTFOLIO_VALUE:,.0f} {cls.PORTFOLIO_CURRENCY}")
        print(f"\nCash Buffer: {cls.CASH_BUFFER['percentage']:.1%}")
        print(f"Smoothing: {cls.TURNOVER_CONTROL['weight_smoothing']:.0%}")
        print(f"\nFacteurs actifs:")
        total = 0
        for name, cfg in cls.FACTORS.items():
            if cfg.get('enabled') and cfg.get('weight', 0) > 0:
                print(f"  - {name}: {cfg['weight']:.0%}")
                total += cfg['weight']
        print(f"  TOTAL: {total:.0%}")
        print("="*70)

Config.validate()
Config.DATE_TODAY_PARSED = datetime.strptime(Config.DATE_TODAY, "%d/%m/%Y")
print(f"[OK] DATE_TODAY = {Config.DATE_TODAY}")
Config.summary()
os.environ['FRED_API_KEY'] = Config.API_KEYS['fred']

print("[OK] Configuration chargée")

[OK] Configuration V3 validée
[OK] DATE_TODAY = 15/03/2026
CONFIGURATION DU PORTEFEUILLE 
Capital: $100,000,000 CAD

Cash Buffer: 2.5%
Smoothing: 40%

Facteurs actifs:
  - momentum: 24%
  - value: 35%
  - profitability: 15%
  - low_volatility: 8%
  - investment: 8%
  - pead: 10%
  TOTAL: 100%
[OK] Configuration chargée


---
# 2. MODULE DATA - Chargement des données
---


Univers d'investissement pour un portefeuille institutionnel:
- Actions nord-américaines (S&P 500, NASDAQ, TSX)
- Obligations nord-américaines (Gouvernement, Corporate, High Yield)
- ETFs thématiques cotés en Amérique du Nord (Crypto, Commodités, Immobilier)

NOTE: On investit via des ETFs cotés aux bourses US/CA, pas directement car non permis
      


In [3]:
# =============================================================================
# CELLULE 2: DATA LOADER AVEC UNIVERS COMPLET (753 TITRES)
# =============================================================================

class DataLoader:
    """Chargement des données avec univers COMPLET de V1."""

    # =========================================================================
    # S&P 500 COMPONENTS (490 titres)
    # =========================================================================
    SP500_COMPONENTS = [
        'A', 'AAPL', 'ABBV', 'ABNB', 'ABT', 'ACGL', 'ACN', 'ADBE', 'ADI', 'ADM',
        'ADP', 'ADSK', 'AEE', 'AEP', 'AES', 'AFL', 'AIG', 'AIZ', 'AJG', 'AKAM',
        'ALB', 'ALGN', 'ALL', 'ALLE', 'AMAT', 'AMCR', 'AMD', 'AME', 'AMGN', 'AMP',
        'AMT', 'AMZN', 'ANET', 'AON', 'AOS', 'APA', 'APD', 'APH', 'APO', 'APP',
        'APTV', 'ARE', 'ARES', 'ATO', 'AVB', 'AVGO', 'AVY', 'AWK', 'AXON', 'AXP',
        'AZO', 'BA', 'BAC', 'BALL', 'BAX', 'BBY', 'BDX', 'BEN', 'BF-B', 'BG',
        'BIIB', 'BK', 'BKNG', 'BKR', 'BLDR', 'BLK', 'BMY', 'BR', 'BRK-B', 'BRO',
        'BSX', 'BX', 'BXP', 'C', 'CAG', 'CAH', 'CARR', 'CAT', 'CB', 'CBOE',
        'CBRE', 'CCI', 'CCL', 'CDNS', 'CDW', 'CEG', 'CF', 'CFG', 'CHD', 'CHRW',
        'CHTR', 'CI', 'CINF', 'CL', 'CLX', 'CMCSA', 'CME', 'CMG', 'CMI', 'CMS',
        'CNC', 'CNP', 'COF', 'COIN', 'COO', 'COP', 'COR', 'COST', 'CPAY', 'CPB',
        'CPRT', 'CPT', 'CRH', 'CRL', 'CRM', 'CRWD', 'CSCO', 'CSGP', 'CSX', 'CTAS',
        'CTRA', 'CTSH', 'CTVA', 'CVNA', 'CVS', 'CVX', 'D', 'DAL', 'DASH', 'DAY',
        'DD', 'DDOG', 'DE', 'DECK', 'DELL', 'DG', 'DGX', 'DHI', 'DHR', 'DIS',
        'DLR', 'DLTR', 'DOC', 'DOV', 'DOW', 'DPZ', 'DRI', 'DTE', 'DUK', 'DVA',
        'DVN', 'DXCM', 'EA', 'EBAY', 'ECL', 'ED', 'EFX', 'EG', 'EIX', 'EL',
        'ELV', 'EME', 'EMR', 'EOG', 'EPAM', 'EQIX', 'EQR', 'EQT', 'ERIE', 'ES',
        'ESS', 'ETN', 'ETR', 'EVRG', 'EW', 'EXC', 'EXE', 'EXPD', 'EXPE', 'EXR',
        'F', 'FANG', 'FAST', 'FCX', 'FDS', 'FDX', 'FE', 'FFIV', 'FICO', 'FIS',
        'FISV', 'FITB', 'FIX', 'FOX', 'FOXA', 'FRT', 'FSLR', 'FTNT', 'FTV', 'GD',
        'GDDY', 'GE', 'GEHC', 'GEN', 'GEV', 'GILD', 'GIS', 'GL', 'GLW', 'GM',
        'GNRC', 'GOOG', 'GOOGL', 'GPC', 'GPN', 'GRMN', 'GS', 'GWW', 'HAL', 'HAS',
        'HBAN', 'HCA', 'HD', 'HIG', 'HII', 'HLT', 'HOLX', 'HON', 'HOOD', 'HPE',
        'HPQ', 'HRL', 'HSIC', 'HST', 'HSY', 'HUBB', 'HUM', 'HWM', 'IBKR', 'IBM',
        'ICE', 'IDXX', 'IEX', 'IFF', 'INCY', 'INTC', 'INTU', 'INVH', 'IP', 'IQV',
        'IR', 'IRM', 'ISRG', 'IT', 'ITW', 'IVZ', 'J', 'JBHT', 'JBL', 'JCI',
        'JKHY', 'JNJ', 'JPM', 'KDP', 'KEY', 'KEYS', 'KHC', 'KIM', 'KKR', 'KLAC',
        'KMB', 'KMI', 'KO', 'KR', 'KVUE', 'L', 'LDOS', 'LEN', 'LH', 'LHX',
        'LII', 'LIN', 'LLY', 'LMT', 'LNT', 'LOW', 'LRCX', 'LULU', 'LUV', 'LVS',
        'LW', 'LYB', 'LYV', 'MA', 'MAA', 'MAR', 'MAS', 'MCD', 'MCHP', 'MCK',
        'MCO', 'MDLZ', 'MDT', 'MET', 'META', 'MGM', 'MKC', 'MLM', 'MMM', 'MNST',
        'MO', 'MOH', 'MOS', 'MPC', 'MPWR', 'MRK', 'MRNA', 'MS', 'MSCI', 'MSFT',
        'MSI', 'MTB', 'MTCH', 'MTD', 'MU', 'NCLH', 'NDAQ', 'NDSN', 'NEE', 'NEM',
        'NFLX', 'NI', 'NKE', 'NOC', 'NOW', 'NRG', 'NSC', 'NTAP', 'NTRS', 'NUE',
        'NVDA', 'NVR', 'NWS', 'NWSA', 'NXPI', 'O', 'ODFL', 'OKE', 'OMC', 'ON',
        'ORCL', 'ORLY', 'OTIS', 'OXY', 'PANW', 'PAYC', 'PAYX', 'PCAR', 'PCG',
        'PEG', 'PEP', 'PFE', 'PFG', 'PG', 'PGR', 'PH', 'PHM', 'PKG', 'PLD',
        'PLTR', 'PM', 'PNC', 'PNR', 'PNW', 'PODD', 'POOL', 'PPG', 'PPL', 'PRU',
        'PSA', 'PSX', 'PTC', 'PWR', 'PYPL', 'QCOM', 'RCL', 'REG', 'REGN', 'RF',
        'RJF', 'RL', 'RMD', 'ROK', 'ROL', 'ROP', 'ROST', 'RSG', 'RTX', 'RVTY',
        'SBAC', 'SBUX', 'SCHW', 'SHW', 'SJM', 'SLB', 'SMCI', 'SNA', 'SNPS', 'SO',
        'SOLV', 'SPG', 'SPGI', 'SRE', 'STE', 'STLD', 'STT', 'STX', 'STZ', 'SW',
        'SWK', 'SWKS', 'SYF', 'SYK', 'SYY', 'T', 'TAP', 'TDG', 'TDY', 'TECH',
        'TEL', 'TER', 'TFC', 'TGT', 'TJX', 'TKO', 'TMO', 'TMUS', 'TPL', 'TPR',
        'TRGP', 'TRMB', 'TROW', 'TRV', 'TSCO', 'TSLA', 'TSN', 'TT', 'TTD', 'TTWO',
        'TXN', 'TXT', 'TYL', 'UAL', 'UBER', 'UDR', 'UHS', 'ULTA', 'UNH', 'UNP',
        'UPS', 'URI', 'USB', 'V', 'VICI', 'VLO', 'VLTO', 'VMC', 'VRSK', 'VRSN',
        'VRTX', 'VST', 'VTR', 'VTRS', 'VZ', 'WAB', 'WAT', 'WBD', 'WDAY', 'WDC',
        'WEC', 'WELL', 'WFC', 'WM', 'WMB', 'WMT', 'WRB', 'WSM', 'WST', 'WTW',
        'WY', 'WYNN', 'XEL', 'XOM', 'XYL', 'YUM', 'ZBH', 'ZBRA', 'ZTS'
    ]

    # Actions US additionnelles (NASDAQ growth)
    US_ADDITIONAL_STOCKS = [
        'MELI', 'TEAM', 'ZS', 'OKTA', 'DKNG', 'ROKU', 'ZM', 'DOCU', 'SNAP',
        'PINS', 'SPOT', 'NET', 'SNOW', 'PATH', 'U', 'RBLX',
        'SOFI', 'UPST', 'AFRM', 'HIMS',
    ]

    # =========================================================================
    # ACTIONS CANADIENNES (100 titres)
    # =========================================================================
    TSX_COMPONENTS = [
        # Financières
        'RY.TO', 'TD.TO', 'BNS.TO', 'BMO.TO', 'CM.TO', 'NA.TO',
        'MFC.TO', 'SLF.TO', 'GWO.TO', 'IAG.TO', 'POW.TO', 'FFH.TO',
        'BN.TO', 'BAM.TO', 'IFC.TO', 'X.TO', 'EQB.TO',
        # Énergie
        'ENB.TO', 'TRP.TO', 'CNQ.TO', 'SU.TO', 'IMO.TO', 'CVE.TO',
        'PPL.TO', 'KEY.TO', 'ARX.TO', 'WCP.TO', 'TVE.TO', 'BTE.TO',
        'TOU.TO', 'VET.TO', 'PEY.TO', 'FRU.TO',
        # Matériaux & Mines
        'ABX.TO', 'NTR.TO', 'FNV.TO', 'WPM.TO', 'AEM.TO', 'K.TO',
        'FM.TO', 'IVN.TO', 'LUN.TO', 'HBM.TO', 'CS.TO',
        'CCL-B.TO', 'WFG.TO',
        # Industriels
        'CNR.TO', 'CP.TO', 'WSP.TO', 'CAE.TO',
        'WCN.TO', 'STN.TO', 'TFII.TO', 'RBA.TO', 'ARE.TO',
        'GIB-A.TO', 'AC.TO',
        # Consommation
        'ATD.TO', 'L.TO', 'MRU.TO', 'EMP-A.TO', 'WN.TO',
        'DOL.TO', 'CTC-A.TO', 'QSR.TO', 'MTY.TO',
        'GIL.TO', 'LSPD.TO',
        # Technologie
        'SHOP.TO', 'OTEX.TO', 'CSU.TO', 'DSG.TO', 'DCBO.TO',
        'BB.TO', 'KXS.TO', 'REAL.TO', 'GSY.TO',
        # Télécommunications
        'BCE.TO', 'T.TO', 'RCI-B.TO', 'QBR-B.TO',
        # Immobilier (REITs)
        'REI-UN.TO', 'HR-UN.TO', 'CAR-UN.TO', 'AP-UN.TO', 'CRT-UN.TO',
        'DIR-UN.TO', 'SRU-UN.TO', 'GRT-UN.TO', 'IIP-UN.TO',
        # Utilities
        'FTS.TO', 'EMA.TO', 'H.TO', 'AQN.TO', 'CPX.TO', 'BEP-UN.TO',
        'NPI.TO', 'TA.TO', 'ALA.TO',
        # Santé
        'WELL.TO',
    ]

    # =========================================================================
    # OBLIGATIONS ETFs
    # =========================================================================
    US_BOND_ETFS = [
        'AGG', 'BND', 'SCHZ',
        'SHY', 'IEI', 'IEF', 'TLT', 'TLH', 'GOVT', 'VGSH', 'VGIT', 'VGLT',
        'TIP', 'VTIP', 'SCHP',
        'LQD', 'VCIT', 'VCSH', 'IGIB', 'IGSB',
        'HYG', 'JNK', 'SHYG', 'USHY', 'HYLB',
        'MUB', 'VTEB', 'HYD',
        'MBB', 'VMBS',
        'FLOT', 'FLRN',
    ]

    CA_BOND_ETFS = [
        'XBB.TO', 'ZAG.TO', 'VAB.TO',
        'XGB.TO', 'ZFL.TO', 'ZFS.TO', 'CLF.TO',
        'XQB.TO', 'ZPS.TO', 'ZPL.TO',
        'XCB.TO', 'ZCS.TO', 'ZLC.TO',
        'XHY.TO', 'ZHY.TO',
        'XRB.TO', 'ZRR.TO',
    ]


    # =========================================================================
    # Cash-equivalent ETF
    # =========================================================================
    CASH_ETFS = ['CASH.TO']

    # =========================================================================
    # ETFs THÉMATIQUES
    # =========================================================================
    CRYPTO_ETFS = [
        'IBIT', 'FBTC', 'BITB', 'GBTC', 'ARKB', 'BITO',
        'ETHA', 'FETH', 'ETHE',
        'BTCX-B.TO', 'BTCC-B.TO', 'ETHX-B.TO', 'ETHH-B.TO',
        'MSTR', 'MARA', 'RIOT', 'CLSK',
    ]

    COMMODITY_ETFS = [
        'DJP', 'GSG', 'PDBC', 'DBC', 'COMT',
        'GLD', 'IAU', 'GLDM', 'GDX', 'GDXJ',
        'SLV', 'SIL',
        'USO', 'BNO', 'UNG',
        'DBA', 'CORN', 'WEAT', 'SOYB',
        'DBB', 'CPER',
    ]

    REAL_ESTATE_ETFS = [
        'VNQ', 'IYR', 'SCHH', 'XLRE', 'MORT',
        'XRE.TO', 'ZRE.TO', 'VRE.TO',
    ]

    SECTOR_THEMATIC_ETFS = [
        'XLK', 'VGT', 'QQQ', 'SMH', 'SOXX', 'IGV',
        'AIQ', 'BOTZ', 'ROBO',
        'XLV', 'VHT', 'IBB', 'XBI',
        'XLF', 'VFH', 'KBE', 'KRE',
        'XLE', 'VDE', 'XOP', 'OIH',
        'ICLN', 'TAN', 'QCLN', 'LIT',
        'XLI', 'VIS',
        'XLY', 'XLP', 'VCR', 'VDC',
        'XLU', 'VPU',
        'XLB', 'VAW',
        'ITA', 'PPA',
        'CIBR', 'HACK', 'BUG',
    ]

    # =========================================================================
    # TICKERS HISTORIQUES — SURVIVORSHIP BIAS FREE
    #  Pour Backtesting et Analyse Factorielle
    # =========================================================================

    DELISTED_TICKERS = [
         # Tickers vérifiés fonctionnels sur Yahoo Finance (février 2025)
         # Acquisitions récentes (données complètes jusqu'au délistage)
         'BUD',      # Anheuser-Busch — acquis par InBev 2008
         'DELL',     # Dell — privatisé 2013 (données jusqu'en 2013)
         'SE',       # Spectra Energy — fusionne avec Enbridge 2017
         'TWX',      # Time Warner — acquis par AT&T 2018
         'CA',       # CA Technologies — acquis par Broadcom 2018
         'EP',       # El Paso Corp — acquis par Kinder Morgan 2012
         'GR',       # Goodrich — acquis par UTC 2012
         'INFO',     # IHS Markit — fusionne avec S&P Global 2022
         'BMC',      # BMC Software — privatisé 2013
         'HTZ',      # Hertz — faillite 2020 (données avant faillite)
         'WB',       # Wachovia — rachat par Wells Fargo 2008
         'WM',       # Washington Mutual (ticker WM = Waste Mgmt, garder pour données)
         'MER',      # Merrill Lynch — rachat par BofA 2008
         'CFC',      # Countrywide Financial — rachat par BofA 2008
         'SBNY',     # Signature Bank — faillite mars 2023
         'FRCB',     # First Republic Bank — faillite mai 2023
         'BKRRF',    # Bed Bath & Beyond — faillite 2023
         'PCG',      # PG&E — faillite 2019 (restructuré)
         'FNMA',     # Fannie Mae (ticker correct, pas FNM)
         'FMCC',     # Freddie Mac (ticker correct, pas FRE)
         'YLO.TO',   # Yellow Media — restructuré 2012
         'CCO.TO',   # Cameco
         'CTC.TO',   # Canadian Tire
     ]

    def __init__(self, config=Config):
        self.config = config
        self.prices = None
        self.returns = None
        self.fx_rate = None
        self.asset_currency = {}
        self.risk_free_rate_series = None
        self.macro_data = {}  # V4: Données macro pour régimes
        # scores ESG
        self.esg_scores = None

    def get_all_tickers(self, backtest_mode=False):
        '''Retourne l'univers des tickers.

        Args:
            backtest_mode: Si True, inclut les tickers delistes/faillites
                          pour eviter le survivorship bias dans le backtest.
                          Si False (defaut), univers live uniquement.
        '''
        us_stocks = self.SP500_COMPONENTS + self.US_ADDITIONAL_STOCKS
        ca_stocks = self.TSX_COMPONENTS
        us_bonds = self.US_BOND_ETFS
        ca_bonds = self.CA_BOND_ETFS
        cash_etf = self.CASH_ETFS
        thematic = (self.CRYPTO_ETFS + self.COMMODITY_ETFS +
                    self.REAL_ESTATE_ETFS + self.SECTOR_THEMATIC_ETFS)

        all_tickers = us_stocks + ca_stocks + us_bonds + ca_bonds + thematic + cash_etf

        # Ajouter les tickers historiques UNIQUEMENT en mode backtest
        if backtest_mode and hasattr(self, 'DELISTED_TICKERS'):
            delisted = self.DELISTED_TICKERS
            # Filtrer les doublons avec l'univers live
            existing = set(all_tickers)
            new_delisted = [t for t in delisted if t not in existing]
            all_tickers = all_tickers + new_delisted
            print(f"  [BACKTEST] +{len(new_delisted)} tickers historiques "
                  f"(survivorship-bias-free)")

        all_tickers = list(dict.fromkeys(all_tickers))  # Dedupliquer

        # Mapper les devises
        currency_map = {}
        for t in all_tickers:
            if t.endswith('.TO'):
                currency_map[t] = 'CAD'
            else:
                currency_map[t] = 'USD'

        self.asset_currency = currency_map

        n_live = len(us_stocks) + len(ca_stocks) + len(us_bonds) + len(ca_bonds) + len(thematic)
        n_total = len(all_tickers)
        print(f"Univers: {len(us_stocks)} US stocks + {len(ca_stocks)} CA stocks + "
              f"{len(us_bonds)} US bonds + {len(ca_bonds)} CA bonds + "
              f"{len(thematic)} thematic = {n_live} live"
              + (f" + {n_total - n_live} historiques" if backtest_mode else ""))

        return all_tickers, currency_map

    def load_data(self, start_date=None, end_date=None, backtest_mode=False):
        '''Charge les données de prix ET de volume.

        Args:
            backtest_mode: Si True, inclut les tickers historiques.
        '''
        if start_date is None:
            start_date = self.config.BACKTEST['start_date'] - timedelta(days=400)
        if end_date is None:
            end_date = self.config.BACKTEST['end_date']

        tickers, currency_map = self.get_all_tickers(backtest_mode=backtest_mode)
        print(f"Chargement de {len(tickers)} tickers...")

        # Le reste est IDENTIQUE a l'existant
        prices, volumes = self._load_yahoo_with_volume(tickers, start_date, end_date)
        fx_rate = self._load_fx_rate(start_date, end_date)
        prices_cad = self._convert_to_cad(prices, currency_map, fx_rate)
        if backtest_mode:
            prices_cad = self._handle_delisted_tickers(prices_cad)
            n_active = prices_cad.iloc[-1].notna().sum()
            n_total = len(prices_cad.columns)
            print(f"  [SURVIVORSHIP] {n_total - n_active} tickers detectes comme delistes")


        volumes_usd = volumes * prices

        returns = prices_cad.pct_change().dropna(how='all')

        self.prices = prices_cad
        self.returns = returns
        self.fx_rate = fx_rate
        self.volumes = volumes
        self.volumes_dollar = volumes_usd
        self.esg_scores = self.load_esg_dataset(prices.columns)
        self.risk_free_rate_series = self._load_risk_free_rate(start_date, end_date)

        print(f"[OK] Donnees: {len(prices_cad)} jours, {len(prices_cad.columns)} titres actifs")
        print(f"[OK] Volumes charges pour {len(volumes.columns)} titres")

        return prices_cad, returns

    def load_esg_dataset(self, tickers):
        """
        Charge ou calcule un score ESG pour chaque ticker.

        Strategie :
        1. Tenter de recuperer le score ESG Sustainalytics via yfinance
           (champ 'esgScores' dans sustainability). Ce score, quand disponible,
           est un vrai score ESG reconnu par l'industrie.
        2. Si indisponible, calculer un proxy base sur :
           - E (40%) : score environnemental par secteur GICS
           - S (30%) : dimension sociale (log du nombre d'employes)
           - G (30%) : gouvernance (log de la capitalisation boursiere)
        3. Le score final est normalise entre 0 et 1 (1 = meilleur).

        Le dictionnaire des secteurs est aussi stocke dans self.sector_map
        pour le controle de diversification sectorielle.
        """

        import yfinance as yf
        import pandas as pd
        import numpy as np

        # Score environnemental par secteur (proxy de dernier recours)
        sector_environment = {
            "Technology": 0.85,
            "Healthcare": 0.80,
            "Financial Services": 0.70,
            "Consumer Defensive": 0.75,
            "Consumer Cyclical": 0.65,
            "Communication Services": 0.70,
            "Industrials": 0.55,
            "Real Estate": 0.60,
            "Utilities": 0.50,
            "Basic Materials": 0.40,
            "Energy": 0.25
        }

        esg_scores = {}
        sector_map = {}
        n_sustainalytics = 0
        n_proxy = 0

        # Supprimer les messages HTTP 404 de yfinance
        import logging, io, sys
        logging.getLogger('yfinance').setLevel(logging.CRITICAL)

        # Test rapide : verifier si l'API Yahoo Finance fonctionne
        api_works = False
        test_ticker = 'AAPL'
        _stderr_bak = sys.stderr
        _stdout_bak = sys.stdout
        sys.stderr = io.StringIO()
        sys.stdout = io.StringIO()
        try:
            _test = yf.Ticker(test_ticker)
            _info = _test.info
            if _info and 'sector' in _info:
                api_works = True
        except (ValueError, KeyError, Exception) as e:
            print(f"[WARN] {type(e).__name__}: {e}")
        sys.stderr = _stderr_bak
        sys.stdout = _stdout_bak

        if api_works:
            print(f"  [ESG] API Yahoo Finance accessible, chargement des scores...")
        else:
            print(f"  [ESG] API Yahoo Finance inaccessible. Utilisation du proxy sectoriel.")

        for ticker in tickers:

            if not api_works:
                sector_map[ticker] = "Unknown"
                esg_scores[ticker] = 0.5
                n_proxy += 1
                continue

            _stderr_bak = sys.stderr
            _stdout_bak = sys.stdout
            sys.stderr = io.StringIO()
            sys.stdout = io.StringIO()
            try:
                tk = yf.Ticker(ticker)
                info = tk.info

                sector = info.get("sector", None)
                sector_map[ticker] = sector if sector else "Unknown"

                sust = tk.sustainability
                if sust is not None and not sust.empty:
                    total_esg_risk = None
                    if 'totalEsg' in sust.index:
                        total_esg_risk = sust.loc['totalEsg'].values[0]
                    elif 'Total ESG Risk score' in sust.index:
                        total_esg_risk = sust.loc['Total ESG Risk score'].values[0]

                    if total_esg_risk is not None and not np.isnan(total_esg_risk):
                        esg_scores[ticker] = max(0.1, 1.0 - total_esg_risk / 50.0)
                        n_sustainalytics += 1
                        continue

                employees = info.get("fullTimeEmployees", 10000)
                marketcap = info.get("marketCap", 1e9)

                E = sector_environment.get(sector, 0.6)
                S = min(0.9, 0.4 + np.log(employees + 1) / 12)
                G = min(0.9, 0.5 + np.log(marketcap + 1) / 35)

                ESG = 0.4 * E + 0.3 * S + 0.3 * G
                esg_scores[ticker] = ESG
                n_proxy += 1

            except (ValueError, KeyError, Exception) as e:
                print(f"[WARN] {type(e).__name__}: {e}")
                sector_map[ticker] = "Unknown"
                esg_scores[ticker] = 0.5
                n_proxy += 1
            finally:
                sys.stderr = _stderr_bak
                sys.stdout = _stdout_bak

        print(f"  [ESG] Scores Sustainalytics: {n_sustainalytics} | "
              f"Proxy: {n_proxy} / {len(tickers)} tickers")

        self.sector_map = pd.Series(sector_map, name="sector")
        return pd.Series(esg_scores, name="esg_score")



    def _handle_delisted_tickers(self, prices):
        '''
        Detecte les tickers delistes (prix constant pendant 30+ jours
        a la fin de la serie) et met les prix a NaN apres le delistage.

        Cela empeche le backtester de detenir un titre mort dont le
        prix est artificiellement constant via (ffill).
        '''
        for col in prices.columns:
            series = prices[col].dropna()
            if len(series) < 30:
                continue

            # Chercher si le prix est constant a la fin
            last_30 = series.tail(30)
            if last_30.nunique() <= 1:
                # Prix constant = probablement deliste
                # Trouver la date ou le prix est devenu constant
                changes = series.diff().ne(0)
                last_change = changes[changes].index[-1] if changes.any() else series.index[0]

                # Mettre NaN apres la derniere variation reelle + 5 jours
                # (laisser 5 jours pour que le dernier trade soit pris en compte)
                cutoff = last_change + timedelta(days=5)
                prices.loc[prices.index > cutoff, col] = np.nan

        return prices

    def _load_yahoo(self, tickers, start, end):
        """Charge les prix depuis Yahoo Finance."""
        # Charger par lots pour éviter les timeouts
        batch_size = 100
        all_data = []

        for i in range(0, len(tickers), batch_size):
            batch = tickers[i:i+batch_size]
            try:
                data = yf.download(batch, start=start, end=end, progress=False, auto_adjust=True, threads=True)
                if isinstance(data.columns, pd.MultiIndex):
                    prices = data['Close']
                else:
                    prices = data
                all_data.append(prices)
            except (ValueError, KeyError, Exception) as e:
                print(f"[WARN] {type(e).__name__}: {e}")
                print(f"  Erreur batch {i//batch_size + 1}: {e}")

        if all_data:
            prices = pd.concat(all_data, axis=1)
            prices = prices.loc[:, ~prices.columns.duplicated()]
            return prices.ffill().dropna(how='all', axis=1)
        return pd.DataFrame()

    # Chargement des prix ET volumes
    def _load_yahoo_with_volume(self, tickers, start, end):
        """Charge les prix ET volumes depuis Yahoo Finance."""
        batch_size = 100
        all_prices = []
        all_volumes = []

        for i in range(0, len(tickers), batch_size):
            batch = tickers[i:i+batch_size]
            try:
                data = yf.download(batch, start=start, end=end, progress=False, auto_adjust=True, threads=True)
                if isinstance(data.columns, pd.MultiIndex):
                    prices = data['Close']
                    volumes = data['Volume']
                else:
                    # Un seul ticker
                    prices = data[['Close']].rename(columns={'Close': batch[0]})
                    volumes = data[['Volume']].rename(columns={'Volume': batch[0]})
                all_prices.append(prices)
                all_volumes.append(volumes)
            except (ValueError, KeyError, Exception) as e:
                print(f"[WARN] {type(e).__name__}: {e}")
                print(f"  Erreur batch {i//batch_size + 1}: {e}")

        if all_prices:
            prices = pd.concat(all_prices, axis=1)
            volumes = pd.concat(all_volumes, axis=1)
            prices = prices.loc[:, ~prices.columns.duplicated()]
            volumes = volumes.loc[:, ~volumes.columns.duplicated()]
            return prices.ffill().dropna(how='all', axis=1), volumes.ffill().fillna(0)
        return pd.DataFrame(), pd.DataFrame()

    def get_excluded_tickers(self) -> set:
        """
        Retourne l'ensemble des tickers à exclure selon la configuration.
        """
        if not hasattr(self.config, 'EXCLUSIONS') or not self.config.EXCLUSIONS.get('enabled', False):
            return set()

        cfg = self.config.EXCLUSIONS
        excluded = set()

        # Tickers explicites
        excluded.update(cfg.get('tickers', []))

        # ESG presets
        for preset_name in cfg.get('apply_esg_presets', []):
            if preset_name in cfg.get('esg_presets', {}):
                excluded.update(cfg['esg_presets'][preset_name])

        return excluded

    def filter_excluded_tickers(self, tickers: list) -> list:
        """
        Filtre une liste de tickers en retirant les exclusions.
        """
        excluded = self.get_excluded_tickers()
        if not excluded:
            return tickers

        filtered = [t for t in tickers if t not in excluded]
        n_excluded = len(tickers) - len(filtered)

        if n_excluded > 0:
            print(f"  [INFO] Exclusions: {n_excluded} tickers retirés ({', '.join(list(excluded)[:5])}...)")

        return filtered

    def _load_fx_rate(self, start, end):
        """Charge le taux de change USD/CAD."""
        fx_data = yf.download('USDCAD=X', start=start, end=end, progress=False, auto_adjust=True)
        if isinstance(fx_data.columns, pd.MultiIndex):
            fx_rate = fx_data['Close']['USDCAD=X']
        else:
            fx_rate = fx_data['Close']
        return fx_rate.ffill().bfill()

    def _convert_to_cad(self, prices, currency_map, fx_rate):
        """Convertit tous les prix en CAD.
        Politique de remplissage : ffill(limit=5) uniquement — pas de bfill.
        Pour le tout debut de serie : premiere valeur disponible comme constante.
        """
        prices_cad = prices.copy()

        # Reindex sur le calendrier des prix puis ffill(limit=5) — pas de bfill
        fx_aligned = fx_rate.reindex(prices.index).ffill(limit=5)

        # Debut de serie : NaN restants → premiere valeur disponible comme constante
        _fx_clean = fx_rate.dropna()
        first_valid_fx = float(_fx_clean.iloc[0]) if len(_fx_clean) > 0 else 1.0
        fx_aligned = fx_aligned.fillna(first_valid_fx)

        for ticker in prices.columns:
            if currency_map.get(ticker) == 'USD':
                prices_cad[ticker] = prices[ticker] * fx_aligned

        # Diagnostic : NaN restants apres conversion CAD
        n_nan   = int(prices_cad.isna().sum().sum())
        n_cells = int(prices_cad.size)
        print(f"  [_convert_to_cad] NaN restants apres conversion : "
              f"{n_nan}/{n_cells} ({n_nan/n_cells:.4%})")

        return prices_cad

    def _load_risk_free_rate(self, start, end):
        """Charge le taux sans risque depuis FRED."""
        api_key = os.environ.get('FRED_API_KEY')
        if not api_key:
            print("[ATTENTION] FRED API key non trouvée, utilisation du taux par défaut")
            return None

        try:
            url = "https://api.stlouisfed.org/fred/series/observations"
            params = {
                'series_id': 'DGS3MO',  # Treasury 3-month
                'api_key': api_key,
                'file_type': 'json',
                'observation_start': start.strftime('%Y-%m-%d'),
                'observation_end': end.strftime('%Y-%m-%d'),
            }
            response = requests.get(url, params=params)
            obs = response.json().get('observations', [])

            if obs:
                series = pd.Series({
                    pd.to_datetime(o['date']): float(o['value']) / 100 if o['value'] != '.' else np.nan
                    for o in obs
                }).dropna()
                print(f"  [OK] Taux sans risque chargé: {len(series)} points")
                return series
        except (ValueError, KeyError, Exception) as e:
            print(f"[WARN] {type(e).__name__}: {e}")
            print(f"  [ATTENTION] Erreur chargement FRED: {e}")

        return None

    def load_vix(self, start, end):
        """Charge le VIX depuis FRED."""
        api_key = os.environ.get('FRED_API_KEY')
        if not api_key:
            return None

        try:
            url = "https://api.stlouisfed.org/fred/series/observations"
            params = {
                'series_id': 'VIXCLS',
                'api_key': api_key,
                'file_type': 'json',
                'observation_start': start.strftime('%Y-%m-%d'),
                'observation_end': end.strftime('%Y-%m-%d'),
            }
            response = requests.get(url, params=params)
            obs = response.json().get('observations', [])

            if obs:
                series = pd.Series({
                    pd.to_datetime(o['date']): float(o['value']) if o['value'] != '.' else np.nan
                    for o in obs
                })
                # ffill(limit=5) uniquement — pas de bfill.
                # Debut de serie : premiere valeur disponible comme constante.
                series = series.ffill(limit=5)
                _first = series.dropna()
                if len(_first) > 0:
                    series = series.fillna(float(_first.iloc[0]))
                return series
        except (ValueError, KeyError, Exception) as e:
            print(f"[WARN] {type(e).__name__}: {e}")
        return None

    def load_macro_data(self, start, end):
        """V4: Charge les données macro pour la détection de régimes."""
        api_key = os.environ.get('FRED_API_KEY')
        if not api_key:
            print("  [ATTENTION] FRED API key non trouvée")
            return {}
        series_map = {'yield_10y': 'DGS10', 'yield_2y': 'DGS2', 'credit_spread_hy': 'BAMLH0A0HYM2'}
        print("  Chargement données macro pour régimes...")
        for name, series_id in series_map.items():
            try:
                url = "https://api.stlouisfed.org/fred/series/observations"
                params = {'series_id': series_id, 'api_key': api_key, 'file_type': 'json',
                         'observation_start': start.strftime('%Y-%m-%d'), 'observation_end': end.strftime('%Y-%m-%d')}
                response = requests.get(url, params=params, timeout=10)
                obs = response.json().get('observations', [])
                if obs:
                    series = pd.Series({pd.to_datetime(o['date']): float(o['value']) if o['value'] != '.' else np.nan for o in obs}).dropna().sort_index()
                    self.macro_data[name] = series
                    print(f"    [OK] {name}: {len(series)} obs")
            except (ValueError, KeyError, Exception) as e:
                print(f"[WARN] {type(e).__name__}: {e}")
                print(f"    [ATTENTION] {name}: {e}")
        return self.macro_data

    def get_fx_exposure_report(self, weights: pd.Series) -> Dict:
        """Calcule l'exposition FX du portefeuille."""
        usd_exposure = 0
        cad_exposure = 0

        for ticker, weight in weights.items():
            if weight > 0:
                currency = self.asset_currency.get(ticker, 'USD')
                if currency == 'USD':
                    usd_exposure += weight
                else:
                    cad_exposure += weight

        total = usd_exposure + cad_exposure
        if total > 0:
            usd_pct = usd_exposure / total
            cad_pct = cad_exposure / total
        else:
            usd_pct = cad_pct = 0

        return {
            'usd_exposure': usd_exposure,
            'cad_exposure': cad_exposure,
            'usd_pct': usd_pct,
            'cad_pct': cad_pct,
        }


# =========================================================================
# CONTRAINTES / COMPLIANCE (Jan 2026)
# =========================================================================
    # NOTE: Le mandat peut être configuré via Config.CONSTRAINTS.
    # - crypto ETFs permis: True/False
    # - exotiques (commodities, etc.) limités à 5% (par défaut selon 1er cours)
    # Cette logique est centralisée ici pour rester "Presentation-ready".

    def classify_ticker(self, t: str) -> str:
        """Retourne une étiquette d'actif pour les contraintes."""
        t = str(t)
        if hasattr(self, "US_BOND_ETFS") and t in self.US_BOND_ETFS:
            return "BOND_ETF"
        if hasattr(self, "CA_BOND_ETFS") and t in self.CA_BOND_ETFS:
            return "BOND_ETF"
        if hasattr(self, "CRYPTO_ETFS") and t in self.CRYPTO_ETFS:
            return "CRYPTO_ETF"
        # Exotiques: ici on considère COMMODITY_ETFS comme exotiques.
        if hasattr(self, "COMMODITY_ETFS") and t in self.COMMODITY_ETFS:
            return "EXOTIC_ETF"
        return "EQUITY"

    def get_compliance_breakdown(self, weights: pd.Series, stock_alloc: float, bond_alloc: float, cash_alloc: float) -> dict:
        """Calcule les expositions par catégories sur le portefeuille total."""
        w = weights.fillna(0.0)
        expo = {"EQUITY": 0.0, "BOND_ETF": 0.0, "CRYPTO_ETF": 0.0, "EXOTIC_ETF": 0.0}
        # Les poids 'weights' sont des scores/poids sur les titres; les allocations stock/bond/cash appliquent l'échelle.
        # Ici, on approxime: les tickers 'BOND_ETF' sont dans la poche bonds; les autres dans stocks.
        for t, wt in w.items():
            if wt <= 0:
                continue
            cls = self.classify_ticker(t)
            if cls == "BOND_ETF":
                expo["BOND_ETF"] += float(wt) * bond_alloc
            else:
                # tout le reste est considéré dans la poche actions
                expo[cls] += float(wt) * stock_alloc

        # Normalisation: les weights ne sont pas forcément déjà normalisés à 1 dans chaque poche.
        # Pour produire une lecture robuste, on renormalise à la somme des expositions "risky" (stocks+bonds), puis on ajoute cash.
        risky = sum(expo.values())
        total = risky + float(cash_alloc)
        if total > 0:
            for k in list(expo.keys()):
                expo[k] = expo[k] / total
        expo["CASH"] = float(cash_alloc) / total if total > 0 else 0.0
        return expo

    def apply_allocation_constraints(self, weights: pd.Series, stock_alloc: float, bond_alloc: float, cash_alloc: float,
                                     allow_crypto: bool = True, max_exotic: float = 0.05, max_crypto: float = 0.03) -> tuple[pd.Series, dict]:
        """Applique des contraintes simples (crypto/exotique) en ajustant les poids et retourne un rapport compliance."""
        w = weights.fillna(0.0).copy()
        w[w < 0] = 0.0

        # Sépare stocks (non-bond) vs bonds
        bond_set = set(getattr(self, "US_BOND_ETFS", [])) | set(getattr(self, "CA_BOND_ETFS", []))
        bond_mask = w.index.isin(list(bond_set))
        stock_mask = ~bond_mask

        # Normalise par poche si possible
        if w[stock_mask].sum() > 0:
            w.loc[stock_mask] = w.loc[stock_mask] / w.loc[stock_mask].sum() * stock_alloc
        if w[bond_mask].sum() > 0:
            w.loc[bond_mask] = w.loc[bond_mask] / w.loc[bond_mask].sum() * bond_alloc

        # Contraintes sur la poche actions (stock_alloc) uniquement
        exotic_set = set(getattr(self, "COMMODITY_ETFS", []))
        crypto_set = set(getattr(self, "CRYPTO_ETFS", []))

        exotic_mask = stock_mask & w.index.isin(list(exotic_set))
        crypto_mask = stock_mask & w.index.isin(list(crypto_set))

        def _cap_group(mask, cap_share_total):
            """Cap une sous-classe à cap_share_total du portefeuille total."""
            nonlocal w
            if stock_alloc <= 0 or w.loc[stock_mask].sum() <= 0:
                return
            current_total_share = float(w.loc[mask].sum()) * stock_alloc  # part du portefeuille total
            if current_total_share <= cap_share_total:
                return
            # facteur de réduction
            if current_total_share > 0:
                scale = cap_share_total / current_total_share
                w.loc[mask] *= scale
                # redistribuer la masse retirée sur le reste de la poche actions
                remainder_mask = stock_mask & (~mask)
                rem_sum = float(w.loc[remainder_mask].sum())
                if rem_sum > 0:
                    w.loc[remainder_mask] *= (1.0 + (1.0 - scale) * float(w.loc[mask].sum()) / rem_sum)

        # Exotiques
        _cap_group(exotic_mask, max_exotic)

        # Crypto (si autorisé)
        if allow_crypto:
            _cap_group(crypto_mask, max_crypto)
        else:
            w.loc[crypto_mask] = 0.0

        # Renormalise poches après caps
        if w[stock_mask].sum() > 0:
            w.loc[stock_mask] = w.loc[stock_mask] / w.loc[stock_mask].sum() * stock_alloc
        if w[bond_mask].sum() > 0:
            w.loc[bond_mask] = w.loc[bond_mask] / w.loc[bond_mask].sum() * bond_alloc

        compliance = self.get_compliance_breakdown(w, stock_alloc, bond_alloc, cash_alloc)
        compliance.update({
            "allow_crypto": bool(allow_crypto),
            "max_exotic": float(max_exotic),
            "max_crypto": float(max_crypto),
        })
        return w, compliance

    def get_liquid_tickers(self, as_of_date, prices=None, volumes_dollar=None):
        """
        Retourne la liste des tickers qui passent les filtres de liquidité.

        Args:
            as_of_date: Date de référence pour le calcul
            prices: DataFrame des prix (optionnel, utilise self.prices si None)
            volumes_dollar: DataFrame des volumes en $ (optionnel)

        Returns:
            list: Tickers éligibles selon les contraintes de liquidité
        """
        if not hasattr(self.config, 'LIQUIDITY') or not self.config.LIQUIDITY.get('enabled', False):
            # Si pas de config liquidité, retourner tous les tickers
            return list(self.prices.columns) if prices is None else list(prices.columns)

        cfg = self.config.LIQUIDITY
        lookback = cfg.get('adv_lookback_days', 20)
        min_adv = cfg.get('min_avg_daily_volume_usd', 1_000_000)

        if prices is None:
            prices = self.prices
        if volumes_dollar is None:
            volumes_dollar = self.volumes_dollar if hasattr(self, 'volumes_dollar') else None

        if volumes_dollar is None:
            print("[ATTENTION] Volumes non disponibles, filtrage liquidité désactivé")
            return list(prices.columns)

        # Filtrer jusqu'à as_of_date
        vol_slice = volumes_dollar.loc[:as_of_date].tail(lookback)

        if len(vol_slice) < lookback // 2:
            print(f"[ATTENTION] Historique volume insuffisant ({len(vol_slice)} jours)")
            return list(prices.columns)

        # Calculer ADV (Average Daily Volume en $)
        adv = vol_slice.mean()

        # Filtrer les titres avec ADV >= minimum
        liquid_tickers = adv[adv >= min_adv].index.tolist()
        return liquid_tickers


    def compute_max_position_weight(self, ticker, as_of_date, portfolio_value=None):
        """
        Calcule le poids maximum d'une position basé sur les contraintes de liquidité.

        Args:
            ticker: Symbole du titre
            as_of_date: Date de référence
            portfolio_value: Valeur du portefeuille (défaut: Config.PORTFOLIO_VALUE)

        Returns:
            float: Poids maximum autorisé (entre 0 et 1)
        """
        # Si liquidité désactivée, retourner le max_weight de la config
        if not hasattr(self.config, 'LIQUIDITY') or not self.config.LIQUIDITY.get('enabled', False):
            return self.config.STOCK_SELECTION.get('max_weight', 0.10)

        cfg = self.config.LIQUIDITY
        if portfolio_value is None:
            portfolio_value = self.config.PORTFOLIO_VALUE

        lookback = cfg.get('adv_lookback_days', 20)
        max_adv_fraction = cfg.get('max_position_adv_fraction', 0.10)
        max_days = cfg.get('max_days_to_liquidate', 5)

        # Vérifier si volumes disponibles
        if not hasattr(self, 'volumes_dollar') or self.volumes_dollar is None:
            return self.config.STOCK_SELECTION.get('max_weight', 0.10)

        if ticker not in self.volumes_dollar.columns:
            return 0.0  # Ticker sans volume = non éligible

        # Récupérer le volume historique
        vol_slice = self.volumes_dollar[ticker].loc[:as_of_date].tail(lookback)
        if len(vol_slice) < 5:
            return 0.0

        adv = vol_slice.mean()

        # Méthode 1: Fraction de l'ADV
        max_position_by_adv = adv * max_adv_fraction

        # Méthode 2: Liquidation en N jours
        max_position_by_days = adv * max_days

        # Prendre le minimum des deux contraintes
        max_position_usd = min(max_position_by_adv, max_position_by_days)

        # Convertir en poids
        max_weight = max_position_usd / portfolio_value

        # Plafonner au max_weight de la config
        config_max = self.config.STOCK_SELECTION.get('max_weight', 0.10)

        return min(max_weight, config_max)

    def get_liquidity_constraints(self, tickers, as_of_date):
        """
        Retourne un dictionnaire {ticker: max_weight} pour tous les tickers.
        Utilisé par l'optimiseur pour imposer les contraintes.

        Args:
            tickers: Liste des tickers à évaluer
            as_of_date: Date de référence

        Returns:
            dict: {ticker: max_weight_allowed}
        """
        constraints = {}
        for ticker in tickers:
            constraints[ticker] = self.compute_max_position_weight(ticker, as_of_date)
        return constraints



print("[OK] Module DataLoader V3 chargé (univers complet: ~750 titres)")

# =============================================================================
# EXCEPT UNIFORMISÉS — DataLoader (blocs modifiés)
# =============================================================================
#   ligne ~372 : WARN sans type(e).__name__
#   ligne ~425 : WARN sans type(e).__name__
#   ligne ~484 : except Exception → triple + WARN
#   ligne ~513 : except Exception → triple + WARN
#   ligne ~622 : except Exception → triple + WARN
#   ligne ~658 : WARN sans type(e).__name__
#   ligne ~680 : except Exception → triple + WARN
# =============================================================================

[OK] Module DataLoader V3 chargé (univers complet: ~750 titres)


---
# 3 MODULE Modèle factoriel - Calculs des facteurs
---


In [4]:
# =============================================================================
# CELLULE 3 : CALCUL DES FACTEURS
# =============================================================================
#
# Quatre facteurs : 52-week high, momentum, profitability, quality.
# Les deux premiers sont des signaux de prix. Les deux derniers utilisent
# des données fondamentales trimestrielles (Yahoo Finance) avec respect
# strict du point-in-time : seules les publications antérieures à la date
# d'évaluation (+ 45 j de délai SEC) sont utilisées.
#
# Références :
#   George & Hwang (2004), Journal of Finance — 52-week high
#   Jegadeesh & Titman (1993), Journal of Finance — momentum
#   Novy-Marx (2013), Journal of Financial Economics — gross profitability
#   Asness, Frazzini & Pedersen (2019), Review of Accounting Studies — QMJ
# =============================================================================

class FactorCalculator:
    """
    Modele factoriel a 6 facteurs avec donnees Compustat/WRDS.

    Facteurs :
        1. Momentum       — Rendement 12m skip 1m (prix Yahoo Finance)
        2. Value           — Earnings Yield TTM via Compustat
        3. Profitability   — GPOA = Gross Profit / Assets (Novy-Marx 2013)
        4. Low Volatility  — Inverse volatilite 12m (prix Yahoo Finance)
        5. Investment/CMA  — Inverse croissance actifs YoY (Fama-French 2015)
        6. PEAD            — Surprise de resultats normalisee (Bernard & Thomas 1989)

    Les poids sont ajustes dynamiquement selon le regime (RISK_ON / RISK_OFF).
    """

    def __init__(self, config=Config):
        self.config = config
        self._fundamental_cache = {}
        self._fundamental_ts_cache = {}
        self._fundamental_load_date = None
        self._compustat_df = None  # Cache du DataFrame Compustat complet
        self._compustat_by_ticker = {}  # Pre-indexe par ticker pour performance
        self._data_loader_ref = None

    # =================================================================
    # Point d'entree principal
    # =================================================================

    def compute_all_factors(self, prices, returns, as_of_date=None, regime=None):
        """Calcule les 6 facteurs a la date as_of_date."""
        if as_of_date is not None:
            prices = prices.loc[:as_of_date]
            returns = returns.loc[:as_of_date]

        factors = {}
        tickers = prices.columns.tolist()

        # Charger Compustat si pas encore fait
        if self._compustat_df is None:
            self._load_compustat_data(tickers)
        # Aussi charger dans le cache legacy pour compatibilite
        if not self._fundamental_cache:
            self._load_fundamental_data(tickers)

        # --- 1. Momentum adaptatif (prix) ---
        # RISK_ON: momentum classique | RISK_OFF: 52-week high
        cfg = self.config.FACTORS.get('momentum', {})
        if cfg.get('enabled'):
            factors['momentum'] = self._compute_momentum(prices, regime=regime)

        # --- 2. Value (Compustat : earnings yield) ---
        cfg = self.config.FACTORS.get('value', {})
        if cfg.get('enabled'):
            factors['value'] = self._compute_value(tickers, as_of_date)

        # --- 3. Profitability / RMW (Compustat : GPOA) ---
        cfg = self.config.FACTORS.get('profitability', {})
        if cfg.get('enabled'):
            factors['profitability'] = self._compute_profitability_gpoa(
                tickers, returns, as_of_date)

        # --- 4. Low Volatility (prix) ---
        cfg = self.config.FACTORS.get('low_volatility', {})
        if cfg.get('enabled'):
            factors['low_volatility'] = self._compute_low_volatility(returns)

        # --- 5. Investment / CMA (Compustat : asset growth) ---
        cfg = self.config.FACTORS.get('investment', {})
        if cfg.get('enabled'):
            factors['investment'] = self._compute_investment_cma(tickers, as_of_date)

        # --- 6. PEAD (Compustat : SUE + report date) ---
        cfg = self.config.FACTORS.get('pead', {})
        if cfg.get('enabled'):
            factors['pead'] = self._compute_pead(tickers, as_of_date)

        return pd.DataFrame(factors)

    # =================================================================
    # Facteurs de prix (Yahoo Finance)
    # =================================================================

    def _compute_momentum(self, prices, regime=None):
        """
        Facteur momentum adaptatif selon le regime de marche.

        RISK_ON  : Momentum classique (rendement 12m skip 1m, Jegadeesh & Titman 1993)
                   Capte la tendance haussiere, signal fort en marche directionnel.
        RISK_OFF : 52-week high (George & Hwang 2004)
                   Ancrage psychologique, moins sujet aux crashes de momentum.
                   Le momentum classique est expose au crash risk lors des reversals
                   (2009, 2020). Le 52w_high est plus stable dans ces periodes.

        Ce switch est justifie par la litterature sur le risk-managed momentum
        (Barroso & Santa-Clara 2015, Daniel & Moskowitz 2016).
        """
        cfg = self.config.FACTORS.get('momentum', {})
        lookback = cfg.get('lookback', 252)

        if len(prices) < lookback:
            return pd.Series(0.5, index=prices.columns)

        if regime == 'RISK_OFF':
            # 52-week high : ratio prix courant / plus haut 52 semaines
            high_52w = prices.tail(lookback).max()
            current = prices.iloc[-1]
            ratio = current / high_52w
            score = ratio.rank(pct=True).fillna(0.5)
        else:
            # Momentum classique : rendement 12m skip 1m
            skip = cfg.get('skip_recent', 21)
            end_idx = len(prices) - skip
            start_idx = max(0, end_idx - lookback)
            ret = (prices.iloc[end_idx] / prices.iloc[start_idx]) - 1
            score = ret.rank(pct=True).fillna(0.5)

        # --- Vol réalisée du facteur momentum (Barroso & Santa-Clara 2015) ---
        # Long-short equal-weight (top-25% vs bottom-25%) sur 126 jours glissants
        _vol_window = 126
        if len(prices) >= _vol_window:
            _daily_rets = prices.pct_change().tail(_vol_window)
            # Score momentum classique sur fenêtre vol (indépendant du régime)
            _mom_126 = (prices.iloc[-1] / prices.iloc[-_vol_window]) - 1
            _q75 = _mom_126.quantile(0.75)
            _q25 = _mom_126.quantile(0.25)
            _winners = _mom_126[_mom_126 >= _q75].index
            _losers  = _mom_126[_mom_126 <= _q25].index
            if len(_winners) > 0 and len(_losers) > 0:
                _ls_rets = (_daily_rets[_winners].mean(axis=1) -
                            _daily_rets[_losers].mean(axis=1))
                self._momentum_vol_realized = _ls_rets.std() * (252 ** 0.5)
            else:
                self._momentum_vol_realized = None
        else:
            self._momentum_vol_realized = None

        return score

    def _compute_profitability_proxy(self, returns):
        """Fallback quand Compustat n'est pas disponible.
        Utilise le Sharpe ratio 6 mois comme proxy de profitabilité."""
        if returns is None or len(returns) < 126:
            return pd.Series(0.5, index=returns.columns if returns is not None else [])
        sharpe = returns.tail(126).mean() / returns.tail(126).std().replace(0, np.nan)
        proxy = sharpe.rank(pct=True).fillna(0.5)
        return 0.5 * proxy + 0.5 * 0.5  # atténué vers 0.5

    def _compute_quality_proxy(self, returns):
        """Fallback quand Compustat n'est pas disponible.
        Utilise l'inverse de la volatilité comme proxy de qualité."""
        if returns is None or len(returns) < 126:
            return pd.Series(0.5, index=returns.columns if returns is not None else [])
        inv_vol = 1.0 / returns.tail(126).std().replace(0, np.nan)
        proxy = inv_vol.rank(pct=True).fillna(0.5)
        return 0.5 * proxy + 0.5 * 0.5  # atténué vers 0.5

    def _compute_low_volatility(self, returns):
        """
        Inverse de la volatilite realisee 12 mois.

        Les titres a faible volatilite surperforment sur base ajustee au risque
        (Black 1972, Frazzini & Pedersen 2014, van Vliet CFA 2024).
        Alpha de 2.1% apres controle pour tous les autres facteurs.
        """
        cfg = self.config.FACTORS.get('low_volatility', {})
        lookback = cfg.get('lookback', 252)
        if len(returns) < lookback:
            return pd.Series(0.5, index=returns.columns)
        vol = returns.tail(lookback).std() * np.sqrt(252)
        inv_vol = 1 / (vol + 0.01)
        return inv_vol.rank(pct=True).fillna(0.5)

    # =================================================================
    # Facteurs fondamentaux (Compustat/WRDS)
    # =================================================================

    def _compute_value(self, tickers, as_of_date=None):
        """
        Facteur Value = Earnings Yield (NI TTM / Market Cap).

        Definition modernisee qui evite les biais des intangibles du
        book-to-market traditionnel (Arnott et al. 2021).
        Correlation negative avec momentum (~-0.3), diversifie le signal.
        """
        scores = pd.Series(np.nan, index=tickers)

        if self._compustat_by_ticker:
            for ticker in tickers:
                val = self._get_compustat_metric(ticker, as_of_date, 'earnings_yield')
                if val is not None and np.isfinite(val):
                    scores[ticker] = val

        # Fallback : utiliser les donnees legacy si disponibles
        missing = scores.isna()
        if missing.sum() > 0 and self._fundamental_cache:
            for ticker in tickers:
                if pd.isna(scores.get(ticker, np.nan)):
                    pm = self._get_fundamental_as_of(ticker, as_of_date, 'profit_margin')
                    roe = self._get_fundamental_as_of(ticker, as_of_date, 'roe')
                    if pm is not None and roe is not None:
                        scores[ticker] = (pm + roe / 4) / 2

        return scores.rank(pct=True).fillna(0.5)

    def _compute_profitability_gpoa(self, tickers, returns=None, as_of_date=None):
        """
        Facteur Profitability/RMW = GPOA (Gross Profit / Total Assets).

        Definition Novy-Marx (2013) — le meilleur predicteur cross-sectionnel
        de rendements selon la litterature. Seul facteur FF5 ayant
        systematiquement delivre un premium (CFA Institute 2022).
        """
        scores = pd.Series(np.nan, index=tickers)

        if self._compustat_by_ticker:
            for ticker in tickers:
                gpoa = self._get_compustat_metric(ticker, as_of_date, 'gpoa')
                if gpoa is not None and np.isfinite(gpoa):
                    scores[ticker] = gpoa

        # Fallback : composite ROE/marges (legacy)
        missing = scores.isna()
        if missing.sum() > 0:
            for ticker in tickers:
                if pd.isna(scores.get(ticker, np.nan)):
                    components, weights_c = [], []
                    roe = self._get_fundamental_as_of(ticker, as_of_date, 'roe')
                    if roe is not None:
                        components.append(min(max(roe / 0.25, 0.0), 1.0))
                        weights_c.append(0.50)
                    gm = self._get_fundamental_as_of(ticker, as_of_date, 'gross_margins')
                    if gm is not None:
                        components.append(min(max(gm / 0.60, 0.0), 1.0))
                        weights_c.append(0.50)
                    if components:
                        total_w = sum(weights_c)
                        scores[ticker] = sum(c*w for c,w in zip(components,weights_c)) / total_w

        # Fallback ultime : Sharpe proxy attenue
        still_missing = scores.isna()
        if still_missing.sum() > 0 and returns is not None:
            if len(returns) >= 126:
                recent = returns.tail(126)
                sharpe = recent.mean() / recent.std() * np.sqrt(252)
                proxy = sharpe.rank(pct=True).fillna(0.5)
                scores[still_missing] = 0.5 * proxy[still_missing] + 0.25

        return scores.rank(pct=True).fillna(0.5)

    def _compute_investment_cma(self, tickers, as_of_date=None):
        """
        Facteur Investment/CMA = inverse de la croissance des actifs totaux YoY.

        Conservative Minus Aggressive (Fama & French 2015).
        Les entreprises qui investissent peu surperforment.
        Inverse car asset_growth bas = score eleve.
        """
        scores = pd.Series(np.nan, index=tickers)

        if self._compustat_by_ticker:
            for ticker in tickers:
                ag = self._get_compustat_metric(ticker, as_of_date, 'asset_growth')
                if ag is not None and np.isfinite(ag):
                    # Inverser : croissance faible = score eleve
                    scores[ticker] = -ag

        return scores.rank(pct=True).fillna(0.5)

    def _compute_pead(self, tickers, as_of_date=None):
        """
        Facteur PEAD = Post-Earnings Announcement Drift.

        Utilise le SUE (Standardized Unexpected Earnings) de Compustat.
        SUE = (EPS_Q - EPS_Q-4) / std(surprises passees)
        (Bernard & Thomas 1989, Ball & Brown 1968)

        Le signal est pondere par la recence de la publication :
        plus la publication est recente, plus le signal est fort
        (le drift se dissipe sur 60-90 jours).
        """
        scores = pd.Series(np.nan, index=tickers)

        if self._compustat_by_ticker:
            now = pd.Timestamp(as_of_date) if as_of_date else pd.Timestamp.now()

            for ticker in tickers:
                tk_df = self._compustat_by_ticker.get(ticker)
                if tk_df is None or tk_df.empty:
                    continue

                # Filtrer les publications avant as_of_date
                if as_of_date is not None:
                    tk_df = tk_df[tk_df['date'] <= as_of_date]
                if tk_df.empty:
                    continue

                # Derniere observation avec SUE
                with_sue = tk_df.dropna(subset=['sue'])
                if with_sue.empty:
                    # Fallback : utiliser la surprise brute
                    with_surp = tk_df.dropna(subset=['earnings_surprise_raw'])
                    if not with_surp.empty:
                        latest = with_surp.iloc[-1]
                        scores[ticker] = latest['earnings_surprise_raw']
                    continue

                latest = with_sue.iloc[-1]
                sue_val = latest['sue']

                # Ponderer par la recence (decay exponentiel sur 90 jours)
                report_date = latest.get('report_date')
                if pd.notna(report_date):
                    report_date = pd.Timestamp(report_date)
                    days_since = (now - report_date).days
                    # Decay : signal plein < 30j, moitie a 60j, quart a 90j
                    recency_weight = np.exp(-days_since / 60)
                    recency_weight = max(recency_weight, 0.1)  # plancher 10%
                    scores[ticker] = sue_val * recency_weight
                else:
                    scores[ticker] = sue_val

        return scores.rank(pct=True).fillna(0.5)

    # =================================================================
    # Score composite (avec poids dynamiques par regime)
    # =================================================================

    def compute_composite_score(self, factors_df, regime=None, regime_score=None):
        """
        Somme ponderee des rangs percentiles, re-classee en percentiles,
        avec penalite proportionnelle au nombre de facteurs manquants.

        Chaque titre est multiplie par (n_valides / n_total_actifs) :
          - titre avec tous les facteurs valides   -> poids plein
          - titre avec 50 % de facteurs manquants  -> score divise par 2
          - titre sans aucun facteur               -> score = 0

        Si un regime est fourni et que FACTOR_REGIME_WEIGHTS est configure,
        les poids sont ajustes dynamiquement (surponderation des facteurs
        defensifs en RISK_OFF, offensifs en RISK_ON).
        """
        # Determiner les poids a utiliser
        regime_weights = getattr(self.config, 'FACTOR_REGIME_WEIGHTS', None)
        _interp_used   = False
        _interp_score  = None

        if (regime_score is not None
                and regime_weights
                and 'RISK_ON'  in regime_weights
                and 'RISK_OFF' in regime_weights):
            # Interpolation linéaire continue entre RISK_OFF (score=-1) et RISK_ON (score=+1)
            _interp_score = max(-1.0, min(1.0, float(regime_score)))
            _t     = (_interp_score + 1.0) / 2.0   # 0 = RISK_OFF, 1 = RISK_ON
            _w_off = regime_weights['RISK_OFF']
            _w_on  = regime_weights['RISK_ON']
            _all_f = set(_w_off) | set(_w_on)
            weights = {
                f: _w_off.get(f, 0.0) + (_w_on.get(f, 0.0) - _w_off.get(f, 0.0)) * _t
                for f in _all_f
                if (_w_off.get(f, 0.0) + _w_on.get(f, 0.0)) > 0
            }
            _interp_used = True
        elif regime and regime_weights and regime in regime_weights:
            weights = regime_weights[regime]
        else:
            weights = {k: v['weight'] for k, v in self.config.FACTORS.items()
                       if v.get('enabled') and v.get('weight', 0) > 0}

        # Facteurs actifs presents dans factors_df
        active_factors = [f for f in weights if f in factors_df.columns]
        n_total = len(active_factors)

        # Diagnostic interpolation (premier appel uniquement)
        if _interp_used and not getattr(self, '_interp_diagnostic_done', False):
            _diag = ', '.join(
                f'{f}={weights[f]:.1%}' for f in active_factors if f in weights
            )
            print(f'Poids interpolés (score={_interp_score:.2f}): {_diag}')
            self._interp_diagnostic_done = True

        # Nombre de facteurs valides (non-NaN) par titre
        if n_total > 0:
            n_valid = factors_df[active_factors].notna().sum(axis=1)
        else:
            n_valid = pd.Series(0, index=factors_df.index)

        # --- Risk-managed momentum (Barroso & Santa-Clara 2015) ---
        _mom_vol = getattr(self, '_momentum_vol_realized', None)
        _use_rmom = getattr(self.config, 'RISK_MANAGED_MOMENTUM', True)
        if _use_rmom and _mom_vol is not None and 'momentum' in weights and 'momentum' in active_factors:
            _target_vol = 0.15
            _mult = _target_vol / _mom_vol
            _mult = max(0.3, min(1.5, _mult))
            weights = dict(weights)  # copie — ne pas muter l'original
            weights['momentum'] = weights['momentum'] * _mult
            # Renormaliser les poids actifs pour que leur somme = 1.0
            _sum_active = sum(weights.get(f, 0) for f in active_factors)
            if _sum_active > 0:
                for _f in active_factors:
                    if _f in weights:
                        weights[_f] /= _sum_active
            _new_mom_w = weights.get('momentum', 0)
            if not getattr(self, '_rmom_diagnostic_done', False):
                print(f'Risk-managed momentum: vol={_mom_vol:.1%}, mult={_mult:.2f}, '
                      f'poids momentum ajusté={_new_mom_w:.1%}')
                self._rmom_diagnostic_done = True

        # --- Diagnostic (premier appel uniquement) ---
        _run_diag = not getattr(self, '_composite_diagnostic_done', False)
        if _run_diag and n_total > 0:
            # Score AVANT : fillna(0.5) — reference avant la modification
            old_comp = pd.Series(0.0, index=factors_df.index)
            old_w = 0.0
            for f, w in weights.items():
                if f in factors_df.columns:
                    old_comp += w * factors_df[f].fillna(0.5)
                    old_w += w
            if old_w > 0:
                old_comp /= old_w
            old_top25  = set(old_comp.rank(pct=True).nlargest(25).index)
            old_n_miss = sum(1 for t in old_top25
                             if n_valid.get(t, 0) < n_total * 0.5)

        # Score NOUVEAU : fillna(0) + penalite proportionnelle
        composite = pd.Series(0.0, index=factors_df.index)
        total_w = 0.0
        for factor, weight in weights.items():
            if factor in factors_df.columns:
                composite += weight * factors_df[factor].fillna(0.0)
                total_w += weight

        if total_w > 0:
            composite = composite / total_w

        # Penalite proportionnelle : score *= (n_valides / n_total_actifs)
        if n_total > 0:
            composite = composite * (n_valid / n_total)

        ranked = composite.rank(pct=True)

        # --- Afficher le diagnostic une seule fois ---
        if _run_diag and n_total > 0:
            new_top25  = set(ranked.nlargest(25).index)
            new_n_miss = sum(1 for t in new_top25
                             if n_valid.get(t, 0) < n_total * 0.5)
            removed = old_top25 - new_top25
            added   = new_top25 - old_top25
            print('\n' + '-' * 60)
            print('[PENALITE FACTEURS MANQUANTS] Diagnostic — premier appel')
            print('-' * 60)
            print(f'  AVANT (fillna=0.5) : {old_n_miss}/25 titres avec >50% facteurs manquants')
            print(f'  APRES (penalite)   : {new_n_miss}/25 titres avec >50% facteurs manquants')
            if removed:
                print(f'  Retires du top-25  : {sorted(removed)}')
            if added:
                print(f'  Ajoutes au top-25  : {sorted(added)}')
            if not removed and not added:
                print('  Composition top-25 identique (aucun titre retire/ajoute)')
            print('-' * 60)
            self._composite_diagnostic_done = True

        return ranked

    # =================================================================
    # Chargement Compustat (nouveau : DataFrame complet)
    # =================================================================

    def _load_compustat_data(self, tickers):
        """Charge le DataFrame Compustat complet en memoire."""
        import os

        search_paths = [
            'compustat_fundamentals.csv',
            '/content/compustat_fundamentals.csv',
            '/content/drive/MyDrive/compustat_fundamentals.csv',
            os.path.join(os.getcwd(), 'compustat_fundamentals.csv'),
        ]

        csv_path = None
        for path in search_paths:
            if os.path.exists(path):
                csv_path = path
                break

        if csv_path is None:
            print("  [COMPUSTAT] Fichier compustat_fundamentals.csv non trouve.")
            self._compustat_df = pd.DataFrame()
            return

        try:
            date_cols = ['date']
            # report_date peut avoir des NaT, on le parse separement
            df = pd.read_csv(csv_path, parse_dates=date_cols)
            if 'report_date' in df.columns:
                df['report_date'] = pd.to_datetime(df['report_date'], errors='coerce')

            tickers_set = set(tickers)
            df_filtered = df[df['yahoo_ticker'].isin(tickers_set)]

            self._compustat_df = df_filtered.sort_values(['yahoo_ticker', 'date'])

            n_tk = df_filtered['yahoo_ticker'].nunique()
            n_new = 0
            for col in ['gpoa', 'asset_growth', 'earnings_yield', 'sue']:
                if col in df_filtered.columns:
                    n_col = df_filtered.dropna(subset=[col])['yahoo_ticker'].nunique()
                    n_new = max(n_new, n_col)

            # Pre-indexer par ticker pour eviter les filtres repetes
            # sur le DataFrame complet (248K lignes) a chaque appel
            self._compustat_by_ticker = {
                tk: grp.sort_values('date')
                for tk, grp in df_filtered.groupby('yahoo_ticker')
            }

            print(f"  [COMPUSTAT v2] {n_tk} tickers charges, "
                  f"nouveaux facteurs: ~{n_new} tickers avec donnees")

        except Exception as e:
            print(f"  [COMPUSTAT] Erreur: {e}")
            self._compustat_df = pd.DataFrame()
            self._compustat_by_ticker = {}

    def _get_compustat_metric(self, ticker, as_of_date, metric):
        """Recupere la derniere valeur d'une metrique Compustat avant as_of_date."""
        tk_data = self._compustat_by_ticker.get(ticker)
        if tk_data is None or tk_data.empty or metric not in tk_data.columns:
            return None

        if as_of_date is not None:
            as_of_ts = pd.Timestamp(as_of_date)
            # Point-in-time : utiliser report_date si disponible
            if 'report_date' in tk_data.columns:
                mask = (tk_data['report_date'] <= as_of_ts) & tk_data[metric].notna()
                valid = tk_data.loc[mask]
                if not valid.empty:
                    return valid.iloc[-1][metric]
            # Fallback : utiliser datadate avec filing_delay
            filing_delay = getattr(self.config, 'FUNDAMENTAL_DATA', {}).get('filing_delay', 45)
            cutoff = as_of_ts - pd.Timedelta(days=filing_delay)
            mask = (tk_data['date'] <= cutoff) & tk_data[metric].notna()
            valid = tk_data.loc[mask]
        else:
            valid = tk_data.dropna(subset=[metric])

        if valid.empty:
            return None
        return valid.iloc[-1][metric]

    # =================================================================
    # Donnees fondamentales — chargement legacy (compatibilite backtest)
    # =================================================================

    def _load_fundamental_data(self, tickers, max_tickers=300):
        """
        Charge les donnees fondamentales avec cascade de sources.

        Ordre de priorite :
        1. Cache memoire (si deja charge dans cette session)
        2. Fichier Compustat CSV pre-telecharge via WRDS
           (compustat_fundamentals.csv, genere par download_wrds_fundamentals.py)
        3. Financial Modeling Prep API (gratuit, 250 req/jour)
        4. Yahoo Finance (fallback, souvent instable)
        5. Proxy prix si aucune source ne fonctionne

        Le fichier Compustat est la source recommandee : donnees auditees,
        standardisees, avec historique trimestriel complet. Il est genere
        hors du notebook pour proteger les identifiants WRDS.
        """
        from datetime import datetime as dt, timedelta
        import os, io, sys, logging

        cache_days = getattr(self.config, 'FUNDAMENTAL_DATA', {}).get('cache_days', 7)

        # Cache memoire
        if (self._fundamental_load_date is not None and
            dt.now() - self._fundamental_load_date < timedelta(days=cache_days)):
            return self._fundamental_cache

        fundamental_data = {
            'roe': {}, 'debt_equity': {}, 'profit_margin': {},
            'current_ratio': {}, 'revenue_growth': {},
            'gross_margins': {}, 'operating_margins': {},
            'beta': {}, 'earnings_growth': {},
        }
        ts_cache = {}

        # Filtrer : actions uniquement
        exclude = set()
        for attr in ['US_BOND_ETFS', 'CA_BOND_ETFS', 'CRYPTO_ETFS',
                      'COMMODITY_ETFS', 'REAL_ESTATE_ETFS', 'SECTOR_THEMATIC_ETFS']:
            if hasattr(DataLoader, attr):
                exclude |= set(getattr(DataLoader, attr))
        stock_tickers = [t for t in tickers if t not in exclude][:max_tickers]

        # --- Source 1 : Fichier Compustat CSV (recommande) ---
        loaded_compustat = self._load_from_compustat_csv(
            stock_tickers, fundamental_data, ts_cache)

        # --- Source 2 : FMP API (si Compustat insuffisant) ---
        loaded_fmp = 0
        if loaded_compustat < len(stock_tickers) * 0.3:
            loaded_fmp = self._load_from_fmp(
                stock_tickers, fundamental_data, ts_cache)

        # --- Source 3 : Yahoo Finance (dernier recours API) ---
        loaded_yf = 0
        if loaded_compustat + loaded_fmp < len(stock_tickers) * 0.1:
            loaded_yf = self._load_from_yahoo(
                stock_tickers, fundamental_data, ts_cache)

        loaded = loaded_compustat + loaded_fmp + loaded_yf

        self._fundamental_cache = fundamental_data
        self._fundamental_ts_cache = ts_cache
        self._fundamental_load_date = dt.now()

        print(f"  [OK] Fondamentaux: {loaded}/{len(stock_tickers)} tickers "
              f"(Compustat: {loaded_compustat}, FMP: {loaded_fmp}, YF: {loaded_yf})")
        print(f"       ROE: {len(fundamental_data['roe'])}, "
              f"D/E: {len(fundamental_data['debt_equity'])}, "
              f"Margins: {len(fundamental_data['gross_margins'])}")
        return fundamental_data

    def _load_from_compustat_csv(self, tickers, fundamental_data, ts_cache):
        """
        Charge les fondamentaux depuis le fichier CSV Compustat pre-telecharge.

        Le fichier compustat_fundamentals.csv est genere par le script
        download_wrds_fundamentals.py (execute en local, pas dans le notebook).
        Il contient les etats financiers trimestriels de Compustat North America
        avec les tickers deja convertis au format Yahoo Finance.

        Avantages par rapport a Yahoo Finance / FMP :
        - Donnees auditees et standardisees (Compustat/S&P)
        - Historique trimestriel complet (3 ans)
        - Aucune requete HTTP (lecture fichier locale)
        - Secteurs GICS inclus (utile pour la diversification sectorielle)
        """
        import os

        # Chemins possibles pour le fichier CSV
        search_paths = [
            'compustat_fundamentals.csv',
            '/compustat_fundamentals.csv',
            '/content/drive/MyDrive/compustat_fundamentals.csv',
            os.path.join(os.getcwd(), 'compustat_fundamentals.csv'),
        ]

        csv_path = None
        for path in search_paths:
            if os.path.exists(path):
                csv_path = path
                break

        if csv_path is None:
            print("  [COMPUSTAT] Fichier compustat_fundamentals.csv non trouve.")
            print("              Generer avec : python download_wrds_fundamentals.py")
            return 0

        try:
            df = pd.read_csv(csv_path, parse_dates=['date'])
            print(f"  [COMPUSTAT] Charge depuis {csv_path} "
                  f"({df['yahoo_ticker'].nunique()} tickers, "
                  f"{len(df)} observations)")
        except Exception as e:
            print(f"  [COMPUSTAT] Erreur lecture: {e}")
            return 0

        # Construire le cache par ticker
        loaded = 0
        tickers_set = set(tickers)

        # Stocker aussi les secteurs GICS dans le DataLoader (pour la
        # diversification sectorielle, evite de dependre de yf.Ticker.info)
        sector_from_compustat = {}

        for yahoo_ticker, grp in df.groupby('yahoo_ticker'):
            if yahoo_ticker not in tickers_set:
                continue

            grp = grp.sort_values('date')

            # Construire le time-series cache
            records = []
            for _, row in grp.iterrows():
                rec = {'date': row['date']}
                for metric in ['roe', 'debt_equity', 'profit_margin',
                               'gross_margins', 'operating_margins', 'current_ratio']:
                    if pd.notna(row.get(metric)):
                        rec[metric] = row[metric]
                if len(rec) > 1:
                    records.append(rec)

            if records:
                ts_cache[yahoo_ticker] = pd.DataFrame(records)
                latest = ts_cache[yahoo_ticker].iloc[-1]
                for metric in ['roe', 'debt_equity', 'profit_margin',
                               'gross_margins', 'operating_margins', 'current_ratio']:
                    if metric in latest and pd.notna(latest[metric]):
                        fundamental_data[metric][yahoo_ticker] = latest[metric]
                loaded += 1

            # Secteur GICS
            sector = grp['sector'].dropna()
            if len(sector) > 0:
                sector_from_compustat[yahoo_ticker] = sector.iloc[-1]

        # Mettre a jour le sector_map du DataLoader si disponible
        if sector_from_compustat and hasattr(self, '_data_loader_ref'):
            dl = self._data_loader_ref
            if hasattr(dl, 'sector_map'):
                for tk, sec in sector_from_compustat.items():
                    if dl.sector_map.get(tk, 'Unknown') == 'Unknown':
                        dl.sector_map[tk] = sec

        print(f"  [COMPUSTAT] {loaded}/{len(tickers)} tickers charges")
        return loaded

    def _load_from_fmp(self, tickers, fundamental_data, ts_cache):
        """
        Charge les fondamentaux depuis Financial Modeling Prep (API gratuite).

        FMP fournit des etats financiers trimestriels pour les actions US
        et internationales. Cle gratuite sur financialmodelingprep.com
        (250 requetes/jour).
        """
        import urllib.request, json as _json

        # Ne charger que les tickers pas encore dans le cache
        missing = [t for t in tickers if t not in ts_cache]
        if not missing:
            return 0

        api_key = getattr(self.config, 'API_KEYS', {}).get('fmp', 'demo')
        base_url = "https://financialmodelingprep.com/api/v3"

        # Test rapide
        try:
            test_url = f"{base_url}/income-statement/AAPL?period=quarter&limit=1&apikey={api_key}"
            req = urllib.request.Request(test_url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=10) as resp:
                test_data = _json.loads(resp.read())
                if not test_data or 'Error' in str(test_data):
                    print(f"  [FMP] API indisponible ou cle invalide.")
                    return 0
        except Exception:
            print(f"  [FMP] API inaccessible.")
            return 0

        print(f"  [FMP] Chargement pour {len(missing)} tickers manquants...")
        loaded = 0

        for idx, ticker in enumerate(missing):
            fmp_ticker = ticker.replace('.TO', '.TSX') if '.TO' in ticker else ticker
            try:
                url_inc = (f"{base_url}/income-statement/{fmp_ticker}"
                           f"?period=quarter&limit=8&apikey={api_key}")
                url_bal = (f"{base_url}/balance-sheet-statement/{fmp_ticker}"
                           f"?period=quarter&limit=8&apikey={api_key}")

                req_inc = urllib.request.Request(url_inc, headers={'User-Agent': 'Mozilla/5.0'})
                req_bal = urllib.request.Request(url_bal, headers={'User-Agent': 'Mozilla/5.0'})

                with urllib.request.urlopen(req_inc, timeout=8) as resp:
                    income_data = _json.loads(resp.read())
                with urllib.request.urlopen(req_bal, timeout=8) as resp:
                    balance_data = _json.loads(resp.read())

                if not income_data or not balance_data:
                    continue

                records = []
                for inc in income_data:
                    rec = {'date': pd.Timestamp(inc.get('date', ''))}
                    revenue = inc.get('revenue', 0) or 0
                    if revenue > 0:
                        gp = inc.get('grossProfit')
                        if gp is not None: rec['gross_margins'] = gp / revenue
                        oi = inc.get('operatingIncome')
                        if oi is not None: rec['operating_margins'] = oi / revenue
                        ni = inc.get('netIncome')
                        if ni is not None: rec['profit_margin'] = ni / revenue
                    bal = next((b for b in balance_data
                                if b.get('date') == inc.get('date')), None)
                    if bal and inc.get('netIncome'):
                        eq = bal.get('totalStockholdersEquity', 0)
                        if eq and eq > 0:
                            rec['roe'] = (inc['netIncome'] / eq) * 4
                        debt = bal.get('totalDebt', 0)
                        if debt is not None and eq and eq > 0:
                            rec['debt_equity'] = (debt / eq) * 100
                        ca = bal.get('totalCurrentAssets', 0)
                        cl = bal.get('totalCurrentLiabilities', 0)
                        if ca and cl and cl > 0:
                            rec['current_ratio'] = ca / cl
                    if len(rec) > 1:
                        records.append(rec)

                if records:
                    ts_cache[ticker] = pd.DataFrame(records).sort_values('date')
                    latest = ts_cache[ticker].iloc[-1]
                    for key in ['roe', 'debt_equity', 'profit_margin',
                                'gross_margins', 'operating_margins', 'current_ratio']:
                        if key in latest and pd.notna(latest[key]):
                            fundamental_data[key][ticker] = latest[key]
                    loaded += 1
            except Exception:
                pass

            if (idx + 1) % 50 == 0:
                print(f"    ... {idx+1}/{len(missing)} ({loaded} OK)")

        print(f"  [FMP] {loaded}/{len(missing)} tickers charges")
        return loaded

    def _load_from_yahoo(self, tickers, fundamental_data, ts_cache):
        """
        Dernier recours : Yahoo Finance avec suppression des erreurs 404.
        """
        import logging, io, sys
        logging.getLogger('yfinance').setLevel(logging.CRITICAL)

        missing = [t for t in tickers if t not in ts_cache
                   and t not in fundamental_data.get('roe', {})]
        if not missing:
            return 0

        print(f"  [YF] Tentative pour {len(missing)} tickers restants...")
        loaded = 0
        consecutive_errors = 0

        for ticker in missing:
            if consecutive_errors >= 20:
                print(f"  [YF] Arret apres {consecutive_errors} echecs consecutifs.")
                break
            _stderr = sys.stderr
            _stdout = sys.stdout
            sys.stderr = io.StringIO()
            sys.stdout = io.StringIO()
            try:
                stock = yf.Ticker(ticker)
                history = self._extract_quarterly_history(stock)
                if history:
                    ts_cache[ticker] = pd.DataFrame(history).sort_values('date')
                    latest = ts_cache[ticker].iloc[-1]
                    for key in ['roe', 'debt_equity', 'profit_margin',
                                'gross_margins', 'operating_margins', 'current_ratio']:
                        if key in latest and pd.notna(latest[key]):
                            fundamental_data[key][ticker] = latest[key]
                    loaded += 1
                    consecutive_errors = 0
                else:
                    consecutive_errors += 1
            except (ValueError, KeyError, Exception) as e:
                print(f"[WARN] {e}")
                consecutive_errors += 1
            finally:
                sys.stderr = _stderr
                sys.stdout = _stdout

        print(f"  [YF] {loaded}/{len(missing)} tickers charges")
        return loaded

    @staticmethod
    def _extract_quarterly_history(stock):
        """
        Extrait les metriques fondamentales depuis Yahoo Finance.
        Supprime les messages HTTP 404.
        """
        import logging, io, sys
        _prev_level = logging.getLogger('yfinance').level
        logging.getLogger('yfinance').setLevel(logging.CRITICAL)
        _stderr = sys.stderr
        _stdout = sys.stdout
        sys.stderr = io.StringIO()
        sys.stdout = io.StringIO()
        try:
            income = stock.quarterly_income_stmt
            balance = stock.quarterly_balance_sheet
        except Exception:
            return []
        finally:
            sys.stderr = _stderr
            sys.stdout = _stdout
            logging.getLogger('yfinance').setLevel(_prev_level)

        if income is None or balance is None:
            return []
        if isinstance(income, pd.DataFrame) and income.empty:
            return []
        if len(income.columns) == 0:
            return []

        records = []
        for col_date in income.columns:
            rec = {'date': pd.Timestamp(col_date)}
            revenue = income.at['Total Revenue', col_date] if 'Total Revenue' in income.index else None
            gross_profit = income.at['Gross Profit', col_date] if 'Gross Profit' in income.index else None
            op_income = income.at['Operating Income', col_date] if 'Operating Income' in income.index else None
            net_income = income.at['Net Income', col_date] if 'Net Income' in income.index else None

            if revenue and revenue > 0:
                if gross_profit is not None: rec['gross_margins'] = gross_profit / revenue
                if op_income is not None: rec['operating_margins'] = op_income / revenue
                if net_income is not None: rec['profit_margin'] = net_income / revenue

            if col_date in balance.columns and net_income is not None:
                equity = balance.at['Stockholders Equity', col_date] \
                    if 'Stockholders Equity' in balance.index else None
                total_debt = balance.at['Total Debt', col_date] \
                    if 'Total Debt' in balance.index else None
                current_assets = balance.at['Current Assets', col_date] \
                    if 'Current Assets' in balance.index else None
                current_liab = balance.at['Current Liabilities', col_date] \
                    if 'Current Liabilities' in balance.index else None
                if equity and equity > 0:
                    rec['roe'] = (net_income / equity) * 4
                if total_debt is not None and equity and equity > 0:
                    rec['debt_equity'] = (total_debt / equity) * 100
                if current_assets and current_liab and current_liab > 0:
                    rec['current_ratio'] = current_assets / current_liab

            if len(rec) > 1:
                records.append(rec)

        return records

    def _get_fundamental_as_of(self, ticker, as_of_date, metric):
        """
        Retourne la dernière valeur publiée de `metric` pour `ticker`
        strictement avant `as_of_date`, avec un délai de 45 jours pour
        simuler le filing delay réglementaire (10-Q / 10-K).

        Retourne None si aucune donnée disponible.
        """
        if ticker not in self._fundamental_ts_cache:
            return self._fundamental_cache.get(metric, {}).get(ticker, None)

        df = self._fundamental_ts_cache[ticker]
        if metric not in df.columns:
            return None

        if as_of_date is not None:
            cutoff = pd.Timestamp(as_of_date) - pd.Timedelta(days=45)
            available = df.loc[df['date'] <= cutoff, metric].dropna()
        else:
            available = df[metric].dropna()

        if len(available) == 0:
            return None
        return available.iloc[-1]

    # =================================================================
    # Facteurs fondamentaux (point-in-time)
    # =================================================================
    def _compute_fundamental_profitability(self, tickers, returns=None,
                                           as_of_date=None):
        """
        Facteur de rentabilité fondamentale (Novy-Marx 2013, Fama & French 2015).

        Composantes :
          ROE             50 %   (Return on Equity annualisé)
          Marge nette     30 %   (Net Income / Revenue)
          Marge brute     20 %   (Gross Profit / Revenue)

        Fallback pour les tickers sans fondamentaux : Sharpe 6 mois atténué
        (50 % proxy + 50 % prior neutre) pour limiter le bruit.
        """
        if not self._fundamental_cache:
            return self._compute_profitability_proxy(returns) \
                if returns is not None else pd.Series(0.5, index=tickers)

        scores = pd.Series(np.nan, index=tickers)

        for ticker in tickers:
            components, weights_c = [], []

            roe = self._get_fundamental_as_of(ticker, as_of_date, 'roe')
            if roe is not None:
                components.append(min(max(roe / 0.25, 0.0), 1.0))
                weights_c.append(0.50)

            pm = self._get_fundamental_as_of(ticker, as_of_date, 'profit_margin')
            if pm is not None:
                components.append(min(max(pm / 0.20, 0.0), 1.0))
                weights_c.append(0.30)

            gm = self._get_fundamental_as_of(ticker, as_of_date, 'gross_margins')
            if gm is not None:
                components.append(min(max(gm / 0.60, 0.0), 1.0))
                weights_c.append(0.20)

            if components:
                total_w = sum(weights_c)
                scores[ticker] = sum(c * w for c, w in zip(components, weights_c)) / total_w

        # Fallback atténué pour les manquants
        missing = scores.isna()
        if missing.sum() > 0 and returns is not None:
            proxy = self._compute_profitability_proxy(returns)
            scores[missing] = 0.5 * proxy[missing] + 0.25

        return scores.rank(pct=True).fillna(0.5)

    def _compute_fundamental_quality(self, tickers, returns=None,
                                      as_of_date=None):
        """
        Facteur de qualité bilancielle (Asness, Frazzini & Pedersen 2019).

        Composantes :
          Sécurité (40 %)    : D/E bas, current ratio élevé
          Marges stables (30 %) : gross margin, operating margin
          Croissance (30 %)  : revenue growth, earnings growth

        Fallback : inverse volatilité atténué (50 % proxy + 50 % prior neutre).
        """
        if not self._fundamental_cache:
            return self._compute_quality_proxy(returns) \
                if returns is not None else pd.Series(0.5, index=tickers)

        scores = pd.Series(np.nan, index=tickers)

        for ticker in tickers:
            components, weights_c = [], []

            # --- Sécurité (40 %) ---
            sec = []
            de = self._get_fundamental_as_of(ticker, as_of_date, 'debt_equity')
            if de is not None:
                sec.append(max(1.0 - de / 300.0, 0.0))
            cr = self._get_fundamental_as_of(ticker, as_of_date, 'current_ratio')
            if cr is not None:
                sec.append(min(max(cr / 2.0, 0.0), 1.0))
            if sec:
                components.append(np.mean(sec))
                weights_c.append(0.40)

            # --- Marges stables (30 %) ---
            mar = []
            gm = self._get_fundamental_as_of(ticker, as_of_date, 'gross_margins')
            if gm is not None:
                mar.append(min(max(gm / 0.60, 0.0), 1.0))
            om = self._get_fundamental_as_of(ticker, as_of_date, 'operating_margins')
            if om is not None:
                mar.append(min(max(om / 0.30, 0.0), 1.0))
            if mar:
                components.append(np.mean(mar))
                weights_c.append(0.30)

            # --- Croissance saine (30 %) ---
            # revenue_growth et earnings_growth ne sont pas disponibles dans
            # l'historique trimestriel brut ; on se limite aux deux piliers
            # ci-dessus quand l'historique est utilisé.

            if components:
                total_w = sum(weights_c)
                scores[ticker] = sum(c * w for c, w in zip(components, weights_c)) / total_w

        # Fallback atténué
        missing = scores.isna()
        if missing.sum() > 0 and returns is not None:
            proxy = self._compute_quality_proxy(returns)
            scores[missing] = 0.5 * proxy[missing] + 0.25

        return scores.rank(pct=True).fillna(0.5)



print("[OK] Module FactorCalculator charge (v2 — 6 facteurs)")


[OK] Module FactorCalculator charge (v2 — 6 facteurs)


In [5]:
# =============================================================================
# CELLULE 3.1 : ANALYSE FACTORIELLE COMPLETE
# =============================================================================
#
# Valide empiriquement les affirmations de la section Justification des poids
# dans Config : corrélations, IC, monotonie, relations non-linéaires.
#
# Exécutée automatiquement dans run_full_backtest_v3().
# ou peut être appelée manuellement pour analyse approfondie :
#     fa = FactorAnalyzer()
#     report = fa.run_full_analysis(output['prices'], output['returns'])
#
# Références :
#   - Grinold & Kahn (2000), "Active Portfolio Management", ch. 6-8
#   - Qian, Hua & Sorensen (2007), "Quantitative Equity Portfolio Management"
#   - De Prado (2018), "Advances in Financial ML", ch. 8
# =============================================================================

class FactorAnalyzer:
    """
    Analyse factorielle complète pour validation empirique et optimisation.

    Conçu pour être présenté en comité d'investissement : chaque test produit
    une conclusion claire (CONFIRME / ATTENTION / A REVOIR) avec les métriques
    quantitatives correspondantes.

    Les noms de facteurs sont lus dynamiquement depuis Config.FACTORS.
    """

    def __init__(self, config=Config):
        self.config = config
        self.factor_calc = FactorCalculator(config)
        # Facteurs de base (sans composite) - dynamique depuis Config
        self._factor_names = [f for f in config.FACTORS if config.FACTORS[f].get('enabled', True)]
        # Facteurs + composite (pour IC, quintiles, etc.)
        self._factor_names_c = self._factor_names + ['composite']

    # =====================================================================
    # 1. PANEL FACTORIEL (calcul sur multiple dates, walk-forward)
    # =====================================================================
    def compute_factor_panel(self, prices, returns, n_snapshots=40):
        """
        Calcule les facteurs à plusieurs dates espacées dans le temps.

        Produit un panel {date: DataFrame(tickers x facteurs)} pour analyse
        statistique cross-sectionnelle répétée (méthode Fama-MacBeth).
        Chaque snapshot utilise UNIQUEMENT les données jusqu'à date (T-1).
        """
        valid_dates = returns.index[252:]
        step = max(1, len(valid_dates) // n_snapshots)
        sample_dates = valid_dates[::step][:n_snapshots]

        panel = []
        for date in tqdm(sample_dates, desc="Panel factoriel"):
            try:
                factors_df = self.factor_calc.compute_all_factors(
                    prices, returns, as_of_date=date
                )
                composite = self.factor_calc.compute_composite_score(factors_df)
                factors_df['composite'] = composite

                loc = returns.index.get_loc(date)
                if loc + 21 >= len(returns):
                    continue
                fwd_ret = returns.iloc[loc+1:loc+22].sum()

                common = factors_df.index.intersection(fwd_ret.index)
                common = common[factors_df.loc[common].notna().all(axis=1)]
                common = common[fwd_ret.loc[common].notna()]

                if len(common) < 30:
                    continue

                panel.append({
                    'date': date,
                    'factors': factors_df.loc[common],
                    'fwd_ret': fwd_ret.loc[common],
                })
            except Exception as e:
                 if panel == []:  # Afficher seulement la première erreur
                     print(f"  [DEBUG] Erreur snapshot {date}: {type(e).__name__}: {e}")
                 continue

        print(f"[OK] Panel : {len(panel)} snapshots, "
              f"~{len(panel[0]['fwd_ret']) if panel else 0} titres/snapshot")
        return panel

    # =====================================================================
    # 2. CORRELATION INTER-FACTEURS (Spearman)
    # =====================================================================
    def correlation_analysis(self, panel):
        """
        Corrélation de rang moyenne entre facteurs sur tout le panel.
        Facteurs lus dynamiquement depuis Config.FACTORS.
        """
        factor_names = self._factor_names
        all_corrs = []

        for snap in panel:
            available = [f for f in factor_names if f in snap['factors'].columns]
            if len(available) < 2:
                continue
            f = snap['factors'][available].dropna()
            if len(f) < 30:
                continue
            all_corrs.append(f.corr(method='spearman'))

        if not all_corrs:
            return {'error': 'Pas assez de données'}

        # Utiliser les facteurs disponibles dans les résultats
        available_factors = sorted(set().union(*(c.columns.tolist() for c in all_corrs)))
        available_factors = [f for f in factor_names if f in available_factors]

        mean_corr = pd.concat(all_corrs).groupby(level=0).mean()
        mean_corr = mean_corr.reindex(index=available_factors, columns=available_factors)

        print("\n" + "="*70)
        print("CORRELATION INTER-FACTEURS (Spearman, moyenne temporelle)")
        print("="*70)
        print(f"Sur {len(all_corrs)} snapshots\n")

        # Affichage
        col_w = max(15, max(len(n) for n in available_factors) + 2)
        print(f"{'':>{col_w}}" + "".join(f"{n:>{col_w}}" for n in available_factors))
        print("-" * (col_w + col_w * len(available_factors)))
        for f1 in available_factors:
            row = f"{f1:>{col_w}}"
            for f2 in available_factors:
                if f1 == f2:
                    row += f"{'1.000':>{col_w}}"
                else:
                    c = mean_corr.loc[f1, f2]
                    row += f"{c:>{col_w}.3f}"
            print(row)

        # Diagnostics : paires les plus correlees
        print(f"\nDiagnostic correlations :")
        import itertools as _it
        for f1, f2 in _it.combinations(available_factors, 2):
            c = mean_corr.loc[f1, f2]
            tag = "[REDONDANCE]" if abs(c) > 0.70 else "[COMPLEMENTAIRE]" if abs(c) < 0.30 else "[CORRELE]"
            if abs(c) > 0.40 or abs(c) > 0.60:
                print(f"  {f1} <-> {f2} : {c:.3f} {tag}")

        return {'mean_corr': mean_corr, 'n_snapshots': len(all_corrs)}

    # =====================================================================
    # 3. INFORMATION COEFFICIENTS (IC)
    # =====================================================================
    def information_coefficients(self, panel):
        """
        IC = Spearman(facteur, rendement forward 21j).
        IC > 0.05 = signal exploitable, IC_IR > 0.40 = signal stable.
        Ref: Grinold & Kahn (2000)
        """
        factor_names = self._factor_names_c
        ic_series = {f: [] for f in factor_names}

        for snap in panel:
            fwd = snap['fwd_ret']
            for fname in factor_names:
                if fname not in snap['factors'].columns:
                    continue
                fv = snap['factors'][fname]
                common = fv.dropna().index.intersection(fwd.dropna().index)
                if len(common) < 30:
                    continue
                ic, _ = spearmanr(fv.loc[common], fwd.loc[common])
                if not np.isnan(ic):
                    ic_series[fname].append(ic)

        print("\n" + "="*70)
        print("INFORMATION COEFFICIENTS (IC)")
        print("="*70)
        print(f"Forward : 21 jours | Methode : Spearman rank\n")

        col_w = max(15, max(len(n) for n in factor_names) + 2)
        print(f"{'Facteur':>{col_w}} {'IC moy':>9} {'IC std':>9} {'IC_IR':>9} "
              f"{'t-stat':>9} {'>0%':>7} {'Verdict':>20}")
        print("-" * (col_w + 65))

        results = {}
        for fname in factor_names:
            ics = ic_series[fname]
            if len(ics) < 5:
                continue
            m = np.mean(ics)
            s = np.std(ics)
            ir = m / s if s > 0 else 0
            t = m / (s / np.sqrt(len(ics))) if s > 0 else 0
            p = np.mean([x > 0 for x in ics]) * 100

            if m > 0.05 and ir > 0.40:
                v = "[OK] Solide"
            elif m > 0.03 and ir > 0.25:
                v = "[OK] Correct"
            elif m > 0:
                v = "[ATTENTION] Faible"
            else:
                v = "[A REVOIR] Negatif"

            print(f"{fname:>{col_w}} {m:>9.4f} {s:>9.4f} {ir:>9.3f} "
                  f"{t:>9.2f} {p:>6.0f}% {v:>20}")

            results[fname] = {
                'ic_mean': m, 'ic_std': s, 'ic_ir': ir,
                't_stat': t, 'pct_positive': p, 'n_obs': len(ics),
                'series': ics,
            }

        # Composite vs meilleur individuel
        if 'composite' in results:
            ci = results['composite']['ic_mean']
            best = max((v['ic_mean'], k) for k, v in results.items()
                       if k != 'composite')
            print(f"\nComposite ({ci:.4f}) vs meilleur individuel "
                  f"({best[1]}: {best[0]:.4f}) : ", end="")
            if ci > best[0]:
                print("[OK] Diversification fonctionne")
            else:
                print("[ATTENTION] Composite ne bat pas le meilleur facteur seul")

        return results

    # =====================================================================
    # 4. IC DECAY (persistance du signal)
    # =====================================================================
    def ic_decay(self, prices, returns):
        """
        IC a differents horizons (5j -> 63j). Justifie le rebalancement hebdo.
        """
        horizons = [5, 10, 21, 42, 63]
        factor_names = self._factor_names_c

        valid_dates = returns.index[252:-63]
        step = max(1, len(valid_dates) // 25)
        sample_dates = valid_dates[::step][:25]

        decay = {f: {h: [] for h in horizons} for f in factor_names}

        for date in sample_dates:
            try:
                fdf = self.factor_calc.compute_all_factors(
                    prices, returns, as_of_date=date)
                fdf['composite'] = self.factor_calc.compute_composite_score(fdf)
                loc = returns.index.get_loc(date)

                for h in horizons:
                    if loc + h >= len(returns):
                        continue
                    fwd = returns.iloc[loc+1:loc+h+1].sum()
                    for fn in factor_names:
                        if fn not in fdf.columns:
                            continue
                        fv = fdf[fn]
                        c = fv.dropna().index.intersection(fwd.dropna().index)
                        if len(c) < 30:
                            continue
                        ic, _ = spearmanr(fv.loc[c], fwd.loc[c])
                        if not np.isnan(ic):
                            decay[fn][h].append(ic)
            except Exception:
                continue

        print("\n" + "="*70)
        print("IC DECAY — Persistance du signal")
        print("="*70)

        col_w = max(15, max(len(n) for n in factor_names) + 2)
        print(f"{'Facteur':>{col_w}}" + "".join(f"{h:>8}j" for h in horizons)
              + f"{'Profil':>14}")
        print("-" * (col_w + 8*len(horizons) + 14))

        for fn in factor_names:
            row = f"{fn:>{col_w}}"
            vals = []
            for h in horizons:
                m = np.mean(decay[fn][h]) if decay[fn][h] else np.nan
                vals.append(m)
                row += f"{m:>8.4f}" if not np.isnan(m) else f"{'N/A':>8}"

            # Profil de decay
            if (not np.isnan(vals[2]) and vals[2] != 0
                    and not np.isnan(vals[4])):
                ratio = vals[4] / vals[2]
                tag = ("Persistant" if ratio > 0.8
                       else "Normal" if ratio > 0.4
                       else "Court terme")
            else:
                tag = "N/A"
            row += f"  {tag:>12}"
            print(row)

        return decay

    # =====================================================================
    # 5. QUINTILES (monotonie du signal)
    # =====================================================================
    def quintile_analysis(self, panel):
        """
        Quintile spreads : Q5 (meilleurs scores) vs Q1 (pires).
        Un facteur sain montre Q5 >> Q1 de facon monotone.
        Affiche MEDIANE (robuste) + MOYENNE (pour comparaison).
        """
        factor_names = self._factor_names_c
        # Collecter les rendements MEDIANS intra-quintile par snapshot
        qrets_median = {f: {q: [] for q in range(1, 6)} for f in factor_names}
        qrets_mean   = {f: {q: [] for q in range(1, 6)} for f in factor_names}

        for snap in panel:
            fwd = snap['fwd_ret']
            for fn in factor_names:
                if fn not in snap['factors'].columns:
                    continue
                fv = snap['factors'][fn]
                common = fv.dropna().index.intersection(fwd.dropna().index)
                if len(common) < 50:
                    continue
                ranks = fv.loc[common].rank(pct=True)
                for q in range(1, 6):
                    lo, hi = (q-1)/5, q/5
                    mask = (ranks > lo) & (ranks <= hi) if q > 1 else (ranks <= hi)
                    subset = fwd.loc[common][mask]
                    r_med = subset.median()
                    r_mean = subset.mean()
                    if not np.isnan(r_med):
                        qrets_median[fn][q].append(r_med)
                    if not np.isnan(r_mean):
                        qrets_mean[fn][q].append(r_mean)

        print("\n" + "="*70)
        print("QUINTILES — Spreads factoriels (rend. forward 21j, annualise)")
        print("="*70)

        col_w = max(15, max(len(n) for n in factor_names) + 2)

        # --- MEDIANE (robuste aux outliers) ---
        print(f"\n{'':>{col_w}}  ** MEDIANE (robuste aux outliers) **")
        print(f"{'Facteur':>{col_w}}" + "".join(f"{'Q'+str(q):>10}" for q in range(1,6))
              + f"{'Q5-Q1':>10} {'Mono?':>10}")
        print("-" * (col_w + 70))

        results = {}
        for fn in factor_names:
            row = f"{fn:>{col_w}}"
            qm = []
            for q in range(1, 6):
                m = np.median(qrets_median[fn][q]) * 12 if qrets_median[fn][q] else np.nan
                qm.append(m)
                row += (f"{m*100:>9.1f}%" if not np.isnan(m) else f"{'N/A':>10}")

            if not np.isnan(qm[0]) and not np.isnan(qm[4]):
                spread = qm[4] - qm[0]
                row += f"{spread*100:>9.1f}%"
                mono = all(qm[i] <= qm[i+1] for i in range(4)
                           if not np.isnan(qm[i]) and not np.isnan(qm[i+1]))
                rough = qm[4] > qm[0] and qm[4] > qm[2]
                row += ("  [OK]" if mono
                        else "  [~]" if rough
                        else "  [NON]")
            else:
                row += f"{'N/A':>10} {'N/A':>10}"
            print(row)
            results[fn] = {'quintile_medians': qm}

        # --- MOYENNE (pour comparaison, sensible aux outliers) ---
        print(f"\n{'':>{col_w}}  ** MOYENNE (sensible aux outliers) **")
        print(f"{'Facteur':>{col_w}}" + "".join(f"{'Q'+str(q):>10}" for q in range(1,6))
              + f"{'Q5-Q1':>10}")
        print("-" * (col_w + 70))

        for fn in factor_names:
            row = f"{fn:>{col_w}}"
            qmean = []
            for q in range(1, 6):
                m = np.mean(qrets_mean[fn][q]) * 12 if qrets_mean[fn][q] else np.nan
                qmean.append(m)
                row += (f"{m*100:>9.1f}%" if not np.isnan(m) else f"{'N/A':>10}")
            if not np.isnan(qmean[0]) and not np.isnan(qmean[4]):
                spread = qmean[4] - qmean[0]
                row += f"{spread*100:>9.1f}%"
            else:
                row += f"{'N/A':>10}"
            print(row)
            if fn in results:
                results[fn]['quintile_means'] = qmean

        return results

    # =====================================================================
    # 6. NON-LINEARITE (Mutual Information vs Spearman)
    # =====================================================================
    def nonlinear_analysis(self, panel):
        """
        Compare dependance lineaire (Spearman) et dependance quelconque (MI).
        MI >> Spearman = relations non-lineaires exploitables.
        Ref: De Prado (2018), ch. 8.
        """
        from sklearn.metrics import mutual_info_score
        from sklearn.preprocessing import KBinsDiscretizer

        factor_names = self._factor_names_c
        res = {f: {'spearman': [], 'mi': []} for f in factor_names}

        for snap in panel:
            fwd = snap['fwd_ret']
            for fn in factor_names:
                if fn not in snap['factors'].columns:
                    continue
                fv = snap['factors'][fn]
                common = fv.dropna().index.intersection(fwd.dropna().index)
                if len(common) < 50:
                    continue
                x = fv.loc[common].values
                y = fwd.loc[common].values
                rho, _ = spearmanr(x, y)
                try:
                    disc = KBinsDiscretizer(
                        n_bins=10, encode='ordinal', strategy='quantile')
                    xd = disc.fit_transform(x.reshape(-1,1)).ravel().astype(int)
                    yd = disc.fit_transform(y.reshape(-1,1)).ravel().astype(int)
                    mi = mutual_info_score(xd, yd)
                except Exception:
                    mi = np.nan
                if not np.isnan(rho):
                    res[fn]['spearman'].append(abs(rho))
                if not np.isnan(mi):
                    res[fn]['mi'].append(mi)

        print("\n" + "="*70)
        print("NON-LINEARITE — Spearman vs Mutual Information")
        print("="*70)

        col_w = max(15, max(len(n) for n in factor_names) + 2)
        print(f"{'Facteur':>{col_w}} {'|Spearman|':>12} {'MI (nats)':>12} "
              f"{'Non-lin?':>15}")
        print("-" * (col_w + 40))

        for fn in factor_names:
            sm = np.mean(res[fn]['spearman']) if res[fn]['spearman'] else np.nan
            mm = np.mean(res[fn]['mi']) if res[fn]['mi'] else np.nan
            row = f"{fn:>{col_w}} {sm:>12.4f} {mm:>12.4f}"
            if sm > 0.001 and not np.isnan(mm):
                ratio = mm / sm
                tag = ("[OUI] Fort" if ratio > 5.0
                       else "[Possible]" if ratio > 2.5
                       else "[Non]")
                row += f"  {tag:>13}"
            else:
                row += f"  {'N/A':>13}"
            print(row)

        return res

    # =====================================================================
    # 7. INTERACTIONS (Double-Sorts)
    # =====================================================================
    def interaction_analysis(self, panel):
        """
        Double-sort par paires de facteurs. Detecte synergies vs redondances.
        """
        factor_names = self._factor_names
        pairs = list(itertools.combinations(factor_names, 2))

        print("\n" + "="*70)
        print("INTERACTIONS FACTORIELLES — Double-Sorts")
        print("="*70)

        results = {}
        for f1, f2 in pairs:
            grid = {'HH': [], 'HL': [], 'LH': [], 'LL': []}
            for snap in panel:
                fwd = snap['fwd_ret']
                if f1 not in snap['factors'].columns or f2 not in snap['factors'].columns:
                    continue
                v1, v2 = snap['factors'][f1], snap['factors'][f2]
                common = (v1.dropna().index.intersection(v2.dropna().index)
                          .intersection(fwd.dropna().index))
                if len(common) < 50:
                    continue
                m1, m2 = v1.loc[common].median(), v2.loc[common].median()
                h1, h2 = v1.loc[common] > m1, v2.loc[common] > m2
                for lab, a, b in [('HH',True,True), ('HL',True,False),
                                   ('LH',False,True), ('LL',False,False)]:
                    r = fwd.loc[common][(h1==a) & (h2==b)].mean()
                    if not np.isnan(r):
                        grid[lab].append(r)

            means = {k: np.mean(v)*12 if v else np.nan for k, v in grid.items()}
            if all(not np.isnan(means[k]) for k in means):
                inter = means['HH'] - means['HL'] - means['LH'] + means['LL']
                syn = means['HH'] > max(means['HL'], means['LH'])
            else:
                inter, syn = np.nan, False

            print(f"\n{f1} x {f2} :")
            print(f"  {'':>10} {f2+' H':>12} {f2+' L':>12}")
            print(f"  {f1+' H':>10} {means['HH']*100:>11.1f}% "
                  f"{means['HL']*100:>11.1f}%")
            print(f"  {f1+' L':>10} {means['LH']*100:>11.1f}% "
                  f"{means['LL']*100:>11.1f}%")
            if not np.isnan(inter):
                tag = ("[SYNERGIE]" if syn and abs(inter) > 0.02
                       else "[REDONDANCE]" if abs(inter) > 0.02
                       else "[Additive]")
                print(f"  Interaction: {inter*100:+.1f}%  {tag}")

            results[(f1, f2)] = {'means': means, 'interaction': inter}

        return results

    # =====================================================================
    # 8. OPTIMISATION DES POIDS
    # =====================================================================
    def optimize_weights(self, ic_results):
        """
        Compare poids actuels vs IC-weighting et IR-weighting.
        Ref: Grinold & Kahn (2000), ch. 14.
        """
        fnames = self._factor_names

        print("\n" + "="*70)
        print("OPTIMISATION DES POIDS FACTORIELS")
        print("="*70)

        curr = {f: self.config.FACTORS[f]['weight'] for f in fnames}

        ics = {f: max(ic_results[f]['ic_mean'], 0.001)
               for f in fnames if f in ic_results}
        t_ic = sum(ics.values())
        ic_w = {f: ics[f]/t_ic for f in ics}

        irs = {f: max(ic_results[f]['ic_ir'], 0.01)
               for f in fnames if f in ic_results}
        t_ir = sum(irs.values())
        ir_w = {f: irs[f]/t_ir for f in irs}

        col_w = max(15, max(len(n) for n in fnames) + 2)
        print(f"\n{'Facteur':>{col_w}} {'Actuel':>10} {'IC-wt':>10} "
              f"{'IR-wt':>10} {'Direction':>15}")
        print("-" * (col_w + 45))

        reco = {}
        for f in fnames:
            c, iw, rw = curr[f], ic_w.get(f,0), ir_w.get(f,0)
            avg = (iw + rw) / 2
            d = ("AUGMENTER" if avg > c * 1.3
                 else "DIMINUER" if avg < c * 0.7
                 else "OK")
            print(f"{f:>{col_w}} {c:>9.0%} {iw:>9.0%} {rw:>9.0%} {d:>15}")
            reco[f] = {'current': c, 'ic_opt': iw, 'ir_opt': rw, 'dir': d}

        # --- Recommandation concrète (compromis données + stabilité) ---
        print("\n--- RECOMMANDATION PRAGMATIQUE ---")
        print("  (compromis entre optimisation empirique et stabilité)")
        suggested = {}
        for f in fnames:
            iw_val = ic_w.get(f, 0)
            rw_val = ir_w.get(f, 0)
            avg_opt = (iw_val + rw_val) / 2
            cur = curr[f]
            # Blend: 50% poids actuel + 50% optimal (prudence vs overfitting)
            sug = round((cur + avg_opt) / 2, 2)
            sug = max(sug, 0.05)  # minimum 5%
            suggested[f] = sug
        # Normaliser à 100%
        total_sug = sum(suggested.values())
        suggested = {f: round(v/total_sug, 2) for f, v in suggested.items()}
        # Ajuster arrondi
        diff = 1.0 - sum(suggested.values())
        max_f = max(suggested, key=suggested.get)
        suggested[max_f] = round(suggested[max_f] + diff, 2)

        print(f"\n{'Facteur':>{col_w}} {'Actuel':>10} {'Suggéré':>10} {'Changement':>12}")
        print("-" * (col_w + 35))
        for f in fnames:
            delta = suggested[f] - curr[f]
            print(f"{f:>{col_w}} {curr[f]:>9.0%} {suggested[f]:>9.0%} {delta:>+11.0%}")
        print(f"\n  Note: Suggéré = 50% actuel + 50% optimal (IR/IC blend)")
        print(f"  Évite l'overfitting vs suivi aveugle des poids actuels")

        reco['suggested'] = suggested
        return reco

    # =====================================================================
    # 9. RAPPORT COMPLET
    # =====================================================================
    def run_full_analysis(self, prices, returns):
        """Point d'entree : execute les 8 analyses et produit le verdict."""

        print("\n" + "#"*70)
        print("#  ANALYSE FACTORIELLE COMPLETE — GSF-3120")
        print("#  Rapport de validation empirique")
        print("#"*70)

        # Pre-charger les fondamentaux UNE FOIS avant le panel
        # (evite que chaque snapshot déclenche le chargement)
        if getattr(self.config, 'USE_FUNDAMENTAL_DATA', False):
            tickers = prices.columns.tolist()
            if not self.factor_calc._fundamental_cache:
                print("[0/7] Pre-chargement des fondamentaux...")
                self.factor_calc._load_fundamental_data(tickers)

        print("\n[1/7] Panel factoriel...")

        panel = self.compute_factor_panel(prices, returns)
        if not panel:
            print("[ERREUR] Pas assez de données pour l'analyse")
            return None

        print("\n[2/7] Correlations...")
        corr = self.correlation_analysis(panel)

        print("\n[3/7] Information Coefficients...")
        ic = self.information_coefficients(panel)

        print("\n[4/7] IC Decay...")
        decay = self.ic_decay(prices, returns)

        print("\n[5/7] Quintiles...")
        quint = self.quintile_analysis(panel)

        print("\n[6/7] Non-linearite...")
        nonlin = self.nonlinear_analysis(panel)

        print("\n[7/7] Interactions...")
        interact = self.interaction_analysis(panel)

        print("\n[BONUS] Optimisation des poids...")
        reco = self.optimize_weights(ic)

        # ===== VERDICT =====
        print("\n" + "#"*70)
        print("#  VERDICT — RECOMMANDATION")
        print("#"*70)

        issues, strengths = [], []
        if 'composite' in ic:
            ci = ic['composite']['ic_mean']
            (strengths if ci > 0.03 else issues).append(
                f"Composite IC = {ci:.4f} "
                f"{'(signal exploitable)' if ci > 0.05 else '(positif)' if ci > 0.03 else '(faible)'}")

        # Check all pairwise correlations for high redundancy
        if corr.get('mean_corr') is not None:
            import itertools as _it
            for f1, f2 in _it.combinations(self._factor_names, 2):
                try:
                    c_val = corr['mean_corr'].loc[f1, f2]
                    if abs(c_val) > 0.70:
                        issues.append(f"{f1}/{f2} rho={c_val:.2f} (quasi-redondants)")
                    elif abs(c_val) < 0.30:
                        strengths.append(f"{f1}/{f2} rho={c_val:.2f} (complementaires)")
                except KeyError:
                    pass

        for fn in self._factor_names:
            if fn in ic and ic[fn]['ic_mean'] < 0.01:
                issues.append(f"{fn} IC ~ 0 — signal très faible")

        print("\nFORCES :")
        for s in strengths:
            print(f"  + {s}")
        print("\nPOINTS D'ATTENTION :")
        for i in issues:
            print(f"  - {i}")

        if len(issues) <= 1:
            print("\n>>> FACTEURS ADEQUATS — poids proches de l'optimal.")
        elif len(issues) <= 3:
            print("\n>>> AJUSTEMENTS RECOMMANDES — enrichir avec fondamentaux.")
        else:
            print("\n>>> REVISION NECESSAIRE — trop de signaux faibles.")

        return {
            'panel': panel, 'correlations': corr, 'ic': ic,
            'decay': decay, 'quintiles': quint, 'nonlinear': nonlin,
            'interactions': interact, 'weight_recommendations': reco,
            'strengths': strengths, 'issues': issues,
        }


print("[OK] Module FactorAnalyzer charge")


[OK] Module FactorAnalyzer charge


---
# 4. Gestion du risque — Détection de régime et overlay adaptatif
---


In [6]:
# =============================================================================
# CELLULE 4: RISK MANAGER V4 - GESTION ADAPTATIVE DU RISQUE
# =============================================================================

from collections import Counter

@dataclass
class RegimeSignal:
    name: str
    value: float
    signal: int  # -1, 0, +1
    confidence: float
    source: str

@dataclass
class RegimeOutput:
    regime: str  # 'RISK_ON' ou 'RISK_OFF' (binaire)
    confidence: float
    composite_score: float
    signals: Dict[str, RegimeSignal]
    exposure_multiplier: float
    overlay_on: bool = False
    overlay_just_activated: bool = False

class RiskManager:
    """ Gestion adaptative du risque avec hystérésis et multiplicateur d'exposition."""

    def __init__(self, config=Config):
        self.config = config
        self._regime_history = []
        self._current_regime = 'RISK_ON'  # Régime courant pour hystérésis
        self._overlay_on = False
        self._cooldown = 0
        self._overlay_activations = []
        # Cache HMM : évite le ré-entraînement à chaque appel
        self._hmm_model           = None
        self._hmm_last_train_date = None
        # Diagnostic redondance HMM vs VIX
        self._hmm_signal_history  = []
        self._vix_signal_history  = []

    def compute_covariance(self, returns, method='shrinkage'):
        lookback = self.config.RISK['cov_lookback']
        returns = returns.tail(lookback).dropna()
        if len(returns) < 50:
            return np.eye(len(returns.columns)) * 0.04
        if method == 'shrinkage':
            lw = LedoitWolf().fit(returns.values)
            return lw.covariance_ * 252
        return returns.cov().values * 252

    def _compute_vix_signal(self, vix):
        if vix is None or len(vix) < 21:
            return RegimeSignal('VIX', np.nan, 0, 0, 'volatility')
        # T-1 pour éviter look-ahead
        current_vix = vix.iloc[-1]

        cfg = self.config.REGIME_DETECTION['vix']
        if current_vix > cfg['risk_off']:
            signal = -1
            confidence = min(1.0, (current_vix - cfg['risk_off']) / (cfg['extreme'] - cfg['risk_off']))
        elif current_vix < cfg['risk_on']:
            signal = 1
            confidence = min(1.0, (cfg['risk_on'] - current_vix) / cfg['risk_on'])
        else:
            signal = 0
            confidence = 0.5
        return RegimeSignal('VIX', current_vix, signal, confidence, 'volatility')

    def _compute_realized_vol_signal(self, returns):
        if returns is None or len(returns) < 64:
            return RegimeSignal('RealizedVol', np.nan, 0, 0, 'volatility')
        # T-1
        returns_t1 = returns
        vol_3m = returns_t1.tail(63).std() * np.sqrt(252) * 100
        lookback = min(252, len(returns_t1))
        rolling_vol = returns_t1.rolling(63).std() * np.sqrt(252) * 100
        rolling_vol = rolling_vol.tail(lookback).dropna()
        if len(rolling_vol) < 20:
            return RegimeSignal('RealizedVol', vol_3m, 0, 0.3, 'volatility')
        percentile = (rolling_vol < vol_3m).mean()
        if percentile > 0.8:
            signal = -1
            confidence = min(1.0, (percentile - 0.8) / 0.2)
        elif percentile < 0.2:
            signal = 1
            confidence = min(1.0, (0.2 - percentile) / 0.2)
        else:
            signal = 0
            confidence = 0.5
        return RegimeSignal('RealizedVol', vol_3m, signal, confidence, 'volatility')

    def _compute_trend_signal(self, prices):
        cfg = self.config.REGIME_DETECTION['trend']
        if prices is None or len(prices) < cfg['sma_slow'] + 21:
            return RegimeSignal('Trend', np.nan, 0, 0, 'trend')
        # T-1
        current_price = prices.iloc[-1]
        sma_200 = prices.tail(cfg['sma_slow']).mean()
        sma_50 = prices.tail(cfg['sma_fast']).mean()

        pct_above_sma = (current_price / sma_200 - 1) * 100
        golden_cross = sma_50 > sma_200
        if current_price > sma_200 and golden_cross:
            signal = 1
            confidence = min(1.0, abs(pct_above_sma) / 10)
        elif current_price < sma_200 and not golden_cross:
            signal = -1
            confidence = min(1.0, abs(pct_above_sma) / 10)
        else:
            signal = 0
            confidence = 0.3
        return RegimeSignal('Trend', pct_above_sma, signal, confidence, 'trend')

    def _compute_momentum_signal(self, prices):
        cfg = self.config.REGIME_DETECTION['trend']
        lookback = cfg['momentum_lookback']
        if prices is None or len(prices) < lookback + 22:
            return RegimeSignal('Momentum', np.nan, 0, 0, 'trend')
        # T-1 avec skip 21 jours
        current = prices.iloc[-23]
        past = prices.iloc[-lookback - 22]
        momentum = (current / past - 1) * 100
        rolling_mom = (prices.shift(22) / prices.shift(lookback + 22) - 1) * 100
        rolling_mom = rolling_mom.tail(252).dropna()
        if len(rolling_mom) > 20:
            percentile = (rolling_mom < momentum).mean()
        else:
            percentile = 0.5
        if percentile > 0.7:
            signal = 1
            confidence = min(1.0, (percentile - 0.7) / 0.3)
        elif percentile < 0.3:
            signal = -1
            confidence = min(1.0, (0.3 - percentile) / 0.3)
        else:
            signal = 0
            confidence = 0.5
        return RegimeSignal('Momentum', momentum, signal, confidence, 'trend')

    def _compute_yield_curve_signal(self, yield_10y, yield_2y):
        if yield_10y is None or yield_2y is None:
            return RegimeSignal('YieldCurve', np.nan, 0, 0, 'yield_curve')
        common_idx = yield_10y.index.intersection(yield_2y.index)
        if len(common_idx) < 21:
            return RegimeSignal('YieldCurve', np.nan, 0, 0, 'yield_curve')
        y10, y2 = yield_10y.loc[common_idx], yield_2y.loc[common_idx]
        # T-1
        spread = y10.iloc[-1] - y2.iloc[-1]
        cfg = self.config.REGIME_DETECTION['yield_curve']
        spread_history = (y10 - y2).tail(63)
        was_inverted_recently = (spread_history < 0).any()
        if spread < cfg['inversion_threshold']:
            signal = -1
            confidence = min(1.0, abs(spread) / 1.0)
        elif spread > cfg['steep_threshold']:
            signal = 1
            confidence = min(1.0, (spread - cfg['steep_threshold']) / 1.0)
        elif was_inverted_recently and spread > 0:
            signal = -1
            confidence = 0.6
        else:
            signal = 0
            confidence = 0.4
        return RegimeSignal('YieldCurve', spread, signal, confidence, 'yield_curve')

    def _compute_credit_spread_signal(self, hy_spread):
        if hy_spread is None or len(hy_spread) < 64:
            return RegimeSignal('CreditSpread', np.nan, 0, 0, 'credit')
        # T-1
        current_spread = hy_spread.iloc[-1]
        cfg = self.config.REGIME_DETECTION['credit']
        lookback = min(cfg['lookback'], len(hy_spread))
        spread_window = hy_spread.tail(lookback)
        percentile = (spread_window < current_spread).mean() * 100
        if percentile < cfg['tight_percentile']:
            signal = 1
            confidence = 0.6
        elif percentile > cfg['wide_percentile']:
            signal = -1
            confidence = min(1.0, (percentile - cfg['wide_percentile']) / 20)
        else:
            signal = 0
            confidence = 0.5
        return RegimeSignal('CreditSpread', current_spread, signal, confidence, 'credit')

    def _compute_breadth_signal(self, prices):
        if prices is None or len(prices) < 201:
            return RegimeSignal('Breadth', np.nan, 0, 0, 'breadth')
        # T-1
        prices_t1 = prices
        sma_200 = prices_t1.tail(200).mean()
        current_prices = prices_t1.iloc[-1]
        valid_stocks = ~(current_prices.isna() | sma_200.isna())
        if valid_stocks.sum() < 20:
            return RegimeSignal('Breadth', np.nan, 0, 0, 'breadth')
        pct_above_sma = (current_prices[valid_stocks] > sma_200[valid_stocks]).mean() * 100
        cfg = self.config.REGIME_DETECTION['breadth']
        if pct_above_sma > cfg['pct_above_sma_bullish']:
            signal = 1
            confidence = min(1.0, (pct_above_sma - cfg['pct_above_sma_bullish']) / 40)
        elif pct_above_sma < cfg['pct_above_sma_bearish']:
            signal = -1
            confidence = min(1.0, (cfg['pct_above_sma_bearish'] - pct_above_sma) / 40)
        else:
            signal = 0
            confidence = 0.5
        return RegimeSignal('Breadth', pct_above_sma, signal, confidence, 'breadth')

    def _compute_hmm_signal(self, returns):
        """
        Signal HMM basé sur les régimes de volatilité.

        Le HMM identifie 2 états cachés basés sur les rendements:
        - État 0: Généralement basse volatilité (RISK_ON)
        - État 1: Généralement haute volatilité (RISK_OFF)

        Cache : le modèle est ré-entraîné seulement toutes les
        retrain_frequency semaines (défaut : 52). Entre deux
        ré-entraînements, predict() est appelé sur le modèle existant.

        Returns:
            RegimeSignal avec signal -1/0/+1 et confidence
        """
        cfg = self.config.REGIME_DETECTION.get('hmm', {})

        if not cfg.get('enabled', False) or not HMM_AVAILABLE:
            return RegimeSignal('HMM', np.nan, 0, 0, 'hmm')

        lookback          = cfg.get('lookback', 504)
        n_states          = cfg.get('n_states', 2)
        n_iter            = cfg.get('n_iter', 100)
        retrain_freq      = cfg.get('retrain_frequency', 52)
        cache_window_days = 7 * retrain_freq   # semaines → jours

        if returns is None or len(returns) < lookback:
            return RegimeSignal('HMM', np.nan, 0, 0, 'hmm')

        try:
            import time as _time

            # Données T-1 pour éviter le look-ahead
            ret_slice    = returns.tail(lookback).values.reshape(-1, 1)
            current_date = pd.Timestamp(returns.index[-1])

            # ── Décision cache ──────────────────────────────────────────────
            _needs_train = (
                self._hmm_model is None
                or self._hmm_last_train_date is None
                or (current_date - self._hmm_last_train_date).days >= cache_window_days
            )

            # Diagnostic : une seule fois, lors du premier appel (toujours un fit)
            _run_diag = not getattr(self, '_hmm_diagnostic_done', False)

            # ── Entraînement (conditionnel) ─────────────────────────────────
            if _needs_train:
                _t0_fit     = _time.perf_counter()
                _seeds      = [42, 43, 44, 45, 46]
                _best_model = None
                _best_score = -np.inf
                _all_preds  = []

                for _seed in _seeds:
                    _m = GaussianHMM(
                        n_components=n_states,
                        covariance_type='full',
                        n_iter=n_iter,
                        random_state=_seed
                    )
                    _m.fit(ret_slice)
                    _all_preds.append(_m.predict(ret_slice))
                    _s = _m.score(ret_slice)
                    if _s > _best_score:
                        _best_score = _s
                        _best_model = _m

                _fit_ms = (_time.perf_counter() - _t0_fit) * 1000

                # Taux d'accord : fraction de jours où les 5 modèles prédisent
                # le même état (labels bruts, sans alignement des états)
                _preds_arr = np.array(_all_preds)   # shape (5, T)
                _accord    = float(np.mean(
                    np.all(_preds_arr == _preds_arr[0], axis=0)
                ))

                # Mettre à jour le cache avec le meilleur modèle
                model                     = _best_model
                self._hmm_model           = _best_model
                self._hmm_last_train_date = current_date

                # Diagnostic au premier ré-entraînement
                if _run_diag:
                    print(f"HMM multi-seeds: meilleur score={_best_score:.1f}, "
                          f"taux accord={_accord:.1%} sur 5 seeds")
            else:
                model   = self._hmm_model
                _fit_ms = None   # pas de fit ce cycle

            # ── Prédiction (toujours exécutée) ──────────────────────────────
            _t0_pred      = _time.perf_counter()
            hidden_states = model.predict(ret_slice)
            _pred_ms      = (_time.perf_counter() - _t0_pred) * 1000

            current_state = hidden_states[-1]

            # Probabilités de l'état actuel
            state_probs = model.predict_proba(ret_slice)[-1]
            confidence  = state_probs[current_state]

            # Identifier quel état est "high vol" vs "low vol"
            state_vars     = np.array([model.covars_[i][0, 0] for i in range(n_states)])
            state_means    = model.means_.flatten()
            high_vol_state = np.argmax(state_vars)
            low_vol_state  = np.argmin(state_vars)

            # Signal: +1 si low vol (RISK_ON), -1 si high vol (RISK_OFF)
            if current_state == low_vol_state:
                signal = 1
            elif current_state == high_vol_state:
                signal = -1
            else:
                signal = 0

            # Stocker les infos pour diagnostic externe
            self._hmm_info = {
                'current_state': int(current_state),
                'state_probs':   state_probs.tolist(),
                'state_means':   state_means.tolist(),
                'state_vars':    state_vars.tolist(),
                'high_vol_state': int(high_vol_state),
            }

            # ── Métriques (premier appel = premier fit) ─────────────────────
            if _run_diag:
                total_days     = 15 * 252
                n_retrains     = max(1, total_days // cache_window_days)
                n_predict_only = total_days - n_retrains
                savings_pct    = n_predict_only / total_days * 100

                print('\n' + '─' * 60)
                print('[HMM CACHE] Métriques — premier appel')
                print('─' * 60)
                print(f'  Fenêtre de cache        : {cache_window_days} jours ({retrain_freq} semaines)')
                print(f'  Backtest 15 ans (~{total_days} jours trading) :')
                print(f'    Fits estimés           : {n_retrains}')
                print(f'    Predict-only estimés   : {n_predict_only}')
                print(f'    Économie de fits       : {savings_pct:.1f}%')
                print(f'  Temps mesuré :')
                print(f'    fit()     : {_fit_ms:.1f} ms')
                print(f'    predict() : {_pred_ms:.1f} ms')
                if _pred_ms > 0:
                    print(f'    Ratio fit/predict : {_fit_ms/_pred_ms:.0f}×')
                print('─' * 60)
                self._hmm_diagnostic_done = True

            return RegimeSignal('HMM', float(current_state), signal, confidence, 'hmm')

        except (ValueError, KeyError, Exception) as e:
            print(f"  [ATTENTION]  Erreur HMM: {e}")
            return RegimeSignal('HMM', np.nan, 0, 0, 'hmm')

    def _compute_composite_score(self, signals):
        """
        Calcule le score composite de régime en agrégeant les signaux
        par catégorie.

        Chaque catégorie (volatility, trend, yield_curve, credit,
        breadth, hmm) reçoit un poids fixe
        (Config.REGIME_DETECTION["signal_weights"]). A l'intérieur
        de chaque catégorie, les signaux sont moyennés par leur
        confiance pour éviter la double-pondération (ex: VIX et vol
        réalisée sont tous deux dans "volatility").

        Le score final est dans [-1, +1] : positif = RISK_ON,
        négatif = RISK_OFF.
        """
        weights = self.config.REGIME_DETECTION["signal_weights"]
        category_signals = {}

        for signal in signals.values():
            cat = signal.source
            if cat not in weights:
                continue
            if cat not in category_signals:
                category_signals[cat] = []
            if not np.isnan(signal.signal):
                category_signals[cat].append(
                    (signal.signal, signal.confidence)
                )

        composite = 0.0
        total_weight = 0.0

        for cat, sigs in category_signals.items():
            if not sigs:
                continue
            w = weights[cat]
            total_conf = sum(conf for _, conf in sigs)
            if total_conf > 0:
                cat_signal = sum(s * c for s, c in sigs) / total_conf
            else:
                cat_signal = sum(s for s, _ in sigs) / len(sigs)
            composite += w * cat_signal
            total_weight += w

        if total_weight > 0:
            composite /= total_weight

        return max(-1.0, min(1.0, composite))

    def _compute_exposure_multiplier(self, regime, confidence,
                                     composite_score, realized_vol):
        """Calcule le multiplicateur d'exposition selon le régime.
        RISK_ON: multiplicateur >= 1.0 (pleine exposition)
        RISK_OFF: multiplicateur < 1.0 (exposition réduite)
        """
        overlay_cfg = self.config.REGIME_DETECTION.get('overlay', {})
        base_mult     = overlay_cfg.get('base_multiplier', 1.0)
        risk_off_mult = overlay_cfg.get('risk_off_multiplier', 0.70)

        if regime == 'RISK_OFF':
            mult = risk_off_mult
            # Ajuster par confiance : plus la confiance est élevée,
            # plus on réduit
            mult = 1.0 - (1.0 - mult) * confidence
        else:
            mult = base_mult

        # Clamp
        return max(0.5, min(1.5, mult))

    def detect_regime(self, prices, vix=None, macro_data=None):
        """Interface simple - retourne string."""
        output = self.detect_regime_full(prices, vix, macro_data)
        return output.regime

    def detect_regime_full(self, prices, vix=None, macro_data=None):
        """Détection complète avec hystérésis + overlay crash-only (ENTRY/EXIT/cooldown) + multiplicateur."""

        # Safety
        if not self.config.REGIME_DETECTION['enabled']:
            return RegimeOutput('RISK_ON', 0.5, 0.0, {}, 1.0, overlay_on=False, overlay_just_activated=False)

        if macro_data is None:
            macro_data = {}

        signals = {}

        # Dernière date disponible dans les séries passées à detect_regime_full
        # (dans ton backtester, prices est déjà tronqué à decision_date = T-1)
        decision_date = prices.index[-1]

        # -------------------------------------------------------------------------
        # 1) VOLATILITÉ
        # -------------------------------------------------------------------------
        if vix is not None:
            signals['vix'] = self._compute_vix_signal(vix)

        spy_col = 'SPY' if hasattr(prices, "columns") and 'SPY' in prices.columns else prices.columns[0]
        spy_returns = prices[spy_col].pct_change().dropna()

        signals['realized_vol'] = self._compute_realized_vol_signal(spy_returns)

        # Vol réalisée actuelle (pour le multiplicateur) - en T-1
        realized_vol = None
        if len(spy_returns) > 64:
            realized_vol = spy_returns.tail(63).std() * np.sqrt(252) * 100

        # -------------------------------------------------------------------------
        # 2) TREND
        # -------------------------------------------------------------------------
        signals['trend'] = self._compute_trend_signal(prices[spy_col])
        signals['momentum'] = self._compute_momentum_signal(prices[spy_col])

        # -------------------------------------------------------------------------
        # 3) YIELD CURVE
        # -------------------------------------------------------------------------
        if 'yield_10y' in macro_data and 'yield_2y' in macro_data:
            signals['yield_curve'] = self._compute_yield_curve_signal(
                macro_data['yield_10y'], macro_data['yield_2y']
            )

        # -------------------------------------------------------------------------
        # 4) CREDIT
        # -------------------------------------------------------------------------
        if 'credit_spread_hy' in macro_data:
            signals['credit'] = self._compute_credit_spread_signal(macro_data['credit_spread_hy'])

        # -------------------------------------------------------------------------
        # 5) BREADTH
        # -------------------------------------------------------------------------
        signals['breadth'] = self._compute_breadth_signal(prices)

        # === Signal HMM (si activé) ===
        if self.config.REGIME_DETECTION.get('hmm', {}).get('enabled', False):
            # Utiliser les rendements du SPY ou de l'indice principal
            if prices is not None and 'SPY' in prices.columns:
                spy_returns = prices['SPY'].pct_change().dropna()
                hmm_signal = self._compute_hmm_signal(spy_returns)
                signals['HMM'] = hmm_signal

        # -------------------------------------------------------------------------
        # Score composite
        # -------------------------------------------------------------------------
        composite_score = float(self._compute_composite_score(signals))

        # -------------------------------------------------------------------------
        # Régime binaire avec hystérésis
        # L'hystérésis évite les oscillations rapides entre RISK_ON et RISK_OFF
        # quand le score composite est proche des seuils. Le régime ne change
        # que si le score franchit le seuil OPPOSÉ au régime courant :
        #   - En RISK_ON : passage à RISK_OFF seulement si score < risk_off_threshold
        #   - En RISK_OFF : passage à RISK_ON seulement si score > risk_on_threshold
        # De plus, le changement doit être confirmé sur N jours consécutifs.
        # -------------------------------------------------------------------------
        thresholds = self.config.REGIME_DETECTION['thresholds']

        if self._current_regime == 'RISK_ON':
            tentative_regime = 'RISK_OFF' if composite_score < thresholds['risk_off'] else 'RISK_ON'
        else:  # RISK_OFF
            tentative_regime = 'RISK_ON' if composite_score > thresholds['risk_on'] else 'RISK_OFF'

        # Confirmation sur N jours
        self._regime_history.append(tentative_regime)
        conf_days_raw = self.config.REGIME_DETECTION['confirmation_days']  # en jours
        reb_freq = self.config.BACKTEST.get('rebalance_freq', 'weekly')

        # Convertir "jours" -> nb d'observations (selon la fréquence de calcul)
        if reb_freq == 'weekly':
            conf_days = max(1, int(np.ceil(conf_days_raw / 5)))   # ~5 jours ouvrés ≈ 1 semaine
        elif reb_freq == 'monthly':
            conf_days = 1
        else:  # daily
            conf_days = int(conf_days_raw)

        if len(self._regime_history) > conf_days:
            self._regime_history = self._regime_history[-conf_days:]


        counts = Counter(self._regime_history)
        confirmed_regime = counts.most_common(1)[0][0]
        self._current_regime = confirmed_regime

        # Confiance (simple)
        confidence = float(min(1.0, abs(composite_score) / 0.5))

        # -------------------------------------------------------------------------
        # OVERLAY CRASH-ONLY (stateful) : ENTRY / EXIT / COOLDOWN
        #
        # L'overlay est un mécanisme de protection additionnel qui ne s'active
        # QUE lors de stress sévère (au moins 2 signaux de stress sur 4).
        # Il fonctionne comme un interrupteur à états :
        #   ENTRY : RISK_OFF confirmé + score de stress >= 2 + confiance >= 70%
        #   EXIT  : score de stress retombe à <= 1
        #   COOLDOWN : après désactivation, période de repos (10 jours)
        #              pour éviter les activations/désactivations répétées
        #
        # Quand l'overlay est actif, le multiplicateur d'exposition réduit la
        # poche actions (ex: ×0.70), redistribuant vers obligations et cash.
        # -------------------------------------------------------------------------
        # Triggers bruts (robustes)
        vix_bad = ('vix' in signals) and (signals['vix'].signal == -1)

        # fast trend + drawdown court terme (sur SPY ou proxy spy_col) ---
        px = prices[spy_col].dropna()

        if len(px) >= 120:
            sma20  = px.rolling(20).mean().iloc[-1]
            sma50  = px.rolling(50).mean().iloc[-1]
            sma100 = px.rolling(100).mean().iloc[-1]
            p_last = px.iloc[-1]

            fast_trend_bad = (sma20 < sma50) or (p_last < sma100)

            roll_max_20 = px.rolling(20).max().iloc[-1]
            dd_20 = (p_last / roll_max_20) - 1.0 if roll_max_20 and roll_max_20 > 0 else 0.0
            dd_bad = (dd_20 <= -0.08)   # -8% sur 20 jours

            trend_bad = bool(fast_trend_bad or dd_bad)
        else:
            # fallback si pas assez d’historique
            trend_bad = ('trend' in signals) and (signals['trend'].signal == -1)

        # --- PATCH 4: vote de stress ---
        vol_bad    = ('realized_vol' in signals) and (signals['realized_vol'].signal == -1)
        credit_bad = ('credit' in signals) and (signals['credit'].signal == -1)
        breadth_bad = ('breadth' in signals) and (signals['breadth'].signal == -1)

        stress_score = int(vix_bad) + int(trend_bad) + int(vol_bad) + int(credit_bad)  # (breadth en option)
        STRESS_ENTRY = 2   # overlay ON si au moins 2 drapeaux sur 4
        STRESS_EXIT  = 1   # overlay OFF si <= 1 drapeau

        # Paramètres (
        CONF_ENTRY = 0.70
        COOLDOWN_DAYS_RAW = 10  # en jours
        reb_freq = self.config.BACKTEST.get('rebalance_freq', 'weekly')

        if reb_freq == 'weekly':
            COOLDOWN_DAYS = max(1, int(np.ceil(COOLDOWN_DAYS_RAW / 5)))  # 10j -> ~2 semaines
        elif reb_freq == 'monthly':
            COOLDOWN_DAYS = 1
        else:  # daily
            COOLDOWN_DAYS = int(COOLDOWN_DAYS_RAW)


        # S'assurer que les attributs existent (au cas où)
        if not hasattr(self, "_overlay_on"):
            self._overlay_on = False
        if not hasattr(self, "_cooldown"):
            self._cooldown = 0
        if not hasattr(self, "_overlay_activations"):
            self._overlay_activations = []

        # --- capture état AVANT modifications ---
        overlay_prev = self._overlay_on

        # Décrémenter cooldown
        if self._cooldown > 0:
            self._cooldown -= 1

        # EXIT:  overlay actif et conditions relâchées
        if self._overlay_on:
            if stress_score <= STRESS_EXIT:
                self._overlay_on = False
                self._cooldown = COOLDOWN_DAYS


        # ENTRY: si overlay inactif, pas en cooldown, et conditions crash-only
        if (not self._overlay_on) and (self._cooldown == 0):
            if (confirmed_regime == 'RISK_OFF') and (stress_score >= STRESS_ENTRY) and (confidence >= CONF_ENTRY):
                self._overlay_on = True


         # --- flag entrée overlay (pour forcer un trade hors calendrier) ---
        overlay_just_activated = (not overlay_prev) and self._overlay_on

        apply_overlay = bool(self._overlay_on)

        # Log: 1 entrée par date (évite doublons)
        # On log seulement quand overlay est ON (ou tu peux log ON/OFF si tu veux)
        if apply_overlay:
            if (len(self._overlay_activations) == 0) or (self._overlay_activations[-1]["date"] != decision_date):
                self._overlay_activations.append({
                    "date": decision_date,
                    "regime": confirmed_regime,
                    "confidence": confidence,
                    "composite_score": composite_score,
                    "vix_bad": bool(vix_bad),
                    "trend_bad": bool(trend_bad),
                    "stress_score": int(stress_score),
                    "vol_bad": bool(vol_bad),
                    "credit_bad": bool(credit_bad),
                })

        # -------------------------------------------------------------------------
        # Multiplicateur d'exposition
        # -------------------------------------------------------------------------
        if not apply_overlay:
            exposure_multiplier = 1.0
        else:
            exposure_multiplier = float(self._compute_exposure_multiplier(
                confirmed_regime, confidence, composite_score, realized_vol
            ))

        # -------------------------------------------------------------------------
        # Output
        # -------------------------------------------------------------------------
        # Log de diagnostic : entrée de l'overlay crash-ONLY
        if overlay_just_activated:
            print(f"[OVERLAY ENTRY] {prices.index[-1].date()} | score={composite_score:.3f} | conf={confidence:.2f}")

        # ── Diagnostic redondance HMM vs VIX (Spearman) ──────────────────
        if 'HMM' in signals and 'vix' in signals:
            self._hmm_signal_history.append(signals['HMM'].signal)
            self._vix_signal_history.append(signals['vix'].signal)
            if len(self._hmm_signal_history) % 252 == 0:
                from scipy import stats as _stats
                _hmm_arr = np.array(self._hmm_signal_history)
                _vix_arr = np.array(self._vix_signal_history)
                _mask = np.isfinite(_hmm_arr) & np.isfinite(_vix_arr)
                if _mask.sum() >= 10:
                    _rho, _pval = _stats.spearmanr(_hmm_arr[_mask], _vix_arr[_mask])
                    _verdict = 'REDONDANT' if abs(_rho) > 0.7 else 'OK'
                    print(f"Corrélation HMM vs VIX (Spearman): rho={_rho:.3f}, "
                          f"p={_pval:.4f}. {_verdict}")

        return RegimeOutput(
            regime=confirmed_regime,
            confidence=confidence,
            composite_score=composite_score,
            signals=signals,
            exposure_multiplier=exposure_multiplier,
            overlay_on=bool(self._overlay_on),
            overlay_just_activated=bool(overlay_just_activated)
        )

print("[OK] Module RiskManager V4 chargé")

[OK] Module RiskManager V4 chargé


---
# 5. Optimisation du portefeuille
---


In [7]:
# =============================================================================
# CELLULE 5: PORTFOLIO OPTIMIZER - MEAN-VARIANCE (P0)
# =============================================================================

class PortfolioOptimizer:
    """
    Optimisation de portefeuille avec support Mean-Variance (Markowitz).

    Méthodes disponibles:
    - 'simple': Pondération par score factoriel (méthode originale)
    - 'mean_variance': Optimisation quadratique avec cvxpy
    """

    def __init__(self, config=Config):
        self.config = config
        self._last_optim_summary = None
        self._prev_weights = None
        self._prev_regime         = None   # Pour smoothing dynamique
        self._smoothing_diag_done = False

        # Vérifier disponibilité cvxpy
        self._cvxpy_available = 'cp' in dir() or CVXPY_AVAILABLE

    def optimize(self, expected_returns, cov_matrix, prev_weights=None,
                 liquidity_constraints=None, data_loader=None, as_of_date=None,
                 current_regime=None):
        """
        Point d'entrée principal pour l'optimisation.

        Args:
            expected_returns: pd.Series des scores/rendements attendus
            cov_matrix: Matrice de covariance (np.array ou pd.DataFrame)
            prev_weights: Poids précédents pour smoothing
            liquidity_constraints: dict {ticker: max_weight} (optionnel)
            data_loader: DataLoader pour calcul contraintes liquidité (optionnel)
            as_of_date: Date pour calcul liquidité (optionnel)

        Returns:
            pd.Series: Poids optimaux
        """
        # Récupérer la méthode d'optimisation
        opt_config = getattr(self.config, 'OPTIMIZATION', {})
        method = opt_config.get('method', 'simple')

        # Calculer les contraintes de liquidité si disponibles
        if liquidity_constraints is None and data_loader is not None and as_of_date is not None:
            if hasattr(data_loader, 'get_liquidity_constraints'):
                liquidity_constraints = data_loader.get_liquidity_constraints(
                    expected_returns.index.tolist(), as_of_date
                )

        # Choisir la méthode
        if method == 'mean_variance' and self._cvxpy_available:
            weights = self._optimize_mean_variance(
                expected_returns, cov_matrix, liquidity_constraints
            )
        else:
            weights = self._optimize_simple(expected_returns, liquidity_constraints)

        # Appliquer smoothing dynamique selon le régime
        if prev_weights is not None:
            tc = self.config.TURNOVER_CONTROL
            _regime_changed = (
                current_regime is not None
                and self._prev_regime is not None
                and current_regime != self._prev_regime
            )
            if _regime_changed:
                smoothing = tc.get('smoothing_on_regime_change', tc['weight_smoothing'])
                if not self._smoothing_diag_done:
                    print(f"Smoothing dynamique: regime change "
                          f"{self._prev_regime} -> {current_regime}, "
                          f"smoothing reduit a {smoothing:.0%}")
                    self._smoothing_diag_done = True
            else:
                smoothing = tc['weight_smoothing']
            prev_aligned = prev_weights.reindex(weights.index).fillna(0)
            weights = smoothing * prev_aligned + (1 - smoothing) * weights
            if weights.sum() > 0:
                weights = weights / weights.sum()

        if current_regime is not None:
            self._prev_regime = current_regime
        self._prev_weights = weights.copy()
        return weights

    def _optimize_simple(self, expected_returns, liquidity_constraints=None):
        """Méthode originale: sélection top-N avec pondération par score."""
        n_stocks = self.config.STOCK_SELECTION['n_stocks']
        min_w = self.config.STOCK_SELECTION['min_weight']
        max_w = self.config.STOCK_SELECTION['max_weight']

        # Filtrer par liquidité si disponible
        if liquidity_constraints is not None:
            eligible = [t for t, max_wt in liquidity_constraints.items() if max_wt >= min_w]
            expected_returns = expected_returns[expected_returns.index.isin(eligible)]

        top_stocks = expected_returns.nlargest(n_stocks)
        selected = top_stocks.index.tolist()

        scores = top_stocks.values
        scores = scores - scores.min() + 0.1
        weights = scores / scores.sum()

        # Appliquer contraintes de poids
        weights = np.clip(weights, min_w, max_w)

        # Appliquer contraintes de liquidité individuelles
        if liquidity_constraints is not None:
            for i, ticker in enumerate(selected):
                if ticker in liquidity_constraints:
                    weights[i] = min(weights[i], liquidity_constraints[ticker])

        weights = weights / weights.sum()

        result = pd.Series(0.0, index=expected_returns.index)
        result[selected] = weights
        return result

    def _optimize_mean_variance(self, expected_returns, cov_matrix, liquidity_constraints=None):
        """
        Optimisation Mean-Variance avec CVXPY.

        Maximise: μ'w - (λ/2) * w'Σw
        Sous contraintes:
        - Σw = 1 (fully invested)
        - w >= min_weight pour positions sélectionnées
        - w <= max_weight (ou contrainte liquidité)
        - w >= 0 (long only)
        """
        import cvxpy as cp

        opt_config = getattr(self.config, 'OPTIMIZATION', {})
        risk_aversion = opt_config.get('risk_aversion', 2.0)
        regularization = opt_config.get('regularization', 0.01)
        objective_type = opt_config.get('objective', 'max_sharpe')

        n_stocks = self.config.STOCK_SELECTION['n_stocks']
        min_w = self.config.STOCK_SELECTION['min_weight']
        max_w = self.config.STOCK_SELECTION['max_weight']

        # Pré-sélection des top scores pour réduire la dimension
        top_candidates = expected_returns.nlargest(min(n_stocks * 2, len(expected_returns)))
        tickers = top_candidates.index.tolist()
        n = len(tickers)

        # Filtrer par liquidité
        if liquidity_constraints is not None:
            tickers = [t for t in tickers if liquidity_constraints.get(t, 0) >= min_w]
            n = len(tickers)

        if n < 5:
            print("  [ATTENTION]  Pas assez de titres liquides, fallback sur méthode simple")
            return self._optimize_simple(expected_returns, liquidity_constraints)

        # Préparer les données
        mu = top_candidates[tickers].values  # Rendements attendus (scores normalisés)

        # Extraire la sous-matrice de covariance
        if isinstance(cov_matrix, pd.DataFrame):
            cov_sub = cov_matrix.loc[tickers, tickers].values
        else:
            # Trouver les indices correspondants
            idx_map = {t: i for i, t in enumerate(expected_returns.index)}
            indices = [idx_map[t] for t in tickers if t in idx_map]
            cov_sub = cov_matrix[np.ix_(indices, indices)]

        # Régularisation pour stabilité numérique
        cov_sub = cov_sub + regularization * np.eye(n)

        # ── Vérification du condition number (stabilité numérique) ────────
        _cov_base = cov_sub - regularization * np.eye(n)
        _reg  = regularization
        _cond = np.linalg.cond(cov_sub)
        for _ in range(3):
            if _cond <= 1e6:
                break
            _reg  = _reg * 10
            cov_sub = _cov_base + _reg * np.eye(n)
            _cond = np.linalg.cond(cov_sub)
        if not getattr(self, '_cond_diagnostic_done', False):
            print(f"Condition number: {_cond:.0f} (regularisation={_reg:.4f})")
            self._cond_diagnostic_done = True

        # Variables d'optimisation
        w = cp.Variable(n)

        # Rendement du portefeuille
        portfolio_return = mu @ w

        # Risque du portefeuille (variance)
        portfolio_variance = cp.quad_form(w, cov_sub)

        # Fonction objectif selon le type
        if objective_type == 'max_sharpe':
            # Maximiser ratio Sharpe ≈ maximiser (rendement - λ * variance)
            objective = cp.Maximize(portfolio_return - (risk_aversion / 2) * portfolio_variance)
        elif objective_type == 'min_variance':
            # Minimiser la variance
            objective = cp.Minimize(portfolio_variance)
        elif objective_type == 'target_return':
            # Minimiser variance pour un rendement cible
            target_vol = opt_config.get('target_volatility', 0.15)
            objective = cp.Maximize(portfolio_return)
            # Contrainte de volatilité ajoutée ci-dessous
        else:
            objective = cp.Maximize(portfolio_return - (risk_aversion / 2) * portfolio_variance)

        # Contraintes
        constraints = [
            cp.sum(w) == 1,      # Fully invested
            w >= 0,              # Long only
        ]

        # Contraintes de poids max (liquidité ou config)
        for i, ticker in enumerate(tickers):
            if liquidity_constraints is not None and ticker in liquidity_constraints:
                max_weight_i = min(max_w, liquidity_constraints[ticker])
            else:
                max_weight_i = max_w
            constraints.append(w[i] <= max_weight_i)

        # Contrainte de volatilité cible (optionnel)
        if objective_type == 'target_return':
            target_vol = opt_config.get('target_volatility', 0.15)
            constraints.append(portfolio_variance <= target_vol**2)

        # Résoudre
        problem = cp.Problem(objective, constraints)
        try:
            problem.solve(solver=cp.OSQP, verbose=False)

            if problem.status not in ['optimal', 'optimal_inaccurate']:
                print(f"  [ATTENTION]  Optimisation non convergée: {problem.status}, fallback simple")
                return self._optimize_simple(expected_returns, liquidity_constraints)

            weights_opt = w.value

            # Nettoyer les poids très faibles
            weights_opt[weights_opt < min_w / 2] = 0


            # Forcer min_weight strict après renormalisation
            for _ in range(5):
                below = weights_opt[(weights_opt > 0) & (weights_opt < min_w)]
                if len(below) == 0:
                    break
                weights_opt[weights_opt < min_w] = 0
                if weights_opt.sum() > 0:
                    weights_opt = weights_opt / weights_opt.sum()


            # Construire le résultat
            result = pd.Series(0.0, index=expected_returns.index)
            for i, ticker in enumerate(tickers):
                if weights_opt[i] > 0:
                    result[ticker] = weights_opt[i]

            # Vérifier qu'on a assez de positions
            n_positions = (result > min_w / 2).sum()
            if n_positions < 5:
                print(f"  [ATTENTION]  Seulement {n_positions} positions, fallback simple")
                return self._optimize_simple(expected_returns, liquidity_constraints)

            return result

        except Exception as e:
            print(f"  [ATTENTION]  Erreur optimisation: {e}, fallback simple")
            return self._optimize_simple(expected_returns, liquidity_constraints)

    # =============================================================================
# MÉTHODE À AJOUTER DANS PortfolioOptimizer (Cell 13)
# Ajouter APRÈS _optimize_mean_variance(), AVANT apply_turnover_control()
# =============================================================================

    def optimize_joint(self, stock_scores, bond_candidates, cov_matrix,
                       stock_alloc, bond_alloc, prev_weights=None,
                       liquidity_constraints=None, current_regime=None):
        """
        Optimisation Mean-Variance CONJOINTE stocks + bonds.

        Maximise: μ'w - (λ/2) * w'Σw
        Sous contraintes:
          - Σ w_stocks = stock_alloc
          - Σ w_bonds  = bond_alloc
          - w >= 0 (long only)
          - w_i <= max_weight (par position)
          - w_i >= min_weight OU w_i = 0 (positions entières)

        La matrice de covariance inclut stocks ET bonds, ce qui permet
        à l'optimiseur d'exploiter les corrélations croisées (ex: bonds
        gouvernementaux montent quand actions baissent → réduit la
        variance du portefeuille total).

        Args:
            stock_scores: pd.Series {ticker: score} pour les actions
            bond_candidates: dict {ticker: relative_weight} du BondSelector
                            Les poids relatifs servent de signal μ pour bonds
            cov_matrix: matrice de covariance couvrant stocks + bonds
            stock_alloc: float (ex: 0.725)
            bond_alloc: float (ex: 0.25)
            prev_weights: pd.Series poids précédents (optionnel)
            liquidity_constraints: dict {ticker: max_weight} (optionnel)

        Returns:
            pd.Series: poids optimaux (stocks + bonds), somme = stock_alloc + bond_alloc
        """
        import cvxpy as cp

        opt_config = getattr(self.config, 'OPTIMIZATION', {})
        risk_aversion = opt_config.get('risk_aversion', 2.0)
        regularization = opt_config.get('regularization', 0.01)

        n_stocks_cfg = self.config.STOCK_SELECTION['n_stocks']
        min_w = self.config.STOCK_SELECTION['min_weight']
        max_w = self.config.STOCK_SELECTION['max_weight']

        # ── Sélection des candidats stocks (top N par score) ────────────
        top_stocks = stock_scores.nlargest(
            min(n_stocks_cfg * 2, len(stock_scores))
        )
        stock_tickers = top_stocks.index.tolist()

        # Filtrer par liquidité
        if liquidity_constraints is not None:
            stock_tickers = [t for t in stock_tickers
                           if liquidity_constraints.get(t, 0) >= min_w]

        # ── Candidats bonds ─────────────────────────────────────────────
        bond_tickers = list(bond_candidates.keys())

        # ── Assemblage univers conjoint ─────────────────────────────────
        all_tickers = stock_tickers + bond_tickers
        n_stocks = len(stock_tickers)
        n_bonds = len(bond_tickers)
        n_total = n_stocks + n_bonds

        if n_stocks < 5:
            print("  [ATTENTION]  Pas assez de stocks, fallback simple")
            return self._optimize_simple(stock_scores, liquidity_constraints)

        # ── Vecteur de rendements attendus μ ────────────────────────────
        # Stocks : scores factoriels (déjà calculés)
        # Bonds  : poids relatifs du BondSelector × scaling factor
        #          Le scaling aligne l'échelle des scores bonds avec stocks
        stock_mu = top_stocks[stock_tickers].values

        # Scaling : les scores bonds sont entre 0-1, on les met à l'échelle
        # du score moyen des stocks pour que l'optimiseur ne les ignore pas
        stock_mean_score = np.abs(stock_mu).mean()
        if stock_mean_score < 1e-8:
            stock_mean_score = 1.0

        bond_mu = np.array([
            bond_candidates.get(t, 0) * stock_mean_score
            for t in bond_tickers
        ])

        mu = np.concatenate([stock_mu, bond_mu])

        # ── Matrice de covariance conjointe ─────────────────────────────
        if isinstance(cov_matrix, pd.DataFrame):
            # Vérifier que tous les tickers sont dans la matrice
            available = [t for t in all_tickers if t in cov_matrix.index]
            missing = [t for t in all_tickers if t not in cov_matrix.index]
            if missing:
                print(f"  [INFO] {len(missing)} tickers absents de la cov: "
                      f"{', '.join(missing[:3])}...")
                # Retirer les manquants
                all_tickers = available
                stock_tickers = [t for t in stock_tickers if t in available]
                bond_tickers = [t for t in bond_tickers if t in available]
                n_stocks = len(stock_tickers)
                n_bonds = len(bond_tickers)
                n_total = n_stocks + n_bonds
                # Reconstruire mu
                stock_mu = top_stocks.reindex(stock_tickers).fillna(0).values
                bond_mu = np.array([
                    bond_candidates.get(t, 0) * stock_mean_score
                    for t in bond_tickers
                ])
                mu = np.concatenate([stock_mu, bond_mu])

            cov_sub = cov_matrix.loc[all_tickers, all_tickers].values
        else:
            # Fallback : construire depuis les indices
            idx_map = {t: i for i, t in enumerate(stock_scores.index)}
            # Bonds might not be in stock_scores index
            cov_sub = np.eye(n_total) * 0.04  # Default
            for i, ti in enumerate(all_tickers):
                for j, tj in enumerate(all_tickers):
                    if ti in idx_map and tj in idx_map:
                        cov_sub[i, j] = cov_matrix[idx_map[ti], idx_map[tj]]

        # Régularisation
        cov_sub = cov_sub + regularization * np.eye(n_total)

        # ── Variables d'optimisation ────────────────────────────────────
        w = cp.Variable(n_total)

        # Indices stocks vs bonds
        stock_idx = list(range(n_stocks))
        bond_idx = list(range(n_stocks, n_total))

        # ── Objectif ────────────────────────────────────────────────────
        portfolio_return = mu @ w
        portfolio_variance = cp.quad_form(w, cov_sub)
        objective = cp.Maximize(
            portfolio_return - (risk_aversion / 2) * portfolio_variance
        )

        # ── Contraintes ─────────────────────────────────────────────────
        constraints = [
            w >= 0,                                    # Long only
            cp.sum(w[stock_idx]) == stock_alloc,       # Σ stocks = stock_alloc
            cp.sum(w[bond_idx]) == bond_alloc,         # Σ bonds  = bond_alloc
        ]

        # Max weight par position
        for i, ticker in enumerate(all_tickers):
            if liquidity_constraints and ticker in liquidity_constraints:
                max_weight_i = min(max_w, liquidity_constraints[ticker])
            else:
                max_weight_i = max_w
            constraints.append(w[i] <= max_weight_i)



        # ── Résoudre ────────────────────────────────────────────────────
        problem = cp.Problem(objective, constraints)
        try:
            print(f"[DEBUG] Entrée optimisation: {n_total} titres "
                  f"({n_stocks} stocks + {n_bonds} bonds)")
            print(f"[DEBUG] Cov matrix shape: {cov_sub.shape}")
            print(f"[DEBUG] stock_alloc={stock_alloc:.3f}  bond_alloc={bond_alloc:.3f}")
            problem.solve(solver=cp.OSQP, verbose=False)

            if problem.status not in ['optimal', 'optimal_inaccurate']:
                print(f"  [ATTENTION]  Optimisation conjointe non convergée: "
                      f"{problem.status}, fallback séparé")
                return self._fallback_separate(
                    stock_scores, bond_candidates, stock_alloc, bond_alloc,
                    cov_matrix, liquidity_constraints
                )

            weights_opt = w.value
            print(f"[DEBUG] Statut solveur: {problem.status}")
            if weights_opt is not None:
                print(f"[DEBUG] Sortie optimisation: "
                      f"{(weights_opt > 0).sum()} positions actives")
                print(f"[DEBUG] Sum weights: {weights_opt.sum():.6f}")
                print(f"[DEBUG] Bonds dans weights: "
                      f"{sum(1 for i in range(n_stocks, n_total) if weights_opt[i] > 0)}")
            else:
                print("[DEBUG] weights_opt = None (solveur a échoué)")

            # ── Post-traitement ─────────────────────────────────────────
            # Étape 1 : supprimer les poids insignifiants (< min_w/2)
            weights_opt[weights_opt < min_w / 2] = 0

            # Étape 2 : nettoyage min_weight — supprime sans redistribuer
            for idx_group in [stock_idx, bond_idx]:
                for _ in range(5):
                    group_w = weights_opt[idx_group].copy()
                    below = (group_w > 0) & (group_w < min_w)
                    if not np.any(below):
                        break
                    group_w[below] = 0
                    weights_opt[idx_group] = group_w

            print(f"[DEBUG] Après nettoyage min_weight: "
                  f"{(weights_opt > 0).sum()} positions, "
                  f"sum={weights_opt.sum():.6f}")

            # Étape 3 : renormalisation avec garde-fou bonds >= MIN_BOND_ALLOC
            _cash_alloc     = max(0.0, 1.0 - stock_alloc - bond_alloc)
            _total_alloc    = stock_alloc + bond_alloc  # = 1.0 - cash
            _min_bond       = float(getattr(self.config, 'MIN_BOND_ALLOC', 0.25))

            _w_stocks = weights_opt[stock_idx].copy()
            _w_bonds  = weights_opt[bond_idx].copy()
            _sum_bonds  = _w_bonds.sum()
            _sum_stocks = _w_stocks.sum()

            if _sum_bonds > 0 and _sum_bonds < _min_bond:
                # Forcer bonds à min_bond_alloc
                _w_bonds = _w_bonds / _sum_bonds * _min_bond
                _bond_alloc_eff  = _min_bond
            elif _sum_bonds > 0:
                _w_bonds = _w_bonds / _sum_bonds * bond_alloc
                _bond_alloc_eff  = bond_alloc
            else:
                _bond_alloc_eff  = 0.0

            _stock_alloc_eff = _total_alloc - _bond_alloc_eff
            if _sum_stocks > 0 and _stock_alloc_eff > 0:
                _w_stocks = _w_stocks / _sum_stocks * _stock_alloc_eff

            weights_opt[stock_idx] = _w_stocks
            weights_opt[bond_idx]  = _w_bonds

            # Étape 4 : reclamper [min_w, max_w] et renormaliser
            for _ in range(3):
                weights_opt = np.clip(weights_opt, 0.0, max_w)
                _below = (weights_opt > 0) & (weights_opt < min_w)
                weights_opt[_below] = 0
                _s = weights_opt.sum()
                if _s > 0:
                    weights_opt = weights_opt / _s * _total_alloc
                if not np.any((weights_opt > 0) & (weights_opt < min_w)):
                    break

            # Construire le résultat
            result = pd.Series(0.0, index=all_tickers)
            for i, ticker in enumerate(all_tickers):
                if weights_opt[i] > 0:
                    result[ticker] = weights_opt[i]

            n_stocks_final = (result[stock_tickers] > min_w / 2).sum()
            n_bonds_final = (result[bond_tickers] > min_w / 2).sum()
            print(f"[DEBUG] Après construction result: "
                  f"{n_stocks_final} stocks + {n_bonds_final} bonds, "
                  f"sum={result.sum():.6f}")
            # Log silencieux — le résumé sera affiché par le backtester
            self._last_optim_summary = {
                'n_stocks': int(n_stocks_final),
                'n_bonds': int(n_bonds_final),
                'w_stocks': float(result[stock_tickers].sum()),
                'w_bonds': float(result[bond_tickers].sum()),
            }

            # Smoothing dynamique selon le régime
            if prev_weights is not None:
                tc = self.config.TURNOVER_CONTROL
                _regime_changed = (
                    current_regime is not None
                    and self._prev_regime is not None
                    and current_regime != self._prev_regime
                )
                if _regime_changed:
                    smoothing = tc.get('smoothing_on_regime_change', tc['weight_smoothing'])
                    if not self._smoothing_diag_done:
                        print(f"Smoothing dynamique: regime change "
                              f"{self._prev_regime} -> {current_regime}, "
                              f"smoothing reduit a {smoothing:.0%}")
                        self._smoothing_diag_done = True
                else:
                    smoothing = tc['weight_smoothing']
                prev_aligned = prev_weights.reindex(result.index).fillna(0)
                # Ne pas appliquer le smoothing si le portefeuille
                # précédent est majoritairement en cash (premier
                # investissement ou état dégénéré)
                cash_ticker = getattr(self.config, "CASH_BUFFER", {}).get(
                    "ticker", "CASH.TO")
                prev_cash_pct = float(prev_aligned.get(cash_ticker, 0))
                if prev_cash_pct > 0.50:
                    print(f"[DEBUG] Smoothing DÉSACTIVÉ: prev cash = "
                          f"{prev_cash_pct:.1%} > 50%")
                    # Ne pas smoother — garder les poids optimisés
                else:
                    result = smoothing * prev_aligned + (1 - smoothing) * result
                    if result.sum() > 0:
                        result = result / result.sum()
                    print(f"[DEBUG] Après smoothing ({smoothing:.0%}): "
                          f"{(result > 0).sum()} positions, sum={result.sum():.6f}")

            if current_regime is not None:
                self._prev_regime = current_regime

            return result

        except Exception as e:
            print(f"  [ATTENTION]  Erreur optimisation conjointe: {e}")
            return self._fallback_separate(
                stock_scores, bond_candidates, stock_alloc, bond_alloc,
                cov_matrix, liquidity_constraints
            )

    def _fallback_separate(self, stock_scores, bond_candidates,
                           stock_alloc, bond_alloc,
                           cov_matrix, liquidity_constraints):
        """
        Fallback : optimise stocks seuls, distribue bonds par BondSelector.
        Utilisé si l'optimisation conjointe échoue.
        """
        print("  [FALLBACK] Optimisation séparée stocks + bonds")
        # Stocks via MV standard
        weights_stocks = self._optimize_mean_variance(
            stock_scores, cov_matrix, liquidity_constraints
        )
        weights_stocks = weights_stocks * stock_alloc

        # Bonds via poids BondSelector
        min_w = self.config.STOCK_SELECTION['min_weight']
        bond_weights = pd.Series({
            t: w * bond_alloc for t, w in bond_candidates.items()
        })
        # Forcer min_weight
        for _ in range(5):
            below = bond_weights[(bond_weights > 0) & (bond_weights < min_w)]
            if len(below) == 0:
                break
            bond_weights[below.index] = 0
            remaining = bond_weights[bond_weights > 0]
            if remaining.sum() > 0:
                bond_weights = remaining / remaining.sum() * bond_alloc

        # Combiner
        result = pd.concat([weights_stocks, bond_weights])
        result = result.groupby(result.index).sum()  # Dédupliquer
        return result

    def apply_turnover_control(self, new_weights, prev_weights, bond_tickers=None):
        """Contrôle du turnover avec no-trade bands par classe d'actif."""
        if prev_weights is None:
            return new_weights

        tc            = self.config.TURNOVER_CONTROL
        threshold     = tc['rebalance_threshold']
        min_trade     = tc['min_trade_size']
        max_turnover  = tc['max_daily_turnover']
        band_stocks   = tc.get('no_trade_band_stocks', 0.015)
        band_bonds    = tc.get('no_trade_band_bonds',  0.005)
        bond_set      = set(bond_tickers) if bond_tickers else set()

        all_tickers = new_weights.index.union(prev_weights.index)
        new_w  = new_weights.reindex(all_tickers).fillna(0)
        prev_w = prev_weights.reindex(all_tickers).fillna(0)

        drift = (new_w - prev_w).abs().sum()

        if drift < threshold:
            return prev_weights

        if drift > max_turnover:
            scale = max_turnover / drift
            new_w = prev_w + scale * (new_w - prev_w)

        changes = new_w - prev_w

        # ── No-trade bands : filtrer les petites déviations par classe ─────────
        # Appliqué UNIQUEMENT aux positions existantes (prev > 0 ET new > 0).
        # Les nouvelles entrées (prev=0) et sorties complètes (new=0) sont
        # toujours autorisées — ce ne sont pas des 'ajustements marginaux'.
        n_total   = len(changes)
        n_skipped = 0
        for ticker in changes.index:
            band = band_bonds if ticker in bond_set else band_stocks
            is_existing = prev_w[ticker] > 0 and new_w[ticker] > 0
            if is_existing and abs(changes[ticker]) <= band:
                changes[ticker] = 0
                n_skipped += 1

        # Diagnostic au premier rebalancement seulement
        if not getattr(self, '_ntb_diag_done', False):
            n_traded = n_total - n_skipped
            print(f"No-trade bands: {n_traded}/{n_total} titres tradés, "
                  f"{n_skipped} sous le seuil")
            self._ntb_diag_done = True

        # min_trade filter (s'applique aux titres qui passent les bandes)
        changes[changes.abs() < min_trade] = 0

        new_w = prev_w + changes

        if new_w.sum() > 0:
            new_w = new_w / new_w.sum()

        return new_w

# =============================================================================
# FONCTIONS D'AJUSTEMENT POST-OPTIMISATION
# =============================================================================
# Ces fonctions s'appliquent apres l'optimisation MV pour integrer des
# contraintes supplementaires (ESG, concentration sectorielle, bornes).
# Elles sont appelees par le backtester et le live execution.

# =============================================================================
# FONCTION UTILITAIRE ESG
# =============================================================================
# Applique un biais ESG (ESG tilt) sur les pondérations du portefeuille.
# Les entreprises avec de meilleurs scores ESG reçoivent un poids plus élevé.

def apply_esg_tilt(weights, esg_scores, strength=0.35):

    esg_scaled = (esg_scores - esg_scores.min()) / (esg_scores.max() - esg_scores.min())

    tilt = 1 + strength * (esg_scaled - esg_scaled.mean())

    weights = weights * tilt
    weights = weights / weights.sum()

    return weights

def apply_sector_cap(weights, sector_map, max_sector_weight=0.25):
    """
    Plafonne la concentration sectorielle du portefeuille.

    Pour chaque secteur GICS depassant le plafond, les poids des titres
    de ce secteur sont reduits proportionnellement. L'excedent est redistribue
    aux secteurs sous le plafond, au prorata de leurs poids existants.

    Args:
        weights: pd.Series, poids des titres (somme = 1)
        sector_map: pd.Series, secteur GICS de chaque ticker
        max_sector_weight: float, plafond par secteur (defaut 0.25 = 25%)

    Returns:
        pd.Series, poids ajustes (somme = 1)
    """
    aligned_sectors = sector_map.reindex(weights.index).fillna("Unknown")
    sector_weights = weights.groupby(aligned_sectors).sum()

    excess_total = 0.0
    for sector, sw in sector_weights.items():
        if sw > max_sector_weight:
            mask = aligned_sectors == sector
            scale = max_sector_weight / sw
            excess_total += (sw - max_sector_weight)
            weights[mask] *= scale

    # Redistribuer l'excedent aux secteurs sous le plafond
    if excess_total > 0:
        under_cap = sector_weights[sector_weights <= max_sector_weight].index
        mask_under = aligned_sectors.isin(under_cap) & (weights > 0)
        if mask_under.sum() > 0:
            under_sum = weights[mask_under].sum()
            weights[mask_under] += weights[mask_under] / under_sum * excess_total

    # Renormaliser pour garantir somme = 1
    weights = weights / weights.sum()
    return weights

def enforce_weight_bounds(weights, min_weight=0.01, max_weight=0.10):
    """
    Force les poids individuels a rester dans les bornes [min_weight, max_weight].

    Les positions negligeables (< min_weight/2) sont mises a zero.
    Les positions actives sont clippees dans [min_weight, max_weight].
    Le tout est renormalise pour sommer a 1.
    """
    weights[weights < min_weight / 2] = 0
    active = weights > 0
    weights[active] = weights[active].clip(lower=min_weight, upper=max_weight)
    if weights.sum() > 0:
        weights = weights / weights.sum()
    return weights


def _redistribute_excess(w, excess, cash_ticker, max_weight, bond_set=None, prefer_bonds=False):
    """
    Redistribue un excédent de poids aux positions actives qui ont de la marge
    sous max_weight, au lieu de tout envoyer au cash.
    Seul le résidu non-distribuable va au cash.

    Si prefer_bonds=True, priorise les obligations (utile après un cap
    qui a réduit des bonds).
    """
    if excess < 1e-12:
        return w

    for _ in range(5):
        eligible = (w.index != cash_ticker) & (w > 0) & (w < max_weight - 1e-9)

        # Si on préfère les bonds, filtrer d'abord sur les bonds
        if prefer_bonds and bond_set is not None:
            bond_eligible = eligible & w.index.isin(bond_set)
            if bond_eligible.any():
                eligible = bond_eligible

        if not eligible.any():
            break

        room = (max_weight - w[eligible]).clip(lower=0.0)
        total_room = float(room.sum())
        if total_room < 1e-12:
            break

        to_distribute = min(excess, total_room)
        w.loc[eligible] += to_distribute * (room / total_room)
        excess -= to_distribute

        if excess < 1e-12:
            break

    # Résidu non-distribuable → cash (normalement très petit)
    if excess > 1e-12:
        w.loc[cash_ticker] += excess

    return w


def finalize_weights_with_bond_floor(
    weights,
    bond_set,
    cash_ticker,
    min_bond_alloc=0.25,
    min_weight=0.01,
    max_weight=0.10,
    cash_target=None,
    decimals=6
):
    """
    Finalise les poids en respectant simultanément:
      - somme exacte = 1
      - obligations >= min_bond_alloc
      - poids individuels <= max_weight (y compris obligations)
      - poids individuels >= min_weight pour les positions non cash conservées
      - cash conservé séparément (proche de cash_target)

    L'excédent issu du capping ou de la suppression de micro-positions
    est redistribué aux positions actives ayant de la marge sous max_weight,
    plutôt que d'être accumulé dans le cash.
    """

    w = weights.copy().astype(float)

    # 0) Nettoyage de base
    w = w[~w.index.duplicated(keep='last')]
    w = w.fillna(0.0)
    w[w < 0] = 0.0

    if cash_ticker not in w.index:
        w.loc[cash_ticker] = 0.0


    # Imposer le cash_target dès le départ
    if cash_target is not None:
        w.loc[cash_ticker] = float(cash_target)

    # S'assurer qu'il y a assez de slots obligataires pour rendre
    # min_bond_alloc atteignable sous max_weight.
    required_bond_slots = int(math.ceil(min_bond_alloc / max_weight))

    existing_bond_slots = [
        t for t in w.index
        if (t in bond_set) and (t != cash_ticker)
    ]

    if len(existing_bond_slots) < required_bond_slots:
        missing_bonds = sorted([
            t for t in bond_set
            if (t not in w.index) and (t != cash_ticker)
        ])
        need = required_bond_slots - len(existing_bond_slots)

        for t in missing_bonds[:need]:
            w.loc[t] = 0.0

    # 1) Supprimer les micro-positions (< min_weight), sauf cash
    #    → redistribuer aux positions actives, pas au cash
    non_cash = w.index != cash_ticker
    small_pos = (w > 0) & (w < min_weight) & non_cash
    if small_pos.any():
        freed = float(w[small_pos].sum())
        w.loc[small_pos] = 0.0
        w = _redistribute_excess(w, freed, cash_ticker, max_weight, bond_set)

    # 2) Cap individuel initial à max_weight (hors cash)
    #    → redistribuer l'excédent aux positions sous le cap
    capped = (w.index != cash_ticker) & (w > max_weight)
    if capped.any():
        excess = float((w[capped] - max_weight).sum())
        w.loc[capped] = max_weight
        w = _redistribute_excess(w, excess, cash_ticker, max_weight, bond_set)

    # 3) Forcer obligations >= min_bond_alloc
    # Algorithme:
    #   a) Calculer l'allocation obligataire courante
    #   b) Si < floor, tranférer du cash/actions jusqu'au floor
    #   c) Renormaliser tout pour que somme = 1.0

    bond_mask = w.index.isin(bond_set)
    current_bond = float(w[bond_mask].sum())
    deficit = float(min_bond_alloc - current_bond)

    if deficit > 1e-12:
        # Stratégie : prendre d'abord du cash, puis actions si nécessaire

        cash_available = max(0.0, float(w.get(cash_ticker, 0.0)))
        take_from_cash = min(deficit, cash_available)
        remaining_deficit = deficit - take_from_cash

        if take_from_cash > 1e-12:
            w.loc[cash_ticker] -= take_from_cash

        # Si encore du déficit, prendre des actions en respectant min_weight
        if remaining_deficit > 1e-12:
            non_bond_non_cash_mask = (~bond_mask) & (w.index != cash_ticker)
            available_to_cut = w[non_bond_non_cash_mask] - min_weight
            available_to_cut = available_to_cut.clip(lower=0.0)
            total_available = float(available_to_cut.sum())

            if total_available > 1e-12:
                take_amount = min(remaining_deficit, total_available)
                cut_fractions = (available_to_cut / total_available) * take_amount
                w[non_bond_non_cash_mask] -= cut_fractions
                remaining_deficit -= take_amount

        # Ajouter le total au fond obligataire (distribuer proportionnellement aux bonds existants)
        bonds_in_portfolio = w[bond_mask] > 1e-12
        if bonds_in_portfolio.any():
            # Distribuer à tous les bonds avec au moins min_weight ou existant
            bond_weights = w[bond_mask & (w > 0)]
            if len(bond_weights) > 0:
                add_fractions = (bond_weights / bond_weights.sum()) * (deficit - remaining_deficit)
                w.loc[bond_mask & (w > 0)] += add_fractions

        # Cas extrême: pas de bonds en portfolio, créer un
        if w[bond_mask].sum() < min_bond_alloc - 1e-12:
            available_bonds = [t for t in bond_set if t in w.index]
            if available_bonds:
                w.loc[available_bonds[0]] += min_bond_alloc - w[bond_mask].sum()

    # 4) NORMALISATION CRITIQUE: la somme peut avoir dévié
    total_before_norm = float(w.sum())
    if abs(total_before_norm - 1.0) > 1e-9:
        # Réduire les poids proportionnellement pour revenir à 1.0
        non_zero = w > 1e-12
        if non_zero.any():
            w[non_zero] = w[non_zero] / total_before_norm

    # 5) Deuxième passe: cap individuel final hors cash
    # redistribuer l'excédent (prioriser les bonds si le floor est serré)
    capped = (w.index != cash_ticker) & (w > max_weight)
    if capped.any():
        excess = float((w[capped] - max_weight).sum())
        w.loc[capped] = max_weight
        w = _redistribute_excess(w, excess, cash_ticker, max_weight, bond_set,
                                 prefer_bonds=(float(w[w.index.isin(bond_set)].sum()) < min_bond_alloc))

    # 5) Supprimer encore les positions < min_weight (hors cash)
    #  redistribuer, pas accumuler au cash
    small_pos = (w > 0) & (w < min_weight) & (w.index != cash_ticker)
    if small_pos.any():
        freed = float(w[small_pos].sum())
        w.loc[small_pos] = 0.0
        w = _redistribute_excess(w, freed, cash_ticker, max_weight, bond_set)

    # 5b) Vidange du cash excédentaire vers les positions actives
    #     Si après toutes les passes le cash dépasse cash_target,
    #     redistribuer le surplus.
    if cash_target is not None:
        cash_excess = float(w.get(cash_ticker, 0.0)) - float(cash_target)
        if cash_excess > 1e-9:
            w.loc[cash_ticker] = float(cash_target)
            w = _redistribute_excess(w, cash_excess, cash_ticker, max_weight, bond_set)

    # 6) Somme exacte = 1 via cash

    total = float(w.sum())
    if abs(total - 1.0) > 1e-12:
        w.loc[cash_ticker] = round(float(w.loc[cash_ticker]) + (1.0 - total), decimals)

    # 7) Arrondi CSV-safe + somme exacte après arrondi
    w = w.round(decimals)
    residual = round(1.0 - float(w.sum()), decimals)
    if abs(residual) >= 10**(-decimals):
        w.loc[cash_ticker] = round(float(w.loc[cash_ticker] + residual), decimals)

    # 7b) L'arrondi peut faire perdre ~1e-6 aux bonds
    bond_mask_final = w.index.isin(bond_set)
    final_bond_check = float(w[bond_mask_final].sum())
    if final_bond_check < min_bond_alloc - 1e-9:
        shortfall = round(min_bond_alloc - final_bond_check, decimals)
        bond_positions = w[bond_mask_final & (w > 0) & (w < max_weight)]
        if len(bond_positions) > 0 and shortfall > 0:
            # Choisir le bond avec le plus de marge sous max_weight
            room = max_weight - bond_positions
            best_bond = room.idxmax()
            add = min(shortfall, round(float(room[best_bond]), decimals))
            if add > 0:
                w.loc[best_bond] = round(float(w.loc[best_bond]) + add, decimals)
                w.loc[cash_ticker] = round(float(w.loc[cash_ticker]) - add, decimals)

    # 8) Sécurité finale
    w[w.abs() < 10**(-decimals)] = 0.0
    w = w[w > 0].copy()

    if cash_ticker in w.index and w.loc[cash_ticker] < 0:
        raise ValueError(f"Cash négatif après finalisation: {w.loc[cash_ticker]:.8f}")
    total_final = float(w.sum())
    residual_final = 1.0 - total_final

    if abs(residual_final) > 1e-12:
        if cash_ticker in w.index:
            w.loc[cash_ticker] = float(w.loc[cash_ticker]) + residual_final
        else:
            # fallback: ajuster la plus grosse position non-cash
            non_cash_idx = [idx for idx in w.index if idx != cash_ticker]
            if len(non_cash_idx) > 0:
                largest = w[non_cash_idx].idxmax()
                w.loc[largest] = float(w.loc[largest]) + residual_final

    # Validation finale stricte
    non_cash_w = w.drop(index=[cash_ticker], errors='ignore')
    if len(non_cash_w) > 0:
        assert (non_cash_w <= max_weight + 1e-9).all(), "Un poids dépasse 10% après finalisation."
        assert (non_cash_w[non_cash_w > 0] >= min_weight - 1e-9).all(), "Un poids actif est < 1% après finalisation."

    final_bond = float(w[w.index.isin(bond_set)].sum())

    # Si le floor n'est pas atteint, on essaie d'ajouter des bonds jusqu'au maximum
    # permis par les contraintes de position, puis on lance un avertissement
    # si c'est vraiment impossible (edge case extrême)
    if final_bond < min_bond_alloc - 1e-6:
        shortfall = min_bond_alloc - final_bond

        # Trouver les bonds avec de la marge sous max_weight
        bond_positions = w[w.index.isin(bond_set)]
        bonds_under_cap = bond_positions[bond_positions < max_weight]

        if len(bonds_under_cap) > 0:
            # Y a-t-il de la place pour ajouter des bonds?
            available_room = float(((max_weight - bonds_under_cap).sum()))
            can_add = min(shortfall, available_room)

            if can_add > 1e-12:
                # Distribuer l'ajout proportionnellement aux bonds qui ont de la marge
                room_pcts = (max_weight - bonds_under_cap) / (max_weight - bonds_under_cap).sum()
                w.loc[bonds_under_cap.index] += can_add * room_pcts
                final_bond = float(w[w.index.isin(bond_set)].sum())

        # Si ON A TOUJOURS pas assez de bonds après cette tentative,
        # vérifier si on peut en créer de nouveaux à partir du cash
        if final_bond < min_bond_alloc - 1e-6 and cash_ticker in w.index:
            remaining_shortfall = min_bond_alloc - final_bond
            available_cash = w.loc[cash_ticker] if w.loc[cash_ticker] > 0.01 else 0

            if available_cash > remaining_shortfall:
                # Prendre du cash pour augmenter les obligations
                cash_to_move = min(remaining_shortfall, available_cash - 0.01)  # Garder min 1% cash

                # Ajouter à un bond qui a de la marge
                available_bonds = w.index[(w.index.isin(bond_set)) & (w > 0) & (w < max_weight)]
                if len(available_bonds) > 0:
                    # Ajouter au plus grand bond déjà en portefeuille
                    biggest_bond = w[available_bonds].idxmax()
                    can_add_to_bond = min(cash_to_move, max_weight - w.loc[biggest_bond])
                    if can_add_to_bond > 1e-12:
                        w.loc[biggest_bond] += can_add_to_bond
                        w.loc[cash_ticker] -= can_add_to_bond
                        final_bond = float(w[w.index.isin(bond_set)].sum())

        # Si TOUJOURS pas assez de bonds après ces tentatives,
        # c'est que le portefeuille n'a pas assez de candidats bonds disponibles.
        # Dans ce cas, on log un WARNING mais on continue (plutôt que d'échouer
        # et d'arrêter le backtest). Cela arrive quand:
        #   - Peu de bonds liquides disponibles
        #   - Bond optimizer a rejeté beaucoup de candidats
        if final_bond < min_bond_alloc - 1e-6:
            print(f"  [ATTENTION] Bond floor non complètement satisfait: {final_bond:.6f} vs {min_bond_alloc:.6f}")
            print(f"  Raison probable: pas assez de candidats bonds liquides ou avec scores positifs")
            print(f"  Nombre de bonds en portefeuille: {len(w[w.index.isin(bond_set) & (w > 0)])}")

    # Renormaliser avant l'assertion finale (au cas où les ajustements de bond floor auraient perturbé la somme)
    total_w = float(w.sum())
    if abs(total_w - 1.0) > 1e-9:
        non_zero = w > 1e-12
        if non_zero.any():
            w[non_zero] = w[non_zero] / total_w

    assert abs(float(w.sum()) - 1.0) <= 10**(-decimals) + 1e-12, f"Somme finale != 1 : {w.sum()}"

    return w


print("[OK] Module PortfolioOptimizer V2 chargé (Mean-Variance + Liquidité)")

[OK] Module PortfolioOptimizer V2 chargé (Mean-Variance + Liquidité)


In [8]:
# =============================================================================
# CELLULE : BOND SELECTOR — Sélection intelligente des obligations
# =============================================================================
#
# Ce module sélectionne les meilleurs ETFs obligataires par catégorie,
# puis les injecte dans l'optimiseur Mean-Variance pour une optimisation
# conjointe stocks + bonds exploitant les corrélations croisées.
#
# ARCHITECTURE :
#   1. BondSelector.select_bond_candidates() : choisit les meilleurs ETFs
#   2. PortfolioOptimizer.optimize_joint()   : MV conjoint stocks + bonds
#   3. generate_rebalancing_journal()        : mise en forme
#
# La sélection intra-catégorie utilise un score composite :
#   Score = 0.50 × momentum_12m + 0.30 × momentum_3m − 0.20 × vol_6m
#
# Anti-turnover : hysteresis de 15% pour éviter le whipsaw entre ETFs
# similaires d'une même catégorie.
#
# =============================================================================
#
# (Amélioration prévu) : INTÉGRATION BLOOMBERG / REFINITIV
# =============================================================================
#
# Avec l'accès aux données Bloomberg (via API blpapi) ou Refinitiv (via
# Eikon/LSEG Data Library), le BondSelector pourrait évoluer vers un
# véritable modèle multi-factoriel obligataire. Voici les facteurs
# et leur implémentation prévue :
#
# 1. CARRY (Rendement à maturité)
#    - Source : Bloomberg field YAS_YLD_SPREAD, Refinitiv TR.YieldToMaturity
#    - Calcul : YTM du portefeuille sous-jacent de l'ETF
#    - Signal : Long les ETFs à haut carry, short les faibles
#    - Impact attendu : +30-50 bps annuels sur la poche bonds
#    - Implémentation : Ajouter dans _compute_bond_scores() comme facteur
#      pondéré à ~30% du score composite bonds
#
# 2. MOMENTUM DE TAUX (Direction des taux souverains)
#    - Source : Bloomberg USGG10YR / USGG2YR, Refinitiv US10YT=RR
#    - Calcul : Variation 3m/12m des taux → signal directionnel
#    - Signal : Taux en baisse → surpondérer duration longue (TLT, ZFL.TO)
#              Taux en hausse → sous-pondérer duration, favoriser floating
#    - Impact attendu : Améliore le timing de la duration de ~15-25 bps
#    - Implémentation : Remplacer le mix fixe REGIME_BOND_MIX par un mix
#      dynamique ajusté selon le momentum de taux
#
# 3. SPREAD MOMENTUM (Variation des spreads de crédit)
#    - Source : Bloomberg field OAS_SPREAD_BID, Refinitiv TR.OASpread
#    - Calcul : Variation 1m/3m du spread OAS (Option-Adjusted Spread)
#    - Signal : Spreads en contraction → surpondérer crédit (HY, IG)
#              Spreads en expansion → sous-pondérer crédit, flight-to-quality
#    - Impact attendu : +20-40 bps en évitant le crédit avant les corrections
#    - Implémentation : Ajouter comme signal dans detect_regime ou comme
#      facteur dans _compute_bond_scores() à ~25% du score composite
#
# 4. ROLLDOWN (Gain de prix le long de la courbe)
#    - Source : Bloomberg courbe YCGT0025 (Gov CA) / YCGT0016 (UST)
#    - Calcul : Différence de taux entre maturité actuelle et maturité - 1an
#    - Signal : Pente de courbe forte → favoriser les ETFs mid-duration
#              Courbe plate/inversée → rester court terme
#    - Impact attendu : +10-20 bps (subtil mais constant)
#    - Implémentation : Nécessite données de courbe par maturité, calculer
#      le rolldown attendu pour chaque catégorie de duration
#
# 5. DURATION TIMING (Tactical duration overlay)
#    - Source : Combinaison de yield curve slope + VIX + regime
#    - Calcul : Si courbe s'aplatit + RISK_OFF → augmenter duration
#              Si courbe se pentifie + RISK_ON → réduire duration
#    - Impact attendu : +20-30 bps par meilleur timing de la duration
#    - Implémentation : Modifier les poids du REGIME_BOND_MIX dynamiquement
#      selon un score de duration timing continu (pas binaire)
#
# ARCHITECTURE CIBLE (avec Bloomberg/Refinitiv) :
#
#   BondFactorCalculator (nouveau module)
#   ├── compute_carry_score(etf_tickers)     : pd.Series
#   ├── compute_rate_momentum()              : float (signal global)
#   ├── compute_spread_momentum()            : float (signal global)
#   ├── compute_rolldown_score(etf_tickers)  : pd.Series
#   └── compute_composite_bond_score()       : pd.Series
#
#   Le score composite bonds remplacerait le scoring simple actuel
#   (momentum + vol) par une combinaison pondérée des 5 facteurs.
#   Les poids recommandés (basés sur la littérature) :
#     Carry: 30%, Momentum de taux: 20%, Spread: 25%, Rolldown: 10%, Vol: 15%
#
#   L'intégration dans l'optimiseur MV conjoint resterat identique :
#   les scores bonds seraient transformés en rendements attendus (μ) au
#   même titre que les scores actions, et la matrice de covariance
#   stocks+bonds exploiterait les corrélations croisées.
#
# =============================================================================


class BondSelector:
    """
    Sélection institutionnelle d'ETFs obligataires par catégories.

    Diversifie la poche bonds selon le régime de marché en choisissant
    le meilleur ETF dans chaque catégorie via un score composite.

    Actuellement limité aux données Yahoo Finance (prix seulement).
    Voir ROADMAP ci-dessus pour l'évolution avec Bloomberg/Refinitiv.
    """

    # ─── Catégories et ETFs disponibles (US + CA) ───────────────────────
    BOND_CATEGORIES = {
        'gov_short': {
            'label': 'Gouvernement court terme (1-3 ans)',
            'tickers': ['SHY', 'VGSH', 'ZFS.TO'],
        },
        'gov_mid': {
            'label': 'Gouvernement moyen terme (3-10 ans)',
            'tickers': ['IEI', 'VGIT', 'IEF', 'GOVT', 'XGB.TO'],
        },
        'gov_long': {
            'label': 'Gouvernement long terme (10+ ans)',
            'tickers': ['TLT', 'VGLT', 'ZFL.TO'],
        },
        'corp_ig': {
            'label': 'Corporate Investment Grade',
            'tickers': ['LQD', 'VCIT', 'VCSH', 'IGIB', 'XCB.TO', 'ZCS.TO'],
        },
        'corp_hy': {
            'label': 'Corporate High Yield',
            'tickers': ['HYG', 'JNK', 'SHYG', 'XHY.TO', 'ZHY.TO'],
        },
        'tips': {
            'label': 'Obligations indexées inflation (TIPS)',
            'tickers': ['TIP', 'VTIP', 'SCHP', 'XRB.TO'],
        },
        'aggregate': {
            'label': 'Marché obligataire agrégé',
            'tickers': ['AGG', 'BND', 'SCHZ', 'XBB.TO', 'ZAG.TO'],
        },
    }

    # ─── Allocation intra-bonds par régime (somme = 1.0) ────────────────
    # RISK_ON  : favorise crédit (spread) et duration courte
    # RISK_OFF : favorise gouvernemental et duration longue (flight-to-quality)
    REGIME_BOND_MIX = {
        'RISK_ON': {
            'gov_short':  0.05,
            'gov_mid':    0.10,
            'gov_long':   0.00,
            'corp_ig':    0.35,
            'corp_hy':    0.20,
            'tips':       0.10,
            'aggregate':  0.20,
        },
        'RISK_OFF': {
            'gov_short':  0.20,
            'gov_mid':    0.25,
            'gov_long':   0.15,
            'corp_ig':    0.15,
            'corp_hy':    0.00,
            'tips':       0.10,
            'aggregate':  0.15,
        },
    }

    # Seuil anti-turnover : ne changer d'ETF que si 15% meilleur
    SWITCH_THRESHOLD = 0.15

    def __init__(self, config):
        self.config = config
        self._prev_selections = {}   # {category: ticker}

    # ═════════════════════════════════════════════════════════════════════
    # SÉLECTION DES CANDIDATS (pour injection dans l'optimiseur MV)
    # ═════════════════════════════════════════════════════════════════════
    def select_bond_candidates(self, regime, available_returns):
        """
        Sélectionne le meilleur ETF par catégorie active pour le régime.

        NE CALCULE PAS les poids — ceux-ci seront déterminés par
        l'optimiseur MV conjoint (stocks + bonds).

        Args:
            regime: 'RISK_ON' ou 'RISK_OFF'
            available_returns: pd.DataFrame des returns (données t-1)

        Returns:
            dict: {ticker: target_relative_weight}
                  target_relative_weight = poids cible RELATIF au sein
                  de la poche bonds (somme = 1.0). Sert de signal μ
                  pour l'optimiseur MV.
        """
        mix = self.REGIME_BOND_MIX.get(regime, self.REGIME_BOND_MIX['RISK_ON'])

        candidates = {}  # {ticker: relative_weight}

        for category, target_pct in mix.items():
            if target_pct <= 0:
                continue

            cat_info = self.BOND_CATEGORIES[category]
            available = [t for t in cat_info['tickers']
                        if t in available_returns.columns]

            if not available:
                continue

            best = self._select_best_etf(available, available_returns, category)

            if best is not None:
                candidates[best] = candidates.get(best, 0) + target_pct

        # Normaliser à 1.0
        if candidates:
            total = sum(candidates.values())
            candidates = {t: w / total for t, w in candidates.items()}
        else:
            # Fallback
            for fb in ['AGG', 'BND', 'XBB.TO', 'ZAG.TO']:
                if fb in available_returns.columns:
                    candidates = {fb: 1.0}
                    break

        return candidates

    # ═════════════════════════════════════════════════════════════════════
    # SCORE COMPOSITE + ANTI-TURNOVER
    # ═════════════════════════════════════════════════════════════════════
    def _select_best_etf(self, tickers, returns, category):
        """
        Score = 0.50 × momentum_12m + 0.30 × momentum_3m − 0.20 × vol_6m

        Anti-turnover : ne change que si le nouveau est 15% meilleur.

        Note: Avec Bloomberg/Refinitiv, ce scoring sera enrichi des facteurs
        carry, spread momentum et rolldown (voir ROADMAP en haut du module).
        """
        scores = {}

        for ticker in tickers:
            ret = returns[ticker].dropna()
            if len(ret) < 63:
                continue

            mom_12m = ret.tail(min(252, len(ret))).sum()
            mom_3m  = ret.tail(63).sum()
            vol_6m  = ret.tail(min(126, len(ret))).std() * np.sqrt(252)

            scores[ticker] = 0.50 * mom_12m + 0.30 * mom_3m - 0.20 * vol_6m

        if not scores:
            return tickers[0] if tickers else None

        best = max(scores, key=scores.get)

        # Hysteresis
        prev = self._prev_selections.get(category)
        if prev and prev in scores and prev != best:
            if abs(scores[prev]) > 1e-8:
                improvement = (scores[best] - scores[prev]) / abs(scores[prev])
                if improvement < self.SWITCH_THRESHOLD:
                    best = prev

        self._prev_selections[category] = best
        return best

    # ═════════════════════════════════════════════════════════════════════
    # RAPPORT
    # ═════════════════════════════════════════════════════════════════════
    def get_selection_report(self, candidates, bond_alloc):
        """Rapport détaillé de la sélection."""
        lines = [f"  BondSelector — {len(candidates)} ETFs candidats:"]
        for ticker, rel_w in sorted(candidates.items(),
                                     key=lambda x: -x[1]):
            cat_label = "?"
            for cat, info in self.BOND_CATEGORIES.items():
                if ticker in info['tickers']:
                    cat_label = info['label']
                    break
            abs_w = rel_w * bond_alloc
            lines.append(
                f"    {ticker:<10} cible: {abs_w:.2%}  ({cat_label})"
            )
        lines.append(f"    TOTAL bonds cible: {bond_alloc:.2%}")
        return "\n".join(lines)


print("[OK] Module BondSelector chargé")

[OK] Module BondSelector chargé


---
# 6. Backtester — Walk-forward rolling
---


In [13]:
# =============================================================================
# CELLULE 6: ROLLING BACKTESTER AVEC CASH BUFFER (FIX KeyError + FIX chevauchements)
# =============================================================================

class RollingBacktester:
    """Backtester avec cash buffer, bonds, taux sans risque, + overlay régimes."""

    def __init__(self, config=Config):
        self.config = config
        self.results = None
        self.weights_history = []
        self.regime_history = []
        self.fx_exposure_history = []
        self.turnover_history = []
        self.regime_details_history = []

    def reset(self):
        """Utile pour tes tests A/B: repartir avec des historiques vides."""
        self.results = None
        self.weights_history = []
        self.regime_history = []
        self.fx_exposure_history = []
        self.turnover_history = []
        self.regime_details_history = []

    def run_walk_forward(self, prices, returns, factor_calc, risk_mgr, optimizer,
                         data_loader, vix_data=None, overlay_enabled=True):
        cfg = self.config.BACKTEST['rolling_window']
        train_months = cfg['train_months']
        test_months = cfg['test_months']
        step_months = cfg['step_months']

        start_date = self.config.BACKTEST['start_date']
        end_date = self.config.BACKTEST['end_date']
        windows = self._generate_windows(start_date, end_date, train_months, test_months, step_months)

        print(f"\n{'='*70}")
        print("WALK-FORWARD BACKTEST V3 (avec cash buffer)")
        print(f"{'='*70}")
        print(f"Fenêtres: {len(windows)}")
        print(f"Train: {train_months}m | Test: {test_months}m | Step: {step_months}m")
        print(f"Cash buffer: {self.config.CASH_BUFFER['percentage']:.1%}")
        print(f"Smoothing: {self.config.TURNOVER_CONTROL['weight_smoothing']:.0%}")

        # --- RESET état RiskManager pour run indépendant (important pour A/B) ---
        risk_mgr._overlay_on = False
        risk_mgr._cooldown = 0
        risk_mgr._overlay_activations = []
        risk_mgr._regime_history = []
        risk_mgr._current_regime = "RISK_ON"
        bond_set = (set(data_loader.US_BOND_ETFS) |
                    set(data_loader.CA_BOND_ETFS))

        # Historique poids précédent pour turnover control
        prev_weights = None
        seen_rebalance_dates = set()

        # --- Diagnostic des fenêtres skippées ---
        skipped_windows = {'no_test_data': 0, 'not_scheduled': 0,
                         'too_few_tickers': 0, 'liquidity_filtered': 0}

        # Reset du diagnostic overlay à chaque run (important pour diagnostic A/B)
        if hasattr(risk_mgr, "_overlay_activations"):
            risk_mgr._overlay_activations = []

        # --- boucle sur les fenêtres ---
        for (train_start, train_end, test_start, test_end) in tqdm(windows, desc="Walk-Forward"):
            test_returns = returns.loc[test_start:test_end]
            if len(test_returns) == 0:
              skipped_windows['no_test_data'] += 1
              continue

            rebalance_dates = self._get_rebalance_dates(test_returns.index)

            for rebal_date in rebalance_dates:
                # Overlap des fenêtres => même date peut apparaître 2 fois.
                if rebal_date in seen_rebalance_dates:
                    continue

                # === LOGIQUE D'EXECUTION (signal daily, trading parcimonieux) ===
                rebal_cfg = getattr(self.config, 'REBALANCING', {})
                schedule = rebal_cfg.get('schedule', 'weekly')
                target_weekday = rebal_cfg.get('day_of_week', 0)

                # Jour programmé ?
                if schedule == 'weekly':
                    is_scheduled_day = (rebal_date.weekday() == target_weekday)
                elif schedule == 'monthly':
                    is_scheduled_day = (rebal_date.day <= 7 and rebal_date.weekday() == target_weekday)
                else:  # daily
                    is_scheduled_day = True

                # ---------------------------------------------------------------
                # PROTECTION LOOK-AHEAD BIAS
                # Toute décision de portefeuille est basée sur l'information
                # disponible STRICTEMENT avant rebal_date. On recule d'un jour
                # (decision_date = T-1) pour simuler le processus réel :
                # le gérant observe les prix de clôture de la veille et passe
                # ses ordres le lendemain à l'ouverture.

                # --- Anti look-ahead: décision à T-1 ---
                if rebal_date not in prices.index:
                    # sécurité: si un rebal_date n'existe pas exactement dans prices.index
                    # on prend la dernière date dispo avant rebal_date
                    pos = prices.index.searchsorted(rebal_date, side="right") - 1
                    if pos <= 0:
                        continue
                    rebal_date_effective = prices.index[pos]
                else:
                    rebal_date_effective = rebal_date

                idx_pos = prices.index.get_loc(rebal_date_effective)
                decision_date = prices.index[idx_pos - 1] if idx_pos > 0 else rebal_date_effective

                available_prices = prices.loc[:decision_date]
                available_returns = returns.loc[:decision_date]

                # --- slices VIX + macro cohérentes T-1 ---
                vix_slice = vix_data.loc[:decision_date] if vix_data is not None else None

                macro_slice = {}
                if hasattr(data_loader, "macro_data") and data_loader.macro_data is not None:
                    for key, series in data_loader.macro_data.items():
                        if series is not None:
                            macro_slice[key] = series.loc[:decision_date]

                # --- régime (calculé tous les jours) ---
                regime_output = risk_mgr.detect_regime_full(available_prices, vix_slice, macro_slice)
                regime = regime_output.regime

                # Force trade UNIQUEMENT à l'entrée de l'overlay crash
                force_trade = bool(overlay_enabled) and bool(getattr(regime_output, "overlay_just_activated", False))

                # Skip => aucun turnover si ni programmé ni urgence
                if (not is_scheduled_day) and (not force_trade):
                    skipped_windows['no_test_data'] += 1
                    continue

                # On enregistre la date de rebal (celle demandée), pas forcément effective
                # mais on utilise rebal_date_effective pour tout aligner au calendrier de prix
                seen_rebalance_dates.add(rebal_date_effective)

                # IMPORTANT: si overlay OFF (test A/B), forcer multiplier=1
                multiplier = 1.0 if not overlay_enabled else float(regime_output.exposure_multiplier)

                # --- allocation stocks/bonds/cash ---
                # 1. Appliquer le multiplicateur aux actions
                # 2. Garantir le floor obligations de 25% STRICTEMENT
                # 3. Le reste va au cash (qui peut absorber les variations)
                # 4. Normaliser pour que somme = 100%

                base_alloc = self.config.REGIME_ALLOCATIONS[regime]
                base_stock = float(base_alloc['stocks'])
                base_bond  = float(base_alloc['bonds'])
                base_cash  = float(base_alloc.get('cash', self.config.CASH_BUFFER['percentage']))
                min_bond_floor = float(self.config.MIN_BOND_ALLOC)

                # Étape 1: Appliquer multiplicateur sur actions
                stock_alloc = base_stock * multiplier

                # Étape 2: Garantir le floor obligations (25%)
                # Les obligations doivent TOUJOURS être >= 25%, peu importe le régime
                bond_alloc = max(min_bond_floor, base_bond + (base_stock - stock_alloc))

                # Étape 3: Cash absorbe le reste
                cash_alloc = 1.0 - stock_alloc - bond_alloc

                # Étape 4: Si cash devient négatif, réduire stocks (jamais bonds en dessous du floor)
                if cash_alloc < 0.0:
                    stock_alloc = max(0.0, stock_alloc + cash_alloc)
                    cash_alloc = 0.0

                # Étape 5: Normalisation finale
                total = stock_alloc + bond_alloc + cash_alloc
                if total <= 0:
                    # Fallback en cas d'erreur extrême
                    stock_alloc, bond_alloc, cash_alloc = base_stock, base_bond, base_cash
                    total = stock_alloc + bond_alloc + cash_alloc

                stock_alloc /= total
                bond_alloc /= total
                cash_alloc /= total

                # Vérification finale AVANT l'optimisation
                assert bond_alloc >= min_bond_floor - 1e-6, \
                    f"[ERREUR CRITIQUE] Bond floor violation: {bond_alloc:.6f} < {min_bond_floor}"
                assert abs(stock_alloc + bond_alloc + cash_alloc - 1.0) < 1e-6, \
                    f"[ERREUR CRITIQUE] Allocation sum != 1.0: {stock_alloc + bond_alloc + cash_alloc:.6f}"

                # --- facteurs + optimisation conjointe ---
                factors = factor_calc.compute_all_factors(available_prices, available_returns, rebal_date_effective, regime=regime)
                _use_binary = getattr(self.config, 'USE_BINARY_REGIME_WEIGHTS', False)
                composite = factor_calc.compute_composite_score(
                    factors, regime=regime,
                    regime_score=None if _use_binary else float(regime_output.composite_score)
                )

                valid_tickers = composite.dropna().index
                valid_tickers = [t for t in valid_tickers if t in available_returns.columns]

                # Séparer actions des bonds dans les scores
                stock_tickers_bt = [t for t in valid_tickers if t not in bond_set]

                if len(stock_tickers_bt) < 10:
                    skipped_windows['no_test_data'] += 1
                    continue

                # --- Filtrage par liquidité ---
                if hasattr(self.config, 'LIQUIDITY') and self.config.LIQUIDITY.get('enabled', False):
                    liquid_tickers = data_loader.get_liquid_tickers(rebal_date_effective)
                    stock_tickers_bt = [t for t in stock_tickers_bt if t in liquid_tickers]

                # --- Filtrage exclusions ---
                if hasattr(data_loader, 'filter_excluded_tickers'):
                    stock_tickers_bt = data_loader.filter_excluded_tickers(stock_tickers_bt)

                if len(stock_tickers_bt) < 10:
                    skipped_windows['no_test_data'] += 1
                    continue

                # --- BondSelector : candidats bonds ---
                bond_selector = BondSelector(self.config)
                bond_candidates = bond_selector.select_bond_candidates(
                    regime, available_returns
                )

                # Fallback: si aucun bond candidat, ajouter des defaults
                if not bond_candidates:
                    defaults = ['AGG', 'BND', 'XBB.TO', 'ZAG.TO']
                    available_defaults = [d for d in defaults if d in available_returns.columns]
                    if available_defaults:
                        weight_per = 1.0 / len(available_defaults)
                        bond_candidates = {d: weight_per for d in available_defaults}

                # --- Covariance conjointe stocks + bonds ---
                all_opt_tickers = stock_tickers_bt + list(bond_candidates.keys())
                all_opt_tickers = [t for t in all_opt_tickers
                                  if t in available_returns.columns]
                cov = risk_mgr.compute_covariance(available_returns[all_opt_tickers])
                cov_df = pd.DataFrame(cov, index=all_opt_tickers,
                                      columns=all_opt_tickers)

                # --- Contraintes de liquidité ---
                liquidity_constraints = None
                if hasattr(data_loader, 'get_liquidity_constraints'):
                    liquidity_constraints = data_loader.get_liquidity_constraints(
                        stock_tickers_bt, rebal_date_effective
                    )

                # --- Optimisation conjointe MV ---
                stock_scores = composite[stock_tickers_bt]
                weights = optimizer.optimize_joint(
                    stock_scores=stock_scores,
                    bond_candidates=bond_candidates,
                    cov_matrix=cov_df,
                    stock_alloc=stock_alloc,
                    bond_alloc=bond_alloc,
                    prev_weights=prev_weights,
                    liquidity_constraints=liquidity_constraints,
                    current_regime=regime,
                    )


                if prev_weights is not None:
                    weights = optimizer.apply_turnover_control(weights, prev_weights, bond_tickers=bond_set)
                    turnover = (weights.reindex(prev_weights.index).fillna(0) - prev_weights).abs().sum()
                else:
                    turnover = 1.0

                # --- ESG tilt (optionnel) ---

                if hasattr(self.config, "ESG") and self.config.ESG.get("enabled", False):

                    weights = apply_esg_tilt(
                        weights,
                        esg_scores=data_loader.esg_scores.loc[weights.index],
                        strength=self.config.ESG.get("tilt_strength", 0.35)
                    )

                # --- Diversification sectorielle ---
                if hasattr(self.config, "SECTOR_DIVERSIFICATION") and \
                        self.config.SECTOR_DIVERSIFICATION.get("enabled", False):
                    weights = apply_sector_cap(
                        weights,
                        sector_map=data_loader.sector_map,
                        max_sector_weight=self.config.SECTOR_DIVERSIFICATION.get(
                            "max_sector_weight", 0.25
                        )
                    )

                # --- Bornes min/max par position sur les actions seulement (cash exempt) ---
                cash_ticker = (self.config.CASH_BUFFER.get('ticker', 'CASH.TO')
                              if isinstance(self.config.CASH_BUFFER, dict) else 'CASH.TO')

                weights = enforce_weight_bounds(
                    weights,
                    min_weight=self.config.STOCK_SELECTION['min_weight'],
                    max_weight=self.config.STOCK_SELECTION['max_weight'],
                )

                # --- Contraintes mandat (crypto/exotiques) ---
                cst = self.config.CONSTRAINTS
                weights, compliance = data_loader.apply_allocation_constraints(
                    weights,
                    stock_alloc=float(stock_alloc),
                    bond_alloc=float(bond_alloc),
                    cash_alloc=float(self.config.CASH_BUFFER['percentage']) if isinstance(self.config.CASH_BUFFER, dict) else float(self.config.CASH_BUFFER),
                    allow_crypto=bool(cst.get('allow_crypto_etfs', True)),
                    max_exotic=float(cst.get('max_exotic_etfs_weight', 0.05)),
                    max_crypto=float(cst.get('max_crypto_etfs_weight', 0.03)),
                )

                # --- Finalisation stricte des poids (inclut bonds <= 10%, bonds >= 25%, somme = 1 exact) ---
                bond_set_full = set(data_loader.US_BOND_ETFS) | set(data_loader.CA_BOND_ETFS)
                cash_ticker = (self.config.CASH_BUFFER.get('ticker', 'CASH.TO')
                              if isinstance(self.config.CASH_BUFFER, dict) else 'CASH.TO')

                weights = finalize_weights_with_bond_floor(
                    weights=weights,
                    bond_set=bond_set_full,
                    cash_ticker=cash_ticker,
                    min_bond_alloc=float(self.config.MIN_BOND_ALLOC),
                    min_weight=float(self.config.STOCK_SELECTION['min_weight']),
                    max_weight=float(self.config.STOCK_SELECTION['max_weight']),
                    cash_target=float(self.config.CASH_BUFFER['percentage'])
                        if isinstance(self.config.CASH_BUFFER, dict)
                        else float(self.config.CASH_BUFFER),
                    decimals=6
                )

                # --- Turnover CAP (optionnel) ---
                cap = cst.get('turnover_cap', None)
                if cap is not None and prev_weights is not None:
                    raw_turn = (weights.reindex(prev_weights.index).fillna(0) - prev_weights).abs().sum()
                    if raw_turn > 0 and raw_turn > cap:
                        alpha = cap / raw_turn
                        weights = prev_weights.fillna(0) * (1 - alpha) + weights.fillna(0) * alpha

                # Re-calcul turnover final (après contraintes/cap)
                if prev_weights is not None:
                    turnover = (weights.reindex(prev_weights.index).fillna(0) - prev_weights).abs().sum()
                else:
                    turnover = 1.0

                # --- Transaction costs (bps) — par poche (stocks vs bonds) ---
                tc_cfg = self.config.TRANSACTION_COSTS
                stocks_bps = float(tc_cfg.get('stocks_bps', 0))
                bonds_bps  = float(tc_cfg.get('bonds_bps', 0))

                # Décomposer le turnover par poche pour appliquer le bon taux
                if prev_weights is not None:
                    delta = (weights.reindex(prev_weights.index).fillna(0) - prev_weights).abs()
                    bond_set_tc = set()
                    if hasattr(data_loader, 'US_BOND_ETFS'):
                        bond_set_tc |= set(data_loader.US_BOND_ETFS)
                    if hasattr(data_loader, 'CA_BOND_ETFS'):
                        bond_set_tc |= set(data_loader.CA_BOND_ETFS)
                    turnover_stocks = float(delta[[t for t in delta.index if t not in bond_set_tc]].sum())
                    turnover_bonds  = float(delta[[t for t in delta.index if t in bond_set_tc]].sum())
                else:
                    # Première allocation : turnover = 100% → ventiler par allocation
                    turnover_stocks = float(stock_alloc)
                    turnover_bonds  = float(bond_alloc)

                tc_cost_variable = (turnover_stocks * stocks_bps + turnover_bonds * bonds_bps) / 10000.0

                # Frais fixes par transaction (back office)
                fixed_cost_per_trade = float(tc_cfg.get('fixed_cost', 0))
                min_trade = self.config.TURNOVER_CONTROL.get('min_trade_size', 0.01)
                if prev_weights is not None:
                    weight_changes = (weights.reindex(prev_weights.index).fillna(0)
                                      - prev_weights).abs()
                    n_trades = int((weight_changes > min_trade).sum())
                else:
                    n_trades = len(weights[weights > 1e-6])
                tc_cost_fixed_cad = float(n_trades * fixed_cost_per_trade)

                # --- FX exposure ---
                fx_exposure = data_loader.get_fx_exposure_report(weights)

                # --- Historiques (1 entrée / rebalance) ---
                self.regime_history.append({'date': rebal_date_effective, 'regime': regime})

                self.regime_details_history.append({
                    'date': rebal_date_effective,
                    'regime': regime,
                    'confidence': float(regime_output.confidence),
                    'composite_score': float(regime_output.composite_score),
                    'multiplier': float(multiplier),
                    'stock_alloc': float(stock_alloc),
                    'bond_alloc': float(bond_alloc),
                    'cash_alloc': float(cash_alloc),
                    'turnover': float(turnover),
                    'tc_cost_variable': float(tc_cost_variable),
                    'compliance': compliance,
                })

                self.weights_history.append({
                    'date': rebal_date_effective,
                    'weights': weights,
                    'regime': regime,
                    'stock_alloc': float(stock_alloc),
                    'bond_alloc': float(bond_alloc),
                    'cash_alloc': float(cash_alloc),
                    'turnover': float(turnover),
                    'tc_cost_variable': float(tc_cost_variable),
                    'tc_cost_fixed_cad': tc_cost_fixed_cad,
                    'compliance': compliance,
                    'fx_exposure': fx_exposure,
                })

                self.fx_exposure_history.append({'date': rebal_date_effective, **fx_exposure})
                self.turnover_history.append({'date': rebal_date_effective, 'turnover': float(turnover)})

                prev_weights = weights.copy()

        #  Résumé des skips
        print(f"\nDiagnostic fenêtres:")
        for reason, count in skipped_windows.items():
            if count > 0:
                print(f"  {reason}: {count} skips")

        # Résumé optimisation
        if hasattr(optimizer, '_last_optim_summary') and optimizer._last_optim_summary:
            s = optimizer._last_optim_summary
            print(f"\nDernier rebalancement: {s['n_stocks']} stocks + {s['n_bonds']} bonds "
                  f"(Σ stocks = {s['w_stocks']:.2%}, Σ bonds = {s['w_bonds']:.2%})")

        # Diagnostic propre des dates de rebalancement
        dates = pd.to_datetime([x["date"] for x in self.weights_history if "date" in x])
        dup = dates[dates.duplicated()]
        print(f"Rebalances enregistrés: {len(dates)} | Dates dupliquées: {len(dup)}")
        if len(dup) > 0:
            print(" Attention: même date rebalancée plusieurs fois -> peut biaiser le backtest.")

        # Agrégation finale (stocks + bonds + cash)
        self.results = self._aggregate_results_with_cash_and_bonds(
            returns,
            data_loader.risk_free_rate_series,
            data_loader
        )
        return self.results

    def _generate_windows(self, start, end, train_months, test_months, step_months):
        """
        Génère les fenêtres walk-forward avec chevauchement contrôlé.

        Le step (3 mois) est inférieur au test (6 mois), ce qui crée un
        chevauchement de 3 mois entre fenêtres consécutives. C'est intentionnel :
        cela augmente le nombre d'observations tout en évitant les trous.
        Les dates déjà traitées sont filtrées dans run_walk_forward()
        via seen_rebalance_dates pour ne pas biaiser les résultats.
        """
        windows = []
        current = start + timedelta(days=train_months * 30)
        while current + timedelta(days=test_months * 30) <= end:
            train_start = current - timedelta(days=train_months * 30)
            train_end = current
            test_start = current
            test_end = min(current + timedelta(days=test_months * 30), end)
            windows.append((train_start, train_end, test_start, test_end))
            current += timedelta(days=step_months * 30)
        return windows

    def _get_rebalance_dates(self, index):
        freq = self.config.BACKTEST['rebalance_freq']
        if freq == 'daily':
            return index.tolist()
        elif freq == 'weekly':
            return [d for d in index if d.weekday() == self.config.BACKTEST['rebalance_day']]
        elif freq == 'monthly':
            return [d for d in index if d.day <= 5 and d.weekday() < 5]
        return index.tolist()

    def _aggregate_results_with_cash_and_bonds(self, returns, risk_free_series, data_loader):
        """Agrège: poche actions + poche bonds + poche cash (rf)."""
        if not self.weights_history:
            return pd.DataFrame()

        # ── BASELINE NAV BRUTE (avant frais de gestion) ──────────────────
        print("\n[BASELINE] Calcul de la NAV brute en cours (sans frais de gestion)...")

        results = []
        portfolio_value = self.config.PORTFOLIO_VALUE
        portfolio_value_gross = self.config.PORTFOLIO_VALUE   # NAV brute parallèle (identique avant modif)
        default_rf = self.config.RISK_FREE_RATE

        # --- Frais de gestion ---
        fee_cfg = getattr(self.config, 'MANAGEMENT_FEES', {})
        mgmt_fee_annual = fee_cfg.get('mgmt_fee_annual', 0.0080)
        hurdle_annual   = fee_cfg.get('hurdle_rate_annual', 0.12)
        perf_fee_rate   = fee_cfg.get('perf_fee_rate', 0.20)
        weekly_hurdle   = (1 + hurdle_annual) ** (1 / 52) - 1   # hurdle hebdomadaire

        nav_start_of_week = float(self.config.PORTFOLIO_VALUE)  # Référence de début de semaine
        total_mgmt_fees   = 0.0
        total_perf_fees   = 0.0

        bond_set = set()
        if hasattr(data_loader, "US_BOND_ETFS"):
            bond_set |= set(data_loader.US_BOND_ETFS)
        if hasattr(data_loader, "CA_BOND_ETFS"):
            bond_set |= set(data_loader.CA_BOND_ETFS)

        for i in range(len(self.weights_history)):
            entry = self.weights_history[i]
            date = entry['date']
            weights = entry['weights']
            stock_alloc = float(entry.get('stock_alloc', 0.0))
            bond_alloc  = float(entry.get('bond_alloc', 0.0))
            cash_alloc  = float(entry.get('cash_alloc', 0.0))

            next_date = self.weights_history[i + 1]['date'] if i < len(self.weights_history) - 1 else returns.index[-1]

            # IMPORTANT: .loc inclut les deux bornes -> on enlève le dernier point (next_date)
            # pour éviter que next_date apparaisse aussi au segment suivant.
            period_ret = returns.loc[date:next_date]
            if i < len(self.weights_history) - 1 and len(period_ret) > 0:
                period_ret = period_ret.iloc[:-1]

            for idx in period_ret.index:
                # --- split weights ---
                w = weights.reindex(period_ret.columns).fillna(0.0)

                bond_cols  = [c for c in w.index if (c in bond_set and w[c] > 0)]
                stock_cols = [c for c in w.index if (c not in bond_set and w[c] > 0)]

                # Actions — utiliser les poids tels quels de l'optimiseur conjoint
                stock_return = 0.0
                if stock_cols and stock_alloc > 0:
                    ws = w[stock_cols]
                    if ws.sum() > 0:
                        ws = ws / ws.sum() * stock_alloc
                    stock_return = float((period_ret.loc[idx, stock_cols] * ws).sum())

                # Bonds — utiliser les poids tels quels de l'optimiseur conjoint
                bond_return = 0.0
                cash_alloc_effective = None
                if bond_cols and bond_alloc > 0:
                    wb = w[bond_cols]
                    if wb.sum() > 0:
                        wb = wb / wb.sum() * bond_alloc
                    bond_return = float((period_ret.loc[idx, bond_cols] * wb).sum())
                else:
                    if bond_alloc > 0:
                        cash_alloc_effective = cash_alloc + bond_alloc

                # Cash
                if risk_free_series is not None and idx in risk_free_series.index:
                    daily_rf = float(risk_free_series.loc[idx]) / 252
                else:
                    daily_rf = float(default_rf) / 252

                if cash_alloc_effective is not None:
                    cash_return = float(cash_alloc_effective) * daily_rf
                else:
                    cash_return = float(cash_alloc) * daily_rf

                daily_ret_gross = stock_return + bond_return + cash_return

               # --- coûts de transaction : débités le premier jour de chaque période ---
                if len(period_ret.index) > 0 and idx == period_ret.index[0]:
                    tc_var = float(entry.get('tc_cost_variable', 0.0))
                    tc_fixed_cad = float(entry.get('tc_cost_fixed_cad', 0.0))
                    # Convertir les frais fixes (en $) en fraction du portefeuille courant
                    tc_fixed = tc_fixed_cad / portfolio_value if portfolio_value > 0 else 0.0
                    tc_cost = tc_var + tc_fixed
                    daily_ret_net = daily_ret_gross - tc_cost
                else:
                    tc_cost = 0.0
                    daily_ret_net = daily_ret_gross

                portfolio_value       *= (1 + daily_ret_net)
                portfolio_value_gross *= (1 + daily_ret_net)  # NAV brute (après TC, avant frais de gestion)

                # --- Frais de gestion (chaque lundi) ---
                mgmt_fee_amount = 0.0
                perf_fee_amount = 0.0
                if hasattr(idx, 'weekday') and idx.weekday() == 0:  # Lundi
                    # 1. Frais de gestion hebdomadaires
                    mgmt_fee_amount = portfolio_value * (mgmt_fee_annual / 52)

                    # 2. Commission de performance: surperformance au-dessus du hurdle hebdo
                    weekly_gross_ret = (
                        (portfolio_value / nav_start_of_week) - 1.0
                        if nav_start_of_week > 0 else 0.0
                    )
                    if weekly_gross_ret > weekly_hurdle:
                        excess = weekly_gross_ret - weekly_hurdle
                        perf_fee_amount = excess * nav_start_of_week * perf_fee_rate

                    # Déduire les frais de la NAV nette
                    portfolio_value -= mgmt_fee_amount + perf_fee_amount
                    total_mgmt_fees += mgmt_fee_amount
                    total_perf_fees += perf_fee_amount

                    # Nouveau point de départ pour la semaine suivante
                    nav_start_of_week = portfolio_value

                results.append({
                    'date': idx,
                    'portfolio_value': portfolio_value,          # NAV nette (après frais gestion)
                    'portfolio_value_gross': portfolio_value_gross,  # NAV brute (avant frais gestion)
                    'return_gross': daily_ret_gross,
                    'return': daily_ret_net,
                    'tc_cost': tc_cost,
                    'mgmt_fee': mgmt_fee_amount,
                    'perf_fee': perf_fee_amount,
                    'stock_return': stock_return,
                    'bond_return': bond_return,
                    'cash_return': cash_return,
                    'regime': entry['regime'],
                    'stock_alloc': stock_alloc,
                    'bond_alloc': bond_alloc,
                    'cash_alloc': cash_alloc,
                })

        df = pd.DataFrame(results)

        # Robustesse si aucun résultat
        if df.empty:
            return df

        # ── RAPPORT FRAIS DE GESTION ─────────────────────────────────────
        if len(df) > 0 and 'portfolio_value_gross' in df.columns:
            nav_brute_finale  = float(df['portfolio_value_gross'].iloc[-1])
            nav_nette_finale  = float(df['portfolio_value'].iloc[-1])
            initial_nav       = float(self.config.PORTFOLIO_VALUE)
            n_years           = max((df.index[-1] - df.index[0]).days / 365.25, 1e-9) if hasattr(df.index[0], 'year') else (len(df) / 252)

            total_fees_usd    = total_mgmt_fees + total_perf_fees
            total_fees_pct_ann = (total_fees_usd / initial_nav) / max(n_years, 1e-9)

            print("\n" + "=" * 70)
            print("RAPPORT FRAIS DE GESTION — BACKTEST")
            print("=" * 70)
            print(f"  NAV brute finale  : ${nav_brute_finale:>18,.2f}  (identique à avant la modif)")
            print(f"  NAV nette finale  : ${nav_nette_finale:>18,.2f}")
            print(f"  Écart (drag fees) : ${nav_brute_finale - nav_nette_finale:>18,.2f}")
            print(f"  ─────────────────────────────────────────────────────")
            print(f"  Frais de gestion  : ${total_mgmt_fees:>18,.2f}")
            print(f"  Commissions perf  : ${total_perf_fees:>18,.2f}")
            print(f"  Revenus gestionnaire: ${total_fees_usd:>16,.2f}  ({total_fees_pct_ann:.3%} ann.)")
            print(f"  ─────────────────────────────────────────────────────")

            # Vérification: NAV nette < NAV brute à CHAQUE date
            violations = df[df['portfolio_value'] > df['portfolio_value_gross'] + 1e-2]
            if len(violations) == 0:
                print(f"  [OK] NAV nette < NAV brute à chaque date ({len(df)} points)")
            else:
                print(f"  [ERREUR] {len(violations)} dates où NAV nette >= NAV brute:")
                print(violations[['portfolio_value', 'portfolio_value_gross']].head(5))
            print("=" * 70)

        df["date"] = pd.to_datetime(df["date"])

        df = (
            df.dropna(subset=["date"])
              .drop_duplicates(subset=["date"], keep="last")
              .sort_values("date")
              .set_index("date")
        )

        # Diagnostics anti-biais (très utile pour les graphiques)
        if not df.index.is_monotonic_increasing:
            df = df.sort_index()

        if df.index.duplicated().any():
            # En principe, le drop_duplicates ci-dessus l'enlève déjà
            df = df[~df.index.duplicated(keep="last")]

        return df


print("[OK] Module RollingBacktester V3 chargé (cash + bonds + FIX KeyError + FIX chevauchements)")


[OK] Module RollingBacktester V3 chargé (cash + bonds + FIX KeyError + FIX chevauchements)


---
# 7. Performance — Analyse et métriques de risque
---


In [9]:
# =============================================================================
# CELLULE 7: PERFORMANCE ANALYZER - FOCUS GESTION DU RISQUE
# =============================================================================

class PerformanceAnalyzer:
    """Analyse de performance focalisée sur la gestion du risque."""

    def __init__(self, config=Config):
        self.config = config

    def full_analysis(self, results, benchmark=None, regime_history=None,
                      fx_history=None, turnover_history=None):
        if len(results) == 0:
            return {'error': 'Pas de résultats'}

        returns = results['return']
        returns_gross = results['return_gross'] if 'return_gross' in results.columns else None
        portfolio_value = results['portfolio_value']

        metrics = self._calculate_metrics(returns, portfolio_value)
        if returns_gross is not None:
            metrics['gross'] = self._calculate_metrics(returns_gross.dropna(), portfolio_value)
            metrics['net_vs_gross'] = {
                'total_cost_estimated': float(results['tc_cost'].fillna(0).sum()) if 'tc_cost' in results.columns else None
            }

        if benchmark is not None and len(benchmark) > 0:
            benchmark_returns = benchmark.pct_change().dropna()
            common_idx = returns.index.intersection(benchmark_returns.index)
            if len(common_idx) > 50:
                metrics['benchmark'] = self._benchmark_analysis(
                    returns.loc[common_idx],
                    benchmark_returns.loc[common_idx]
                )

        if regime_history:
            reg_df = pd.DataFrame(regime_history)

            # Sécuriser + normaliser dates
            reg_df = reg_df.dropna(subset=["date", "regime"]).copy()
            reg_df["date"] = pd.to_datetime(reg_df["date"]).dt.tz_localize(None)

            # Trier + enlever doublons (garder le dernier régime observé pour une date)
            reg_df = reg_df.sort_values("date")
            reg_df = reg_df.drop_duplicates(subset="date", keep="last")

            # Série monotone (exigée par reindex(method='ffill'))
            regimes = pd.Series(reg_df["regime"].values, index=reg_df["date"]).sort_index()

            spy_returns = benchmark.pct_change().dropna() if benchmark is not None else None
            metrics["regime_risk"] = self._regime_risk_analysis(returns, regimes, spy_returns)


        if fx_history:
            metrics['fx_analysis'] = self._fx_analysis(fx_history)

        if turnover_history:
            metrics['turnover_analysis'] = self._turnover_analysis(turnover_history)

        return metrics

    def _calculate_metrics(self, returns, portfolio_value):
        rf = self.config.RISK_FREE_RATE

        total_return = (portfolio_value.iloc[-1] / portfolio_value.iloc[0]) - 1
        n_years = len(returns) / 252
        annual_return = (1 + total_return) ** (1 / n_years) - 1 if n_years > 0 else 0

        vol = returns.std() * np.sqrt(252)

        cumulative = (1 + returns).cumprod()
        rolling_max = cumulative.cummax()
        drawdown = (cumulative - rolling_max) / rolling_max
        max_dd = drawdown.min()

        excess_return = annual_return - rf
        sharpe = excess_return / vol if vol > 0 else 0

        downside_returns = returns[returns < 0]
        downside_vol = downside_returns.std() * np.sqrt(252) if len(downside_returns) > 0 else vol
        sortino = excess_return / downside_vol if downside_vol > 0 else 0

        calmar = annual_return / abs(max_dd) if max_dd != 0 else 0

        var_95 = returns.quantile(0.05)
        cvar_95 = returns[returns <= var_95].mean()

        return {
            'total_return': total_return,
            'annual_return': annual_return,
            'volatility': vol,
            'sharpe_ratio': sharpe,
            'sortino_ratio': sortino,
            'calmar_ratio': calmar,
            'max_drawdown': max_dd,
            'var_95': var_95,
            'cvar_95': cvar_95,
            'win_rate': (returns > 0).mean(),
            'n_days': len(returns),
        }

    def _benchmark_analysis(self, port_returns, bench_returns):
        """Analyse vs benchmark avec Tracking Error et Information Ratio."""

        # Capture ratio (existant)
        down_days = bench_returns < 0
        up_days = bench_returns > 0

        downside_capture = (port_returns[down_days].mean() / bench_returns[down_days].mean()) * 100 if down_days.sum() > 0 else np.nan
        upside_capture = (port_returns[up_days].mean() / bench_returns[up_days].mean()) * 100 if up_days.sum() > 0 else np.nan

        # Régression CAPM
        X = sm.add_constant(bench_returns)
        model = sm.OLS(port_returns, X).fit()

        # === NOUVEAU: Tracking Error et Information Ratio ===
        # Tracking Error = écart-type des rendements actifs (différence port - bench)
        active_returns = port_returns - bench_returns
        tracking_error_daily = active_returns.std()
        tracking_error_annual = tracking_error_daily * np.sqrt(252)

        # Information Ratio = Alpha / Tracking Error
        # Utilise le rendement actif moyen annualisé
        active_return_annual = active_returns.mean() * 252
        information_ratio = active_return_annual / tracking_error_annual if tracking_error_annual > 0 else 0

        # Corrélation avec le benchmark
        correlation = port_returns.corr(bench_returns)

        # Active Share approximation (basé sur R²)
        # Active Share ≈ sqrt(1 - R²) pour une approximation simple
        active_share_approx = np.sqrt(1 - model.rsquared) if model.rsquared < 1 else 0

        return {
            'alpha_annual': model.params.iloc[0] * 252,
            'alpha_tstat': model.tvalues.iloc[0],
            'alpha_pvalue': model.pvalues.iloc[0],
            'beta': model.params.iloc[1],
            'beta_tstat': model.tvalues.iloc[1],
            'r_squared': model.rsquared,
            'downside_capture': downside_capture,
            'upside_capture': upside_capture,
            'tracking_error_daily': tracking_error_daily,
            'tracking_error_annual': tracking_error_annual,
            'information_ratio': information_ratio,
            'active_return_annual': active_return_annual,
            'correlation': correlation,
            'active_share_approx': active_share_approx,
        }

    def _regime_risk_analysis(self, returns, regimes, benchmark_returns=None):
        """Analyse de l'impact du régime sur le RISQUE."""

        weekly_returns = (1 + returns).resample('W-FRI').prod() - 1
        regimes = regimes[~regimes.index.duplicated(keep="last")].sort_index()
        reg_series = regimes.reindex(weekly_returns.index, method='ffill').shift(1)

        common_idx = weekly_returns.index.intersection(reg_series.dropna().index)
        ret_weekly = weekly_returns.loc[common_idx]
        reg_weekly = reg_series.loc[common_idx]

        if benchmark_returns is not None:
            bench_weekly = (1 + benchmark_returns).resample('W-FRI').prod() - 1
            bench_weekly = bench_weekly.reindex(common_idx)
        else:
            bench_weekly = None

        results = {'n_weeks': len(ret_weekly)}

        risk_by_regime = {}
        for regime in ["RISK_OFF", "RISK_ON"]:
            mask = reg_weekly == regime
            r = ret_weekly[mask]

            if len(r) < 5:
                continue

            vol_weekly = r.std()
            var_95 = r.quantile(0.05)
            cvar_95 = r[r <= var_95].mean() if len(r[r <= var_95]) > 0 else var_95

            # Calcul explicite des semaines extrêmes
            n_below_4 = (r < -0.04).sum()
            n_below_5 = (r < -0.05).sum()
            pct_below_4 = n_below_4 / len(r) * 100
            pct_below_5 = n_below_5 / len(r) * 100

            if bench_weekly is not None:
                bench_r = bench_weekly[mask]
                down_weeks = bench_r < 0
                downside_capture = (r[down_weeks].mean() / bench_r[down_weeks].mean()) * 100 if down_weeks.sum() > 0 else np.nan
                up_weeks = bench_r > 0
                upside_capture = (r[up_weeks].mean() / bench_r[up_weeks].mean()) * 100 if up_weeks.sum() > 0 else np.nan
            else:
                downside_capture, upside_capture = np.nan, np.nan

            risk_by_regime[regime] = {
                'count': len(r),
                'mean_weekly': r.mean(),
                'mean_annual': r.mean() * 52,
                'vol_weekly': vol_weekly,
                'vol_annual': vol_weekly * np.sqrt(52),
                'var_95': var_95,
                'cvar_95': cvar_95,
                'worst_week': r.min(),
                'best_week': r.max(),
                'n_below_minus4': n_below_4,
                'n_below_minus5': n_below_5,
                'pct_below_minus4': pct_below_4,
                'pct_below_minus5': pct_below_5,
                'downside_capture': downside_capture,
                'upside_capture': upside_capture,
            }

        results['risk_by_regime'] = risk_by_regime

        # Test: régime prédit la volatilité future
        try:
            future_vol = ret_weekly.rolling(4).std().shift(-4) * np.sqrt(52)
            vol_df = pd.DataFrame({'future_vol': future_vol, 'regime': reg_weekly}).dropna()

            if len(vol_df) > 20:
                vol_df['is_risk_on'] = (vol_df['regime'] == 'RISK_ON').astype(float)
                X = sm.add_constant(vol_df['is_risk_on'])
                model = sm.OLS(vol_df['future_vol'], X).fit(cov_type='HAC', cov_kwds={'maxlags': 8})

                results['vol_prediction'] = {
                    'coef_risk_on': model.params.get('is_risk_on', np.nan),
                    'pvalue': model.pvalues.get('is_risk_on', np.nan),
                    'vol_risk_on': vol_df[vol_df['regime'] == 'RISK_ON']['future_vol'].mean(),
                    'vol_risk_off': vol_df[vol_df['regime'] == 'RISK_OFF']['future_vol'].mean(),
                }
        except (ValueError, KeyError, Exception) as e:
            print(f"[WARN] {e}")

        # Test: régime prédit |retour|
        try:
            abs_ret = ret_weekly.abs()
            vol_df = pd.DataFrame({'abs_ret': abs_ret, 'regime': reg_weekly}).dropna()

            if len(vol_df) > 20:
                vol_df['is_risk_on'] = (vol_df['regime'] == 'RISK_ON').astype(float)
                X = sm.add_constant(vol_df['is_risk_on'])
                model = sm.OLS(vol_df['abs_ret'], X).fit(cov_type='HAC', cov_kwds={'maxlags': 8})

                results['abs_ret_prediction'] = {
                    'coef_risk_on': model.params.get('is_risk_on', np.nan),
                    'pvalue': model.pvalues.get('is_risk_on', np.nan),
                    'mean_abs_risk_on': vol_df[vol_df['regime'] == 'RISK_ON']['abs_ret'].mean(),
                    'mean_abs_risk_off': vol_df[vol_df['regime'] == 'RISK_OFF']['abs_ret'].mean(),
                }
        except (ValueError, KeyError, Exception) as e:
            print(f"[WARN] {e}")

        return results

    def _fx_analysis(self, fx_history):
        df = pd.DataFrame(fx_history)
        return {'avg_usd': df['usd_pct'].mean(), 'avg_cad': df['cad_pct'].mean()}

    def _turnover_analysis(self, turnover_history):
        turnovers = [t['turnover'] for t in turnover_history]
        return {
            'mean': np.mean(turnovers),
            'annualized': np.mean(turnovers) * 52,
            'cost_annual': np.mean(turnovers) * 52 * 0.001,
        }

    def generate_report(self, metrics):
        report = []
        report.append("=" * 70)
        report.append("RAPPORT DE PERFORMANCE - GSF-3120 V4")
        report.append("=" * 70)

        report.append(f"\n RENDEMENTS")
        report.append(f"  Rendement total:      {metrics['total_return']:.2%}")
        report.append(f"  Rendement annualisé:  {metrics['annual_return']:.2%}")

        report.append(f"\n RISQUE")
        report.append(f"  Volatilité:           {metrics['volatility']:.2%}")
        report.append(f"  Max Drawdown:         {metrics['max_drawdown']:.2%}")
        report.append(f"  VaR (95%) [daily]:    {metrics['var_95']:.2%}")
        report.append(f"  CVaR (95%) [daily]:   {metrics['cvar_95']:.2%}")

        report.append(f"\n RATIOS")
        report.append(f"  Sharpe Ratio:         {metrics['sharpe_ratio']:.2f}")
        report.append(f"  Sortino Ratio:        {metrics['sortino_ratio']:.2f}")
        report.append(f"  Calmar Ratio:         {metrics['calmar_ratio']:.2f}")

        if 'benchmark' in metrics:
            b = metrics['benchmark']
            report.append(f"\n VS BENCHMARK")
            report.append(f"  Alpha (annualisé):    {b['alpha_annual']:.2%} (p={b.get('alpha_pvalue', 0):.3f})")
            report.append(f"  Beta:                 {b['beta']:.2f}")
            report.append(f"  R²:                   {b['r_squared']:.2%}")
            report.append(f"  Corrélation:          {b.get('correlation', 0):.2%}")
            report.append(f"  Tracking Error (ann): {b.get('tracking_error_annual', 0):.2%}")
            report.append(f"  Information Ratio:    {b.get('information_ratio', 0):.2f}")
            report.append(f"  Downside Capture:     {b['downside_capture']:.0f}%")
            report.append(f"  Upside Capture:       {b['upside_capture']:.0f}%")

        if 'regime_risk' in metrics:
            rr = metrics['regime_risk']
            report.append(f"\n RISQUE PAR RÉGIME (Hebdo)")

            if 'risk_by_regime' in rr:
                for regime in ["RISK_OFF", "RISK_ON"]:
                    if regime in rr['risk_by_regime']:
                        r = rr['risk_by_regime'][regime]
                        report.append(f"  {regime}: {r['count']}sem | Vol:{r['vol_annual']*100:.1f}% | CVaR:{r['cvar_95']*100:.2f}%")

            if 'vol_prediction' in rr:
                vp = rr['vol_prediction']
                report.append(f"\n   Prédiction Vol: p={vp['pvalue']:.4f}")
                if vp['pvalue'] < 0.05:
                    report.append(f"   Régime prédit la volatilité!")

        if 'turnover_analysis' in metrics:
            to = metrics['turnover_analysis']
            report.append(f"\n TURNOVER: {to['annualized']:.0%} ann. | Coût: {to['cost_annual']:.2%}")

        report.append("\n" + "=" * 70)
        return "\n".join(report)

print("[OK] Module PerformanceAnalyzer V4 chargé")

[OK] Module PerformanceAnalyzer V4 chargé


---
# 7.1. Stress Testing — Périodes historiques de crise
---


In [10]:
# =============================================================================
# CELLULE 7.1: STRESS TESTING FORMEL
# =============================================================================

class StressTestAnalyzer:
    """
    Analyse de stress testing sur des périodes historiques spécifiques.
    """

    # Périodes de stress historiques
    STRESS_PERIODS = {
        'COVID_Crash': {
            'start': '2020-02-19',
            'end': '2020-03-23',
            'description': 'COVID-19 market crash',
            'benchmark_drop': -0.34,  # SPY drop approximatif
        },
        'COVID_Recovery': {
            'start': '2020-03-24',
            'end': '2020-08-31',
            'description': 'Post-COVID recovery rally',
            'benchmark_return': 0.55,
        },
        'Rate_Hike_2022': {
            'start': '2022-01-03',
            'end': '2022-10-12',
            'description': '2022 Fed rate hike bear market',
            'benchmark_drop': -0.25,
        },
        'Banking_Crisis_2023': {
            'start': '2023-03-08',
            'end': '2023-03-24',
            'description': 'SVB/Regional banking crisis',
            'benchmark_drop': -0.05,
        },
        'Tech_Correction_2024': {
            'start': '2024-07-10',
            'end': '2024-08-05',
            'description': 'Summer 2024 tech correction',
            'benchmark_drop': -0.08,
        },
    }

    def __init__(self, config=Config):
        self.config = config

    def analyze_stress_periods(self, results, benchmark=None):
        """
        Analyse la performance du portefeuille pendant chaque période de stress.

        Returns:
            dict: Résultats par période de stress
        """
        stress_results = {}

        for period_name, period_info in self.STRESS_PERIODS.items():
            start = pd.to_datetime(period_info['start'])
            end = pd.to_datetime(period_info['end'])

            # Vérifier si la période est dans nos données
            if start < results.index.min() or end > results.index.max():
                continue

            # Extraire les données de la période
            period_data = results.loc[start:end]

            if len(period_data) < 5:
                continue

            # Calculer les métriques
            period_returns = period_data['return']
            cumulative_return = (1 + period_returns).prod() - 1
            max_dd = self._calculate_max_drawdown(period_returns)
            volatility = period_returns.std() * np.sqrt(252)

            # Benchmark comparison
            if benchmark is not None:
                bench_period = benchmark.loc[start:end]
                if len(bench_period) > 1:
                    bench_return = (bench_period.iloc[-1] / bench_period.iloc[0]) - 1
                    relative_perf = cumulative_return - bench_return
                else:
                    bench_return = np.nan
                    relative_perf = np.nan
            else:
                bench_return = period_info.get('benchmark_drop', np.nan)
                relative_perf = cumulative_return - bench_return if not np.isnan(bench_return) else np.nan

            stress_results[period_name] = {
                'description': period_info['description'],
                'start': start,
                'end': end,
                'n_days': len(period_data),
                'portfolio_return': cumulative_return,
                'benchmark_return': bench_return,
                'relative_performance': relative_perf,
                'max_drawdown': max_dd,
                'volatility_annual': volatility,
                'worst_day': period_returns.min(),
                'best_day': period_returns.max(),
            }

        return stress_results

    def _calculate_max_drawdown(self, returns):
        """Calcule le max drawdown sur une série de rendements."""
        cumulative = (1 + returns).cumprod()
        rolling_max = cumulative.cummax()
        drawdown = (cumulative - rolling_max) / rolling_max
        return drawdown.min()

    def run_hypothetical_scenarios(self, results, weights_history):
        """
        Simule des scénarios hypothétiques de stress.

        Scénarios:
        - Market crash (-20% sur 1 mois)
        - Volatility spike (VIX +100%)
        - Yield curve inversion
        - Credit spread widening
        """
        scenarios = {}

        # Obtenir les poids actuels
        if not weights_history:
            return scenarios

        current_weights = weights_history[-1]['weights']
        current_weights = current_weights[current_weights > 0]

        # Scénario 1: Market Crash uniforme
        scenarios['uniform_crash_10pct'] = {
            'description': 'Crash uniforme de -10%',
            'portfolio_impact': -0.10,  # Simplifié
            'assumption': 'Tous les actifs baissent de 10%',
        }

        scenarios['uniform_crash_20pct'] = {
            'description': 'Crash uniforme de -20%',
            'portfolio_impact': -0.20,
            'assumption': 'Tous les actifs baissent de 20%',
        }

        # Scénario 2: Rotation sectorielle (tech crash)
        # Identifier les positions tech
        tech_tickers = ['AAPL', 'MSFT', 'GOOGL', 'GOOG', 'META', 'NVDA', 'AMD', 'AMZN',
                       'TSLA', 'NFLX', 'CRM', 'ADBE', 'NOW', 'SNOW', 'NET']
        tech_weight = current_weights[current_weights.index.isin(tech_tickers)].sum()

        scenarios['tech_crash_30pct'] = {
            'description': 'Tech sector crash -30%',
            'tech_exposure': tech_weight,
            'portfolio_impact': -tech_weight * 0.30,
            'assumption': 'Secteur tech baisse de 30%, reste stable',
        }

        # Scénario 3: Flight to quality (bonds up, stocks down)
        scenarios['flight_to_quality'] = {
            'description': 'Flight to quality',
            'assumption': 'Actions -15%, Obligations +5%',
            'stock_alloc': weights_history[-1].get('stock_alloc', 0.875),
            'bond_alloc': weights_history[-1].get('bond_alloc', 0.10),
        }

        s = scenarios['flight_to_quality']
        s['portfolio_impact'] = s['stock_alloc'] * (-0.15) + s['bond_alloc'] * 0.05

        return scenarios

    def generate_stress_report(self, stress_results, scenarios):
        """Génère un rapport de stress testing."""
        report = []
        report.append("=" * 70)
        report.append("RAPPORT DE STRESS TESTING")
        report.append("=" * 70)

        report.append("\n📉 PÉRIODES HISTORIQUES DE STRESS")
        report.append("-" * 70)

        for period_name, data in stress_results.items():
            report.append(f"\n{period_name}: {data['description']}")
            report.append(f"  Période: {data['start'].strftime('%Y-%m-%d')} → {data['end'].strftime('%Y-%m-%d')} ({data['n_days']} jours)")
            report.append(f"  Rendement Portfolio: {data['portfolio_return']:.2%}")
            report.append(f"  Rendement Benchmark: {data['benchmark_return']:.2%}")
            report.append(f"  Performance Relative: {data['relative_performance']:+.2%}")
            report.append(f"  Max Drawdown: {data['max_drawdown']:.2%}")
            report.append(f"  Pire journée: {data['worst_day']:.2%}")

        report.append("\n" + "=" * 70)
        report.append(" SCÉNARIOS HYPOTHÉTIQUES")
        report.append("-" * 70)

        for scenario_name, data in scenarios.items():
            report.append(f"\n{scenario_name}: {data['description']}")
            report.append(f"  Hypothèse: {data['assumption']}")
            report.append(f"  Impact estimé: {data['portfolio_impact']:.2%}")

        report.append("\n" + "=" * 70)

        return "\n".join(report)

print("[OK] Module StressTestAnalyzer chargé")

[OK] Module StressTestAnalyzer chargé


---
# 8. Visualisation — Dashboard et graphiques
---


In [11]:
# =============================================================================
# CELLULE 8: VISUALISATION
# =============================================================================

class Visualizer:
    """Graphiques de performance."""

    @staticmethod
    def create_dashboard(results, benchmark=None, regime_history=None, fx_history=None):
        results = results.copy()
        results.index = pd.to_datetime(results.index)

        results = (
            results[~results.index.duplicated(keep="last")]
            .sort_index()
        )

        fig = make_subplots(
            rows=3, cols=2,
            subplot_titles=(
                'Performance Cumulative', 'Rolling Sharpe (6M)',
                'Drawdown', 'Distribution des Rendements',
                'Exposition FX', 'Allocation Actions/Cash'
            ),
            vertical_spacing=0.08,
            horizontal_spacing=0.08
        )

        if len(results) == 0:
            return fig

        returns = results['return']
        portfolio_value = results['portfolio_value']

        # 1. Performance cumulative
        norm_port = portfolio_value / portfolio_value.iloc[0] * 100
        fig.add_trace(
            go.Scatter(x=norm_port.index, y=norm_port.values, name='Portefeuille V3', line=dict(color='blue', width=2)),
            row=1, col=1
        )

        # Aligner et tracer le benchmark si disponible
        if benchmark is not None:
            benchmark = benchmark.copy()
            benchmark.index = pd.to_datetime(benchmark.index)
            benchmark = benchmark[~benchmark.index.duplicated(keep="last")].sort_index()

            bench_aligned = benchmark.reindex(norm_port.index).ffill()


        # 2. Rolling Sharpe
        rolling_sharpe = returns.rolling(126).mean() / returns.rolling(126).std() * np.sqrt(252)
        fig.add_trace(
            go.Scatter(x=rolling_sharpe.index, y=rolling_sharpe.values, name='Sharpe 6M', line=dict(color='green')),
            row=1, col=2
        )
        fig.add_hline(y=1.0, line_dash="dash", line_color="red", row=1, col=2)
        fig.add_hline(y=0.5, line_dash="dot", line_color="orange", row=1, col=2)

        # 3. Drawdown
        cumulative = (1 + returns).cumprod()
        drawdown = (cumulative - cumulative.cummax()) / cumulative.cummax() * 100
        fig.add_trace(
            go.Scatter(x=drawdown.index, y=drawdown.values, fill='tozeroy', name='Drawdown', line=dict(color='red')),
            row=2, col=1
        )

        # 4. Distribution
        fig.add_trace(
            go.Histogram(x=returns.values * 100, nbinsx=50, name='Rendements (%)', marker_color='blue'),
            row=2, col=2
        )

        # 5. Exposition FX
        if fx_history:
            fx_df = pd.DataFrame(fx_history).copy()

            # sécuriser la colonne date
            fx_df["date"] = pd.to_datetime(fx_df["date"])
            fx_df = (
                fx_df.dropna(subset=["date"])
                    .drop_duplicates(subset=["date"], keep="last")
                    .set_index("date")
                    .sort_index()
            )

            fig.add_trace(
                go.Scatter(x=fx_df.index, y=fx_df['usd_pct'] * 100, name='USD %', line=dict(color='green')),
                row=3, col=1
            )
            fig.add_trace(
                go.Scatter(x=fx_df.index, y=fx_df['cad_pct'] * 100, name='CAD %', line=dict(color='red')),
                row=3, col=1
            )


        # 6. Allocation
        if 'stock_alloc' in results.columns:
            fig.add_trace(
                go.Scatter(x=results.index, y=results['stock_alloc'] * 100, name='Actions %', line=dict(color='blue')),
                row=3, col=2
            )
            fig.add_trace(
                go.Scatter(x=results.index, y=results['cash_alloc'] * 100, name='Cash %', line=dict(color='orange')),
                row=3, col=2
            )

        fig.update_layout(
            height=900,
            showlegend=True,
            title_text="Dashboard GSF-3120 V3 (avec Cash Buffer)",
            template='plotly_white'
        )

        return fig

print("[OK] Module Visualizer chargé")


[OK] Module Visualizer chargé


In [14]:
# =============================================================================
# CELLULE : VALIDATION OUT-OF-SAMPLE PURE
# =============================================================================
# Test le plus rigoureux contre la suroptimisation :
#   - FIGER tous les hyperparametres sur la periode IN-SAMPLE (2012-2019)
#   - TESTER sur la periode OUT-OF-SAMPLE (2020-2026) sans re-entrainement
#   - Comparer les metriques IS vs OOS pour detecter la degradation
#
# Methodologie :
#   Le walk-forward "normal" valide les decisions LOCALES (selection de titres,
#   timing des regimes) fenetre par fenetre. Mais les HYPERPARAMETRES globaux
#   (poids factoriels, seuils de regime, parametres RMOM) sont fixes sur toute
#   la periode — ils ont potentiellement vu l'OOS implicitement.
#
#   Ce test resout ce probleme : on execute deux backtests independants avec
#   les MEMES hyperparametres (Config inchange), mais sur des periodes
#   separees. Si les metriques sont stables, les hyperparametres ne sont pas
#   suroptimises.
#
# Interpretation :
#   - Sharpe OOS >= 70% du Sharpe IS → CONFIRME (pas de suroptimisation)
#   - Sharpe OOS entre 50-70% du Sharpe IS → ACCEPTABLE (degradation moderee)
#   - Sharpe OOS < 50% du Sharpe IS → ATTENTION (suroptimisation possible)
#
# Reference : Bailey, Borwein, Lopez de Prado & Zhu (2017),
#             "The Probability of Backtest Overfitting"
#
# Pattern identique aux validations precedentes (cellules 16-20) :
#   - Donnees chargees une seule fois
#   - Copies isolees de Config via _copy_config()
#   - Config global inchange a la sortie
#
# Temps d'execution : ~10-15 minutes (2 backtests complets)
# =============================================================================

import copy
import types
import time

def run_oos_validation():
    """
    Validation Out-of-Sample pure.

    Split temporel :
      - IN-SAMPLE  : 2010-01-01 → 2019-12-31 (calibration)
      - OUT-OF-SAMPLE : 2020-01-01 → 2026-12-22 (test)

    Les hyperparametres (poids factoriels, seuils regime, RMOM) sont
    IDENTIQUES sur les deux periodes — c'est le point cle du test.
    """

    OOS_SPLIT_DATE = datetime(2020, 1, 1)

    print("\n" + "=" * 70)
    print("VALIDATION OUT-OF-SAMPLE PURE")
    print("=" * 70)
    print(f"Split : IN-SAMPLE avant {OOS_SPLIT_DATE.strftime('%Y-%m-%d')}")
    print(f"         OUT-OF-SAMPLE apres {OOS_SPLIT_DATE.strftime('%Y-%m-%d')}")
    print(f"Hyperparametres : FIGES (Config inchange entre IS et OOS)")
    print("=" * 70)

    # ── 1. Charger les donnees une seule fois ────────────────────────────
    print("\n[1/4] Chargement des donnees...")
    data_loader = DataLoader()
    prices, returns = data_loader.load_data(backtest_mode=True)

    vix_data = data_loader.load_vix(
        Config.BACKTEST['start_date'] - timedelta(days=365),
        Config.BACKTEST['end_date']
    )
    data_loader.load_macro_data(
        Config.BACKTEST['start_date'] - timedelta(days=365),
        Config.BACKTEST['end_date']
    )
    print(f"  Donnees : {prices.shape[0]} jours x {prices.shape[1]} tickers")

    # ── 2. Creer les configs IS et OOS ───────────────────────────────────
    # Memes hyperparametres, seules les dates changent
    def _make_period_config(start_date, end_date):
        """Copie isolee de Config avec dates modifiees."""
        ns = types.SimpleNamespace()
        for attr in dir(Config):
            if attr.startswith('__'):
                continue
            val = getattr(Config, attr)
            if callable(val) and not isinstance(val, (dict, list, tuple)):
                continue
            try:
                setattr(ns, attr, copy.deepcopy(val))
            except Exception:
                setattr(ns, attr, val)

        # Modifier uniquement les dates
        ns.BACKTEST = copy.deepcopy(Config.BACKTEST)
        ns.BACKTEST['start_date'] = start_date
        ns.BACKTEST['end_date'] = end_date

        # Desactiver liquidite pour accelerer (comme les autres validations)
        ns.LIQUIDITY = copy.deepcopy(Config.LIQUIDITY)
        ns.LIQUIDITY['enabled'] = False

        return ns

    cfg_is = _make_period_config(
        Config.BACKTEST['start_date'],  # 2010-01-01
        OOS_SPLIT_DATE                   # 2020-01-01
    )
    cfg_oos = _make_period_config(
        OOS_SPLIT_DATE - timedelta(days=24 * 30),  # 2018-01 (besoin de 24m de training)
        Config.BACKTEST['end_date']                  # 2026-12-22
    )

    # ── 3. Executer les deux backtests ───────────────────────────────────
    results_dict = {}

    for label, cfg in [("IN-SAMPLE (2012-2019)", cfg_is),
                        ("OUT-OF-SAMPLE (2020-2026)", cfg_oos)]:

        print(f"\n{'─' * 60}")
        print(f"  BACKTEST — {label}")
        print(f"  Periode : {cfg.BACKTEST['start_date'].strftime('%Y-%m-%d')} → "
              f"{cfg.BACKTEST['end_date'].strftime('%Y-%m-%d')}")
        print(f"  Hyperparametres : IDENTIQUES au modele principal")
        print(f"{'─' * 60}")

        t0 = time.time()

        factor_calc = FactorCalculator(cfg)
        risk_mgr = RiskManager(cfg)
        optimizer = PortfolioOptimizer(cfg)
        backtester = RollingBacktester(cfg)

        results = backtester.run_walk_forward(
            prices, returns, factor_calc, risk_mgr, optimizer,
            data_loader, vix_data, overlay_enabled=True
        )

        elapsed = time.time() - t0
        print(f"\n  [OK] {label} — {len(results)} jours en {elapsed:.0f}s")

        if results.empty:
            print(f"  [ERREUR] Aucun resultat genere pour {label}")
            results_dict[label] = None
            continue

        # ── Calcul des metriques ─────────────────────────────────────────
        ret = results['return']
        pv = results['portfolio_value']
        rf = Config.RISK_FREE_RATE

        n_years = max(len(ret) / 252, 1e-9)
        total_ret = float(pv.iloc[-1] / pv.iloc[0]) - 1
        ann_ret = float((1 + total_ret) ** (1 / n_years) - 1)
        vol = float(ret.std() * np.sqrt(252))
        sharpe = float((ann_ret - rf) / vol) if vol > 0 else 0.0

        cumret = (1 + ret).cumprod()
        drawdown = (cumret - cumret.cummax()) / cumret.cummax()
        max_dd = float(drawdown.min())

        downside = ret[ret < 0]
        sortino_vol = float(downside.std() * np.sqrt(252)) if len(downside) > 0 else vol
        sortino = float((ann_ret - rf) / sortino_vol) if sortino_vol > 0 else 0.0
        calmar = float(ann_ret / abs(max_dd)) if max_dd != 0 else 0.0

        var_95 = float(np.percentile(ret.dropna(), 5))
        cvar_95 = float(ret[ret <= var_95].mean()) if len(ret[ret <= var_95]) > 0 else var_95

        # Turnover
        if backtester.regime_details_history:
            avg_turn = float(np.mean([e.get('turnover', 0) for e in backtester.regime_details_history]) * 52)
        else:
            avg_turn = float('nan')

        # Regime distribution
        if backtester.regime_details_history:
            regimes = [e.get('regime', 'N/A') for e in backtester.regime_details_history]
            risk_off_pct = regimes.count('RISK_OFF') / max(len(regimes), 1) * 100
        else:
            risk_off_pct = float('nan')

        # Win rate
        win_rate = float((ret > 0).sum() / len(ret) * 100) if len(ret) > 0 else 0

        results_dict[label] = {
            'n_days': len(ret),
            'n_years': n_years,
            'ann_ret': ann_ret,
            'vol': vol,
            'sharpe': sharpe,
            'sortino': sortino,
            'calmar': calmar,
            'max_dd': max_dd,
            'var_95': var_95,
            'cvar_95': cvar_95,
            'turnover': avg_turn,
            'risk_off_pct': risk_off_pct,
            'win_rate': win_rate,
            'total_ret': total_ret,
        }

    # ── 4. Tableau comparatif ────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("RESULTATS : IN-SAMPLE vs OUT-OF-SAMPLE")
    print("=" * 70)

    labels = list(results_dict.keys())
    if any(v is None for v in results_dict.values()):
        print("[ERREUR] Un des backtests n'a produit aucun resultat.")
        return results_dict

    is_data = results_dict[labels[0]]
    oos_data = results_dict[labels[1]]

    print(f"\n{'Metrique':<28} {'IN-SAMPLE':>14} {'OUT-OF-SAMPLE':>14} {'Ratio OOS/IS':>14} {'Verdict':>10}")
    print("─" * 82)

    metrics_to_compare = [
        ('Jours de trading', 'n_days', '{:.0f}', False),
        ('Rendement annualise', 'ann_ret', '{:.2%}', True),
        ('Volatilite', 'vol', '{:.2%}', False),
        ('Ratio de Sharpe', 'sharpe', '{:.3f}', True),
        ('Ratio de Sortino', 'sortino', '{:.3f}', True),
        ('Ratio de Calmar', 'calmar', '{:.3f}', True),
        ('Max Drawdown', 'max_dd', '{:.2%}', False),
        ('VaR 95% (daily)', 'var_95', '{:.3%}', False),
        ('CVaR 95% (daily)', 'cvar_95', '{:.3%}', False),
        ('Turnover annualise', 'turnover', '{:.0f}%', False),
        ('% RISK_OFF', 'risk_off_pct', '{:.1f}%', False),
        ('Win rate', 'win_rate', '{:.1f}%', False),
    ]

    key_verdicts = []

    for name, key, fmt, check_ratio in metrics_to_compare:
        is_val = is_data[key]
        oos_val = oos_data[key]

        is_str = fmt.format(is_val)
        oos_str = fmt.format(oos_val)

        if check_ratio and is_val != 0:
            ratio = oos_val / is_val
            ratio_str = f"{ratio:.2f}x"

            if ratio >= 0.70:
                verdict = "[OK]"
            elif ratio >= 0.50:
                verdict = "[~OK]"
            else:
                verdict = "[WARN]"
            key_verdicts.append((name, ratio, verdict))
        else:
            ratio_str = "—"
            verdict = ""

        print(f"{name:<28} {is_str:>14} {oos_str:>14} {ratio_str:>14} {verdict:>10}")

    # ── 5. Diagnostic final ──────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("DIAGNOSTIC DE SUROPTIMISATION")
    print("=" * 70)

    sharpe_is = is_data['sharpe']
    sharpe_oos = oos_data['sharpe']
    sharpe_ratio = sharpe_oos / sharpe_is if sharpe_is != 0 else float('nan')

    calmar_is = is_data['calmar']
    calmar_oos = oos_data['calmar']
    calmar_ratio = calmar_oos / calmar_is if calmar_is != 0 else float('nan')

    dd_is = abs(is_data['max_dd'])
    dd_oos = abs(oos_data['max_dd'])

    print(f"\n  Sharpe IS : {sharpe_is:.3f}  |  Sharpe OOS : {sharpe_oos:.3f}  |  Ratio : {sharpe_ratio:.2f}x")
    print(f"  Calmar IS : {calmar_is:.3f}  |  Calmar OOS : {calmar_oos:.3f}  |  Ratio : {calmar_ratio:.2f}x")
    print(f"  Max DD IS : {is_data['max_dd']:.2%}   |  Max DD OOS : {oos_data['max_dd']:.2%}")

    # Verdict global
    print("\n  Criteres de validation :")

    checks_passed = 0
    total_checks = 4

    # Check 1 : Sharpe ratio
    if sharpe_ratio >= 0.70:
        print(f"  [OK] Sharpe OOS/IS = {sharpe_ratio:.2f}x >= 0.70 → Pas de suroptimisation")
        checks_passed += 1
    elif sharpe_ratio >= 0.50:
        print(f"  [~OK] Sharpe OOS/IS = {sharpe_ratio:.2f}x (0.50-0.70) → Degradation moderee")
        checks_passed += 0.5
    else:
        print(f"  [WARN] Sharpe OOS/IS = {sharpe_ratio:.2f}x < 0.50 → Suroptimisation possible")

    # Check 2 : Calmar ratio
    if calmar_ratio >= 0.70:
        print(f"  [OK] Calmar OOS/IS = {calmar_ratio:.2f}x >= 0.70 → Stable")
        checks_passed += 1
    elif calmar_ratio >= 0.50:
        print(f"  [~OK] Calmar OOS/IS = {calmar_ratio:.2f}x (0.50-0.70) → Degradation moderee")
        checks_passed += 0.5
    else:
        print(f"  [WARN] Calmar OOS/IS = {calmar_ratio:.2f}x < 0.50 → Instable")

    # Check 3 : Drawdown ne doit pas exploser
    if dd_oos <= dd_is * 1.5:
        print(f"  [OK] Max DD OOS ({oos_data['max_dd']:.2%}) <= 1.5x IS ({is_data['max_dd']:.2%})")
        checks_passed += 1
    else:
        print(f"  [WARN] Max DD OOS ({oos_data['max_dd']:.2%}) > 1.5x IS ({is_data['max_dd']:.2%})")

    # Check 4 : Rendement OOS positif
    if oos_data['ann_ret'] > 0:
        print(f"  [OK] Rendement OOS positif ({oos_data['ann_ret']:.2%})")
        checks_passed += 1
    else:
        print(f"  [WARN] Rendement OOS negatif ({oos_data['ann_ret']:.2%})")

    # Verdict
    print(f"\n  Score : {checks_passed}/{total_checks} criteres passes")

    if checks_passed >= 3.5:
        print("\n  ══════════════════════════════════════════════════════════════")
        print("  VERDICT : MODELE VALIDE OUT-OF-SAMPLE")
        print("  Les hyperparametres ne sont pas suroptimises.")
        print("  La performance OOS est coherente avec la performance IS.")
        print("  ══════════════════════════════════════════════════════════════")
    elif checks_passed >= 2.5:
        print("\n  ══════════════════════════════════════════════════════════════")
        print("  VERDICT : MODELE PARTIELLEMENT VALIDE")
        print("  Degradation moderee — acceptable mais a surveiller.")
        print("  ══════════════════════════════════════════════════════════════")
    else:
        print("\n  ══════════════════════════════════════════════════════════════")
        print("  VERDICT : ATTENTION — SUROPTIMISATION POSSIBLE")
        print("  La performance OOS se degrade significativement vs IS.")
        print("  Les hyperparametres pourraient etre suroptimises.")
        print("  ══════════════════════════════════════════════════════════════")

    # Contexte pour la presentation
    print("\n  Note pour la presentation :")
    print("  - Nos hyperparametres sont fixes par la litterature (pas par optimisation)")
    print("  - Ce test confirme/infirme que l'ARCHITECTURE globale generalise")
    print("  - Le walk-forward valide les decisions LOCALES (56 fenetres)")
    print("  - Ce test valide les decisions GLOBALES (hyperparametres)")

    return results_dict

# ─────────────────────────────────────────────────────────────────────────────
# EXECUTION : decommenter la ligne ci-dessous pour lancer la validation
# Temps estime : ~10-15 minutes
# ─────────────────────────────────────────────────────────────────────────────
oos_results = run_oos_validation()


VALIDATION OUT-OF-SAMPLE PURE
Split : IN-SAMPLE avant 2020-01-01
         OUT-OF-SAMPLE apres 2020-01-01
Hyperparametres : FIGES (Config inchange entre IS et OOS)

[1/4] Chargement des donnees...
  [BACKTEST] +20 tickers historiques (survivorship-bias-free)
Univers: 518 US stocks + 100 CA stocks + 32 US bonds + 17 CA bonds + 86 thematic = 753 live + 21 historiques
Chargement de 774 tickers...
  [_convert_to_cad] NaN restants apres conversion : 337477/3422844 (9.8595%)
  [SURVIVORSHIP] 1 tickers detectes comme delistes
  [ESG] API Yahoo Finance accessible, chargement des scores...
  [ESG] Scores Sustainalytics: 0 | Proxy: 773 / 773 tickers
  [OK] Taux sans risque chargé: 4323 points
[OK] Donnees: 4428 jours, 773 titres actifs
[OK] Volumes charges pour 774 titres
  Chargement données macro pour régimes...
    [OK] yield_10y: 4300 obs
    [OK] yield_2y: 4300 obs
    [OK] credit_spread_hy: 4493 obs
  Donnees : 4428 jours x 773 tickers

─────────────────────────────────────────────────────

Walk-Forward:   0%|          | 0/31 [00:00<?, ?it/s]

  [COMPUSTAT v2] 667 tickers charges, nouveaux facteurs: ~563 tickers avec donnees
  [COMPUSTAT] Charge depuis compustat_fundamentals.csv (16856 tickers, 248480 observations)
  [COMPUSTAT] 295/300 tickers charges
  [OK] Fondamentaux: 295/300 tickers (Compustat: 295, FMP: 0, YF: 0)
       ROE: 276, D/E: 276, Margins: 295
Risk-managed momentum: vol=45.3%, mult=0.33, poids momentum ajusté=11.4%

------------------------------------------------------------
[PENALITE FACTEURS MANQUANTS] Diagnostic — premier appel
------------------------------------------------------------
  AVANT (fillna=0.5) : 0/25 titres avec >50% facteurs manquants
  APRES (penalite)   : 0/25 titres avec >50% facteurs manquants
  Composition top-25 identique (aucun titre retire/ajoute)
------------------------------------------------------------
[DEBUG] Entrée optimisation: 43 titres (37 stocks + 6 bonds)
[DEBUG] Cov matrix shape: (43, 43)
[DEBUG] stock_alloc=0.725  bond_alloc=0.250
[DEBUG] Statut solveur: optimal
[DEBU

Walk-Forward:   3%|▎         | 1/31 [00:15<07:56, 15.87s/it]

[DEBUG] Entrée optimisation: 48 titres (42 stocks + 6 bonds)
[DEBUG] Cov matrix shape: (48, 48)
[DEBUG] stock_alloc=0.725  bond_alloc=0.250
[DEBUG] Statut solveur: optimal
[DEBUG] Sortie optimisation: 11 positions actives
[DEBUG] Sum weights: 0.975000
[DEBUG] Bonds dans weights: 3
[DEBUG] Après nettoyage min_weight: 11 positions, sum=0.975000
[DEBUG] Après construction result: 8 stocks + 3 bonds, sum=0.975000
[DEBUG] Après smoothing (40%): 17 positions, sum=1.000000
[DEBUG] Entrée optimisation: 50 titres (44 stocks + 6 bonds)
[DEBUG] Cov matrix shape: (50, 50)
[DEBUG] stock_alloc=0.725  bond_alloc=0.250
[DEBUG] Statut solveur: optimal
[DEBUG] Sortie optimisation: 21 positions actives
[DEBUG] Sum weights: 0.975000
[DEBUG] Bonds dans weights: 3
[DEBUG] Après nettoyage min_weight: 11 positions, sum=0.975000
[DEBUG] Après construction result: 8 stocks + 3 bonds, sum=0.975000
[DEBUG] Après smoothing (40%): 18 positions, sum=1.000000
[DEBUG] Entrée optimisation: 50 titres (44 stocks + 6 bond

...

Walk-Forward:  96%|█████████▌| 23/24 [04:03<00:10, 10.60s/it]

[DEBUG] Entrée optimisation: 56 titres (50 stocks + 6 bonds)
[DEBUG] Cov matrix shape: (56, 56)
[DEBUG] stock_alloc=0.725  bond_alloc=0.250
[DEBUG] Statut solveur: optimal
[DEBUG] Sortie optimisation: 28 positions actives
[DEBUG] Sum weights: 0.975000
[DEBUG] Bonds dans weights: 5
[DEBUG] Après nettoyage min_weight: 13 positions, sum=0.971440
[DEBUG] Après construction result: 10 stocks + 3 bonds, sum=0.975000
[DEBUG] Après smoothing (40%): 19 positions, sum=1.000000
[DEBUG] Entrée optimisation: 56 titres (50 stocks + 6 bonds)
[DEBUG] Cov matrix shape: (56, 56)
[DEBUG] stock_alloc=0.725  bond_alloc=0.250
[DEBUG] Statut solveur: optimal
[DEBUG] Sortie optimisation: 45 positions actives
[DEBUG] Sum weights: 0.975000
[DEBUG] Bonds dans weights: 4
[DEBUG] Après nettoyage min_weight: 15 positions, sum=0.965663
[DEBUG] Après construction result: 12 stocks + 3 bonds, sum=0.975000
[DEBUG] Après smoothing (40%): 23 positions, sum=1.000000
[DEBUG] Entrée optimisation: 55 titres (49 stocks + 6 bo

Walk-Forward: 100%|██████████| 24/24 [04:15<00:00, 10.63s/it]

[DEBUG] Entrée optimisation: 55 titres (49 stocks + 6 bonds)
[DEBUG] Cov matrix shape: (55, 55)
[DEBUG] stock_alloc=0.725  bond_alloc=0.250
[DEBUG] Statut solveur: optimal
[DEBUG] Sortie optimisation: 31 positions actives
[DEBUG] Sum weights: 0.975000
[DEBUG] Bonds dans weights: 6
[DEBUG] Après nettoyage min_weight: 15 positions, sum=0.975000
[DEBUG] Après construction result: 12 stocks + 3 bonds, sum=0.975000
[DEBUG] Après smoothing (40%): 19 positions, sum=1.000000

Diagnostic fenêtres:
  no_test_data: 2448 skips

Dernier rebalancement: 12 stocks + 3 bonds (Σ stocks = 72.50%, Σ bonds = 25.00%)
Rebalances enregistrés: 314 | Dates dupliquées: 0

[BASELINE] Calcul de la NAV brute en cours (sans frais de gestion)...



RAPPORT FRAIS DE GESTION — BACKTEST
  NAV brute finale  : $    247,682,552.83  (identique à avant la modif)
  NAV nette finale  : $    139,885,265.20
  Écart (drag fees) : $    107,797,287.63
  ─────────────────────────────────────────────────────
  Frais de gestion  : $      6,551,504.96
  Commissions perf  : $     59,150,091.88
  Revenus gestionnaire: $   65,701,596.84  (8.746% ann.)
  ─────────────────────────────────────────────────────
  [OK] NAV nette < NAV brute à chaque date (1893 points)

  [OK] OUT-OF-SAMPLE (2020-2026) — 1586 jours en 258s

RESULTATS : IN-SAMPLE vs OUT-OF-SAMPLE

Metrique                          IN-SAMPLE  OUT-OF-SAMPLE   Ratio OOS/IS    Verdict
──────────────────────────────────────────────────────────────────────────────────
Jours de trading                       3631           1586              —           
Rendement annualise                   7.11%          5.57%          0.78x       [OK]
Volatilite                           12.36%         16.03%     

---
# 9. Exécution du backtest principal
---


In [ ]:
# =============================================================================
# BACKTEST PRINCIPAL —
#
# Cette cellule exécute le backtest walk-forward complet (2006-2025) incluant :
#   - Chargement de l'univers (~750 titres), VIX et données macro
#   - Walk-forward sur ~75 fenêtres (24m train / 6m test / 3m step)
#   - Détection de régime, optimisation MV, contrôle du turnover
#   - Stress testing sur périodes historiques (COVID, 2022, SVB, etc.)
#   - Rapport de performance et dashboard interactif
#
# Temps d'exécution estimé : ~5-10 minutes
# Résultats complets disponibles dans l'historique Colab et le rapport final.
# Décommenter pour relancer :
# output = run_full_backtest_v3()
# =============================================================================
#
# =============================================================================
# CELLULE 10: EXÉCUTION PRINCIPALE
# =============================================================================


def run_full_backtest_v3():
    """Exécute le backtest complet."""

    Config.LIQUIDITY['enabled'] = False
    print("\n" + "="*70)
    print("GSF-3120 - BACKTEST COMPLET")
    print("="*70)
    # 1. Charger les données
    print("\n[1/7] Chargement des données...")
    data_loader = DataLoader()
    prices, returns = data_loader.load_data(backtest_mode=True)

    # Charger VIX
    print("\n[2/7] Chargement VIX...")
    vix_data = data_loader.load_vix(
        Config.BACKTEST['start_date'] - timedelta(days=365),
        Config.BACKTEST['end_date']
    )
    if vix_data is not None:
        print(f"  VIX chargé: {len(vix_data)} points")
    else:
        print("  [ATTENTION] VIX non disponible")

    # Charger données macro pour régimes
    print("\n[3/7] Chargement données macro...")
    data_loader.load_macro_data(
        Config.BACKTEST['start_date'] - timedelta(days=365),
        Config.BACKTEST['end_date']
    )

    # 2. Initialiser les modules
    print("\n[3/7] Initialisation des modules...")
    factor_calc = FactorCalculator()
    risk_mgr = RiskManager()
    optimizer = PortfolioOptimizer()
    backtester = RollingBacktester()

    # 3. Exécuter le backtest
    print("\n[4/7] Exécution du backtest walk-forward...")
    results = backtester.run_walk_forward(
        prices, returns, factor_calc, risk_mgr, optimizer, data_loader, vix_data
    )
    # Sécurité : si aucun résultat n'est produit, on arrête immédiatement.
    if len(results) == 0:
        print("[ERREUR] Aucun résultat généré")
        return None

    # Diagnostic des régimes
    regime_df = pd.DataFrame(backtester.regime_details_history)
    print("=== DIAGNOSTIC RÉGIMES ===")
    print(f"\nDistribution des régimes:")
    print(regime_df['regime'].value_counts())

    print(f"\nScore composite moyen par régime:")
    print(regime_df.groupby('regime')['composite_score'].mean())
    print(f"\nConfiance moyenne par régime:")
    print(regime_df.groupby('regime')['confidence'].mean())


    print(f"\n10 dernières détections:")
    print(regime_df[['date', 'regime', 'composite_score', 'confidence']].tail(10))

    # 4. Benchmark
    print("\n[5/7] Analyse de performance...")
    # ---------------------------------------------------------------------
    # Benchmark (priorité: 60/40 si possible, sinon SPY)
    # ---------------------------------------------------------------------
    # Règle choisie :
    # - si SPY est disponible et qu'un ETF obligataire l'est aussi,
    #   on construit un benchmark 60% actions / 40% obligations ;
    # - sinon, on utilise seulement SPY comme référence actions.
    benchmark = None
    if 'SPY' in prices.columns:
        bond_candidates = ['AGG', 'BND', 'IEF', 'TLT', 'XBB.TO', 'ZAG.TO']
        bond_t = next((b for b in bond_candidates if b in prices.columns), None)
        if bond_t is not None:
            benchmark = 0.60 * prices['SPY'] + 0.40 * prices[bond_t]
        else:
            benchmark = prices['SPY']

        if benchmark is not None:
            benchmark = benchmark.loc[results.index[0]:results.index[-1]]

    # -------------------------------------------------------------------------
    # Analyse complète de performance
    # -------------------------------------------------------------------------
    # PerformanceAnalyzer calcule les métriques globales :
    # rendement, risque, drawdown, Sharpe, turnover, exposition FX, etc
    analyzer = PerformanceAnalyzer()
    metrics = analyzer.full_analysis(
        results,
        benchmark,
        backtester.regime_history,
        backtester.fx_exposure_history,
        backtester.turnover_history
    )

    # 5b. Stress Testing
    print("\n[5b/7] Stress Testing...")
    stress_analyzer = StressTestAnalyzer()
    stress_results = stress_analyzer.analyze_stress_periods(results, benchmark)
    scenarios = stress_analyzer.run_hypothetical_scenarios(results, backtester.weights_history)
    stress_report = stress_analyzer.generate_stress_report(stress_results, scenarios)
    print(stress_report)

    # Rapport
    report = analyzer.generate_report(metrics)
    print(report)

    # 6. Graphiques
    print("\n[6/7] Génération des graphiques.")
    fig = Visualizer.create_dashboard(
        results,
        benchmark,
        backtester.regime_history,
        backtester.fx_exposure_history
    )
    # Force browser renderer to avoid nbformat issues
    pio.renderers.default = "browser"
    fig.show()


    # 7. Analyse factorielle complète
    print("\n [7/7] Analyse factorielle complète.")
    fa = FactorAnalyzer()
    # Réutiliser le cache fondamental déjà chargé par le backtest
    if hasattr(factor_calc, '_fundamental_cache') and factor_calc._fundamental_cache:
         fa.factor_calc._fundamental_cache = factor_calc._fundamental_cache
         fa.factor_calc._fundamental_ts_cache = getattr(factor_calc, '_fundamental_ts_cache', {})
         fa.factor_calc._fundamental_load_date = getattr(factor_calc, '_fundamental_load_date', None)
    factor_report = fa.run_full_analysis(prices, returns)



    # Résumé comparatif
    print("\n" + "="*70)
    print("Conclusion")
    print("="*70)
    print(f"""
    {'Métrique':<30} {'V3':<15}
    {'-'*60}
    {'Cash Buffer':<30} {'2.5%':<15}
    {'Univers':<30} {f'{len(prices.columns)} titres':<15}
    {'Smoothing':<30} {'60%':<15}
    {'Rendement annualisé':<30} {f"{metrics['annual_return']:.1%}":<15}
    {'Sharpe Ratio':<30} {f"{metrics['sharpe_ratio']:.2f}":<15}
    {'Max Drawdown':<30}  {f"{metrics['max_drawdown']:.1%}":<15}
    """)

    if 'turnover_analysis' in metrics:
        to = metrics['turnover_analysis']
        turnover_ann = f"{to['annualized']:.0%}"
        turnover_med = f"{to['mean']:.1%}"
        print(f"    {'Turnover annualisé':<30} {turnover_ann:<15}")
        print(f"    {'Turnover médian':<30} {turnover_med:<15}")
    if 'fx_analysis' in metrics:
        fx = metrics['fx_analysis']
        usd_exp = f"{fx['avg_usd']:.1%}"

    Config.LIQUIDITY['enabled'] = False

    return {
        'results': results,
        'metrics': metrics,
        'backtester': backtester,
        'benchmark': benchmark,
        'data_loader': data_loader,
        'regime_history': backtester.regime_details_history,
        'prices': prices,
        'returns': returns,
        'vix_data': vix_data,
        'stress_results': stress_results,
        'stress_scenarios': scenarios,
        'factor_report': factor_report,
        'factor_calc': factor_calc,
        'risk_mgr': risk_mgr,
    }

# Executer et stocker dans variable globale
output = run_full_backtest_v3()



GSF-3120 - BACKTEST COMPLET

[1/7] Chargement des données...
  [BACKTEST] +20 tickers historiques (survivorship-bias-free)
Univers: 518 US stocks + 100 CA stocks + 32 US bonds + 17 CA bonds + 86 thematic = 753 live + 21 historiques
Chargement de 774 tickers...



1 Failed download:
['BR']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


  [_convert_to_cad] NaN restants apres conversion : 339884/3422844 (9.9299%)
  [SURVIVORSHIP] 1 tickers detectes comme delistes
  [ESG] API Yahoo Finance accessible, chargement des scores...
  [ESG] Scores Sustainalytics: 0 | Proxy: 773 / 773 tickers
  [OK] Taux sans risque chargé: 4323 points
[OK] Donnees: 4428 jours, 773 titres actifs
[OK] Volumes charges pour 774 titres

[2/7] Chargement VIX...
  VIX chargé: 4487 points

[3/7] Chargement données macro...
  Chargement données macro pour régimes...
    [OK] yield_10y: 4300 obs
    [OK] yield_2y: 4300 obs
    [OK] credit_spread_hy: 4493 obs

[3/7] Initialisation des modules...

[4/7] Exécution du backtest walk-forward...

WALK-FORWARD BACKTEST V3 (avec cash buffer)
Fenêtres: 56
Train: 24m | Test: 6m | Step: 3m
Cash buffer: 2.5%
Smoothing: 40%


Walk-Forward:   0%|          | 0/56 [00:00<?, ?it/s]

  [COMPUSTAT v2] 667 tickers charges, nouveaux facteurs: ~563 tickers avec donnees
  [COMPUSTAT] Charge depuis compustat_fundamentals.csv (16856 tickers, 248480 observations)
  [COMPUSTAT] 295/300 tickers charges
  [OK] Fondamentaux: 295/300 tickers (Compustat: 295, FMP: 0, YF: 0)
       ROE: 276, D/E: 276, Margins: 295
Risk-managed momentum: vol=45.3%, mult=0.33, poids momentum ajusté=11.4%

------------------------------------------------------------
[PENALITE FACTEURS MANQUANTS] Diagnostic — premier appel
------------------------------------------------------------
  AVANT (fillna=0.5) : 0/25 titres avec >50% facteurs manquants
  APRES (penalite)   : 0/25 titres avec >50% facteurs manquants
  Composition top-25 identique (aucun titre retire/ajoute)
------------------------------------------------------------
[DEBUG] Entrée optimisation: 56 titres (50 stocks + 6 bonds)
[DEBUG] Cov matrix shape: (56, 56)
[DEBUG] stock_alloc=0.725  bond_alloc=0.250
[DEBUG] Statut solveur: optimal
[DEBU

...

  [COMPUSTAT v2] 667 tickers charges, nouveaux facteurs: ~563 tickers avec donnees


Panel factoriel:   2%|▎         | 1/40 [00:01<00:46,  1.19s/it]

Risk-managed momentum: vol=9.0%, mult=1.50, poids momentum ajusté=32.1%

------------------------------------------------------------
[PENALITE FACTEURS MANQUANTS] Diagnostic — premier appel
------------------------------------------------------------
  AVANT (fillna=0.5) : 0/25 titres avec >50% facteurs manquants
  APRES (penalite)   : 0/25 titres avec >50% facteurs manquants
  Composition top-25 identique (aucun titre retire/ajoute)
------------------------------------------------------------


Panel factoriel: 100%|██████████| 40/40 [00:25<00:00,  1.58it/s]


[OK] Panel : 40 snapshots, ~773 titres/snapshot

[2/7] Correlations...

CORRELATION INTER-FACTEURS (Spearman, moyenne temporelle)
Sur 40 snapshots

                        momentum           value   profitability  low_volatility      investment            pead
----------------------------------------------------------------------------------------------------------------
        momentum           1.000           0.029           0.371          -0.058          -0.096           0.234
           value           0.029           1.000          -0.072           0.107           0.036          -0.050
   profitability           0.371          -0.072           1.000           0.132          -0.049           0.067
  low_volatility          -0.058           0.107           0.132           1.000           0.065           0.039
      investment          -0.096           0.036          -0.049           0.065           1.000          -0.087
            pead           0.234          -0.050           0.

In [ ]:
# =============================================================================
# CELLULE : ANALYSE SECTORIELLE, SENSIBILITE FACTORIELLE ET ML
# =============================================================================
#
# Cette cellule ajoute trois analyses manquantes au systeme de backtest :
#
# 1. Attribution sectorielle : decompose la performance du portefeuille par
#    secteur GICS pour identifier quels secteurs contribuent positivement ou
#    negativement, et detecter les surconcentrations historiques.
#
# 2. Sensibilite factorielle : teste l'impact de la modification des poids
#    factoriels (momentum, 52w_high, quality, profitability) sur la performance
#    du portefeuille via un grid search.
#
# 3. Detection de regime par ML : un modele Random Forest entraine sur des
#    indicateurs de marche (VIX, spreads, momentum, volatilite) pour predire
#    le regime optimal. Compare avec le detecteur de regime existant.
#
# Prerequis : le rolling backtester (cellule 17) doit avoir ete execute
# et ses resultats doivent etre disponibles dans la variable backtest_results.
# Le DataLoader doit avoir sector_map charge .
# =============================================================================



# =========================================================================
# 1. ATTRIBUTION SECTORIELLE
# =========================================================================

class SectorAnalyzer:
    """
    Analyse la contribution de chaque secteur GICS a la performance
    du portefeuille sur la periode de backtest.

    Produit :
    - Rendement moyen par secteur
    - Poids moyen par secteur dans le temps
    - Contribution a la performance totale (poids * rendement)
    - Identification des periodes de surconcentration
    """

    def __init__(self, config, data_loader):
        self.config = config
        self.data_loader = data_loader

    def compute_sector_attribution(self, backtest_results, returns):
        """
        Decompose la performance du portefeuille par secteur.

        Args:
            backtest_results: liste de dicts retournee par le rolling backtester,
                              chaque dict contient 'weights' (pd.Series) et
                              'decision_date' (pd.Timestamp)
            returns: pd.DataFrame des rendements quotidiens

        Returns:
            dict avec :
                - sector_contributions: pd.DataFrame (secteur x statistiques)
                - sector_weights_ts: pd.DataFrame (date x secteur)
                - concentration_flags: list de periodes ou un secteur > 30%
        """
        if not hasattr(self.data_loader, 'sector_map'):
            print("[ATTENTION] sector_map non disponible. "
                  "Executer load_esg_dataset d'abord.")
            return None

        sector_map = self.data_loader.sector_map
        bond_set = (set(self.data_loader.US_BOND_ETFS) |
                    set(self.data_loader.CA_BOND_ETFS))

        # Accumuler les poids sectoriels et contributions a chaque pas
        records = []

        for entry in backtest_results:
            if 'weights' not in entry or entry['weights'] is None:
                continue

            weights = entry['weights']
            date = entry.get('date', entry.get('decision_date', None))
            if date is None:
                continue

            # Mapper chaque ticker a son secteur (bonds -> "Obligations")
            for ticker, w in weights.items():
                if w < 1e-6:
                    continue
                if ticker in bond_set:
                    sector = "Obligations"
                elif ticker == self.config.CASH_BUFFER.get('ticker', 'CASH.TO'):
                    sector = "Cash"
                else:
                    sector = sector_map.get(ticker, "Unknown")

                # Rendement du titre sur la periode suivante
                if ticker in returns.columns and date in returns.index:
                    loc = returns.index.get_loc(date)
                    if loc + 1 < len(returns):
                        ret = returns.iloc[loc + 1].get(ticker, 0.0)
                    else:
                        ret = 0.0
                else:
                    ret = 0.0

                records.append({
                    'date': date,
                    'sector': sector,
                    'ticker': ticker,
                    'weight': float(w),
                    'return': float(ret),
                    'contribution': float(w) * float(ret)
                })

        if not records:
            print("[INFO] Aucune donnee d'attribution disponible.")
            return None

        df = pd.DataFrame(records)

        # --- Poids moyen par secteur dans le temps ---
        sector_weights_ts = df.pivot_table(
            index='date', columns='sector',
            values='weight', aggfunc='sum'
        ).fillna(0)

        # --- Contribution cumulee par secteur ---
        sector_contrib_ts = df.pivot_table(
            index='date', columns='sector',
            values='contribution', aggfunc='sum'
        ).fillna(0)

        # --- Statistiques resumees ---
        summary = pd.DataFrame({
            'poids_moyen': sector_weights_ts.mean(),
            'poids_max': sector_weights_ts.max(),
            'contribution_totale': sector_contrib_ts.sum(),
            'contribution_moyenne': sector_contrib_ts.mean(),
            'nb_periodes': (sector_weights_ts > 0.01).sum()
        }).sort_values('contribution_totale', ascending=False)

        # --- Detection de surconcentration (> 30% dans un secteur) ---
        max_cap = self.config.SECTOR_DIVERSIFICATION.get(
            'max_sector_weight', 0.25
        ) if hasattr(self.config, 'SECTOR_DIVERSIFICATION') else 0.30

        concentration_flags = []
        for col in sector_weights_ts.columns:
            breaches = sector_weights_ts[sector_weights_ts[col] > max_cap + 0.05]
            if len(breaches) > 0:
                concentration_flags.append({
                    'sector': col,
                    'n_breaches': len(breaches),
                    'max_weight': sector_weights_ts[col].max(),
                    'dates': breaches.index.tolist()[:5]
                })

        return {
            'summary': summary,
            'sector_weights_ts': sector_weights_ts,
            'sector_contributions_ts': sector_contrib_ts,
            'concentration_flags': concentration_flags
        }

    def print_report(self, results):
        """Affiche le rapport d'attribution sectorielle."""
        if results is None:
            return

        print(f"\n{'='*70}")
        print("ATTRIBUTION SECTORIELLE")
        print(f"{'='*70}")

        print("\n--- Contribution par secteur ---")
        summary = results['summary']
        for sector in summary.index:
            row = summary.loc[sector]
            print(f"  {sector:25s} | Poids moy: {row['poids_moyen']:6.1%} | "
                  f"Max: {row['poids_max']:6.1%} | "
                  f"Contrib: {row['contribution_totale']:+8.4f}")

        if results['concentration_flags']:
            print("\n--- Alertes de surconcentration ---")
            for flag in results['concentration_flags']:
                print(f"  {flag['sector']:25s} | "
                      f"{flag['n_breaches']} depassements | "
                      f"Max: {flag['max_weight']:.1%}")
        else:
            print("\n  Aucune surconcentration detectee.")

        print(f"{'='*70}\n")


# =========================================================================
# 2. SENSIBILITE FACTORIELLE
# =========================================================================

class FactorSensitivityAnalyzer:
    """
    Evalue l'impact de la modification des poids factoriels sur la
    performance du portefeuille.

    Au lieu de relancer le backtest complet (trop lent), cette classe
    recalcule le score composite avec differentes combinaisons de poids
    et mesure la correlation de rang avec le score original. Cela donne
    une approximation de la sensibilite sans cout de calcul excessif.

    Pour une analyse plus precise, un sous-ensemble de periodes est
    utilise pour relancer le scoring + selection + mesure de performance.

    AVERTISSEMENT : Cet outil est un DIAGNOSTIC uniquement.
    Ne PAS utiliser les resultats pour calibrer les poids du modele live
    — risque de sur-optimisation (data mining).
    """

    def __init__(self, config, factor_calc):
        self.config = config
        self.factor_calc = factor_calc

    def run_sensitivity_grid(self, factors, returns, n_top=25):
        """
        Teste differentes combinaisons de poids factoriels et mesure
        la performance moyenne des top-N titres selectionnes.

        Args:
            factors: dict de pd.DataFrame, un par facteur
                     (index = dates, colonnes = tickers)
                     OU dict de pd.Series (scores cross-sectionnels a une date)
            returns: pd.DataFrame des rendements (index = dates, colonnes = tickers)
            n_top: int, nombre de titres selectionnes (defaut 25)

        Returns:
            pd.DataFrame avec colonnes [momentum, 52w_high, quality,
            profitability, sharpe_approx, return_mean, volatility]
        """
        print("[AVERTISSEMENT] Le grid search est un outil de DIAGNOSTIC. "
              "Ne PAS utiliser les résultats pour calibrer les poids du "
              "modèle live — risque de sur-optimisation (data mining).")
        # Grille de poids a tester
        # On fait varier chaque facteur de 0 a 0.50 par pas de 0.10
        # tout en normalisant pour que la somme = 1
        factor_names = [k for k, v in self.config.FACTORS.items()
                        if v.get('enabled') and v.get('weight', 0) > 0]
        n_factors = len(factor_names)

        # Generer les combinaisons
        from itertools import product
        steps = [0.0, 0.10, 0.20, 0.30, 0.40, 0.50]
        combos = []
        for combo in product(steps, repeat=n_factors):
            if sum(combo) > 0:
                total = sum(combo)
                normalized = tuple(round(c / total, 3) for c in combo)
                if normalized not in combos:
                    combos.append(normalized)

        print(f"[SENSIBILITE] {len(combos)} combinaisons de poids a tester "
              f"sur {n_factors} facteurs")

        # Si factors est un dict de Series (un seul snapshot), on travaille
        # avec ce snapshot. Si c'est un dict de DataFrames, on prend les
        # 12 derniers mois pour evaluer.

        # Determiner le format des facteurs
        sample_factor = next(iter(factors.values()))
        if isinstance(sample_factor, pd.DataFrame):
            # Multi-dates : echantillonner 12 dates espacees
            dates = sample_factor.index
            step_size = max(1, len(dates) // 12)
            eval_dates = dates[::step_size][-12:]
            multi_date = True
        else:
            eval_dates = [None]
            multi_date = False

        results = []

        for combo in combos:
            weights_dict = dict(zip(factor_names, combo))

            period_returns = []

            for eval_date in eval_dates:
                # Calculer le score composite avec ces poids
                composite = pd.Series(0.0)

                for fname, fw in weights_dict.items():
                    if fname not in factors or fw < 1e-6:
                        continue

                    if multi_date:
                        raw = factors[fname].loc[eval_date].dropna()
                    else:
                        raw = factors[fname].dropna()

                    if len(raw) < 10:
                        continue

                    ranked = raw.rank(pct=True)
                    composite = composite.reindex(
                        composite.index.union(ranked.index)
                    ).fillna(0)
                    composite[ranked.index] += fw * ranked

                if len(composite) < n_top:
                    continue

                # Selectionner les top-N
                top_tickers = composite.nlargest(n_top).index.tolist()

                # Rendement moyen equal-weight sur la periode suivante
                if multi_date and eval_date in returns.index:
                    loc = returns.index.get_loc(eval_date)
                    # Prendre les 5 jours suivants (une semaine de trading)
                    end_loc = min(loc + 6, len(returns))
                    if end_loc > loc + 1:
                        valid = [t for t in top_tickers
                                 if t in returns.columns]
                        if valid:
                            period_ret = returns.iloc[loc+1:end_loc][valid].mean(
                                axis=1
                            ).sum()
                            period_returns.append(period_ret)

            if len(period_returns) >= 3:
                arr = np.array(period_returns)
                mean_ret = arr.mean()
                vol = arr.std() if len(arr) > 1 else 1e-6
                sharpe = mean_ret / vol if vol > 1e-8 else 0.0

                row = dict(zip(factor_names, combo))
                row['return_mean'] = mean_ret
                row['volatility'] = vol
                row['sharpe_approx'] = sharpe
                row['n_periods'] = len(period_returns)
                results.append(row)

        if not results:
            print("[INFO] Pas assez de donnees pour l'analyse de sensibilite.")
            return None

        df_results = pd.DataFrame(results)
        df_results = df_results.sort_values('sharpe_approx', ascending=False)

        return df_results

    def print_report(self, df_results):
        """Affiche le rapport de sensibilite factorielle."""
        if df_results is None or len(df_results) == 0:
            return

        factor_names = [c for c in df_results.columns
                        if c not in ['return_mean', 'volatility',
                                     'sharpe_approx', 'n_periods']]

        print(f"\n{'='*70}")
        print("SENSIBILITE AUX POIDS FACTORIELS")
        print(f"{'='*70}")

        # Poids actuels
        current = {k: v['weight'] for k, v in self.config.FACTORS.items()
                  if v.get('enabled') and v.get('weight', 0) > 0}
        print(f"\n  Poids actuels : {current}")

        # Top 5 combinaisons
        print(f"\n--- Top 5 combinaisons (Sharpe approximatif) ---")
        for i, (_, row) in enumerate(df_results.head(5).iterrows()):
            weights_str = " | ".join(
                f"{fn}: {row[fn]:.0%}" for fn in factor_names
            )
            print(f"  #{i+1} {weights_str} | "
                  f"Sharpe: {row['sharpe_approx']:.3f} | "
                  f"Ret: {row['return_mean']:+.4f}")

        # Pire 3 combinaisons
        print(f"\n--- Pire 3 combinaisons ---")
        for i, (_, row) in enumerate(df_results.tail(3).iterrows()):
            weights_str = " | ".join(
                f"{fn}: {row[fn]:.0%}" for fn in factor_names
            )
            print(f"  #{i+1} {weights_str} | "
                  f"Sharpe: {row['sharpe_approx']:.3f} | "
                  f"Ret: {row['return_mean']:+.4f}")

        # Sensibilite marginale : correlation entre chaque poids et le Sharpe
        print(f"\n--- Sensibilite marginale (correlation poids <-> Sharpe) ---")
        for fn in factor_names:
            corr = df_results[fn].corr(df_results['sharpe_approx'])
            direction = "+" if corr > 0.05 else ("-" if corr < -0.05 else "~")
            print(f"  {fn:20s} : r = {corr:+.3f}  [{direction}]")

        print(f"{'='*70}\n")


# =========================================================================
# 3. DETECTION DE REGIME PAR ML (RANDOM FOREST)
# =========================================================================

class MLRegimeDetector:
    """
    Entraine un Random Forest pour predire le regime de marche optimal
    (bull, bear, neutral) a partir d'indicateurs techniques et macro.

    Le modele est entraine sur les donnees historiques en utilisant le
    rendement forward du marche pour labelliser les regimes (supervision).
    Il est ensuite compare au detecteur de regime existant (HMM/rule-based).

    Ceci n'est pas destine a remplacer le detecteur existant, mais a
    valider si un modele ML apporte de la valeur ajoutee.
    """

    def __init__(self, config):
        self.config = config
        self.model = None
        self.feature_names = None

    def prepare_features(self, prices, vix_data=None, risk_free=None):
        """
        Construit la matrice de features pour le modele ML.

        Features utilisees :
        - Rendement du marche (proxy: moyenne equal-weight) sur 5, 21, 63 jours
        - Volatilite realisee sur 21 et 63 jours
        - Ratio vol courte / vol longue (acceleration de la vol)
        - VIX niveau et variation 21j (si disponible)
        - Momentum du marche (rendement 252j)
        - Breadth (% de titres au-dessus de leur SMA 200j)

        Args:
            prices: pd.DataFrame des prix
            vix_data: pd.Series du VIX (optionnel)
            risk_free: pd.Series du taux sans risque (optionnel)

        Returns:
            pd.DataFrame des features (index = dates)
        """
        # Rendement equal-weight du marche
        returns = prices.pct_change()
        mkt_ret = returns.mean(axis=1)

        features = pd.DataFrame(index=prices.index)

        # Rendements sur differentes fenetres
        features['ret_21d'] = mkt_ret.rolling(21).sum()
        features['ret_63d'] = mkt_ret.rolling(63).sum()
        features['ret_252d'] = mkt_ret.rolling(252).sum()

        # Volatilite realisee
        features['vol_21d'] = mkt_ret.rolling(21).std() * np.sqrt(252)
        features['vol_63d'] = mkt_ret.rolling(63).std() * np.sqrt(252)

        # Ratio vol courte / vol longue (> 1 = vol en acceleration)
        features['vol_ratio'] = features['vol_21d'] / features['vol_63d'].replace(0, np.nan)

        # Breadth : % de titres au-dessus de leur SMA 200
        sma200 = prices.rolling(200).mean()
        above_sma = (prices > sma200).sum(axis=1)
        features['breadth'] = above_sma / prices.shape[1]

        # Drawdown du marche
        cumret = (1 + mkt_ret).cumprod()
        running_max = cumret.cummax()
        features['drawdown'] = (cumret / running_max) - 1

        # VIX (si disponible)
        if vix_data is not None:
            if isinstance(vix_data, pd.DataFrame):
                _vix_series = vix_data.iloc[:, 0]
            else:
                _vix_series = vix_data
            vix_aligned = _vix_series.reindex(prices.index, method='ffill')
            features['vix'] = vix_aligned
            features['vix_change_21d'] = vix_aligned.pct_change(21)

            # VIX term structure = VIX - VIX3M (si les deux colonnes existent)
            if isinstance(vix_data, pd.DataFrame) and 'VIX3M' in vix_data.columns:
                _vix_col  = vix_data['VIX'].reindex(prices.index, method='ffill')                             if 'VIX' in vix_data.columns else vix_aligned
                _vix3m    = vix_data['VIX3M'].reindex(prices.index, method='ffill')
                features['vix_term_structure'] = _vix_col - _vix3m
            else:
                features['vix_term_structure'] = np.nan
        else:
            features['vix_term_structure'] = np.nan

        # Jours depuis le dernier plus haut du marché
        _peak_dates  = prices.index.to_series().where(cumret.eq(running_max))
        _peak_filled = _peak_dates.ffill()
        features['days_since_peak'] = (
            prices.index.to_series() - _peak_filled
        ).dt.days

        return features.dropna()

    def prepare_labels(self, prices, vix_data=None,
                       vix_risk_off=25, dd_risk_off=0.05,
                       vix_risk_on=20, breadth_risk_on=0.60):
        """
        Labels observables au jour t (aucun look-ahead).

        RISK_OFF : VIX > vix_risk_off  ET  drawdown > dd_risk_off
        RISK_ON  : VIX < vix_risk_on   ET  breadth  > breadth_risk_on
        neutral  : sinon

        Args:
            prices:          pd.DataFrame des prix
            vix_data:        pd.Series du VIX (optionnel)
            vix_risk_off:    seuil VIX pour RISK_OFF (defaut 25)
            dd_risk_off:     seuil drawdown pour RISK_OFF (defaut 5%)
            vix_risk_on:     seuil VIX pour RISK_ON  (defaut 20)
            breadth_risk_on: seuil breadth pour RISK_ON (defaut 60%)

        Returns:
            pd.Series de labels ('RISK_ON', 'RISK_OFF', 'neutral')
        """
        returns  = prices.pct_change()
        mkt_ret  = returns.mean(axis=1)

        # Drawdown du marché (série passée uniquement)
        cumret      = (1 + mkt_ret).cumprod()
        running_max = cumret.cummax()
        drawdown    = (cumret / running_max.replace(0, np.nan)) - 1   # négatif

        # Breadth : % de titres au-dessus de leur SMA200
        sma200  = prices.rolling(200).mean()
        breadth = (prices > sma200).sum(axis=1) / prices.shape[1]

        # VIX aligné (si disponible)
        if vix_data is not None:
            if isinstance(vix_data, pd.DataFrame):
                vix_col = vix_data.iloc[:, 0]
            else:
                vix_col = vix_data
            vix_aligned = vix_col.reindex(prices.index, method='ffill')
        else:
            vix_aligned = pd.Series(np.nan, index=prices.index)

        # Labels basés sur critères observables (T)
        labels = pd.Series('neutral', index=prices.index)

        risk_off_mask = (vix_aligned > vix_risk_off) & (-drawdown > dd_risk_off)
        risk_on_mask  = (vix_aligned < vix_risk_on)  & (breadth > breadth_risk_on)

        labels[risk_off_mask] = 'RISK_OFF'
        labels[risk_on_mask & ~risk_off_mask] = 'RISK_ON'

        # Supprimer les lignes où VIX est indisponible (début de série)
        valid = vix_aligned.notna() & breadth.notna() & drawdown.notna()
        return labels[valid]

    def train(self, prices, vix_data=None, risk_free=None,
              train_ratio=0.7):
        """
        Entraine le Random Forest sur les donnees historiques.

        Utilise une separation temporelle (pas de shuffle) pour eviter
        le look-ahead bias.

        Args:
            prices: pd.DataFrame des prix
            vix_data: pd.Series du VIX
            risk_free: pd.Series du taux sans risque
            train_ratio: float, proportion des donnees pour l'entrainement

        Returns:
            dict avec les metriques d'evaluation
        """
        from sklearn.ensemble import RandomForestClassifier
        from sklearn.metrics import classification_report, accuracy_score

        features = self.prepare_features(prices, vix_data, risk_free)
        labels = self.prepare_labels(prices, vix_data=vix_data)

        # Aligner features et labels
        common_idx = features.index.intersection(labels.index)
        X = features.loc[common_idx]
        y = labels.loc[common_idx]

        if len(X) < 100:
            print("[ML] Pas assez de donnees pour entrainer le modele "
                  f"({len(X)} observations)")
            return None

        # Split temporel
        split = int(len(X) * train_ratio)
        X_train, X_test = X.iloc[:split], X.iloc[split:]
        y_train, y_test = y.iloc[:split], y.iloc[split:]

        self.feature_names = X.columns.tolist()

        # Entrainement
        self.model = RandomForestClassifier(
            n_estimators=200,
            max_depth=5,
            min_samples_leaf=20,
            class_weight='balanced',
            random_state=42,
            n_jobs=-1
        )
        self.model.fit(X_train, y_train)

        # Evaluation out-of-sample
        y_pred_train = self.model.predict(X_train)
        y_pred_test = self.model.predict(X_test)

        results = {
            'train_accuracy': accuracy_score(y_train, y_pred_train),
            'test_accuracy': accuracy_score(y_test, y_pred_test),
            'train_size': len(X_train),
            'test_size': len(X_test),
            'class_report': classification_report(
                y_test, y_pred_test, output_dict=True
            ),
            'feature_importance': dict(zip(
                self.feature_names,
                self.model.feature_importances_
            )),
            'regime_distribution_train': y_train.value_counts().to_dict(),
            'regime_distribution_test': y_test.value_counts().to_dict(),
            'predictions_test': pd.Series(
                y_pred_test, index=y_test.index
            ),
            'actuals_test': y_test
        }

        return results

    def predict_current_regime(self, prices, vix_data=None, risk_free=None):
        """
        Predit le regime actuel avec le modele entraine.

        Returns:
            dict avec regime predit et probabilites
        """
        if self.model is None:
            print("[ML] Modele non entraine.")
            return None

        features = self.prepare_features(prices, vix_data, risk_free)
        if len(features) == 0:
            return None

        last_features = features.iloc[[-1]]
        prediction = self.model.predict(last_features)[0]
        probas = self.model.predict_proba(last_features)[0]
        classes = self.model.classes_

        return {
            'regime': prediction,
            'probabilities': dict(zip(classes, probas)),
            'date': features.index[-1]
        }

    def print_report(self, results):
        """Affiche le rapport d'entrainement du modele ML."""
        if results is None:
            return

        print(f"\n{'='*70}")
        print("DETECTION DE REGIME PAR ML (RANDOM FOREST)")
        print(f"{'='*70}")

        print(f"\n  Entrainement : {results['train_size']} observations")
        print(f"  Test (OOS)   : {results['test_size']} observations")
        print(f"  Accuracy train : {results['train_accuracy']:.1%}")
        print(f"  Accuracy test  : {results['test_accuracy']:.1%}")

        # Metriques par classe
        print(f"\n--- Performance par regime (out-of-sample) ---")
        report = results['class_report']
        for regime in ['bull', 'neutral', 'bear']:
            if regime in report:
                r = report[regime]
                print(f"  {regime:10s} | Precision: {r['precision']:.2f} | "
                      f"Recall: {r['recall']:.2f} | F1: {r['f1-score']:.2f} | "
                      f"Support: {r['support']}")

        # Feature importance
        print(f"\n--- Importance des features ---")
        fi = sorted(results['feature_importance'].items(),
                    key=lambda x: x[1], reverse=True)
        for fname, importance in fi:
            bar = "#" * int(importance * 50)
            print(f"  {fname:20s} : {importance:.3f}  {bar}")

        # Comparaison avec le detecteur existant (si possible)
        print(f"\n--- Distribution des regimes ---")
        print(f"  Train : {results['regime_distribution_train']}")
        print(f"  Test  : {results['regime_distribution_test']}")

        # Signaux de surapprentissage
        gap = results['train_accuracy'] - results['test_accuracy']
        if gap > 0.15:
            print(f"\n  [ATTENTION] Ecart train/test = {gap:.1%} "
                  f"-> risque de surapprentissage")
        elif gap < 0.05:
            print(f"\n  Ecart train/test = {gap:.1%} -> modele stable")

        print(f"{'='*70}\n")


# =========================================================================
# EXECUTION DES ANALYSES
# =========================================================================

def run_all_analyses(config, data_loader, factor_calc, risk_mgr,
                     backtest_results, prices, returns,
                     vix_data=None, risk_free=None):
    """
    Execute les trois analyses et affiche les rapports.

    Args:
        config: objet Config
        data_loader: DataLoader avec sector_map et esg_scores
        factor_calc: FactorCalculator
        risk_mgr: RiskManager
        backtest_results: liste de dicts du rolling backtester
        prices: pd.DataFrame des prix
        returns: pd.DataFrame des rendements
        vix_data: pd.Series du VIX
        risk_free: pd.Series du taux sans risque
    """
    print(f"\n{'#'*70}")
    print("# ANALYSES AVANCEES : SECTEURS, FACTEURS, ML")
    print(f"{'#'*70}")

    # --- 1. Attribution sectorielle ---
    sector_analyzer = SectorAnalyzer(config, data_loader)
    sector_results = sector_analyzer.compute_sector_attribution(
        backtest_results, returns
    )
    sector_analyzer.print_report(sector_results)

    # --- 2. Sensibilite factorielle ---
    sensitivity_analyzer = FactorSensitivityAnalyzer(config, factor_calc)

    # Calculer les facteurs sur les dernieres donnees disponibles
    decision_date = prices.index[-1]
    factors_df = factor_calc.compute_all_factors(
        prices, returns, decision_date
    )

    # Convertir le DataFrame en dict de Series (format attendu par le grid search)
    if isinstance(factors_df, pd.DataFrame):
        factors_dict = {col: factors_df[col] for col in factors_df.columns}
    else:
        factors_dict = factors_df

    sensitivity_results = sensitivity_analyzer.run_sensitivity_grid(
        factors_dict, returns, n_top=config.STOCK_SELECTION['n_stocks']
    )
    sensitivity_analyzer.print_report(sensitivity_results)

    # --- 3. ML regime detection ---
    ml_detector = MLRegimeDetector(config)
    ml_results = ml_detector.train(
        prices, vix_data=vix_data, risk_free=risk_free
    )
    ml_detector.print_report(ml_results)

    # Prediction du regime actuel
    current = ml_detector.predict_current_regime(
        prices, vix_data=vix_data, risk_free=risk_free
    )
    if current:
        print(f"\n  Regime ML actuel : {current['regime']}")
        print(f"  Probabilites    : {current['probabilities']}")
        print(f"  Date            : {current['date'].strftime('%Y-%m-%d')}")

    return {
        'sector': sector_results,
        'sensitivity': sensitivity_results,
        'ml_regime': ml_results,
        'ml_current': current,
        'ml_detector': ml_detector
    }

# =============================================================================
# EXECUTION DES ANALYSES AVANCEES
# =============================================================================
# Les analyses utilisent les resultats du backtest principal (variable 'output'
# definie dans la cellule precedente). Si le backtest n'a pas ete execute,
# cette section est ignoree.


if 'output' in dir() and output is not None:
    _bt = output.get('backtester', None)
    _wh = _bt.weights_history if _bt and hasattr(_bt, 'weights_history') else []

    if len(_wh) > 0:
        analysis_results = run_all_analyses(
            config=Config,
            data_loader=output['data_loader'],
            factor_calc=output['factor_calc'],
            risk_mgr=output['risk_mgr'],
            backtest_results=_wh,
            prices=output['prices'],
            returns=output['returns'],
            vix_data=output.get('vix_data', None),
            risk_free=output['data_loader'].risk_free_rate_series
        )
    else:
        print("[INFO] Aucun historique de poids disponible pour les analyses avancees.")
else:
    print("[INFO] Executer le backtest principal (cellule precedente) avant les analyses.")



# décommenté l'execution au besoin, car les analyses sont longues et on veut d'abord valider le backtest avant de les lancer.



######################################################################
# ANALYSES AVANCEES : SECTEURS, FACTEURS, ML
######################################################################

ATTRIBUTION SECTORIELLE

--- Contribution par secteur ---
  Technology                | Poids moy:  19.0% | Max:  37.0% | Contrib:  +0.2314
  Healthcare                | Poids moy:  19.3% | Max:  29.5% | Contrib:  +0.0611
  Consumer Defensive        | Poids moy:  16.1% | Max:  36.7% | Contrib:  +0.0494
  Financial Services        | Poids moy:   5.0% | Max:  19.4% | Contrib:  +0.0462
  Consumer Cyclical         | Poids moy:   3.1% | Max:  23.7% | Contrib:  +0.0291
  Communication Services    | Poids moy:   2.6% | Max:   8.5% | Contrib:  +0.0281
  Obligations               | Poids moy:  30.8% | Max:  57.9% | Contrib:  +0.0179
  Utilities                 | Poids moy:   0.1% | Max:   5.2% | Contrib:  +0.0030
  Energy                    | Poids moy:   0.4% | Max:  31.2% | Contrib:  +0.0027
  Real Estate  

---
# 10. Trading live — Algorithme de rééquilibrage
---


In [ ]:
# =============================================================================
# CELLULE 10:  LIVE EXECUTION — Algorithme de rééquilibrage en temps réel
# =============================================================================
#
# Ce module reproduit EXACTEMENT la logique du backtester (Cellule 7) pour
# générer les ordres de rééquilibrage du portefeuille en temps réel.
#
# Cohérence avec le backtest :
#   Décision sur données T-1 (clôture de la veille)
#   Détection de régime multi-signaux (identique à RiskManager)
#   Scores factoriels et optimisation Mean-Variance (identique)
#   Contraintes de liquidité (ADV, max position)
#   Contrôle du turnover + smoothing
#   Compliance mandat (crypto ≤5%, exotiques ≤5%)
#   Cash buffer permanent (2.5%) via ETF money-market (CASH.TO)
#   Allocation obligataire minimum 25% (garde-fou post-pipeline)
#   Output CSV conforme au format exigé (5 colonnes)
#
# SOURCES DE DONNÉES :
#   Yahoo Finance est utilisé omme source de prix.
#   Tous les prix sont convertis en CAD via
#   le taux USDCAD (DataLoader._convert_to_cad).
#
# FORMAT CSV DE SORTIE (5 colonnes) :
#   1. TICKER   : Identifiant Yahoo Finance du titre
#   2. WEIGHTRB : Pondération du titre après rebalancement (somme = 1)
#   3. QTYRB    : Quantité de titres détenus après rebalancement
#   4. VALUERB  : Valeur en CAD du titre après rebalancement
#   5. FEES     : Frais de transaction (financés par le cash buffer)
#
# Contraintes vérifiées :
#   - Somme(WEIGHTRB) = 1
#   - QTYRB × prix_clôture_t-1 ≈ VALUERB
#   - FEES >= 0
#   - Allocation bonds >= Config.MIN_BOND_ALLOC (25%)
#   - Poids min 1%, max 10% par position
#
# TICKERS — CORRESPONDANCE BLOOMBERG / REFINITIV / YAHOO :
#   Bloomberg         Refinitiv (RIC)    Yahoo Finance
#   AAPL US Equity    AAPL.OQ            AAPL
#   MSFT US Equity    MSFT.OQ            MSFT
#   RY CT Equity      RY.TO              RY.TO
#   AGG US Equity     AGG.P              AGG
#   XBB CT Equity     XBB.TO             XBB.TO
#   (Les tickers .TO sont identiques entre Refinitiv et Yahoo)
#
# =============================================================================

class LiveExecutor:
    """
    Module d'exécution live — miroir exact du RollingBacktester.

    Génère le journal de rebalancement CSV conforme aux exigences du cours,
    en réutilisant les mêmes modules que le backtest (FactorCalculator,
    RiskManager, PortfolioOptimizer, DataLoader).
    """

    # Fichier de persistance de l'état du portefeuille
    PORTFOLIO_STATE_FILE = 'portfolio_state.csv'

    # Fichier CSV de sortie (journal de rebalancement pour le prof)
    REBALANCING_OUTPUT_FILE = 'rebalancing_journal.csv'

    def __init__(self, config=Config):
        self.config = config
        self.data_loader = DataLoader(config)
        self.factor_calc = FactorCalculator(config)
        self.risk_mgr = RiskManager(config)
        self.optimizer = PortfolioOptimizer(config)

    # =========================================================================
    # 1. CHARGEMENT DES DONNÉES (identique au backtester)
    # =========================================================================
    def load_market_data(self, lookback_days=900):
        """
        Charge les données nécessaires avec un historique suffisant
        pour les calculs factoriels (252j) et la covariance (252j).
        Args:
        lookback_days: profondeur historique utilisée pour construire
                           l’univers de décision.

        """
        end_date = self.config.DATE_TODAY_PARSED
        start_date = end_date - timedelta(days=lookback_days)

        print(f"\n{'='*70}")
        print(f"CHARGEMENT DES DONNÉES — {end_date.strftime('%Y-%m-%d')}")
        print(f"{'='*70}")

        # Prix et volumes
        print("[1/4] Prix et volumes...")
        self.prices, self.returns = self.data_loader.load_data(
            start_date=start_date, end_date=end_date
        )

        # VIX
        print("[2/4] VIX...")
        self.vix_data = self.data_loader.load_vix(start_date, end_date)

        # Données macro (yield curve, credit spreads)
        print("[3/4] Données macro...")
        self.data_loader.load_macro_data(start_date, end_date)

        # Taux sans risque (déjà chargé dans load_data)
        print("[4/4] Taux sans risque...")

        print(f"\n[OK] Données chargées: {len(self.prices)} jours, "
              f"{len(self.prices.columns)} titres")

        return self.prices, self.returns

    # =========================================================================
    # 2. ÉTAT DU PORTEFEUILLE — Chargement depuis CSV précédent
    # =========================================================================
    def load_portfolio_state(self):
        """
        Charge l'état précédent du portefeuille.

        Si un fichier portfolio_state.csv existe :
          → Réévalue les positions aux prix de clôture t-1
          → Calcule les poids actuels (après drift de marché)

        Sinon (premier investissement) :
          → Capital initial, aucune position

        Returns:
            tuple: (total_market_value, current_weights_series, current_units_series)
              - total_market_value : float, valeur totale actuelle
              - current_weights    : pd.Series {TICKER: poids}, somme = 1
              - current_units      : pd.Series {TICKER: nb unités détenues}
        """
        # Dernier prix disponible = prix de clôture t-1 utilisé pour valoriser
        # les positions déjà détenues.
        prices_t1 = self.prices.iloc[-1]

        search_paths = [
            self.PORTFOLIO_STATE_FILE,                               # répertoire courant (/content/)
            f'/content/{self.PORTFOLIO_STATE_FILE}',                 # racine Colab (upload manuel)
            f'/content/drive/MyDrive/{self.PORTFOLIO_STATE_FILE}',   # Google Drive (si monté)
            os.path.join(os.getcwd(), self.PORTFOLIO_STATE_FILE),    # répertoire de travail courant
        ]

        found_path = None

        for path in search_paths:
            if os.path.exists(path):
                found_path = path
                print(f"  [INFO] Fichier portfolio_state trouvé : {found_path}")
                break

        if found_path is None:
            print(f"  [INFO] Chemins recherchés :")
            for p in search_paths:
                print(f"    {p}")

        if found_path is not None:
            # -----------------------------------------------------------
            # CORRECTIF 2 : sep=None détecte automatiquement ; ou ,
            # -----------------------------------------------------------
            df_prev = pd.read_csv(found_path, sep=None, engine='python')
            print(f"  [INFO] Colonnes détectées : {df_prev.columns.tolist()}")

            # --- Compatibilité ancien/nouveau format ---
            # Nouveau format : TICKER, WEIGHTRB, QTYRB, VALUERB
            # Ancien format  : TICKER, WEIGHTRB, VALUERCAD, VALUERB
            if 'QTYRB' in df_prev.columns:
                qty_col = 'QTYRB'
            elif 'VALUERB' in df_prev.columns and 'VALUERCAD' in df_prev.columns:
                qty_col = 'VALUERB'
                print("  [INFO] Ancien format détecté — migration automatique")
            else:
                print("  [ATTENTION]  Fichier portfolio_state.csv mal formé, "
                      "traité comme investissement initial.")
                return self.config.PORTFOLIO_VALUE, pd.Series(dtype=float), pd.Series(dtype=float)
            # Réévaluation aux prix t-1
            def get_price(ticker):
                return prices_t1.get(ticker, np.nan)

            df_prev['price_t1'] = df_prev['TICKER'].apply(get_price)

            # Exclure les tickers sans prix (delisted, erreur données)
            missing = df_prev[df_prev['price_t1'].isna()]
            if len(missing) > 0:
                tickers_missing = missing['TICKER'].tolist()
                print(f"  [ATTENTION]  {len(missing)} tickers sans prix: "
                      f"{', '.join(tickers_missing[:5])}... — positions liquidées.")
                df_prev = df_prev[df_prev['price_t1'].notna()].copy()

            # Valeur de marché actuelle de chaque position
            df_prev['market_value'] = df_prev[qty_col] * df_prev['price_t1']
            total_market_value = df_prev['market_value'].sum()

            if total_market_value <= 0:
                print("  [ATTENTION]  Valeur totale ≤ 0, traité comme investissement initial.")
                return self.config.PORTFOLIO_VALUE, pd.Series(dtype=float), pd.Series(dtype=float)

            # Poids actuels (après drift de marché)
            current_weights = pd.Series(
                (df_prev['market_value'] / total_market_value).values,
                index=df_prev['TICKER'].values
            )

            # Unités détenues
            current_units = pd.Series(
                df_prev[qty_col].values,
                index=df_prev['TICKER'].values
            )

            pnl = total_market_value - self.config.PORTFOLIO_VALUE
            print(f"\n ÉTAT DU PORTEFEUILLE (réévalué aux prix t-1):")
            print(f"  Valeur actuelle  : {total_market_value:,.2f} CAD")
            print(f"  P&L cumulatif    : {pnl:+,.2f} CAD ({pnl/self.config.PORTFOLIO_VALUE:+.2%})")
            print(f"  Positions        : {(current_weights > 1e-6).sum()}")

            return total_market_value, current_weights, current_units

        else:
            print(f"\n Premier investissement — Capital: "
                  f"{self.config.PORTFOLIO_VALUE:,.0f} CAD")
            return self.config.PORTFOLIO_VALUE, pd.Series(dtype=float), pd.Series(dtype=float)

    # =========================================================================
    # 3. PORTEFEUILLE CIBLE — Miroir exact du backtester
    # =========================================================================
    def compute_target_portfolio(self, prev_weights_stocks=None):
        """
        Calcule le portefeuille cible — logique identique au backtester.

        La décision est prise sur les données T-1 (clôture de la veille),
        conformément à la protection anti look-ahead du backtest.

        Args:
            prev_weights_stocks: pd.Series, poids précédents (poche actions
                                 uniquement, normalisés) pour le smoothing.
                                 None si premier investissement.

        Returns:
            dict avec weights, allocations, régime, turnover, compliance
        """
        print(f"\n{'='*70}")
        print(f"CALCUL DU PORTEFEUILLE CIBLE")
        print(f"{'='*70}")

        # =================================================================
        # ÉTAPE 1 — Données T-1 (identique au backtester)
        # =================================================================
        decision_date = self.prices.index[-1]
        print(f"[1/6] Date de décision (clôture t-1): "
              f"{decision_date.strftime('%Y-%m-%d')}")

        available_prices = self.prices.loc[:decision_date]
        available_returns = self.returns.loc[:decision_date]

        vix_slice = (self.vix_data.loc[:decision_date]
                     if self.vix_data is not None else None)

        macro_slice = {}
        if hasattr(self.data_loader, 'macro_data'):
            for key, series in self.data_loader.macro_data.items():
                if series is not None:
                    macro_slice[key] = series.loc[:decision_date]

        # =================================================================
        # ÉTAPE 2 — Détection de régime (identique au backtester)
        # =================================================================
        print("[2/6] Détection de régime...")
        regime_output = self.risk_mgr.detect_regime_full(
            available_prices, vix_slice, macro_slice
        )
        regime = regime_output.regime
        multiplier = regime_output.exposure_multiplier

        print(f"  Régime        : {regime}")
        print(f"  Confiance     : {regime_output.confidence:.2f}")
        print(f"  Score comp.   : {regime_output.composite_score:.3f}")
        print(f"  Multiplicateur: {multiplier:.2f}")
        print(f"  Overlay crash : {' [ATTENTION] ACTIF ' if regime_output.overlay_on else 'inactif'}")

        # =================================================================
        # ÉTAPE 3 — Allocation par régime (identique au backtester)
        # =================================================================
        print("[3/6] Allocation par régime...")
        base_alloc = self.config.REGIME_ALLOCATIONS[regime]
        base_stock = float(base_alloc['stocks'])
        base_bond = float(base_alloc['bonds'])
        base_cash = float(base_alloc.get('cash',
                          self.config.CASH_BUFFER['percentage']))

        stock_alloc = base_stock * multiplier
        bond_alloc = base_bond + (base_stock - stock_alloc)
        cash_alloc = base_cash

        if bond_alloc < 0:
            cash_alloc += abs(bond_alloc)
            bond_alloc = 0.0

        total = stock_alloc + bond_alloc + cash_alloc
        stock_alloc /= total
        bond_alloc /= total
        cash_alloc /= total

        # Garde-fou bonds >= 25% (APRES overlay)
        if bond_alloc < self.config.MIN_BOND_ALLOC:
            deficit = self.config.MIN_BOND_ALLOC - bond_alloc
            stock_alloc = max(0.0, stock_alloc - deficit)
            bond_alloc = self.config.MIN_BOND_ALLOC
            total2 = stock_alloc + bond_alloc + cash_alloc
            stock_alloc /= total2
            bond_alloc /= total2
            cash_alloc /= total2


        print(f"  Actions: {stock_alloc:.1%} | Obligations: {bond_alloc:.1%} "
              f"| Cash: {cash_alloc:.1%}")

        # =================================================================
        # ÉTAPE 4 — Scores factoriels ACTIONS
        # =================================================================
        print("[4/8] Scores factoriels (actions)...")
        factors = self.factor_calc.compute_all_factors(
            available_prices, available_returns, decision_date, regime=regime
        )
        _use_binary = getattr(self.config, 'USE_BINARY_REGIME_WEIGHTS', False)
        composite = self.factor_calc.compute_composite_score(
            factors, regime=regime,
            regime_score=None if _use_binary else float(regime_output.composite_score)
        )
       # Ensemble des tickers obligataires pour séparer actions et bonds
        bond_set = (set(self.data_loader.US_BOND_ETFS) |
                    set(self.data_loader.CA_BOND_ETFS))

        valid_tickers = composite.dropna().index.tolist()
        valid_tickers = [t for t in valid_tickers
                         if t in available_returns.columns]

        # Garder seulement les actions pour les scores
        stock_tickers = [t for t in valid_tickers if t not in bond_set]

        # Filtrage liquidité
        if (hasattr(self.config, 'LIQUIDITY') and
                self.config.LIQUIDITY.get('enabled', False)):
            liquid_tickers = self.data_loader.get_liquid_tickers(decision_date)
            stock_tickers = [t for t in stock_tickers if t in liquid_tickers]
        # Exclusions éventuelles définies dans le DataLoader
        if hasattr(self.data_loader, 'filter_excluded_tickers'):
            stock_tickers = self.data_loader.filter_excluded_tickers(stock_tickers)

        if len(stock_tickers) < 20:
            print(f"  [ATTENTION]  Seulement {len(stock_tickers)} actions valides!")
            return None

        print(f"  Actions éligibles avant pre-filtrage: {len(stock_tickers)}")
        stock_scores = composite[stock_tickers]

        # Pre-filtrage top-N : même logique qu'optimize_joint en interne.
        # Ceci assure que compute_covariance reçoit ~50 titres (pas 700+)
        # → matrice de covariance mieux conditionnée (n > p).
        _n_stocks_cfg = self.config.STOCK_SELECTION['n_stocks']
        _n_top = min(_n_stocks_cfg * 2, len(stock_tickers))
        stock_tickers = stock_scores.nlargest(_n_top).index.tolist()
        stock_scores  = composite[stock_tickers]
        print(f"  Pré-filtrage top-{_n_top}: {len(stock_tickers)} stocks retenus")

        # =================================================================
        # ÉTAPE 5 — BondSelector (sélection des candidats)
        # =================================================================
        print("[5/8] Sélection des bonds...")
        bond_selector = BondSelector(self.config)
        bond_candidates = bond_selector.select_bond_candidates(
            regime, available_returns
        )
        print(bond_selector.get_selection_report(bond_candidates, bond_alloc))

        # =================================================================
        # ÉTAPE 6 — Covariance conjointe + Optimisation MV
        # =================================================================
        print("[6/8] Optimisation conjointe stocks + bonds...")

        all_opt_tickers = stock_tickers + list(bond_candidates.keys())
        all_opt_tickers = [t for t in all_opt_tickers
                          if t in available_returns.columns]

        cov = self.risk_mgr.compute_covariance(
            available_returns[all_opt_tickers]
        )
        cov_df = pd.DataFrame(cov, index=all_opt_tickers,
                              columns=all_opt_tickers)

        liq_constraints = None
        if hasattr(self.data_loader, 'get_liquidity_constraints'):
            liq_constraints = self.data_loader.get_liquidity_constraints(
                stock_tickers, decision_date
            )

        weights = self.optimizer.optimize_joint(
            stock_scores=stock_scores,
            bond_candidates=bond_candidates,
            cov_matrix=cov_df,
            stock_alloc=stock_alloc,
            bond_alloc=bond_alloc,
            prev_weights=prev_weights_stocks,
            liquidity_constraints=liq_constraints,
            current_regime=regime,
            )


        # =================================================================
        # ÉTAPE 5 — Contrôle du turnover (identique au backtester)
        # =================================================================
        print("[7/8] Contrôle du turnover...")

        if prev_weights_stocks is not None and len(prev_weights_stocks) > 0:
            bond_set_full = set(self.data_loader.US_BOND_ETFS) | set(self.data_loader.CA_BOND_ETFS)
            cash_ticker = self.config.CASH_BUFFER.get('ticker', 'CASH.TO')

            # Comparer seulement la poche actions pour estimer le turnover,
            # sans modifier le portefeuille cible complet.
            new_stock_w = weights[
                ~weights.index.isin(list(bond_set_full) + [cash_ticker])
            ].copy()

            if new_stock_w.sum() > 0:
                new_stock_w = new_stock_w / new_stock_w.sum()

            turnover = (
                new_stock_w.reindex(prev_weights_stocks.index).fillna(0.0)
                - prev_weights_stocks
            ).abs().sum()
        else:
            turnover = 1.0

        print(f"  Turnover estimé: {turnover:.1%}")

        # --- Integration ESG ---
        if hasattr(self.config, "ESG") and self.config.ESG.get("enabled", False):
            weights = apply_esg_tilt(
                weights,
                esg_scores=self.data_loader.esg_scores.loc[weights.index],
                strength=self.config.ESG.get("tilt_strength", 0.35)
            )

        # =================================================================
        # ÉTAPE 6 — Compliance mandat (identique au backtester)
        # =================================================================
        print("[8/8] Vérification compliance...")
        cst = self.config.CONSTRAINTS
        weights, compliance = self.data_loader.apply_allocation_constraints(
            weights,
            stock_alloc=stock_alloc,
            bond_alloc=bond_alloc,
            cash_alloc=self.config.CASH_BUFFER['percentage'],
            allow_crypto=bool(cst.get('allow_crypto_etfs', True)),
            max_exotic=float(cst.get('max_exotic_etfs_weight', 0.05)),
            max_crypto=float(cst.get('max_crypto_etfs_weight', 0.05)),
        )

        # --- Diversification sectorielle ---
        if hasattr(self.config, "SECTOR_DIVERSIFICATION") and \
                self.config.SECTOR_DIVERSIFICATION.get("enabled", False):
            weights = apply_sector_cap(
                weights,
                sector_map=self.data_loader.sector_map,
                max_sector_weight=self.config.SECTOR_DIVERSIFICATION.get(
                    "max_sector_weight", 0.25
                )
            )

        # --- Bornes min/max par position ---
        weights = enforce_weight_bounds(
            weights,
            min_weight=self.config.STOCK_SELECTION['min_weight'],
            max_weight=self.config.STOCK_SELECTION['max_weight']
        )


        # Turnover cap
        cap = cst.get('turnover_cap', None)
        if cap is not None and prev_weights_stocks is not None:
            raw_turn = (weights.reindex(prev_weights_stocks.index).fillna(0)
                        - prev_weights_stocks).abs().sum()
            if raw_turn > cap:
                alpha = cap / raw_turn
                weights = (prev_weights_stocks.fillna(0) * (1 - alpha)
                           + weights.fillna(0) * alpha)
                print(f"  [ATTENTION]  Turnover cappé à {cap:.0%}")

        # Finalisation stricte finale pour garantir:
        # - max 10% par position
        # - min 1%
        # - bond floor
        # - cash target:
        bond_set_full = set(self.data_loader.US_BOND_ETFS) | set(self.data_loader.CA_BOND_ETFS)
        cash_ticker = self.config.CASH_BUFFER.get('ticker', 'CASH.TO')

        weights = finalize_weights_with_bond_floor(
            weights=weights,
            bond_set=bond_set_full,
            cash_ticker=cash_ticker,
            min_bond_alloc=float(self.config.MIN_BOND_ALLOC),
            min_weight=float(self.config.STOCK_SELECTION['min_weight']),
            max_weight=float(self.config.STOCK_SELECTION['max_weight']),
            cash_target=float(self.config.CASH_BUFFER['percentage']),
            decimals=6
        )

        #Debug de controle pour la finalisation stricte finale
        active = weights[(weights > 1e-8) & (weights.index != cash_ticker)]
        above_max = active[active > self.config.STOCK_SELECTION['max_weight'] + 1e-9]

        if len(above_max) > 0:
            print("[DEBUG] Positions > max_weight après finalisation:")
            print(above_max.sort_values(ascending=False))

        n_positions = (weights > self.config.STOCK_SELECTION['min_weight'] / 2).sum()
        print(f"  Positions actives: {n_positions}")


        return {
            'weights': weights,
            'stock_alloc': stock_alloc,
            'bond_alloc': bond_alloc,
            'cash_alloc': cash_alloc,
            'regime': regime,
            'regime_output': regime_output,
            'turnover': turnover,
            'compliance': compliance,
            'decision_date': decision_date,
        }

    # =========================================================================
    # 4. JOURNAL DE REBALANCEMENT — Format CSV exigé par le cours
    # =========================================================================
    def generate_rebalancing_journal(self, target, total_market_value,
                                     current_weights):
        """
        Génère le journal de rebalancement au format CSV exigé.

        Format (5 colonnes) :
          TICKER, WEIGHTRB, QTYRB, VALUERB, FEES

        Contraintes respectées :
          - Somme(WEIGHTRB) = 1
          - QTYRB × prix_clôture_t-1 ≈ VALUERB
          - FEES >= 0 (financés par le cash buffer)
          - Allocation bonds >= Config.MIN_BOND_ALLOC (25%)
          - Poids min 1%, max 10% par position

        Args:
            target: dict retourné par compute_target_portfolio
            total_market_value: valeur actuelle du portefeuille (après PnL)
            current_weights: pd.Series {TICKER: poids_t-1}

        Returns:
            pd.DataFrame: journal conforme au format exigé (5 colonnes)

        """
        weights = target['weights']
        stock_alloc = target['stock_alloc']
        bond_alloc = target['bond_alloc']
        cash_alloc = target['cash_alloc']

        # Prix de clôture t-1 (base pour tous les calculs de quantités)
        prices_t1 = self.prices.iloc[-1]

        def is_cad_ticker(t):
            return t.endswith((".TO", ".NE", ".V", ".CN"))

        fx = self.data_loader._load_fx_rate(
            start=self.prices.index[-2].date(),
            end=self.prices.index[-1].date() + timedelta(days=1)
        )
        fx_t1 = float(fx.iloc[-1])

        def get_price_cad_t1(ticker):
            p = prices_t1.get(ticker, np.nan)
            if np.isnan(p):
                return np.nan
            return p if is_cad_ticker(ticker) else p * fx_t1

        # Classification des titres
        bond_set = (set(self.data_loader.US_BOND_ETFS) |
                    set(self.data_loader.CA_BOND_ETFS))
        # =================================================================
        # Construction des poids WEIGHTRB (stocks + bonds + cash = 100%)
        # IMPORTANT: on ne renormalise plus séparément stocks et bonds,
        # car cela peut pousser certaines obligations au-dessus de 10%.
        # On finalise directement les poids du portefeuille cible.
        # =================================================================
        cash_ticker = self.config.CASH_BUFFER['ticker']
        bond_set_full = set(self.data_loader.US_BOND_ETFS) | set(self.data_loader.CA_BOND_ETFS)


        final_weights = weights.copy()


        # Vérification informative
        actual_bond_w = float(final_weights[final_weights.index.isin(bond_set_full)].sum())
        if actual_bond_w < self.config.MIN_BOND_ALLOC - 1e-6:
            print(f"  [ATTENTION] Bonds = {actual_bond_w:.2%} < min "
                  f"{self.config.MIN_BOND_ALLOC:.0%}")


        all_tickers = sorted(set(
            list(final_weights.index) +
            list(current_weights.index)
        ))

        def force_sum_exact_safe(w: pd.Series, cash_ticker: str, max_w: float, decimals=6):
            w = w / w.sum()
            w = w.round(decimals)
            r = 1.0 - float(w.sum())
            if abs(r) < 10**(-decimals):
                return w

            # priorité au cash (évite de casser les contraintes sur les actifs risqués)
            if cash_ticker in w.index and w[cash_ticker] + r >= 0:
                w[cash_ticker] = round(float(w[cash_ticker] + r), decimals)
                return w

            # sinon: ticker avec le plus de slack sous max_w
            slack = (max_w - w).where((max_w - w) > 0, -np.inf)
            idx = slack.idxmax()
            w[idx] = round(float(w[idx] + r), decimals)
            return w

        # Aligner les séries sur l'union complète
        weightrb = final_weights.reindex(all_tickers).fillna(0.0)
        weightrb = weightrb.round(6)

        residual = round(1.0 - float(weightrb.sum()), 6)
        if abs(residual) >= 1e-6:
            cash_ticker = self.config.CASH_BUFFER['ticker']
            weightrb.loc[cash_ticker] = round(
                float(weightrb.get(cash_ticker, 0.0) + residual), 6
            )

        weight_prev = current_weights.reindex(all_tickers).fillna(0.0)
        # =================================================================
        # Calculs conformes aux exigences
        # =================================================================

        # 1. Delta de poids (pour mask_keep)
        weightdelta = weightrb - weight_prev

        # 2. Valeur en CAD (= VALUERB dans le CSV)
        value_cad = weightrb * total_market_value


        # 3. Prix de clôture t-1 pour chaque ticker
        def get_price_t1(ticker):
            p = prices_t1.get(ticker, np.nan)
            return p if not np.isnan(p) else np.nan

        price_t1 = pd.Series(
            {t: get_price_t1(t) for t in all_tickers}
        )


        # 4. VALUERB = VALUERCAD / prix_clôture_{t-1}
        qty = value_cad / price_t1
        # ---------------------------------------------------------------------
        # Calcul des frais de transaction
        # --------------------------------------------------------------------
        tc_cfg = self.config.TRANSACTION_COSTS
        if tc_cfg.get('enabled', False):
            bond_set_fees = (set(self.data_loader.US_BOND_ETFS) |
                             set(self.data_loader.CA_BOND_ETFS))
            min_trade = self.config.TURNOVER_CONTROL.get('min_trade_size', 0.01)
            fixed_cost = float(tc_cfg.get('fixed_cost',0))
            fees = pd.Series(0.0, index=all_tickers)
            for ticker in all_tickers:
                delta_w = abs(weightdelta.get(ticker,0))

                #Ignore les micro-trades
                if delta_w < min_trade:
                  weightdelta[ticker] = 0
                  continue

                delta_cad = delta_w * total_market_value
                if ticker in bond_set_fees:
                    bps = float(tc_cfg.get('bonds_bps', 0))
                else:
                    bps = float(tc_cfg.get('stocks_bps', 0))
                fees[ticker] = delta_cad * (bps / 10000.0)

                #Frais fixes de 1000$ par ticker effectivement transigé
                if abs(weightdelta.get(ticker,0))>1e-6:
                  fees[ticker] += fixed_cost
            total_fees = fees.sum()

            # Financer depuis le cash buffer
            cash_ticker = self.config.CASH_BUFFER['ticker']
            if cash_ticker in weightrb.index and total_fees > 0:
                cash_value = weightrb[cash_ticker] * total_market_value
                if cash_value >= total_fees:
                    weightrb[cash_ticker] -= total_fees / total_market_value
                    weightrb = weightrb / weightrb.sum()
                    value_cad = weightrb * total_market_value
                    qty = value_cad / price_t1
                    print(f"  Frais totaux: {total_fees:,.2f} CAD — financés par {cash_ticker}")
                else:
                    print(f"  [ATTENTION] Frais ({total_fees:,.2f}) > cash dispo ({cash_value:,.2f})")
        else:
            fees = pd.Series(0.0, index=all_tickers)


        # =================================================================
        # Assemblage du DataFrame final (format exigé)
        # =================================================================
        df_journal = pd.DataFrame({
            'TICKER': all_tickers,
            'WEIGHTRB': weightrb.values,
            'QTYRB': qty.values,
            'VALUERB': value_cad.values,
            'FEES': fees.values,

        })

        # Retirer les lignes où le ticker n'a pas de prix et n'a aucune
        # position (ni avant ni après)
        mask_keep = (df_journal['WEIGHTRB'].abs() > 1e-8)
        df_journal = df_journal[mask_keep].copy()

        # Trier par poids décroissant
        df_journal = df_journal.sort_values('WEIGHTRB', ascending=False)
        df_journal = df_journal.reset_index(drop=True)

        return df_journal

    # =========================================================================
    # 5. VALIDATION MATHÉMATIQUE
    # =========================================================================
    def validate_journal(self, df_journal, total_market_value, current_weights):
        """
        Validation conforme aux exigences du cours (format 5 colonnes) :
          1. Somme(WEIGHTRB) = 1
          2. QTYRB × prix_t1 ≈ VALUERB
          3. FEES >= 0
        """
        print(f"\n{'='*70}")
        print("VALIDATION MATHÉMATIQUE DU JOURNAL")
        print(f"{'='*70}")

        all_ok = True

        # --- 1. Somme(WEIGHTRB) = 1 ---
        sum_w = df_journal['WEIGHTRB'].sum()
        check_1 = np.isclose(sum_w, 1.0, atol=1e-4)
        print(f"  Somme(WEIGHTRB) = {sum_w:.6f} : "
              f"{'[OK]' if check_1 else '[ERREUR]'}")
        if not check_1:
            all_ok = False

        # --- 2. Cohérence QTYRB × prix_t1 ≈ VALUERB ---
        prices_t1 = self.prices.iloc[-1]
        errors = []
        for _, row in df_journal.iterrows():
            t = row['TICKER']
            p = prices_t1.get(t, np.nan)
            if np.isnan(p):
                continue

            if abs(row['QTYRB']) > 1e-8:
                recalc = row['QTYRB'] * p
                if not np.isclose(recalc, row['VALUERB'], atol=1.0):
                    errors.append((t, row['VALUERB'], recalc))

        check_2 = len(errors) == 0
        print(f"  QTYRB × prix_t1 ≈ VALUERB : "
              f"{'[OK]' if check_2 else '[ERREUR]'}")
        if not check_2:
            for t, expected, got in errors[:5]:
                print(f"    {t}: attendu {expected:,.2f}, recalculé {got:,.2f}")
            all_ok = False

        # --- 3. FEES >= 0 ---
        check_3 = (df_journal['FEES'] >= 0).all()
        print(f"  FEES >= 0 partout : {'[OK]' if check_3 else '[ERREUR]'}")
        if not check_3:
            all_ok = False

        # --- 4. Nombre de positions (informatif) ---
        n_pos = (df_journal['WEIGHTRB'] > 1e-4).sum()
        print(f"  Positions actives : {n_pos}")

        # --- 4b. Min weight (1%) — aucune position active < min_weight ---
        min_w = self.config.STOCK_SELECTION['min_weight']
        cash_ticker = self.config.CASH_BUFFER['ticker']
        active = df_journal[
            (df_journal['WEIGHTRB'] > 1e-4) &
            (~df_journal['TICKER'].isin([cash_ticker]))
        ]
        below_min = active[active['WEIGHTRB'] < min_w - 1e-4]
        check_4b = len(below_min) == 0
        print(f"  Min weight ({min_w:.0%}) respecté : "
              f"{'[OK]' if check_4b else '[ERREUR] ' + str(len(below_min)) + ' positions'}")
        if not check_4b:
            for _, row in below_min.iterrows():
                print(f"    {row['TICKER']}: {row['WEIGHTRB']:.2%} < {min_w:.0%}")
            all_ok = False

        # --- 4c. Max weight (10%) — aucune position > max_weight ---
        max_w = self.config.STOCK_SELECTION['max_weight']
        above_max = active[active['WEIGHTRB'] > max_w + 1e-4]
        check_4c = len(above_max) == 0
        print(f"  Max weight ({max_w:.0%}) respecté : "
              f"{'[OK]' if check_4c else '[ERREUR] ' + str(len(above_max)) + ' positions'}")
        if not check_4c:
            for _, row in above_max.iterrows():
                print(f"    {row['TICKER']}: {row['WEIGHTRB']:.2%} > {max_w:.0%}")
            all_ok = False


        # --- 5. Bonds >= 25% (informatif) ---
        bond_set = (set(self.data_loader.US_BOND_ETFS) |
                    set(self.data_loader.CA_BOND_ETFS))
        bond_w = df_journal[df_journal['TICKER'].isin(bond_set)]['WEIGHTRB'].sum()
        check_bonds = bond_w >= self.config.MIN_BOND_ALLOC - 1e-4
        print(f"  Poids bonds = {bond_w:.2%} (min {self.config.MIN_BOND_ALLOC:.0%}) : "
              f"{'[OK]' if check_bonds else '[ERREUR]'}")
        if not check_bonds:
            all_ok = False

        # Conclusion
        print(f"\n  {'[OK] JOURNAL VALIDE' if all_ok else '[ATTENTION]  ANOMALIES DÉTECTÉES'}")
        print(f"{'='*70}")

        return all_ok

    # =========================================================================
    # 6. FRAIS, SAUVEGARDE ET RAPPORT
    # =========================================================================
    def compute_weekly_fees(self, total_market_value, weekly_gross_return=None):
        """
        Calcule les frais de gestion et la commission de performance pour la semaine.

        Appelé chaque lundi lors de l'exécution live.

        Args:
            total_market_value: NAV actuelle (avant déduction des frais).
            weekly_gross_return: rendement brut de la semaine (float, optionnel).
                                 Si None, seuls les frais fixes sont calculés.

        Returns:
            dict avec 'mgmt_fee', 'perf_fee', 'total_fees'
        """
        fee_cfg = getattr(self.config, 'MANAGEMENT_FEES', {})
        mgmt_fee_annual  = fee_cfg.get('mgmt_fee_annual',  0.0080)
        hurdle_annual    = fee_cfg.get('hurdle_rate_annual', 0.12)
        perf_fee_rate    = fee_cfg.get('perf_fee_rate', 0.20)
        weekly_hurdle    = (1 + hurdle_annual) ** (1 / 52) - 1

        mgmt_fee  = total_market_value * (mgmt_fee_annual / 52)
        perf_fee  = 0.0

        if weekly_gross_return is not None and weekly_gross_return > weekly_hurdle:
            excess   = weekly_gross_return - weekly_hurdle
            perf_fee = excess * total_market_value * perf_fee_rate

        total_fees = mgmt_fee + perf_fee

        print(f"\n{'='*70}")
        print("FRAIS DE GESTION — SEMAINE EN COURS")
        print(f"{'='*70}")
        print(f"  NAV avant frais          : ${total_market_value:>15,.2f} CAD")
        print(f"  Frais de gestion (0.80%/52): ${mgmt_fee:>13,.2f} CAD")
        if weekly_gross_return is not None:
            print(f"  Rendement brut semaine   : {weekly_gross_return:>13.4%}")
            print(f"  Hurdle hebdomadaire      : {weekly_hurdle:>13.4%}")
        if perf_fee > 0:
            print(f"  Commission performance   : ${perf_fee:>13,.2f} CAD  (20% surplus)")
        else:
            print(f"  Commission performance   : ${'0':>13}  (sous le hurdle)")
        print(f"  Total frais gestionnaire : ${total_fees:>13,.2f} CAD")
        print(f"  NAV après frais          : ${total_market_value - total_fees:>15,.2f} CAD")
        print(f"{'='*70}")

        return {'mgmt_fee': mgmt_fee, 'perf_fee': perf_fee, 'total_fees': total_fees}

    def save_and_report(self, df_journal, target, total_market_value):
        """
        Sauvegarde le CSV de sortie et affiche le rapport.

        Produit deux fichiers :
          1. rebalancing_journal.csv → CSV exigé par le cours (5 colonnes)
          2. portfolio_state.csv    → État interne pour le prochain cycle
        """
        decision_date = target['decision_date']

        # =================================================================
        # 1. Sauvegarder le journal (format exigé)
        # =================================================================
        df_journal.to_csv(self.REBALANCING_OUTPUT_FILE, index=False)
        print(f"\n Journal sauvegardé : {self.REBALANCING_OUTPUT_FILE}")

        # =================================================================
        # 2. Sauvegarder l'état pour le prochain cycle
        #    (TICKER + VALUERB pour réévaluation au prochain run)
        # =================================================================
        df_state = df_journal[['TICKER', 'WEIGHTRB', 'QTYRB', 'VALUERB']].copy()
        df_state = df_state[df_state['QTYRB'].abs() > 1e-8]
        df_state.to_csv(self.PORTFOLIO_STATE_FILE, index=False)
        print(f" État sauvegardé   : {self.PORTFOLIO_STATE_FILE}")

        try:
            from google.colab import files
            files.download(self.PORTFOLIO_STATE_FILE)
            files.download(self.REBALANCING_OUTPUT_FILE)
            print(" Téléchargement lancé automatiquement.")
        except ImportError:
          print(" [INFO] Pas dans Colab - fichiers disponibles localement")

        # =================================================================
        # 3. Rapport de positionnement
        # =================================================================
        print(f"\n{'='*90}")
        print(f"RAPPORT DE POSITIONNEMENT — "
              f"{self.config.DATE_TODAY_PARSED.strftime('%Y-%m-%d')}")
        print(f"Décision basée sur clôture du "
              f"{decision_date.strftime('%Y-%m-%d')}")
        print(f"{'='*90}")

        # Régime
        ro = target['regime_output']
        print(f"\n RÉGIME: {target['regime']} | "
              f"Confiance: {ro.confidence:.0%} | "
              f"Overlay: {'ACTIF' if ro.overlay_on else 'OFF'}")

        # Allocation
        print(f"\n ALLOCATION:")
        print(f"  Actions     : {target['stock_alloc']:.1%}")
        print(f"  Obligations : {target['bond_alloc']:.1%}")
        print(f"  Cash        : {target['cash_alloc']:.1%}")

        # Cash
        cash_ticker = self.config.CASH_BUFFER['ticker']
        cash_row = df_journal[df_journal['TICKER'] == cash_ticker]
        if len(cash_row) > 0:
            print(f"\n LIQUIDITÉS: "
                  f"{cash_row['VALUERB'].values[0]:,.2f} CAD "
                  f"({cash_row['WEIGHTRB'].values[0]:.2%})")

        # Calcul local des deltas
        prev_w = current_weights if hasattr(self, '_last_current_weights') else pd.Series(dtype=float)
        # On peut recalculer depuis le journal :
        print(f"\n POSITIONS ET ORDRES:")
        print(f"  {'Ticker':<12} {'Poids':>10} {'Valeur CAD':>15} "
              f"{'Quantité':>12} {'Frais':>10}")
        print("  " + "-" * 62)
        for _, row in df_journal.iterrows():
            if row['WEIGHTRB'] > 1e-4:
                print(f"  {row['TICKER']:<12} {row['WEIGHTRB']:>9.2%} "
                      f"{row['VALUERB']:>14,.0f} "
                      f"{row['QTYRB']:>11,.2f} "
                      f"{row['FEES']:>9,.2f}")

         # position final
        cash_ticker = self.config.CASH_BUFFER['ticker']
        positions = df_journal[
            (df_journal['TICKER'] != cash_ticker) &
            (df_journal['WEIGHTRB'] > 1e-4)
        ]
        print(f"\n POSITIONS FINALES ({len(positions)} titres hors cash):")
        print(f"  {'Ticker':<12} {'Poids':>10} {'Valeur CAD':>15} {'Quantité':>12}")
        print("  " + "-" * 52)
        for _, row in positions.head(20).iterrows():
             print(f"  {row['TICKER']:<12} {row['WEIGHTRB']:>9.2%} "
                  f"{row['VALUERB']:>14,.0f} {row['QTYRB']:>11,.2f}")
        if len(positions) > 20:
            print(f"  ... et {len(positions) - 20} autres positions")

        # Turnover
        print(f"\n Turnover: {target['turnover']:.1%}")
        print(f"\n{'='*90}")
    # =========================================================================
    # 7. EXÉCUTION COMPLÈTE
    # =========================================================================
    def run(self):
        """
        Point d'entrée principal — exécute le cycle complet :
          1. Charger les données de marché
          2. Charger l'état actuel du portefeuille
          3. Calculer le portefeuille cible (identique au backtester)
          4. Générer le journal de rebalancement (format CSV exigé)
          5. Valider mathématiquement
          6. Sauvegarder et rapporter

        Returns:
            dict avec journal, target, validation
        """
        print("\n" + "=" * 70)
        print("GSF-3120 — RÉÉQUILIBRAGE LIVE")
        print(f"Date: {self.config.DATE_TODAY_PARSED.strftime('%Y-%m-%d')}")
        print("=" * 70)

        # 1. Données
        self.load_market_data()

        asof = self.config.DATE_TODAY_PARSED.date()
        can_trade_ca = is_session(TSX_CAL, asof)
        can_trade_us = is_session(NYSE_CAL, asof)
        print(f"[CAL] TSX open={can_trade_ca} | NYSE open={can_trade_us}")

        # 2. État actuel
        total_market_value, current_weights, current_units = (
            self.load_portfolio_state()
        )

        # Extraire les poids actions pour le smoothing du backtester
        cash_ticker = self.config.CASH_BUFFER['ticker']
        prev_weights_stocks = None
        if len(current_weights) > 0:
            bond_set = (set(self.data_loader.US_BOND_ETFS) |
                        set(self.data_loader.CA_BOND_ETFS))
            stock_w = current_weights[
                ~current_weights.index.isin(list(bond_set) + [cash_ticker])
            ]
            if stock_w.sum() > 0:
                prev_weights_stocks = stock_w / stock_w.sum()

        # 3. Portefeuille cible
        target = self.compute_target_portfolio(prev_weights_stocks)
        if target is None:
            print("[ATTENTION]  Impossible de calculer le portefeuille cible.")
            return None
        min_turnover = self.config.TURNOVER_CONTROL.get('rebalance_threshold',0.01)
        if target['turnover'] < min_turnover and len(current_weights)>0:
          print(f"\n[SKIP] Turnover estimé ({target['turnover']:.2%}) < seuil ({min_turnover:.2%})")
          print("  → Aucun rebalancement effectué, portefeuille inchangé.")
          bond_set_full = set(self.data_loader.US_BOND_ETFS) | set(self.data_loader.CA_BOND_ETFS)

          print("\n[DEBUG compute_target_portfolio]")
          print("weights.index sample:", list(weights.index[:20]))
          print("bond tickers in weights:", [t for t in weights.index if t in bond_set_full])
          print("sum bonds in target weights:", float(weights[weights.index.isin(bond_set_full)].sum()))
          print("n bonds in target weights:", int(weights.index.isin(bond_set_full).sum()))
          return {'journal':None, 'target':target,
                  'total_market_value': total_market_value, 'is_valid':True}
        # 3b. Frais de gestion live (si lundi)
        import datetime as _dt
        _today = self.config.DATE_TODAY_PARSED
        if isinstance(_today, _dt.datetime):
            _weekday = _today.weekday()
        else:
            _weekday = _today.weekday() if hasattr(_today, 'weekday') else -1
        if _weekday == 0:
            self.compute_weekly_fees(total_market_value)

        # 3c. No-trade bands (cohérence avec le backtester)
        if len(current_weights) > 0:
            _bond_set_ntb = (set(self.data_loader.US_BOND_ETFS) |
                             set(self.data_loader.CA_BOND_ETFS))
            target['weights'] = self.optimizer.apply_turnover_control(
                target['weights'], current_weights,
                bond_tickers=list(_bond_set_ntb)
            )

        # 4. Journal de rebalancement
        df_journal = self.generate_rebalancing_journal(
            target, total_market_value, current_weights
        )

        # 5. Validation
        is_valid = self.validate_journal(
            df_journal, total_market_value, current_weights
        )

        # 6. Rapport et sauvegarde
        self.save_and_report(df_journal, target, total_market_value)

        return {
            'journal': df_journal,
            'target': target,
            'total_market_value': total_market_value,
            'is_valid': is_valid,
        }


# =============================================================================
# EXÉCUTION
# =============================================================================

print("[OK] Module LiveExecutor chargé")


# Décommenter pour exécuter le rééquilibrage:
output_live = LiveExecutor().run()

[OK] Module LiveExecutor chargé

GSF-3120 — RÉÉQUILIBRAGE LIVE
Date: 2026-03-15

CHARGEMENT DES DONNÉES — 2026-03-15
[1/4] Prix et volumes...
Univers: 518 US stocks + 100 CA stocks + 32 US bonds + 17 CA bonds + 86 thematic = 753 live
Chargement de 754 tickers...
  [_convert_to_cad] NaN restants apres conversion : 974/475020 (0.2050%)
  [ESG] API Yahoo Finance inaccessible. Utilisation du proxy sectoriel.
  [ESG] Scores Sustainalytics: 0 | Proxy: 754 / 754 tickers
  [OK] Taux sans risque chargé: 613 points
[OK] Donnees: 630 jours, 754 titres actifs
[OK] Volumes charges pour 754 titres
[2/4] VIX...
[3/4] Données macro...
  Chargement données macro pour régimes...
    [OK] yield_10y: 613 obs
    [OK] yield_2y: 613 obs
    [OK] credit_spread_hy: 646 obs
[4/4] Taux sans risque...

[OK] Données chargées: 630 jours, 754 titres
[CAL] TSX open=False | NYSE open=False
  [INFO] Fichier portfolio_state trouvé : portfolio_state.csv
  [INFO] Colonnes détectées : ['TICKER', 'WEIGHTRB', 'QTYRB', 'VALU

---
# 11. Validation — Contraintes de liquidité et optimisation
---


In [ ]:
# =============================================================================
# VALIDATION CONTRAINTES DE LIQUIDITÉ ET OPTIMISATION
# Commenté pour l'exécution live — nécessite les résultats du backtest.
#
# Cette cellule vérifie :
#   - Le bon fonctionnement des filtres de liquidité (ADV, max position)
#   - La méthode d'optimisation (Mean-Variance vs simple)
#   - Le respect des contraintes de poids min/max par position
#
# Décommenter après avoir exécuté le backtest (cellule 24) pour executé:
# =============================================================================
#
# =============================================================================
# CELLULE VALIDATION: CONTRAINTES DE LIQUIDITÉ + OPTIMISATION
# =============================================================================

# Les variables sont dans le dictionnaire 'output', pas directement accessibles
"""
if 'output' not in dir() or output is None:
    print("[ATTENTION]  Exécutez d'abord la cellule du backtest principal (run_full_backtest_v3)")
else:
    # Extraire les variables depuis output
    backtester = output['backtester']
    data_loader = output['data_loader']
    results = output['results']

    print("="*70)
    print("VALIDATION CONTRAINTES DE LIQUIDITÉ")
    print("="*70)

    if not hasattr(Config, 'LIQUIDITY'):
        print("[ATTENTION]  Config LIQUIDITY non définie")
    elif not Config.LIQUIDITY.get('enabled', False):
        print("[ATTENTION]  Contraintes de liquidité désactivées")
    else:
        print(f"[OK] Liquidité activée:")
        print(f"   - Min ADV: ${Config.LIQUIDITY['min_avg_daily_volume_usd']:,.0f}")
        print(f"   - Max fraction ADV: {Config.LIQUIDITY['max_position_adv_fraction']:.0%}")
        print(f"   - Max jours liquidation: {Config.LIQUIDITY['max_days_to_liquidate']}")

        # Stats sur les positions
        if backtester.weights_history:
            last_w = backtester.weights_history[-1]['weights']
            active = last_w[last_w > 0]
            print(f"\n Dernier rééquilibrage:")
            print(f"   - Positions actives: {len(active)}")
            print(f"   - Poids max: {active.max():.2%}")
            print(f"   - Poids min: {active.min():.2%}")
            print(f"   - Poids moyen: {active.mean():.2%}")

    print("\n" + "="*70)
    print("MÉTHODE D'OPTIMISATION")
    print("="*70)

    if not hasattr(Config, 'OPTIMIZATION'):
        print("Méthode: SIMPLE (pondération par score factoriel)")
    else:
        method = Config.OPTIMIZATION.get('method', 'simple')
        print(f"Méthode: {method.upper()}")
        if method == 'mean_variance':
            print(f"   - Objectif: {Config.OPTIMIZATION.get('objective', 'max_sharpe')}")
            print(f"   - Aversion au risque: {Config.OPTIMIZATION.get('risk_aversion', 2.0)}")
            try:
                import cvxpy as cp
                print("   [OK] CVXPY disponible")
            except ImportError:
                print("   [ATTENTION]  CVXPY non installé - fallback sur méthode simple")

    print("\n" + "="*70)
    print("RÉSUMÉ PERFORMANCE")
    print("="*70)
    metrics = output.get('metrics', {})
    print(f"   Rendement annualisé: {metrics.get('annual_return', 0):.2%}")
    print(f"   Sharpe Ratio: {metrics.get('sharpe_ratio', 0):.2f}")
    print(f"   Max Drawdown: {metrics.get('max_drawdown', 0):.2%}")
    print(f"   Volatilité: {metrics.get('volatility', 0):.2%}")

    """

'\nif \'output\' not in dir() or output is None:\n    print("[ATTENTION]  Exécutez d\'abord la cellule du backtest principal (run_full_backtest_v3)")\nelse:\n    # Extraire les variables depuis output\n    backtester = output[\'backtester\']\n    data_loader = output[\'data_loader\']\n    results = output[\'results\']\n\n    print("="*70)\n    print("VALIDATION CONTRAINTES DE LIQUIDITÉ")\n    print("="*70)\n\n    if not hasattr(Config, \'LIQUIDITY\'):\n        print("[ATTENTION]  Config LIQUIDITY non définie")\n    elif not Config.LIQUIDITY.get(\'enabled\', False):\n        print("[ATTENTION]  Contraintes de liquidité désactivées")\n    else:\n        print(f"[OK] Liquidité activée:")\n        print(f"   - Min ADV: ${Config.LIQUIDITY[\'min_avg_daily_volume_usd\']:,.0f}")\n        print(f"   - Max fraction ADV: {Config.LIQUIDITY[\'max_position_adv_fraction\']:.0%}")\n        print(f"   - Max jours liquidation: {Config.LIQUIDITY[\'max_days_to_liquidate\']}")\n\n        # Stats sur les po

---
# 12. Validation — Pouvoir prédictif du détecteur de régimes
---


In [ ]:
# =============================================================================
# VALIDATION OUT-OF-SAMPLE DU POUVOIR PRÉDICTIF DES RÉGIMES
# Commenté pour l'exécution live — nécessite les résultats du backtest.
#
# Cette cellule teste si le modèle de détection de régimes a un réel
# pouvoir prédictif en splitant les données temporellement (50/50) :
#   - Calibration sur la première moitié
#   - Test sur la seconde moitié
#   - Vérification que le ratio Vol(RISK_ON)/Vol(RISK_OFF) est stable
#
# Résultat clé : pouvoir prédictif confirmé out-of-sample si le ratio
# de volatilité entre régimes est > 1.5 sur les deux périodes.
#
# Décommenter après avoir exécuté le backtest (cellule 24) :
# validate_regime_predictive_power(output['results'], output['regime_history'])

# =============================================================================
# VALIDATION OUT-OF-SAMPLE DU POUVOIR PRÉDICTIF (du model de detection des régimes)
# =============================================================================

def build_regime_series(regime_history):
    reg_df = pd.DataFrame(regime_history)
    reg_df = reg_df.dropna(subset=["date", "regime"]).copy()
    reg_df["date"] = pd.to_datetime(reg_df["date"]).dt.tz_localize(None)
    reg_df = reg_df.sort_values("date")
    reg_df = reg_df.drop_duplicates(subset="date", keep="last")
    return pd.Series(reg_df["regime"].values, index=reg_df["date"]).sort_index()


def validate_regime_predictive_power(results, regime_history, split_ratio=0.5):
    """
    Test out-of-sample: calibrer sur première moitié, tester sur seconde.
    """
    print("="*70)
    print("VALIDATION OUT-OF-SAMPLE")
    print("="*70)

    returns = results['return']
    regimes = build_regime_series(regime_history)

    # Split temporel
    n = len(returns)
    split_idx = int(n * split_ratio)

    returns_train = returns.iloc[:split_idx]
    returns_test = returns.iloc[split_idx:]

    regimes_train = regimes[regimes.index <= returns_train.index[-1]]
    regimes_test = regimes[regimes.index > returns_train.index[-1]]

    print(f"Train: {len(returns_train)} jours | Test: {len(returns_test)} jours")

    # Calculer vol par régime sur chaque période
    def vol_by_regime(rets, regs):
        weekly_ret = (1 + rets).resample('W-FRI').prod() - 1
        regs = regs[~regs.index.duplicated(keep="last")].sort_index()
        reg_weekly = regs.reindex(weekly_ret.index, method='ffill').shift(1)

        result = {}
        for regime in ['RISK_ON', 'RISK_OFF']:
            mask = reg_weekly == regime
            r = weekly_ret[mask].dropna()
            if len(r) > 5:
                result[regime] = r.std() * np.sqrt(52)
        return result

    vol_train = vol_by_regime(returns_train, regimes_train)
    vol_test = vol_by_regime(returns_test, regimes_test)

    print(f"\n Volatilité annualisée par régime:")
    print(f"{'Régime':<12} {'Train':<12} {'Test':<12} {'Cohérent?':<10}")
    print("-" * 50)

    for regime in ['RISK_ON', 'RISK_OFF']:
        v_train = vol_train.get(regime, np.nan)
        v_test = vol_test.get(regime, np.nan)

        # Cohérent si même direction (RISK_ON > RISK_OFF dans les deux)
        coherent = "[OK]" if not np.isnan(v_train) and not np.isnan(v_test) else "❓"

        print(f"{regime:<12} {v_train*100:>10.1f}% {v_test*100:>10.1f}% {coherent:<10}")

    # Test: le ratio RISK_ON/RISK_OFF est-il stable ?
    if 'RISK_ON' in vol_train and 'RISK_OFF' in vol_train:
        ratio_train = vol_train['RISK_ON'] / vol_train['RISK_OFF']
        ratio_test = vol_test.get('RISK_ON', np.nan) / vol_test.get('RISK_OFF', 1)

        print(f"\n Ratio Vol(RISK_ON) / Vol(RISK_OFF):")
        print(f"  Train: {ratio_train:.2f}x")
        print(f"  Test:  {ratio_test:.2f}x")

        if ratio_train > 1.5 and ratio_test > 1.5:
            print("  [OK] Pouvoir prédictif CONFIRMÉ out-of-sample")
        elif ratio_test > 1.2:
            print("  [ATTENTION]  Pouvoir prédictif PARTIEL out-of-sample")
        else:
            print("  [ATTENTION]  Pouvoir prédictif NON CONFIRMÉ out-of-sample")

# Exécuter,
# validate_regime_predictive_power(output['results'], output['regime_history'])

---
# 13. Diagnostic — Performance de l'overlay
---


In [ ]:
# =============================================================================
# DIAGNOSTIC COMPLET — OVERLAY ET GESTION DU RISQUE
# Commenté pour l'exécution live — nécessite les résultats du backtest.
#
# Cette cellule analyse en détail l'efficacité de l'overlay crash-only :
#   - Distribution des régimes et transitions
#   - Variation du multiplicateur d'exposition
#   - Volatilité, CVaR et downside capture par régime
#   - Test statistique : le régime prédit-il la volatilité future ?
#   - Test statistique : le régime prédit-il la magnitude des retours ?
#
# Résultats détaillés dans le rapport final (section Gestion du risque).
#
# Décommenter après avoir exécuté le backtest (cellule 24) :
# [... tout le code de la cellule a ete 30 commenté ...]
#
# =============================================================================
# DIAGNOSTIC COMPLET - OVERLAY ET RISQUE  (detection des Régimes)
# =============================================================================

"""
# Récupérer les données depuis output
metrics = output['metrics']
regime_history = output.get('regime_history', None)

# Si regime_history n'est pas dans output, le récupérer depuis results
if regime_history is None:
    # Essayer de construire depuis metrics
    print("[ATTENTION]  regime_history non disponible directement")
    regime_history = []

# Créer DataFrame des régimes
if regime_history:
    regime_df = pd.DataFrame(regime_history)
else:
    # Fallback: créer depuis metrics si possible
    regime_df = None

print("\n" + "="*70)
print("DIAGNOSTIC OVERLAY ET GESTION DU RISQUE")
print("="*70)

# 1. DISTRIBUTION DES RÉGIMES
if regime_df is not None and len(regime_df) > 0:
    print("\n DISTRIBUTION DES RÉGIMES:")
    print(regime_df['regime'].value_counts())

    # MULTIPLICATEURS
    if 'multiplier' in regime_df.columns:
        print("\n MULTIPLICATEURS:")
        print(f"  Min: {regime_df['multiplier'].min():.3f} | Max: {regime_df['multiplier'].max():.3f}")
        print(f"  Mean: {regime_df['multiplier'].mean():.3f} | Std: {regime_df['multiplier'].std():.3f}")

        print("\n PAR RÉGIME:")
        print(regime_df.groupby('regime')[['multiplier', 'confidence']].agg(['mean', 'min', 'max']))

        # Variation
        mult_std = regime_df['multiplier'].std()
        if mult_std < 0.05:
            print(f"\n[ATTENTION]  Multiplicateur quasi-constant (std={mult_std:.4f}) → Overlay INACTIF")
        else:
            print(f"\n[OK] Multiplicateur varie (std={mult_std:.4f}) → Overlay ACTIF")

    # TRANSITIONS
    regimes_list = regime_df['regime'].tolist()
    transitions = sum(1 for i in range(1, len(regimes_list)) if regimes_list[i] != regimes_list[i-1])
    print(f"\n Transitions: {transitions} sur {len(regimes_list)} semaines ({transitions/len(regimes_list)*100:.1f}%)")

# 2. RISQUE PAR RÉGIME
print("\n" + "-"*70)
print(" ANALYSE DE RISQUE PAR RÉGIME")
print("-"*70)

if 'regime_risk' in metrics:
    rr = metrics['regime_risk']

    # Stats par régime
    if 'risk_by_regime' in rr:
        print(f"\n{'Régime':<12} {'N':>6} {'Vol Ann':>10} {'CVaR 95%':>10} {'Worst Wk':>10} {'<-4%':>8} {'<-5%':>8}")
        print("-" * 70)

        for regime in ["RISK_OFF", "RISK_ON"]:
            if regime in rr['risk_by_regime']:
                r = rr['risk_by_regime'][regime]
                pct4 = r.get('pct_below_minus4', 0)
                pct5 = r.get('pct_below_minus5', 0)
                print(f"{regime:<12} {r['count']:>6} {r['vol_annual']*100:>9.1f}% {r['cvar_95']*100:>9.2f}% {r['worst_week']*100:>9.2f}% {pct4:>7.1f}% {pct5:>7.1f}%")

        # Évaluation de l'overlay
        print("\n[OK] ÉVALUATION DE L'OVERLAY:")
        ro = rr['risk_by_regime'].get('RISK_ON', {})
        rf = rr['risk_by_regime'].get('RISK_OFF', {})

        if ro and rf:
            # CVaR
            if rf.get('cvar_95', -999) > ro.get('cvar_95', 0):
                print(f"  [OK] RISK_OFF a meilleur CVaR: {rf['cvar_95']*100:.2f}% vs {ro['cvar_95']*100:.2f}%")
            else:
                print(f"  [ATTENTION]  RISK_OFF n'améliore pas le CVaR")

            # Volatilité
            if rf.get('vol_annual', 999) < ro.get('vol_annual', 0):
                print(f"  [OK] RISK_OFF a volatilité plus basse: {rf['vol_annual']*100:.1f}% vs {ro['vol_annual']*100:.1f}%")

            # Downside capture
            if not np.isnan(rf.get('downside_capture', np.nan)):
                if rf['downside_capture'] < 100:
                    print(f"  [OK] RISK_OFF capture {rf['downside_capture']:.0f}% des baisses (protection)")
            if not np.isnan(ro.get('upside_capture', np.nan)):
                if ro['upside_capture'] > 90:
                    print(f"  [OK] RISK_ON capture {ro['upside_capture']:.0f}% des hausses (participation)")

    # Prédiction volatilité
    if 'vol_prediction' in rr:
        vp = rr['vol_prediction']
        print(f"\n RÉGIME → VOLATILITÉ FUTURE (4 sem):")
        print(f"  Vol future RISK_ON:  {vp.get('vol_risk_on', 0)*100:.1f}%")
        print(f"  Vol future RISK_OFF: {vp.get('vol_risk_off', 0)*100:.1f}%")
        print(f"  P-value: {vp.get('pvalue', 1):.4f}")
        if vp.get('pvalue', 1) < 0.05:
            print("  [OK] Régime prédit significativement la volatilité!")
        elif vp.get('pvalue', 1) < 0.10:
            print("  [ATTENTION]  Tendance (p < 0.10)")

    # Prédiction |retour|
    if 'abs_ret_prediction' in rr:
        ar = rr['abs_ret_prediction']
        print(f"\n RÉGIME → |RETOUR| HEBDO:")
        print(f"  |Ret| RISK_ON:  {ar.get('mean_abs_risk_on', 0)*100:.2f}%")
        print(f"  |Ret| RISK_OFF: {ar.get('mean_abs_risk_off', 0)*100:.2f}%")
        print(f"  P-value: {ar.get('pvalue', 1):.4f}")
        if ar.get('pvalue', 1) < 0.05:
            print("  [OK] Régime prédit la magnitude des retours!")

else:
    print("[ATTENTION]  'regime_risk' non trouvé dans metrics")

print("\n" + "="*70)

"""

'\n# Récupérer les données depuis output\nmetrics = output[\'metrics\']\nregime_history = output.get(\'regime_history\', None)\n\n# Si regime_history n\'est pas dans output, le récupérer depuis results\nif regime_history is None:\n    # Essayer de construire depuis metrics\n    print("[ATTENTION]  regime_history non disponible directement")\n    regime_history = []\n\n# Créer DataFrame des régimes\nif regime_history:\n    regime_df = pd.DataFrame(regime_history)\nelse:\n    # Fallback: créer depuis metrics si possible\n    regime_df = None\n\nprint("\n" + "="*70)\nprint("DIAGNOSTIC OVERLAY ET GESTION DU RISQUE")\nprint("="*70)\n\n# 1. DISTRIBUTION DES RÉGIMES\nif regime_df is not None and len(regime_df) > 0:\n    print("\n DISTRIBUTION DES RÉGIMES:")\n    print(regime_df[\'regime\'].value_counts())\n\n    # MULTIPLICATEURS\n    if \'multiplier\' in regime_df.columns:\n        print("\n MULTIPLICATEURS:")\n        print(f"  Min: {regime_df[\'multiplier\'].min():.3f} | Max: {regime_df[\'

---
# 14. Test A/B — Overlay ON vs OFF
---


In [ ]:
# =============================================================================
# TEST A/B : OVERLAY ON vs OFF
# Commenté pour l'exécution live — exécute deux backtests complets (~20 min).
#
# Cette cellule compare rigoureusement la stratégie avec et sans overlay :
#   - Backtest identique avec overlay activé vs désactivé
#   - Comparaison : rendement, volatilité, Sharpe, drawdown, VaR, CVaR
#   - Capture ratios (upside/downside) vs benchmark
#   - Conclusion sur la valeur ajoutée de l'overlay
#
# Ce test est essentiel pour justifier l'overlay devant le comité
# d'investissement. Résultats dans le rapport final.
#
# Décommenter après avoir exécuté le backtest (cellule 24) :
# =============================================================================
# [... tout le code de la cellule 32 commenté ...]
# =============================================================================
# TEST A/B: OVERLAY ON vs OFF
# =============================================================================

"""
# Extraire les données depuis output
prices = output['prices']
returns = output['returns']
vix_data = output['vix_data']
data_loader = output['data_loader']
benchmark = output['benchmark']

def run_backtest_ab(overlay_enabled=True):
    # Run backtest avec ou sans overlay.

    print(f"  Running OVERLAY {'ON' if overlay_enabled else 'OFF'}...", end=" ")

    # Initialiser modules
    factor_calc = FactorCalculator(Config)
    risk_mgr = RiskManager(Config)
    optimizer = PortfolioOptimizer(Config)
    analyzer = PerformanceAnalyzer(Config)
    backtester = RollingBacktester(Config)

    # MODIFIER LE MULTIPLICATEUR SI OVERLAY OFF
    if not overlay_enabled:
        risk_mgr._compute_exposure_multiplier = lambda *args, **kwargs: 1.0

    # Run backtest
    results = backtester.run_walk_forward(
        prices, returns, factor_calc, risk_mgr, optimizer, data_loader, vix_data
    )

    # Récupérer les historiques (avec fallback si attribut n'existe pas)
    regime_history = getattr(backtester, 'regime_details_history', [])
    fx_history = getattr(backtester, 'fx_history', None)
    turnover_history = getattr(backtester, 'turnover_history', None)

    # Analyser
    metrics = analyzer.full_analysis(
        results,
        benchmark=benchmark,
        regime_history=regime_history,
        fx_history=fx_history,
        turnover_history=turnover_history
    )

    print("✓")

    return {
        'results': results,
        'metrics': metrics,
        'regime_history': regime_history,
    }

# =============================================================================
# EXÉCUTION
# =============================================================================

print("="*70)
print("TEST A/B: OVERLAY ON vs OFF")
print("="*70)

# A) Overlay OFF
print("\n Test 1: OVERLAY OFF")
output_off = run_backtest_ab(overlay_enabled=False)
metrics_off = output_off['metrics']

# B) Overlay ON
print(" Test 2: OVERLAY ON")
output_on = run_backtest_ab(overlay_enabled=True)
metrics_on = output_on['metrics']

# =============================================================================
# COMPARAISON
# =============================================================================

print("\n" + "="*70)
print("COMPARAISON A/B")
print("="*70)

print(f"\n{'Métrique':<25} {'OVERLAY OFF':>15} {'OVERLAY ON':>15} {'Diff':>12}")
print("-"*70)

# Rendements
print(f"{'Rendement Total':<25} {metrics_off['total_return']*100:>14.2f}% {metrics_on['total_return']*100:>14.2f}% {(metrics_on['total_return']-metrics_off['total_return'])*100:>+11.2f}%")
print(f"{'Rendement Annualisé':<25} {metrics_off['annual_return']*100:>14.2f}% {metrics_on['annual_return']*100:>14.2f}% {(metrics_on['annual_return']-metrics_off['annual_return'])*100:>+11.2f}%")

# Risque
print(f"{'Volatilité':<25} {metrics_off['volatility']*100:>14.2f}% {metrics_on['volatility']*100:>14.2f}% {(metrics_on['volatility']-metrics_off['volatility'])*100:>+11.2f}%")
print(f"{'Max Drawdown':<25} {metrics_off['max_drawdown']*100:>14.2f}% {metrics_on['max_drawdown']*100:>14.2f}% {(metrics_on['max_drawdown']-metrics_off['max_drawdown'])*100:>+11.2f}%")
print(f"{'VaR 95%':<25} {metrics_off['var_95']*100:>14.2f}% {metrics_on['var_95']*100:>14.2f}% {(metrics_on['var_95']-metrics_off['var_95'])*100:>+11.2f}%")
print(f"{'CVaR 95%':<25} {metrics_off['cvar_95']*100:>14.2f}% {metrics_on['cvar_95']*100:>14.2f}% {(metrics_on['cvar_95']-metrics_off['cvar_95'])*100:>+11.2f}%")

# Ratios
print(f"{'Sharpe Ratio':<25} {metrics_off['sharpe_ratio']:>15.2f} {metrics_on['sharpe_ratio']:>15.2f} {metrics_on['sharpe_ratio']-metrics_off['sharpe_ratio']:>+12.3f}")
print(f"{'Sortino Ratio':<25} {metrics_off['sortino_ratio']:>15.2f} {metrics_on['sortino_ratio']:>15.2f} {metrics_on['sortino_ratio']-metrics_off['sortino_ratio']:>+12.3f}")
print(f"{'Calmar Ratio':<25} {metrics_off['calmar_ratio']:>15.2f} {metrics_on['calmar_ratio']:>15.2f} {metrics_on['calmar_ratio']-metrics_off['calmar_ratio']:>+12.3f}")

# Benchmark (avec vérification)
if 'benchmark' in metrics_off and 'benchmark' in metrics_on:
    b_off = metrics_off['benchmark']
    b_on = metrics_on['benchmark']
    if 'downside_capture' in b_off and 'downside_capture' in b_on:
        print(f"{'Downside Capture':<25} {b_off['downside_capture']:>14.0f}% {b_on['downside_capture']:>14.0f}% {b_on['downside_capture']-b_off['downside_capture']:>+11.0f}%")
    if 'upside_capture' in b_off and 'upside_capture' in b_on:
        print(f"{'Upside Capture':<25} {b_off['upside_capture']:>14.0f}% {b_on['upside_capture']:>14.0f}% {b_on['upside_capture']-b_off['upside_capture']:>+11.0f}%")

# Multiplicateur
regime_df_on = pd.DataFrame(output_on['regime_history'])
if len(regime_df_on) > 0 and 'multiplier' in regime_df_on.columns:
    print(f"\n Multiplicateur (ON): Min={regime_df_on['multiplier'].min():.2f} | Max={regime_df_on['multiplier'].max():.2f} | Mean={regime_df_on['multiplier'].mean():.2f}")

# =============================================================================
# CONCLUSION
# =============================================================================

print("\n" + "="*70)
print("CONCLUSION")
print("="*70)

sharpe_diff = metrics_on['sharpe_ratio'] - metrics_off['sharpe_ratio']
dd_diff = metrics_on['max_drawdown'] - metrics_off['max_drawdown']
ret_diff = metrics_on['annual_return'] - metrics_off['annual_return']

if sharpe_diff > 0:
    print(f"[OK] Overlay améliore Sharpe: {sharpe_diff:+.3f}")
else:
    print(f"[ATTENTION]  Overlay dégrade Sharpe: {sharpe_diff:+.3f}")

if dd_diff > 0:  # max_drawdown est négatif, donc plus grand = moins pire
    print(f"[OK] Overlay réduit Max DD: {dd_diff*100:+.2f}%")
else:
    print(f"[ATTENTION]  Overlay augmente Max DD: {dd_diff*100:+.2f}%")

if ret_diff < -0.01:
    print(f"[ATTENTION]  Overlay coûte: {abs(ret_diff)*100:.2f}% rendement annuel")
elif ret_diff > 0.01:
    print(f"[OK] Overlay ajoute: {ret_diff*100:.2f}% rendement annuel")

print("\n VERDICT:")
if sharpe_diff > 0 and dd_diff >= 0:
    print("   → [OK] GARDER l'overlay (améliore Sharpe ET Drawdown)")
elif sharpe_diff > 0:
    print("   → [OK] GARDER l'overlay (améliore Sharpe)")
elif dd_diff > 0.01:
    print("   → [ATTENTION]  GARDER pour protection (DD amélioré mais Sharpe neutre/négatif)")
else:
    print("   → [ATTENTION]  REVOIR l'overlay (pas d'amélioration)")

print("="*70)

"""

'\n# Extraire les données depuis output\nprices = output[\'prices\']\nreturns = output[\'returns\']\nvix_data = output[\'vix_data\']\ndata_loader = output[\'data_loader\']\nbenchmark = output[\'benchmark\']\n\ndef run_backtest_ab(overlay_enabled=True):\n    # Run backtest avec ou sans overlay.\n\n    print(f"  Running OVERLAY {\'ON\' if overlay_enabled else \'OFF\'}...", end=" ")\n\n    # Initialiser modules\n    factor_calc = FactorCalculator(Config)\n    risk_mgr = RiskManager(Config)\n    optimizer = PortfolioOptimizer(Config)\n    analyzer = PerformanceAnalyzer(Config)\n    backtester = RollingBacktester(Config)\n\n    # MODIFIER LE MULTIPLICATEUR SI OVERLAY OFF\n    if not overlay_enabled:\n        risk_mgr._compute_exposure_multiplier = lambda *args, **kwargs: 1.0\n\n    # Run backtest\n    results = backtester.run_walk_forward(\n        prices, returns, factor_calc, risk_mgr, optimizer, data_loader, vix_data\n    )\n\n    # Récupérer les historiques (avec fallback si attribut n\

---
# 15. Simulations Monte Carlo pour scénarios futurs
---


In [ ]:
# =============================================================================
# MONTE CARLO SIMULATIONS POUR SCÉNARIOS FUTURS
# =============================================================================

class MonteCarloSimulator:
    """
    Simule des mouvements de marché futurs pour évaluer la performance
    du modèle dans différents environnements.

    Méthodes :
    - Simulations historiques (bootstrap des rendements passés)
    - Simulations paramétriques (distributions normales multivariées)
    - Scénarios de stress (événements extrêmes)
    """

    def __init__(self, config=Config):
        self.config = config

    def simulate_portfolio_paths(self, current_weights, returns, n_simulations=1000,
                                n_days=252, method='bootstrap'):
        """
        Simule des trajectoires futures du portefeuille.

        Args:
            current_weights: pd.Series, poids actuels
            returns: pd.DataFrame, rendements historiques
            n_simulations: int, nombre de simulations
            n_days: int, horizon (jours)
            method: 'bootstrap' ou 'parametric'

        Returns:
            pd.DataFrame: (n_simulations, n_days) avec valeurs finales
        """
        # Alignement des données
        common_tickers = current_weights.index.intersection(returns.columns)
        w = current_weights[common_tickers]
        ret = returns[common_tickers].dropna()

        if len(ret) < 100:
            print("[ERREUR] Pas assez de données historiques")
            return None

        # Rendement du portefeuille historique
        port_ret = (ret * w).sum(axis=1)

        paths = []

        if method == 'bootstrap':
            # Bootstrap des rendements passés
            for _ in range(n_simulations):
                # Échantillonnage avec remise
                sampled = np.random.choice(port_ret.values, size=n_days, replace=True)
                # Cumulé
                path = np.cumprod(1 + sampled)
                paths.append(path[-1])  # Valeur finale

        elif method == 'parametric':
            # Distribution normale multivariée
            mu = port_ret.mean()
            sigma = port_ret.std()
            # Simuler rendements quotidiens
            sim_returns = np.random.normal(mu, sigma, (n_simulations, n_days))
            # Cumuler
            paths = np.prod(1 + sim_returns, axis=1)

        return pd.Series(paths, name='final_value')

    def stress_test_scenarios(self, current_weights, returns, scenarios=None):
        """
        Teste des scénarios de stress spécifiques.

        Args:
            current_weights: pd.Series, poids actuels
            returns: pd.DataFrame, rendements historiques
            scenarios: dict de scénarios {'nom': {'tickers': {ticker: shock}}}

        Returns:
            dict: résultats par scénario
        """
        if scenarios is None:
            # Scénarios par défaut
            scenarios = {
                'crash_2008': {
                    'description': 'Crash 2008-like (-50% actions, -20% bonds)',
                    'shocks': {'stocks': -0.5, 'bonds': -0.2}
                },
                'covid_2020': {
                    'description': 'COVID 2020 (-30% actions, +5% bonds)',
                    'shocks': {'stocks': -0.3, 'bonds': 0.05}
                },
                'stagflation': {
                    'description': 'Stagflation (+10% inflation, -20% actions)',
                    'shocks': {'stocks': -0.2, 'bonds': -0.1}
                },
                'rate_hike': {
                    'description': 'Hausse taux (+2% taux, -15% actions, -5% bonds)',
                    'shocks': {'stocks': -0.15, 'bonds': -0.05}
                }
            }

        results = {}

        # Classifier les tickers
        bond_set = set()  # À remplir selon DataLoader
        # Pour simplifier, supposer que les ETFs bonds sont connus
        bond_tickers = ['AGG', 'BND', 'XBB.TO', 'ZAG.TO']  # Exemple

        for name, scenario in scenarios.items():
            shocks = scenario['shocks']

            # Appliquer les chocs
            stressed_weights = current_weights.copy()

            if 'stocks' in shocks:
                stock_mask = ~current_weights.index.isin(bond_tickers)
                stressed_weights[stock_mask] *= (1 + shocks['stocks'])

            if 'bonds' in shocks:
                bond_mask = current_weights.index.isin(bond_tickers)
                stressed_weights[bond_mask] *= (1 + shocks['bonds'])

            # Renormaliser
            stressed_weights = stressed_weights / stressed_weights.sum()

            # Impact sur la valeur
            impact = (stressed_weights - current_weights).sum()

            results[name] = {
                'description': scenario['description'],
                'impact_pct': impact,
                'stressed_weights': stressed_weights
            }

        return results

    def analyze_regime_performance(self, backtest_results, regime_history):
        """
        Analyse la performance par régime pour identifier les forces/faiblesses.

        Args:
            backtest_results: liste de dicts du backtester
            regime_history: liste des régimes

        Returns:
            dict: stats par régime
        """
        if backtest_results is None:
            return None
        if hasattr(backtest_results, 'empty') and backtest_results.empty:
            return None
        if isinstance(backtest_results, (list, tuple, dict)) and len(backtest_results) == 0:
            return None

        # Si c'est un DataFrame déjà construit depuis un backtest, on accepte aussi
        if isinstance(backtest_results, pd.DataFrame) and backtest_results.empty:
            return None

        df = pd.DataFrame(backtest_results)

        # Alignement du régime avec le tracking des résultats:
        # - backtest_results est souvent journalier (n > 3000)
        # - regime_history est typiquement périodique (hebdo/rebalancement, n ~ 700)
        # Nous ajustons par date plutôt que par index 1:1 pour éviter ValueError.
        reg_df = pd.DataFrame(regime_history)
        if 'regime' not in reg_df.columns:
            raise ValueError("regime_history doit contenir une colonne 'regime'")

        if 'date' in reg_df.columns:
            reg_df['date'] = pd.to_datetime(reg_df['date'])
            reg_df = reg_df.sort_values('date').drop_duplicates(subset='date', keep='last')

            if 'date' in df.columns:
                df['date'] = pd.to_datetime(df['date'])
                df = df.sort_values('date')
                merged = pd.merge_asof(
                    df, reg_df[['date', 'regime']],
                    on='date',
                    direction='backward',
                    tolerance=pd.Timedelta('30D')
                )
                df['regime'] = merged['regime'].fillna('UNKNOWN')
            elif isinstance(df.index, pd.DatetimeIndex):
                regimes = pd.Series(reg_df['regime'].values, index=reg_df['date'])
                regimes = regimes.reindex(df.index, method='ffill', fill_value='UNKNOWN')
                df['regime'] = regimes.values
            else:
                if len(df) == len(reg_df):
                    df['regime'] = reg_df['regime'].values
                else:
                    # Pas d'alignement temporel possible; on laisse NaN
                    df['regime'] = 'UNKNOWN'
        else:
            # Pas de date de régime: alignement strict
            if len(df) == len(reg_df):
                df['regime'] = reg_df['regime'].values
            else:
                df['regime'] = 'UNKNOWN'

        stats = {}
        for regime in df['regime'].unique():
            regime_data = df[df['regime'] == regime]

            # champs facultatifs avec fallback
            avg_return = regime_data['return'].mean() if 'return' in regime_data.columns else np.nan
            volatility = regime_data['return'].std() if 'return' in regime_data.columns else np.nan
            sharpe = (avg_return / volatility) if (not np.isnan(avg_return) and not np.isnan(volatility) and volatility > 0) else 0
            max_drawdown = ((regime_data['portfolio_value'] / regime_data['portfolio_value'].cummax() - 1).min()) if 'portfolio_value' in regime_data.columns else np.nan
            avg_turnover = regime_data['turnover'].mean() if 'turnover' in regime_data.columns else np.nan

            stats[regime] = {
                'n_periods': len(regime_data),
                'avg_return': avg_return,
                'volatility': volatility,
                'sharpe': sharpe,
                'max_drawdown': max_drawdown,
                'avg_turnover': avg_turnover
            }

        return stats

print("[OK] Module MonteCarloSimulator chargé")



# Après avoir exécuté le backtest principal (output disponible)
if 'output' in dir():
    simulator = MonteCarloSimulator()

    # Simuler des trajectoires futures
    current_weights = output['backtester'].weights_history[-1]['weights']
    paths = simulator.simulate_portfolio_paths(
        current_weights, output['returns'], n_simulations=500
    )

    # Tests de stress
    stress_results = simulator.stress_test_scenarios(
        current_weights, output['returns']
    )

    # Analyse par régime
    regime_stats = simulator.analyze_regime_performance(
        output['results'], output['regime_history']
    )

    print("Simulations terminées!")

# 1) paths
print("=== Monte Carlo paths (final values) ===")
print(paths.describe())

# 2) stress test
print("\n=== Stress tests ===")
for s, r in stress_results.items():
    print(f"{s}: {r}")

# 3) regime stats
print("\n=== Regime stats ===")
print(pd.DataFrame(regime_stats).T)


[OK] Module MonteCarloSimulator chargé
Simulations terminées!
=== Monte Carlo paths (final values) ===
count    500.000000
mean       1.176724
std        0.135157
min        0.848635
25%        1.086163
50%        1.160620
75%        1.254504
max        1.778683
Name: final_value, dtype: float64

=== Stress tests ===
crash_2008: {'description': 'Crash 2008-like (-50% actions, -20% bonds)', 'impact_pct': np.float64(-2.654126918244515e-16), 'stressed_weights': ABBV       0.015058
AGG        0.111797
AMGN       0.015058
BND        0.113373
BNS.TO     0.012456
CL         0.015974
CLX        0.015974
CSCO       0.027569
DELL       0.027502
DG         0.012099
FOXA       0.012705
GDDY       0.027393
GEHC       0.015058
GILD       0.015058
GOVT       0.063072
GRMN       0.027557
HCA        0.015058
HIMS       0.015058
HOLX       0.015058
HSY        0.015974
IBM        0.027536
IDXX       0.015058
JBL        0.027393
JNJ        0.015057
KO         0.015974
KR         0.015972
L.TO       0.0159

---
# 16. Comparaison — 4 Facteurs vs 6 Facteurs


In [ ]:
# =============================================================================
# CELLULE 16 : COMPARAISON 4 FACTEURS vs 6 FACTEURS
# =============================================================================
# Exécute deux backtests walk-forward identiques avec des sous-ensembles
# de facteurs différents, puis affiche un tableau comparatif côte à côte.
#
# IMPORTANT : N'utilise que des copies isolées de Config via SimpleNamespace.
#             Le pipeline principal (Config, data_loader, etc.) est inchangé.
# =============================================================================

# ─────────────────────────────────────────────────────────────────────────────
# UTILITAIRE : copie isolée de Config
# ─────────────────────────────────────────────────────────────────────────────

def _copy_config(factor_subset: dict) -> types.SimpleNamespace:
    """
    Retourne un SimpleNamespace miroir de Config avec uniquement les facteurs
    de factor_subset actifs (poids renormalisés à 1.0).
    """
    ns = types.SimpleNamespace()

    # Copie profonde de tous les attributs non-callable
    for attr in dir(Config):
        if attr.startswith('__'):
            continue
        val = getattr(Config, attr)
        if callable(val) and not isinstance(val, (dict, list, tuple)):
            continue
        try:
            setattr(ns, attr, copy.deepcopy(val))
        except Exception:
            setattr(ns, attr, val)

    # ── Remplacer FACTORS (seuls les facteurs du subset sont actifs) ──────
    total_w = sum(factor_subset.values())
    new_factors = {}
    for name, cfg in Config.FACTORS.items():
        new_cfg = copy.deepcopy(cfg)
        if name in factor_subset and total_w > 0:
            new_cfg['enabled'] = True
            new_cfg['weight']  = round(factor_subset[name] / total_w, 8)
        else:
            new_cfg['enabled'] = False
            new_cfg['weight']  = 0.0
        new_factors[name] = new_cfg
    ns.FACTORS = new_factors

    # ── Renormaliser FACTOR_REGIME_WEIGHTS pour les régimes ───────────────
    new_regime_w = {}
    for regime, rw in Config.FACTOR_REGIME_WEIGHTS.items():
        sub_total = sum(v for k, v in rw.items() if k in factor_subset)
        sub_total = sub_total if sub_total > 0 else 1.0
        new_regime_w[regime] = {
            k: round(v / sub_total, 8) if k in factor_subset else 0.0
            for k, v in rw.items()
        }
    ns.FACTOR_REGIME_WEIGHTS = new_regime_w

    # ── Garantir que les attributs critiques pour les bonds sont copiés ──
    # MIN_BOND_ALLOC est un attribut de classe simple (float), pas un dict
    ns.MIN_BOND_ALLOC = getattr(Config, 'MIN_BOND_ALLOC', 0.25)

    # S'assurer que REGIME_ALLOCATIONS est copié correctement
    ns.REGIME_ALLOCATIONS = copy.deepcopy(Config.REGIME_ALLOCATIONS)

    # S'assurer que STOCK_SELECTION est copié (pour min_weight, max_weight)
    ns.STOCK_SELECTION = copy.deepcopy(Config.STOCK_SELECTION)

    # S'assurer que TURNOVER_CONTROL est copié
    ns.TURNOVER_CONTROL = copy.deepcopy(Config.TURNOVER_CONTROL)

    # S'assurer que MANAGEMENT_FEES est copié
    ns.MANAGEMENT_FEES = copy.deepcopy(Config.MANAGEMENT_FEES)

    # S'assurer que TRANSACTION_COSTS est copié
    ns.TRANSACTION_COSTS = copy.deepcopy(Config.TRANSACTION_COSTS)

    # Garder la liquidité copiée, puis la désactiver pour accélérer
    ns.LIQUIDITY = copy.deepcopy(Config.LIQUIDITY)
    ns.LIQUIDITY['enabled'] = False

    return ns


# ─────────────────────────────────────────────────────────────────────────────
# MOTEUR : un seul backtest avec n'importe quel subset de facteurs
# ─────────────────────────────────────────────────────────────────────────────

def _run_backtest_subset(factor_subset: dict, label: str,
                         prices, returns, data_loader, vix_data) -> dict | None:
    """
    Lance un backtest walk-forward complet avec le subset de facteurs donné.
    Utilise une copie isolée de Config — Config global inchangé.
    """
    tmp_cfg = _copy_config(factor_subset)

    total_w = sum(factor_subset.values())
    norm    = {k: v / total_w for k, v in factor_subset.items()}

    print(f"\n{'─'*60}")
    print(f"  BACKTEST — {label}")
    print(f"  Facteurs actifs :")
    for k, w in norm.items():
        print(f"    {k:<18} {w:.1%}")
    print(f"{'─'*60}")

    t0 = time.time()

    factor_calc = FactorCalculator(tmp_cfg)
    risk_mgr    = RiskManager(tmp_cfg)
    optimizer   = PortfolioOptimizer(tmp_cfg)
    backtester  = RollingBacktester(tmp_cfg)

    results = backtester.run_walk_forward(
        prices, returns, factor_calc, risk_mgr, optimizer,
        data_loader, vix_data, overlay_enabled=True
    )

    elapsed = time.time() - t0
    print(f"\n  [OK] {label} — {len(results)} jours en {elapsed:.0f}s")

    if results.empty:
        print(f"  [ERREUR] Aucun résultat généré pour {label}")
        return None

    # ── Calcul des métriques ──────────────────────────────────────────────
    ret  = results['return']
    pv   = results['portfolio_value']
    rf   = Config.RISK_FREE_RATE

    n_years   = max(len(ret) / 252, 1e-9)
    total_ret = float(pv.iloc[-1] / pv.iloc[0]) - 1
    ann_ret   = float((1 + total_ret) ** (1 / n_years) - 1)
    vol       = float(ret.std() * np.sqrt(252))
    sharpe    = float((ann_ret - rf) / vol) if vol > 0 else 0.0

    cumret   = (1 + ret).cumprod()
    drawdown = (cumret - cumret.cummax()) / cumret.cummax()
    max_dd   = float(drawdown.min())

    downside    = ret[ret < 0]
    sortino_vol = float(downside.std() * np.sqrt(252)) if len(downside) > 0 else vol
    sortino     = float((ann_ret - rf) / sortino_vol) if sortino_vol > 0 else 0.0
    calmar      = float(ann_ret / abs(max_dd)) if max_dd != 0 else 0.0

    if backtester.regime_details_history:
        avg_turn_ann = float(
            np.mean([e['turnover'] for e in backtester.regime_details_history]) * 52
        )
    else:
        avg_turn_ann = float('nan')

    total_mgmt = float(results['mgmt_fee'].sum()) if 'mgmt_fee' in results.columns else float('nan')
    total_perf = float(results['perf_fee'].sum()) if 'perf_fee' in results.columns else float('nan')

    return {
        'label':           label,
        'n_factors':       len(factor_subset),
        'total_return':    total_ret,
        'annual_return':   ann_ret,
        'volatility':      vol,
        'sharpe_ratio':    sharpe,
        'sortino_ratio':   sortino,
        'calmar_ratio':    calmar,
        'max_drawdown':    max_dd,
        'turnover_ann':    avg_turn_ann,
        'mgmt_fees_total': total_mgmt,
        'perf_fees_total': total_perf,
        'nav_finale':      float(pv.iloc[-1]),
        'results':         results,
        'backtester':      backtester,
    }


# ─────────────────────────────────────────────────────────────────────────────
# POINT D'ENTRÉE
# ─────────────────────────────────────────────────────────────────────────────

def run_factor_comparison():
    """
    Charge les données une seule fois, exécute les deux backtests,
    puis affiche le tableau comparatif.

    Config global : inchangé à la sortie de la fonction.
    """
    print("\n" + "=" * 70)
    print("COMPARAISON 4 FACTEURS vs 6 FACTEURS")
    print("=" * 70)

    # Pas de modification de Config global — chaque backtest utilise sa propre copie

    # ── 1. Chargement unique des données ─────────────────────────────────
    print("\n[1/3] Chargement des données (partagées entre les 2 backtests)...")
    data_loader = DataLoader(Config)
    prices, returns = data_loader.load_data(backtest_mode=True)

    vix_data = data_loader.load_vix(
        Config.BACKTEST['start_date'] - timedelta(days=365),
        Config.BACKTEST['end_date']
    )
    data_loader.load_macro_data(
        Config.BACKTEST['start_date'] - timedelta(days=365),
        Config.BACKTEST['end_date']
    )
    print(f"  Données : {len(prices)} jours, {len(prices.columns)} titres")

    # ── 2. Définir les subsets de facteurs ────────────────────────────────
    orig = Config.FACTORS

    # 4 facteurs : momentum, value, low_vol, PEAD
    factors_4 = {
        'momentum':       orig['momentum']['weight'],
        'value':          orig['value']['weight'],
        'low_volatility': orig['low_volatility']['weight'],
        'pead':           orig['pead']['weight'],
    }

    # 6 facteurs : tous les facteurs actifs dans Config
    factors_6 = {
        k: v['weight']
        for k, v in orig.items()
        if v.get('enabled', False) and v.get('weight', 0) > 0
    }

    # ── 3a. Backtest 4 facteurs ───────────────────────────────────────────
    print("\n[2/3] Backtest A — 4 facteurs (momentum, value, low_vol, PEAD)...")
    r4 = _run_backtest_subset(
        factors_4,
        "4F : momentum / value / low_vol / PEAD",
        prices, returns, data_loader, vix_data
    )

    # ── 3b. Backtest 6 facteurs ───────────────────────────────────────────
    print("\n[3/3] Backtest B — 6 facteurs (modèle complet)...")
    r6 = _run_backtest_subset(
        factors_6,
        "6F : + profitability / investment",
        prices, returns, data_loader, vix_data
    )

    if r4 is None or r6 is None:
        print("[ERREUR] Un des backtests n'a pas produit de résultats.")
        return None

    # ── 4. Tableau comparatif ─────────────────────────────────────────────
    METRICS = [
        ('annual_return',   'Rendement annualisé',        '.2%',  True),
        ('volatility',      'Volatilité annualisée',       '.2%',  False),
        ('sharpe_ratio',    'Ratio de Sharpe',             '.3f',  True),
        ('sortino_ratio',   'Ratio de Sortino',            '.3f',  True),
        ('calmar_ratio',    'Ratio de Calmar',             '.3f',  True),
        ('max_drawdown',    'Drawdown maximum',            '.2%',  False),
        ('turnover_ann',    'Turnover annualisé',          '.1%',  False),
        ('nav_finale',      'NAV finale ($)',              ',.0f', True),
    ]

    has_fees = not np.isnan(r4.get('mgmt_fees_total', float('nan')))
    if has_fees:
        METRICS += [
            ('mgmt_fees_total', 'Frais gestion totaux ($)',    ',.0f', False),
            ('perf_fees_total', 'Commissions perf totales ($)',',.0f', False),
        ]

    C1 = 30
    C2 = 22
    SEP = "─" * (C1 + C2 * 2 + 16)

    print("\n\n" + "=" * len(SEP))
    print("TABLEAU COMPARATIF — 4F vs 6F")
    print("=" * len(SEP))
    print(f"{'MÉTRIQUE':<{C1}}  {'4 FACTEURS':>{C2}}  {'6 FACTEURS':>{C2}}  {'DELTA (6F−4F)'}")
    print(SEP)

    for key, label, fmt, higher_is_better in METRICS:
        v4 = r4.get(key, float('nan'))
        v6 = r6.get(key, float('nan'))

        if np.isnan(v4) or np.isnan(v6):
            s4, s6, ds = 'N/A', 'N/A', 'N/A'
        else:
            s4    = format(v4, fmt)
            s6    = format(v6, fmt)
            delta = v6 - v4
            sign  = '+' if delta >= 0 else ''
            if abs(delta) < 1e-9:
                arrow = ' ='
            elif (higher_is_better and delta > 0) or (not higher_is_better and delta < 0):
                arrow = '▲'
            else:
                arrow = '▼'
            ds = f"{arrow} {sign}{format(delta, fmt)}"

        print(f"{label:<{C1}}  {s4:>{C2}}  {s6:>{C2}}  {ds}")

    print(SEP)

    sharpe_d = r6['sharpe_ratio'] - r4['sharpe_ratio']
    dd_d     = r6['max_drawdown']  - r4['max_drawdown']
    turn_d   = r6['turnover_ann']  - r4['turnover_ann']

    print("\nVERDICT :")
    verdict = []
    if   sharpe_d >  0.05: verdict.append(f"  ▲ Sharpe supérieur avec 6F (+{sharpe_d:.3f})")
    elif sharpe_d < -0.05: verdict.append(f"  ▼ Sharpe inférieur avec 6F ({sharpe_d:+.3f})")
    else:                   verdict.append(f"  = Sharpe quasi-identique (Δ = {sharpe_d:+.3f})")

    if   dd_d < -0.02: verdict.append(f"  ▼ Drawdown aggravé avec 6F (Δ = {dd_d:+.2%})")
    elif dd_d >  0.02: verdict.append(f"  ▲ Drawdown réduit avec 6F  (Δ = {dd_d:+.2%})")
    else:               verdict.append(f"  = Drawdown quasi-identique  (Δ = {dd_d:+.2%})")

    if   turn_d >  0.10: verdict.append(f"  ▼ Turnover plus élevé avec 6F (+{turn_d:.1%})")
    elif turn_d < -0.10: verdict.append(f"  ▲ Turnover réduit avec 6F ({turn_d:+.1%})")
    else:                 verdict.append(f"  = Turnover quasi-identique (Δ = {turn_d:+.1%})")

    for line in verdict:
        print(line)

    print(f"\n[OK] Comparaison terminée. Config global inchangé.")
    return {'4f': r4, '6f': r6}


comparison = run_factor_comparison()



COMPARAISON 4 FACTEURS vs 6 FACTEURS

[1/3] Chargement des données (partagées entre les 2 backtests)...
  [BACKTEST] +20 tickers historiques (survivorship-bias-free)
Univers: 518 US stocks + 100 CA stocks + 32 US bonds + 17 CA bonds + 86 thematic = 753 live + 21 historiques
Chargement de 774 tickers...
  [_convert_to_cad] NaN restants apres conversion : 339885/3427272 (9.9171%)
  [SURVIVORSHIP] 1 tickers detectes comme delistes
  [ESG] API Yahoo Finance accessible, chargement des scores...
  [ESG] Scores Sustainalytics: 0 | Proxy: 774 / 774 tickers
  [OK] Taux sans risque chargé: 4323 points
[OK] Donnees: 4428 jours, 774 titres actifs
[OK] Volumes charges pour 774 titres
  Chargement données macro pour régimes...
    [OK] yield_10y: 4300 obs
    [OK] yield_2y: 4300 obs
    [OK] credit_spread_hy: 4493 obs
  Données : 4428 jours, 774 titres

[2/3] Backtest A — 4 facteurs (momentum, value, low_vol, PEAD)...

────────────────────────────────────────────────────────────
  BACKTEST — 4F : m

Walk-Forward:   0%|          | 0/56 [00:00<?, ?it/s]

  [COMPUSTAT v2] 668 tickers charges, nouveaux facteurs: ~564 tickers avec donnees
  [COMPUSTAT] Charge depuis compustat_fundamentals.csv (16856 tickers, 248480 observations)
  [COMPUSTAT] 295/300 tickers charges
  [OK] Fondamentaux: 295/300 tickers (Compustat: 295, FMP: 0, YF: 0)
       ROE: 276, D/E: 276, Margins: 295
Risk-managed momentum: vol=45.3%, mult=0.33, poids momentum ajusté=15.9%

------------------------------------------------------------
[PENALITE FACTEURS MANQUANTS] Diagnostic — premier appel
------------------------------------------------------------
  AVANT (fillna=0.5) : 0/25 titres avec >50% facteurs manquants
  APRES (penalite)   : 0/25 titres avec >50% facteurs manquants
  Composition top-25 identique (aucun titre retire/ajoute)
------------------------------------------------------------
[DEBUG] Entrée optimisation: 56 titres (50 stocks + 6 bonds)
[DEBUG] Cov matrix shape: (56, 56)
[DEBUG] stock_alloc=0.725  bond_alloc=0.250
[DEBUG] Statut solveur: optimal
[DEBU

Walk-Forward:   2%|▏         | 1/56 [00:09<08:42,  9.49s/it]

[DEBUG] Entrée optimisation: 56 titres (50 stocks + 6 bonds)
[DEBUG] Cov matrix shape: (56, 56)
[DEBUG] stock_alloc=0.725  bond_alloc=0.250
[DEBUG] Statut solveur: optimal
[DEBUG] Sortie optimisation: 53 positions actives
[DEBUG] Sum weights: 0.975000
[DEBUG] Bonds dans weights: 3
[DEBUG] Après nettoyage min_weight: 11 positions, sum=0.975000
[DEBUG] Après construction result: 8 stocks + 3 bonds, sum=0.975000
[DEBUG] Après smoothing (40%): 22 positions, sum=1.000000
[DEBUG] Entrée optimisation: 56 titres (50 stocks + 6 bonds)
[DEBUG] Cov matrix shape: (56, 56)
[DEBUG] stock_alloc=0.725  bond_alloc=0.250
[DEBUG] Statut solveur: optimal
[DEBUG] Sortie optimisation: 47 positions actives
[DEBUG] Sum weights: 0.975000
[DEBUG] Bonds dans weights: 3
[DEBUG] Après nettoyage min_weight: 11 positions, sum=0.975000
[DEBUG] Après construction result: 8 stocks + 3 bonds, sum=0.975000
[DEBUG] Après smoothing (40%): 21 positions, sum=1.000000
[DEBUG] Entrée optimisation: 56 titres (50 stocks + 6 bond

...

Walk-Forward:  98%|█████████▊| 55/56 [09:20<00:10, 10.43s/it]

[DEBUG] Entrée optimisation: 56 titres (50 stocks + 6 bonds)
[DEBUG] Cov matrix shape: (56, 56)
[DEBUG] stock_alloc=0.725  bond_alloc=0.250
[DEBUG] Statut solveur: optimal
[DEBUG] Sortie optimisation: 44 positions actives
[DEBUG] Sum weights: 0.975000
[DEBUG] Bonds dans weights: 6
[DEBUG] Après nettoyage min_weight: 13 positions, sum=0.975000
[DEBUG] Après construction result: 10 stocks + 3 bonds, sum=0.975000
[DEBUG] Après smoothing (40%): 16 positions, sum=1.000000
[DEBUG] Entrée optimisation: 56 titres (50 stocks + 6 bonds)
[DEBUG] Cov matrix shape: (56, 56)
[DEBUG] stock_alloc=0.725  bond_alloc=0.250
[DEBUG] Statut solveur: optimal
[DEBUG] Sortie optimisation: 36 positions actives
[DEBUG] Sum weights: 0.975000
[DEBUG] Bonds dans weights: 6
[DEBUG] Après nettoyage min_weight: 13 positions, sum=0.975000
[DEBUG] Après construction result: 10 stocks + 3 bonds, sum=0.975000
[DEBUG] Après smoothing (40%): 16 positions, sum=1.000000
[DEBUG] Entrée optimisation: 56 titres (50 stocks + 6 bo

Walk-Forward: 100%|██████████| 56/56 [09:31<00:00, 10.20s/it]

[DEBUG] Entrée optimisation: 56 titres (50 stocks + 6 bonds)
[DEBUG] Cov matrix shape: (56, 56)
[DEBUG] stock_alloc=0.725  bond_alloc=0.250
[DEBUG] Statut solveur: optimal
[DEBUG] Sortie optimisation: 30 positions actives
[DEBUG] Sum weights: 0.975000
[DEBUG] Bonds dans weights: 3
[DEBUG] Après nettoyage min_weight: 13 positions, sum=0.975000
[DEBUG] Après construction result: 10 stocks + 3 bonds, sum=0.975000
[DEBUG] Après smoothing (40%): 20 positions, sum=1.000000

Diagnostic fenêtres:
  no_test_data: 5706 skips

Dernier rebalancement: 10 stocks + 3 bonds (Σ stocks = 72.50%, Σ bonds = 25.00%)
Rebalances enregistrés: 712 | Dates dupliquées: 0

[BASELINE] Calcul de la NAV brute en cours (sans frais de gestion)...



RAPPORT FRAIS DE GESTION — BACKTEST
  NAV brute finale  : $    902,758,362.89  (identique à avant la modif)
  NAV nette finale  : $    321,923,661.66
  Écart (drag fees) : $    580,834,701.22
  ─────────────────────────────────────────────────────
  Frais de gestion  : $     25,261,158.79
  Commissions perf  : $    197,281,356.66
  Revenus gestionnaire: $  222,542,515.45  (14.317% ann.)
  ─────────────────────────────────────────────────────
  [OK] NAV nette < NAV brute à chaque date (3917 points)

  [OK] 6F : + profitability / investment — 3631 jours en 579s


TABLEAU COMPARATIF — 4F vs 6F
MÉTRIQUE                                    4 FACTEURS              6 FACTEURS  DELTA (6F−4F)
──────────────────────────────────────────────────────────────────────────────────────────
Rendement annualisé                              8.00%                   8.46%  ▲ +0.46%
Volatilité annualisée                           12.52%                  12.40%  ▲ -0.12%
Ratio de Sharpe                       

---
# 17. Validation — Risk-Managed Momentum

Compare le modèle complet **avec** vs **sans** risk-managed momentum
(Barroso & Santa-Clara 2015).

- **Backtest A** : `_momentum_vol_realized` forcé à `None` — poids momentum fixe
- **Backtest B** : comportement par défaut — poids momentum scalé par `0.15 / σ`

Métriques : Sharpe, volatilité, drawdown maximum, rendement annualisé,
rendement mars 2020, drawdown mars 2020.

In [ ]:
# =============================================================================
# CELLULE 17 : VALIDATION — RISK-MANAGED MOMENTUM
# =============================================================================
# Compare le modèle complet AVEC vs SANS risk-managed momentum.
# Pattern identique à run_factor_comparison() : données chargées une seule fois,
# Config global inchangé à la sortie.
# =============================================================================

def run_momentum_risk_mgmt_validation():
    """
    Backtest A : Risk-managed momentum DÉSACTIVÉ (_momentum_vol_realized = None)
    Backtest B : Risk-managed momentum ACTIF (comportement par défaut)

    Métriques : Sharpe, vol, max drawdown, rendement annualisé,
    rendement mars 2020, drawdown mars 2020.
    """
    print("\n" + "=" * 70)
    print("VALIDATION — RISK-MANAGED MOMENTUM (Barroso & Santa-Clara 2015)")
    print("=" * 70)

    # ── Sous-classe qui neutralise le risk-managed momentum ───────────────
    class _FCNoRiskMgmt(FactorCalculator):
        """FactorCalculator avec _momentum_vol_realized forcé à None."""
        def _compute_momentum(self, prices, regime=None):
            score = super()._compute_momentum(prices, regime)
            self._momentum_vol_realized = None   # désactive le scaling de poids
            return score

    # ── Runner interne ────────────────────────────────────────────────────
    def _run_rmom_variant(label: str, use_rmom: bool,
                          prices, returns, data_loader, vix_data) -> dict | None:
        """Lance un backtest avec ou sans risk-managed momentum."""
        factors_all = {
            k: v['weight']
            for k, v in Config.FACTORS.items()
            if v.get('enabled', False) and v.get('weight', 0) > 0
        }
        tmp_cfg = _copy_config(factors_all)

        print(f"\n{'─'*60}")
        print(f"  BACKTEST — {label}")
        print(f"  Risk-managed momentum : {'ACTIF' if use_rmom else 'DÉSACTIVÉ'}")
        print(f"{'─'*60}")

        t0 = time.time()

        fc_class    = FactorCalculator if use_rmom else _FCNoRiskMgmt
        factor_calc = fc_class(tmp_cfg)
        risk_mgr    = RiskManager(tmp_cfg)
        optimizer   = PortfolioOptimizer(tmp_cfg)
        backtester  = RollingBacktester(tmp_cfg)

        results = backtester.run_walk_forward(
            prices, returns, factor_calc, risk_mgr, optimizer,
            data_loader, vix_data, overlay_enabled=True
        )

        elapsed = time.time() - t0
        print(f"\n  [OK] {label} — {len(results)} jours en {elapsed:.0f}s")

        if results.empty:
            print(f"  [ERREUR] Aucun résultat pour {label}")
            return None

        # ── Métriques globales ────────────────────────────────────────────
        ret  = results['return']
        pv   = results['portfolio_value']
        rf   = Config.RISK_FREE_RATE

        n_years   = max(len(ret) / 252, 1e-9)
        total_ret = float(pv.iloc[-1] / pv.iloc[0]) - 1
        ann_ret   = float((1 + total_ret) ** (1 / n_years) - 1)
        vol       = float(ret.std() * np.sqrt(252))
        sharpe    = float((ann_ret - rf) / vol) if vol > 0 else 0.0

        cumret   = (1 + ret).cumprod()
        drawdown = (cumret - cumret.cummax()) / cumret.cummax()
        max_dd   = float(drawdown.min())

        # ── Mars 2020 ─────────────────────────────────────────────────────
        mar2020 = results.loc[
            (results.index >= '2020-03-01') & (results.index <= '2020-03-31')
        ]
        if len(mar2020) > 0:
            mar_ret    = float((1 + mar2020['return']).prod() - 1)
            mar_cumret = (1 + mar2020['return']).cumprod()
            mar_dd     = float(
                ((mar_cumret - mar_cumret.cummax()) / mar_cumret.cummax()).min()
            )
        else:
            mar_ret = float('nan')
            mar_dd  = float('nan')

        return {
            'label':         label,
            'use_rmom':      use_rmom,
            'annual_return': ann_ret,
            'volatility':    vol,
            'sharpe_ratio':  sharpe,
            'max_drawdown':  max_dd,
            'mar2020_ret':   mar_ret,
            'mar2020_dd':    mar_dd,
            'results':       results,
        }

    # ── Sauvegarde et désactivation temporaire de la liquidité ───────────
    _liq_backup = copy.deepcopy(Config.LIQUIDITY)
    Config.LIQUIDITY['enabled'] = False

    try:
        # ── 1. Chargement unique des données ─────────────────────────────
        print("\n[1/3] Chargement des données (partagées entre les 2 backtests)...")
        data_loader = DataLoader(Config)
        prices, returns = data_loader.load_data(backtest_mode=True)

        vix_data = data_loader.load_vix(
            Config.BACKTEST['start_date'] - timedelta(days=365),
            Config.BACKTEST['end_date']
        )
        data_loader.load_macro_data(
            Config.BACKTEST['start_date'] - timedelta(days=365),
            Config.BACKTEST['end_date']
        )
        print(f"  Données : {len(prices)} jours, {len(prices.columns)} titres")

        # ── 2. Backtest A — sans risk-managed momentum ───────────────────
        print("\n[2/3] Backtest A — Risk-managed momentum DÉSACTIVÉ...")
        rA = _run_rmom_variant(
            "A : Sans risk-managed momentum", False,
            prices, returns, data_loader, vix_data
        )

        # ── 3. Backtest B — avec risk-managed momentum ───────────────────
        print("\n[3/3] Backtest B — Risk-managed momentum ACTIF...")
        rB = _run_rmom_variant(
            "B : Avec risk-managed momentum", True,
            prices, returns, data_loader, vix_data
        )

    finally:
        # ── Restauration garantie de Config ──────────────────────────────
        Config.LIQUIDITY = _liq_backup

    if rA is None or rB is None:
        print("[ERREUR] Un des backtests n'a pas produit de résultats.")
        return None

    # ── 4. Tableau comparatif ─────────────────────────────────────────────
    METRICS = [
        ('annual_return', 'Rendement annualisé',    '.2%', True),
        ('volatility',    'Volatilité annualisée',  '.2%', False),
        ('sharpe_ratio',  'Ratio de Sharpe',        '.3f', True),
        ('max_drawdown',  'Drawdown maximum',       '.2%', False),
        ('mar2020_ret',   'Rendement mars 2020',    '.2%', True),
        ('mar2020_dd',    'Drawdown mars 2020',     '.2%', False),
    ]

    C1  = 28
    C2  = 22
    SEP = "─" * (C1 + C2 * 2 + 16)

    print("\n\n" + "=" * len(SEP))
    print("TABLEAU COMPARATIF — Sans vs Avec Risk-Managed Momentum")
    print("=" * len(SEP))
    print(f"{'MÉTRIQUE':<{C1}}  {'SANS RMOM':>{C2}}  {'AVEC RMOM':>{C2}}  {'DELTA (B−A)'}")
    print(SEP)

    for key, label, fmt, higher_is_better in METRICS:
        vA = rA.get(key, float('nan'))
        vB = rB.get(key, float('nan'))

        if np.isnan(vA) or np.isnan(vB):
            sA, sB, ds = 'N/A', 'N/A', 'N/A'
        else:
            sA    = format(vA, fmt)
            sB    = format(vB, fmt)
            delta = vB - vA
            sign  = '+' if delta >= 0 else ''
            if abs(delta) < 1e-9:
                arrow = ' ='
            elif (higher_is_better and delta > 0) or (not higher_is_better and delta < 0):
                arrow = '▲'   # amélioration avec RMOM
            else:
                arrow = '▼'   # dégradation avec RMOM
            ds = f"{arrow} {sign}{format(delta, fmt)}"

        print(f"{label:<{C1}}  {sA:>{C2}}  {sB:>{C2}}  {ds}")

    print(SEP)

    # ── Verdict automatique ───────────────────────────────────────────────
    sharpe_d = rB['sharpe_ratio']  - rA['sharpe_ratio']
    dd_d     = rB['max_drawdown']  - rA['max_drawdown']
    mar20_d  = rB.get('mar2020_ret', float('nan')) - rA.get('mar2020_ret', float('nan'))

    print("\nVERDICT :")
    verdict = []

    if   sharpe_d >  0.05: verdict.append(f"  ▲ Sharpe amélioré par le risk-managed momentum (+{sharpe_d:.3f})")
    elif sharpe_d < -0.05: verdict.append(f"  ▼ Sharpe dégradé par le risk-managed momentum ({sharpe_d:+.3f})")
    else:                   verdict.append(f"  = Sharpe quasi-identique (Δ = {sharpe_d:+.3f})")

    if   dd_d < -0.02: verdict.append(f"  ▼ Max drawdown aggravé par le risk-managed momentum (Δ = {dd_d:+.2%})")
    elif dd_d >  0.02: verdict.append(f"  ▲ Max drawdown réduit par le risk-managed momentum (Δ = {dd_d:+.2%})")
    else:               verdict.append(f"  = Max drawdown quasi-identique (Δ = {dd_d:+.2%})")

    if not np.isnan(mar20_d):
        if   mar20_d >  0.01: verdict.append(f"  ▲ Meilleure résistance en mars 2020 (Δ ret = {mar20_d:+.2%})")
        elif mar20_d < -0.01: verdict.append(f"  ▼ Moins bonne résistance en mars 2020 (Δ ret = {mar20_d:+.2%})")
        else:                  verdict.append(f"  = Performance mars 2020 quasi-identique (Δ = {mar20_d:+.2%})")

    for line in verdict:
        print(line)

    print(f"\n[OK] Validation terminée. Config.LIQUIDITY restauré.")
    return {'A': rA, 'B': rB}


# run_momentum_risk_mgmt_validation()


---
# 18. Validation — Valeur ajoutée du ML

Compare trois configurations de détection de régime :

- **Config A** : Sans ML — règles uniquement (VIX, trend, yield curve, credit, breadth)
- **Config B** : HMM activé (comportement par défaut)
- **Config C** : HMM + Random Forest entraîné sur l'historique du backtest

Critère de valeur ajoutée : Sharpe(B) > Sharpe(A) + 0.05, Sharpe(C) > Sharpe(B) + 0.05.

In [ ]:
# =============================================================================
# CELLULE 18 : VALIDATION — VALEUR AJOUTÉE DU ML
# =============================================================================
# Compare 3 configurations de détection de régime.
# Pattern identique à run_factor_comparison() et
# run_momentum_risk_mgmt_validation() : données chargées une seule fois,
# copies isolées de Config, try/finally restauration.
# =============================================================================
import copy

def run_ml_value_validation():
    """
    Config A : Sans ML  — HMM désactivé, règles seules.
    Config B : HMM seul — comportement par défaut.
    Config C : HMM + Random Forest (MLRegimeDetector entraîné post-backtest).

    Tableau : Sharpe / vol / max_dd / ann_ret / turnover sur 3 colonnes.
    Verdict : Sharpe(B) > Sharpe(A) + 0.05 ? Idem C vs B ?
    """
    print("\n" + "=" * 70)
    print("VALIDATION — VALEUR AJOUTÉE DU ML (HMM / Random Forest)")
    print("=" * 70)

    # ── Runner interne ────────────────────────────────────────────────────
    def _run_ml_variant(label: str, hmm_enabled: bool,
                        prices, returns, data_loader, vix_data) -> dict | None:
        """Lance un backtest avec la config ML demandée."""
        factors_all = {
            k: v['weight']
            for k, v in Config.FACTORS.items()
            if v.get('enabled', False) and v.get('weight', 0) > 0
        }
        tmp_cfg = _copy_config(factors_all)

        # ── Ajuster le HMM dans la copie de Config ────────────────────────
        if not hmm_enabled:
            tmp_cfg.REGIME_DETECTION = copy.deepcopy(Config.REGIME_DETECTION)
            tmp_cfg.REGIME_DETECTION['hmm'] = copy.deepcopy(
                Config.REGIME_DETECTION.get('hmm', {})
            )
            tmp_cfg.REGIME_DETECTION['hmm']['enabled'] = False
            # Redistribuer le poids HMM sur les signaux rule-based
            sw = copy.deepcopy(tmp_cfg.REGIME_DETECTION.get('signal_weights', {}))
            hmm_w = sw.pop('hmm', 0.0)
            rule_signals = [k for k in sw if k != 'hmm']
            if rule_signals and hmm_w > 0:
                bonus = hmm_w / len(rule_signals)
                for k in rule_signals:
                    sw[k] = round(sw[k] + bonus, 8)
            sw['hmm'] = 0.0
            tmp_cfg.REGIME_DETECTION['signal_weights'] = sw

        print(f"\n{'─'*60}")
        print(f"  BACKTEST — {label}")
        print(f"  HMM : {'ACTIF' if hmm_enabled else 'DÉSACTIVÉ'}")
        print(f"{'─'*60}")

        t0 = time.time()

        factor_calc = FactorCalculator(tmp_cfg)
        risk_mgr    = RiskManager(tmp_cfg)
        optimizer   = PortfolioOptimizer(tmp_cfg)
        backtester  = RollingBacktester(tmp_cfg)

        results = backtester.run_walk_forward(
            prices, returns, factor_calc, risk_mgr, optimizer,
            data_loader, vix_data, overlay_enabled=True
        )

        elapsed = time.time() - t0
        print(f"\n  [OK] {label} — {len(results)} jours en {elapsed:.0f}s")

        if results.empty:
            print(f"  [ERREUR] Aucun résultat pour {label}")
            return None

        # ── Métriques ─────────────────────────────────────────────────────
        ret  = results['return']
        pv   = results['portfolio_value']
        rf   = Config.RISK_FREE_RATE

        n_years   = max(len(ret) / 252, 1e-9)
        total_ret = float(pv.iloc[-1] / pv.iloc[0]) - 1
        ann_ret   = float((1 + total_ret) ** (1 / n_years) - 1)
        vol       = float(ret.std() * np.sqrt(252))
        sharpe    = float((ann_ret - rf) / vol) if vol > 0 else 0.0
        cumret    = (1 + ret).cumprod()
        drawdown  = (cumret - cumret.cummax()) / cumret.cummax()
        max_dd    = float(drawdown.min())

        if backtester.regime_details_history:
            turnover_ann = float(
                np.mean([e['turnover'] for e in backtester.regime_details_history]) * 52
            )
        else:
            turnover_ann = float('nan')

        return {
            'label':         label,
            'hmm_enabled':   hmm_enabled,
            'annual_return': ann_ret,
            'volatility':    vol,
            'sharpe_ratio':  sharpe,
            'max_drawdown':  max_dd,
            'turnover_ann':  turnover_ann,
            'results':       results,
            'backtester':    backtester,
        }

    # ── Sauvegarde Config ─────────────────────────────────────────────────
    _liq_backup = copy.deepcopy(Config.LIQUIDITY)
    Config.LIQUIDITY['enabled'] = False

    try:
        # ── 1. Chargement unique des données ─────────────────────────────
        print("\n[1/4] Chargement des données (partagées entre les 3 backtests)...")
        data_loader = DataLoader(Config)
        prices, returns = data_loader.load_data(backtest_mode=True)

        vix_data = data_loader.load_vix(
            Config.BACKTEST['start_date'] - timedelta(days=365),
            Config.BACKTEST['end_date']
        )
        data_loader.load_macro_data(
            Config.BACKTEST['start_date'] - timedelta(days=365),
            Config.BACKTEST['end_date']
        )
        print(f"  Données : {len(prices)} jours, {len(prices.columns)} titres")

        # ── 2. Config A — Sans ML (HMM désactivé) ────────────────────────
        print("\n[2/4] Config A — HMM désactivé (règles seules)...")
        rA = _run_ml_variant(
            "A : Sans ML (règles seules)", False,
            prices, returns, data_loader, vix_data
        )

        # ── 3. Config B — HMM activé ─────────────────────────────────────
        print("\n[3/4] Config B — HMM activé...")
        rB = _run_ml_variant(
            "B : HMM activé", True,
            prices, returns, data_loader, vix_data
        )

        # ── 4. Config C — HMM + Random Forest ────────────────────────────
        print("\n[4/4] Config C — HMM + Random Forest...")
        rC = _run_ml_variant(
            "C : HMM + RF", True,
            prices, returns, data_loader, vix_data
        )

    finally:
        Config.LIQUIDITY = _liq_backup

    if rA is None or rB is None or rC is None:
        print("[ERREUR] Un des backtests n'a pas produit de résultats.")
        return None

    # ── 5. Entraîner le MLRegimeDetector sur les résultats de C ──────────
    print("\n[ML] Entraînement du RandomForest sur l'historique du backtest C...")
    try:
        ml_detector = MLRegimeDetector(Config)
        ml_results  = ml_detector.train(prices, vix_data=vix_data)
        if ml_results:
            acc_train = ml_results['train_accuracy']
            acc_test  = ml_results['test_accuracy']
            print(f"  Accuracy train : {acc_train:.1%}")
            print(f"  Accuracy test  : {acc_test:.1%}")
            gap = acc_train - acc_test
            if gap > 0.15:
                print(f"  [ATTENTION] Écart train/test = {gap:.1%} → risque de surapprentissage")
            rC['ml_train_acc'] = acc_train
            rC['ml_test_acc']  = acc_test
        else:
            print("  [INFO] Entraînement ML impossible (données insuffisantes).")
            rC['ml_train_acc'] = float('nan')
            rC['ml_test_acc']  = float('nan')
    except (ValueError, KeyError, Exception) as e:
        print(f"[WARN] {type(e).__name__}: {e}")
        rC['ml_train_acc'] = float('nan')
        rC['ml_test_acc']  = float('nan')

    # ── 6. Tableau comparatif ─────────────────────────────────────────────
    METRICS = [
        ('annual_return', 'Rendement annualisé',   '.2%', True),
        ('volatility',    'Volatilité annualisée', '.2%', False),
        ('sharpe_ratio',  'Ratio de Sharpe',       '.3f', True),
        ('max_drawdown',  'Drawdown maximum',      '.2%', False),
        ('turnover_ann',  'Turnover annualisé',    '.1%', False),
    ]

    C1  = 26
    C2  = 18
    SEP = "─" * (C1 + C2 * 3 + 18)

    print("\n\n" + "=" * len(SEP))
    print("TABLEAU COMPARATIF — Sans ML / HMM / HMM+RF")
    print("=" * len(SEP))
    print(f"{'MÉTRIQUE':<{C1}}  {'SANS ML (A)':>{C2}}  {'HMM (B)':>{C2}}  "
          f"{'HMM+RF (C)':>{C2}}  {'Δ B−A / C−B'}")
    print(SEP)

    for key, label, fmt, higher_is_better in METRICS:
        vA = rA.get(key, float('nan'))
        vB = rB.get(key, float('nan'))
        vC = rC.get(key, float('nan'))

        if any(np.isnan(v) for v in [vA, vB, vC]):
            sA, sB, sC, ds = 'N/A', 'N/A', 'N/A', 'N/A'
        else:
            sA = format(vA, fmt)
            sB = format(vB, fmt)
            sC = format(vC, fmt)
            dBA = vB - vA
            dCB = vC - vB
            def _arrow(d):
                if abs(d) < 1e-9: return ' ='
                return '▲' if (higher_is_better and d > 0) or (not higher_is_better and d < 0) else '▼'
            sign_ba = '+' if dBA >= 0 else ''
            sign_cb = '+' if dCB >= 0 else ''
            ds = (f"{_arrow(dBA)}{sign_ba}{format(dBA, fmt)} / "
                  f"{_arrow(dCB)}{sign_cb}{format(dCB, fmt)}")

        print(f"{label:<{C1}}  {sA:>{C2}}  {sB:>{C2}}  {sC:>{C2}}  {ds}")

    # Ligne ML accuracy (Config C uniquement)
    acc_t = rC.get('ml_test_acc', float('nan'))
    if not np.isnan(acc_t):
        print(f"{'Accuracy RF (OOS)':<{C1}}  {'—':>{C2}}  {'—':>{C2}}  "
              f"{format(acc_t, '.1%'):>{C2}}")

    print(SEP)

    # ── Verdict automatique ───────────────────────────────────────────────
    sharpe_BA = rB['sharpe_ratio'] - rA['sharpe_ratio']
    sharpe_CB = rC['sharpe_ratio'] - rB['sharpe_ratio']

    print("\nVERDICT :")
    if sharpe_BA > 0.05:
        print(f"  ▲ HMM apporte une valeur ajoutée significative vs règles seules "
              f"(ΔSharpe = {sharpe_BA:+.3f})")
    elif sharpe_BA < -0.05:
        print(f"  ▼ HMM dégrade la performance vs règles seules "
              f"(ΔSharpe = {sharpe_BA:+.3f})")
    else:
        print(f"  = HMM n'apporte pas de valeur significative vs règles seules "
              f"(ΔSharpe = {sharpe_BA:+.3f}, seuil = 0.05)")

    if sharpe_CB > 0.05:
        print(f"  ▲ Random Forest apporte une valeur ajoutée au-delà du HMM "
              f"(ΔSharpe = {sharpe_CB:+.3f})")
    elif sharpe_CB < -0.05:
        print(f"  ▼ Random Forest dégrade la performance vs HMM seul "
              f"(ΔSharpe = {sharpe_CB:+.3f})")
    else:
        print(f"  = Random Forest n'apporte pas de valeur significative au-delà du HMM "
              f"(ΔSharpe = {sharpe_CB:+.3f}, seuil = 0.05)")

    print(f"\n[OK] Validation terminée. Config.LIQUIDITY restauré.")
    return {'A': rA, 'B': rB, 'C': rC}


# run_ml_value_validation()


---
# 19. Validation — Smoothing dynamique

Compare deux configurations de smoothing :

- **Backtest A** : `smoothing_on_regime_change = weight_smoothing` (0.40) — pas de smoothing dynamique
- **Backtest B** : `smoothing_on_regime_change = 0.15` (comportement par défaut)

Verdict : turnover réduit de > 5 % sans perte de Sharpe > 0.03 → smoothing dynamique justifié.

In [ ]:
# =============================================================================
# CELLULE 19 : VALIDATION — SMOOTHING DYNAMIQUE
# =============================================================================
# Compare le modèle AVEC vs SANS smoothing réduit lors des changements de régime.
# Pattern identique aux validations précédentes.
# =============================================================================
import copy

def run_smoothing_validation():
    """
    Backtest A : smoothing_on_regime_change = weight_smoothing (0.40)
                 → pas de smoothing dynamique, comportement uniforme
    Backtest B : smoothing_on_regime_change = 0.15 (par défaut actuel)
                 → transitions de régime plus réactives

    Tableau : Sharpe / turnover / max_drawdown.
    Verdict : turnover baisse > 5 % sans perte Sharpe > 0.03 → justifié.
    """
    print("\n" + "=" * 70)
    print("VALIDATION — SMOOTHING DYNAMIQUE (changements de régime)")
    print("=" * 70)

    def _run_smoothing_variant(label: str, smoothing_regime: float,
                               prices, returns, data_loader, vix_data) -> dict | None:
        """Lance un backtest avec la valeur de smoothing_on_regime_change donnée."""
        factors_all = {
            k: v['weight']
            for k, v in Config.FACTORS.items()
            if v.get('enabled', False) and v.get('weight', 0) > 0
        }
        tmp_cfg = _copy_config(factors_all)

        # Injecter la valeur de smoothing_on_regime_change
        tmp_cfg.TURNOVER_CONTROL = copy.deepcopy(Config.TURNOVER_CONTROL)
        tmp_cfg.TURNOVER_CONTROL['smoothing_on_regime_change'] = smoothing_regime

        print(f"\n{'─'*60}")
        print(f"  BACKTEST — {label}")
        print(f"  smoothing_on_regime_change = {smoothing_regime:.0%}")
        print(f"{'─'*60}")

        t0 = time.time()

        factor_calc = FactorCalculator(tmp_cfg)
        risk_mgr    = RiskManager(tmp_cfg)
        optimizer   = PortfolioOptimizer(tmp_cfg)
        backtester  = RollingBacktester(tmp_cfg)

        results = backtester.run_walk_forward(
            prices, returns, factor_calc, risk_mgr, optimizer,
            data_loader, vix_data, overlay_enabled=True
        )

        elapsed = time.time() - t0
        print(f"\n  [OK] {label} — {len(results)} jours en {elapsed:.0f}s")

        if results.empty:
            print(f"  [ERREUR] Aucun résultat pour {label}")
            return None

        ret  = results['return']
        pv   = results['portfolio_value']
        rf   = Config.RISK_FREE_RATE

        n_years   = max(len(ret) / 252, 1e-9)
        total_ret = float(pv.iloc[-1] / pv.iloc[0]) - 1
        ann_ret   = float((1 + total_ret) ** (1 / n_years) - 1)
        vol       = float(ret.std() * np.sqrt(252))
        sharpe    = float((ann_ret - rf) / vol) if vol > 0 else 0.0
        cumret    = (1 + ret).cumprod()
        max_dd    = float(((cumret - cumret.cummax()) / cumret.cummax()).min())

        if backtester.regime_details_history:
            turnover_ann = float(
                np.mean([e['turnover'] for e in backtester.regime_details_history]) * 52
            )
        else:
            turnover_ann = float('nan')

        return {
            'label':         label,
            'smoothing':     smoothing_regime,
            'annual_return': ann_ret,
            'volatility':    vol,
            'sharpe_ratio':  sharpe,
            'max_drawdown':  max_dd,
            'turnover_ann':  turnover_ann,
        }

    _liq_backup = copy.deepcopy(Config.LIQUIDITY)
    Config.LIQUIDITY['enabled'] = False

    try:
        print("\n[1/3] Chargement des données...")
        data_loader = DataLoader(Config)
        prices, returns = data_loader.load_data(backtest_mode=True)
        vix_data = data_loader.load_vix(
            Config.BACKTEST['start_date'] - timedelta(days=365),
            Config.BACKTEST['end_date']
        )
        data_loader.load_macro_data(
            Config.BACKTEST['start_date'] - timedelta(days=365),
            Config.BACKTEST['end_date']
        )
        print(f"  Données : {len(prices)} jours, {len(prices.columns)} titres")

        base_smoothing = Config.TURNOVER_CONTROL['weight_smoothing']

        print("\n[2/3] Backtest A — pas de smoothing dynamique (uniforme 0.40)...")
        rA = _run_smoothing_variant(
            "A : Sans smoothing dynamique", base_smoothing,
            prices, returns, data_loader, vix_data
        )

        print("\n[3/3] Backtest B — smoothing dynamique 0.15 sur changement de régime...")
        rB = _run_smoothing_variant(
            "B : Avec smoothing dynamique (0.15)", 0.15,
            prices, returns, data_loader, vix_data
        )

    finally:
        Config.LIQUIDITY = _liq_backup

    if rA is None or rB is None:
        print("[ERREUR] Un des backtests n'a pas produit de résultats.")
        return None

    METRICS = [
        ('sharpe_ratio',  'Ratio de Sharpe',       '.3f', True),
        ('turnover_ann',  'Turnover annualisé',     '.1%', False),
        ('max_drawdown',  'Drawdown maximum',       '.2%', False),
        ('annual_return', 'Rendement annualisé',    '.2%', True),
        ('volatility',    'Volatilité annualisée',  '.2%', False),
    ]

    C1  = 26
    C2  = 22
    SEP = "─" * (C1 + C2 * 2 + 16)

    print("\n\n" + "=" * len(SEP))
    print("TABLEAU COMPARATIF — Sans vs Avec Smoothing Dynamique")
    print("=" * len(SEP))
    print(f"{'MÉTRIQUE':<{C1}}  {'SANS SMTH DYN (A)':>{C2}}  "
          f"{'AVEC SMTH DYN (B)':>{C2}}  {'DELTA (B−A)'}")
    print(SEP)

    for key, label, fmt, higher_is_better in METRICS:
        vA = rA.get(key, float('nan'))
        vB = rB.get(key, float('nan'))
        if np.isnan(vA) or np.isnan(vB):
            sA, sB, ds = 'N/A', 'N/A', 'N/A'
        else:
            sA    = format(vA, fmt)
            sB    = format(vB, fmt)
            delta = vB - vA
            sign  = '+' if delta >= 0 else ''
            if abs(delta) < 1e-9:
                arrow = ' ='
            elif (higher_is_better and delta > 0) or (not higher_is_better and delta < 0):
                arrow = '▲'
            else:
                arrow = '▼'
            ds = f"{arrow} {sign}{format(delta, fmt)}"
        print(f"{label:<{C1}}  {sA:>{C2}}  {sB:>{C2}}  {ds}")

    print(SEP)

    # ── Verdict automatique ───────────────────────────────────────────────
    sharpe_d   = rB['sharpe_ratio'] - rA['sharpe_ratio']
    turn_A     = rA.get('turnover_ann', float('nan'))
    turn_B     = rB.get('turnover_ann', float('nan'))
    turn_delta = turn_B - turn_A if not (np.isnan(turn_A) or np.isnan(turn_B)) else float('nan')

    print("\nVERDICT :")
    if np.isnan(turn_delta):
        print("  ? Données de turnover insuffisantes pour conclure.")
    else:
        turnover_reduced = turn_delta < -0.05 * turn_A   # baisse relative > 5 %
        sharpe_preserved = sharpe_d > -0.03

        if turnover_reduced and sharpe_preserved:
            print(f"  ▲ Smoothing dynamique JUSTIFIÉ :")
            print(f"    Turnover réduit de {abs(turn_delta):.1%} "
                  f"(Δ = {turn_delta:+.1%}) sans perte significative de Sharpe "
                  f"(ΔSharpe = {sharpe_d:+.3f})")
        elif not turnover_reduced:
            print(f"  = Réduction de turnover insuffisante (Δ = {turn_delta:+.1%}, "
                  f"seuil > 5% de {turn_A:.1%})")
        else:
            print(f"  ▼ Perte de Sharpe trop importante pour justifier le smoothing "
                  f"(ΔSharpe = {sharpe_d:+.3f}, seuil = −0.03)")

    print(f"\n[OK] Validation terminée. Config.LIQUIDITY restauré.")
    return {'A': rA, 'B': rB}


# run_smoothing_validation()


---
# 20. Validation — Interpolation continue des poids factoriels

Compare deux configurations de pondération factorielle :

- **Backtest A (Binaire)** : switch discret RISK_ON / RISK_OFF (regime_score ignoré)
- **Backtest B (Continu)** : interpolation linéaire selon composite_score ∈ [-1, 1]

Verdict : si le turnover baisse de > 10 % sans perte de Sharpe > 0.03, l'interpolation continue est justifiée.


In [ ]:
# =============================================================================
# CELLULE 20 : VALIDATION — INTERPOLATION CONTINUE DES POIDS FACTORIELS
# =============================================================================
# Compare le modèle AVEC interpolation continue vs switch binaire RISK_ON/RISK_OFF.
# Pattern identique aux validations précédentes.
# =============================================================================

def run_interpolation_validation():
    """
    Backtest A : poids binaires — compute_composite_score reçoit regime_score=None
                 → switch discret RISK_ON / RISK_OFF uniquement
    Backtest B : interpolation continue — compute_composite_score reçoit
                 regime_score=composite_score ∈ [-1, 1] (comportement par défaut)

    Tableau : Sharpe / turnover / max_drawdown / rendement / volatilité.
    Verdict : turnover baisse > 10 % sans perte Sharpe > 0.03 → interpolation justifiée.
    """
    print("\n" + "=" * 70)
    print("VALIDATION — INTERPOLATION CONTINUE DES POIDS FACTORIELS")
    print("=" * 70)

    # ── Sous-classe qui force le fallback binaire ──────────────────────────────
    class _FCBinaryWeights(FactorCalculator):
        """Override compute_composite_score : ne transmet JAMAIS regime_score."""
        def compute_composite_score(self, factors_df, regime=None, regime_score=None):
            return super().compute_composite_score(factors_df, regime=regime,
                                                   regime_score=None)

    def _run_interp_variant(label: str, use_binary: bool,
                            prices, returns, data_loader, vix_data) -> dict | None:
        """Lance un backtest avec ou sans interpolation continue."""
        factors_all = {
            k: v['weight']
            for k, v in Config.FACTORS.items()
            if v.get('enabled', False) and v.get('weight', 0) > 0
        }
        tmp_cfg = _copy_config(factors_all)

        print(f"\n{'─'*60}")
        print(f"  BACKTEST — {label}")
        mode = "binaire (regime_score=None)" if use_binary else "continue (regime_score=composite_score)"
        print(f"  Mode : {mode}")
        print(f"{'─'*60}")

        t0 = time.time()

        factor_calc = _FCBinaryWeights(tmp_cfg) if use_binary else FactorCalculator(tmp_cfg)
        risk_mgr    = RiskManager(tmp_cfg)
        optimizer   = PortfolioOptimizer(tmp_cfg)
        backtester  = RollingBacktester(tmp_cfg)

        results = backtester.run_walk_forward(
            prices, returns, factor_calc, risk_mgr, optimizer,
            data_loader, vix_data, overlay_enabled=True
        )

        elapsed = time.time() - t0
        print(f"\n  [OK] {label} — {len(results)} jours en {elapsed:.0f}s")

        if results.empty:
            print(f"  [ERREUR] Aucun résultat pour {label}")
            return None

        ret  = results['return']
        pv   = results['portfolio_value']
        rf   = Config.RISK_FREE_RATE

        n_years   = max(len(ret) / 252, 1e-9)
        total_ret = float(pv.iloc[-1] / pv.iloc[0]) - 1
        ann_ret   = float((1 + total_ret) ** (1 / n_years) - 1)
        vol       = float(ret.std() * np.sqrt(252))
        sharpe    = float((ann_ret - rf) / vol) if vol > 0 else 0.0
        cumret    = (1 + ret).cumprod()
        max_dd    = float(((cumret - cumret.cummax()) / cumret.cummax()).min())

        if backtester.regime_details_history:
            turnover_ann = float(
                np.mean([e['turnover'] for e in backtester.regime_details_history]) * 52
            )
        else:
            turnover_ann = float('nan')

        return {
            'label':         label,
            'use_binary':    use_binary,
            'annual_return': ann_ret,
            'volatility':    vol,
            'sharpe_ratio':  sharpe,
            'max_drawdown':  max_dd,
            'turnover_ann':  turnover_ann,
        }

    _liq_backup = copy.deepcopy(Config.LIQUIDITY)
    Config.LIQUIDITY['enabled'] = False

    try:
        print("\n[1/3] Chargement des données...")
        data_loader = DataLoader(Config)
        prices, returns = data_loader.load_data(backtest_mode=True)
        vix_data = data_loader.load_vix(
            Config.BACKTEST['start_date'] - timedelta(days=365),
            Config.BACKTEST['end_date']
        )
        data_loader.load_macro_data(
            Config.BACKTEST['start_date'] - timedelta(days=365),
            Config.BACKTEST['end_date']
        )
        print(f"  Données : {len(prices)} jours, {len(prices.columns)} titres")

        print("\n[2/3] Backtest A — poids binaires (regime_score=None)...")
        rA = _run_interp_variant(
            "A : Binaire (switch RISK_ON/RISK_OFF)", use_binary=True,
            prices=prices, returns=returns, data_loader=data_loader, vix_data=vix_data
        )

        print("\n[3/3] Backtest B — interpolation continue (regime_score=composite_score)...")
        rB = _run_interp_variant(
            "B : Continu (interpolation linéaire)", use_binary=False,
            prices=prices, returns=returns, data_loader=data_loader, vix_data=vix_data
        )

    finally:
        Config.LIQUIDITY = _liq_backup

    if rA is None or rB is None:
        print("[ERREUR] Un des backtests n'a pas produit de résultats.")
        return None

    METRICS = [
        ('sharpe_ratio',  'Ratio de Sharpe',       '.3f', True),
        ('turnover_ann',  'Turnover annualisé',     '.1%', False),
        ('max_drawdown',  'Drawdown maximum',       '.2%', False),
        ('annual_return', 'Rendement annualisé',    '.2%', True),
        ('volatility',    'Volatilité annualisée',  '.2%', False),
    ]

    C1  = 26
    C2  = 22
    SEP = "─" * (C1 + C2 * 2 + 16)

    print("\n\n" + "=" * len(SEP))
    print("TABLEAU COMPARATIF — Binaire vs Interpolation Continue")
    print("=" * len(SEP))
    print(f"{'MÉTRIQUE':<{C1}}  {'BINAIRE (A)':>{C2}}  "
          f"{'CONTINU (B)':>{C2}}  {'DELTA (B−A)'}")
    print(SEP)

    for key, label, fmt, higher_is_better in METRICS:
        vA = rA.get(key, float('nan'))
        vB = rB.get(key, float('nan'))
        if np.isnan(vA) or np.isnan(vB):
            sA, sB, ds = 'N/A', 'N/A', 'N/A'
        else:
            sA    = format(vA, fmt)
            sB    = format(vB, fmt)
            delta = vB - vA
            sign  = '+' if delta >= 0 else ''
            if abs(delta) < 1e-9:
                arrow = ' ='
            elif (higher_is_better and delta > 0) or (not higher_is_better and delta < 0):
                arrow = '▲'
            else:
                arrow = '▼'
            ds = f"{arrow} {sign}{format(delta, fmt)}"
        print(f"{label:<{C1}}  {sA:>{C2}}  {sB:>{C2}}  {ds}")

    print(SEP)

    # ── Verdict automatique ───────────────────────────────────────────────────
    sharpe_d   = rB['sharpe_ratio'] - rA['sharpe_ratio']
    turn_A     = rA.get('turnover_ann', float('nan'))
    turn_B     = rB.get('turnover_ann', float('nan'))
    turn_delta = turn_B - turn_A if not (np.isnan(turn_A) or np.isnan(turn_B)) else float('nan')

    print("\nVERDICT :")
    if np.isnan(turn_delta):
        print("  ? Données de turnover insuffisantes pour conclure.")
    else:
        turnover_reduced = turn_A > 0 and turn_delta < -0.10 * turn_A   # baisse relative > 10 %
        sharpe_preserved = sharpe_d > -0.03

        if turnover_reduced and sharpe_preserved:
            print(f"  ▲ Interpolation continue JUSTIFIÉE :")
            print(f"    Turnover réduit de {abs(turn_delta):.1%} "
                  f"(Δ = {turn_delta:+.1%}) sans perte significative de Sharpe "
                  f"(ΔSharpe = {sharpe_d:+.3f})")
        elif not turnover_reduced:
            print(f"  = Réduction de turnover insuffisante (Δ = {turn_delta:+.1%}, "
                  f"seuil > 10% de {turn_A:.1%})")
        else:
            print(f"  ▼ Perte de Sharpe trop importante pour justifier l'interpolation "
                  f"(ΔSharpe = {sharpe_d:+.3f}, seuil = −0.03)")

    print(f"\n[OK] Validation terminée. Config.LIQUIDITY restauré.")
    return {'A': rA, 'B': rB}


run_interpolation_validation()



VALIDATION — INTERPOLATION CONTINUE DES POIDS FACTORIELS

[1/3] Chargement des données...
  [BACKTEST] +20 tickers historiques (survivorship-bias-free)
Univers: 518 US stocks + 100 CA stocks + 32 US bonds + 17 CA bonds + 86 thematic = 753 live + 21 historiques
Chargement de 774 tickers...
  [_convert_to_cad] NaN restants apres conversion : 339885/3427272 (9.9171%)
  [SURVIVORSHIP] 1 tickers detectes comme delistes
  [ESG] API Yahoo Finance accessible, chargement des scores...


## A. Tests unitaires

In [ ]:

"""
# Tests rapides des modules

def run_tests():
    # Exécute les tests de base.
    print("Running tests...")

    # Test 1: Covariance
    test_returns = pd.DataFrame(
        np.random.randn(100, 5) * 0.02,
        columns=['A', 'B', 'C', 'D', 'E']
    )
    rm = RiskManager()
    cov = rm.compute_covariance(test_returns)
    assert cov.shape == (5, 5), "Covariance shape error"
    print("  [OK] Covariance matrix")

    # Test 2: Optimization
    exp_ret = np.array([0.1, 0.12, 0.08, 0.15, 0.09])
    opt = PortfolioOptimizer()
    weights = opt.optimize(exp_ret, cov)
    assert abs(weights.sum() - 1.0) < 0.01, "Weights don't sum to 1"
    assert all(weights >= 0), "Negative weights"
    print("  [OK] Portfolio optimization")

    # Test 3: Metrics
    test_rets = pd.Series(np.random.randn(252) * 0.01)
    metrics = PerformanceMetrics.calculate(test_rets)
    assert 'sharpe_ratio' in metrics, "Missing Sharpe ratio"
    print("  [OK] Performance metrics")

    print("\nAll tests passed!")

run_tests()

"""

## B. Modification des contraintes dynamiques

Pour adapter le système aux nouvelles contraintes du client, modifier la section `DYNAMIC_CONSTRAINTS` dans la configuration:

In [ ]:

"""
# Exemple: Appliquer une nouvelle contrainte de volatilité max

# Modifier la configuration
Config.DYNAMIC_CONSTRAINTS['max_volatility'] = 0.12  # 12% au lieu de 15%

print("Nouvelle contrainte appliquée:")
print(f"  max_volatility = {Config.DYNAMIC_CONSTRAINTS.get('max_volatility')}")

# Pour exclure un secteur:
# Config.DYNAMIC_CONSTRAINTS['excluded_sectors'] = ['Energy']

# Pour augmenter l'allocation min en obligations:
# Config.DYNAMIC_CONSTRAINTS['min_bonds'] = 0.40

"""